# 1\.5\.6 Construct Modeling Feature Sets

We start with shared imports, paths, and modeling controls so the rest of the notebook uses one consistent feature\-selection setup\.

In [1]:
# ---------------------------------------------------------------------
# Imports and notebook display settings
# ---------------------------------------------------------------------

from pathlib import Path
import gc
import re
import time
import warnings

import duckdb
import numpy as np
import pandas as pd

try:
    import plotly.express as px
except ImportError:
    px = None

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 200)

/root/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------------------------------------------------
# Project paths and output controls
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

INPUT_DIR = PIPELINE_DATA_DIR / "1.5.5.final_tables"
OUTPUT_DIR = PIPELINE_DATA_DIR / "1.5.6.final_tables"
INTERMEDIATE_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.5.6.intermediate_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PANEL_PATH = INPUT_DIR / "variability_anomaly_feature_panel.parquet"
INPUT_MANIFEST_PATH = INPUT_DIR / "variability_anomaly_feature_manifest.csv"
INPUT_QA_PATH = INPUT_DIR / "variability_anomaly_feature_qa_summary.csv"

OUTPUT_RAW_MODELING_PANEL_PATH = OUTPUT_DIR / "modeling_feature_panel_raw.parquet"
OUTPUT_SCALED_MODELING_PANEL_PATH = OUTPUT_DIR / "modeling_feature_panel_scaled.parquet"
OUTPUT_FEATURE_CATALOG_PATH = OUTPUT_DIR / "modeling_feature_catalog.csv"
OUTPUT_FEATURE_SET_DEFINITIONS_PATH = OUTPUT_DIR / "modeling_feature_set_definitions.csv"
OUTPUT_FEATURE_FAMILY_SUMMARY_PATH = OUTPUT_DIR / "modeling_feature_family_summary.csv"
OUTPUT_QA_PATH = OUTPUT_DIR / "modeling_feature_set_qa_summary.csv"

FEATURE_INVENTORY_PATH = INTERMEDIATE_OUTPUT_DIR / "feature_inventory.csv"
MODELING_SAMPLE_PATH = INTERMEDIATE_OUTPUT_DIR / "modeling_diagnostic_sample.parquet"
WITHIN_FAMILY_REDUNDANCY_PATH = INTERMEDIATE_OUTPUT_DIR / "within_family_redundancy_summary.csv"
INCREMENTAL_CONTRIBUTION_PATH = INTERMEDIATE_OUTPUT_DIR / "incremental_feature_family_contribution.csv"
FEATURE_SELECTION_DECISIONS_PATH = INTERMEDIATE_OUTPUT_DIR / "feature_selection_decisions.csv"

WRITE_OUTPUTS = True
REBUILD_MODELING_SAMPLE = False
REBUILD_REDUNDANCY_ANALYSIS = False
REBUILD_INCREMENTAL_CONTRIBUTION_ANALYSIS = False


def duckdb_path(path):
    """Convert a pathlib path to a DuckDB-friendly string."""
    return str(path).replace("\\", "/")


required_input_paths = {
    "1.5.5 anomaly-severity feature panel": INPUT_PANEL_PATH,
    "1.5.5 anomaly-severity feature manifest": INPUT_MANIFEST_PATH,
    "1.5.5 anomaly-severity QA summary": INPUT_QA_PATH,
}

for label, path in required_input_paths.items():
    print(f"{label}: {path.exists()} | {path}")

missing_input_paths = [
    path
    for path in required_input_paths.values()
    if not path.exists()
]

assert not missing_input_paths, (
    f"Missing required 1.5.6 input files: {missing_input_paths}"
)

1.5.5 anomaly-severity feature panel: True | pipeline_data/1.5.5.final_tables/variability_anomaly_feature_panel.parquet
1.5.5 anomaly-severity feature manifest: True | pipeline_data/1.5.5.final_tables/variability_anomaly_feature_manifest.csv
1.5.5 anomaly-severity QA summary: True | pipeline_data/1.5.5.final_tables/variability_anomaly_feature_qa_summary.csv


In [3]:
# ---------------------------------------------------------------------
# Analytical grain, metadata, and core modeling metrics
# ---------------------------------------------------------------------

GRAIN_COLUMNS = [
    "taxi_zone_id",
    "date",
    "temporal_bucket",
]

ROW_IDENTIFIER_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "date",
    "year",
    "month",
    "day_of_week",
    "temporal_bucket",
    "pre_post_cp",
]

VALID_CP_PERIOD_VALUES = {"pre_cp", "post_cp"}

NON_MODELING_COLUMNS = set(
    ROW_IDENTIFIER_COLUMNS
    + [
        "geometry",
        "centroid_geometry",
        "pickup_geometry",
        "dropoff_geometry",
    ]
)

CORE_MODELING_METRICS = [
    "bus_trip_count",
    "avg_bus_speed",
    "subway_ridership",
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
]

print(f"Core modeling metrics: {len(CORE_MODELING_METRICS)}")
print(f"Analytical grain: {' × '.join(GRAIN_COLUMNS)}")

Core modeling metrics: 7
Analytical grain: taxi_zone_id × date × temporal_bucket


These controls define how features will be grouped, compared, and carried into modality\-specific modeling sets\.

In [4]:
# ---------------------------------------------------------------------
# Feature-family, modality, and selection controls
# ---------------------------------------------------------------------

MODALITY_METRIC_PREFIXES = {
    "bus": [
        "bus_",
        "avg_bus_",
    ],
    "subway": [
        "subway_",
    ],
    "taxi": [
        "taxi_",
    ],
    "fhvhv": [
        "fhvhv_",
    ],
    "multimodal": [
        "interaction_",
        "cross_modal_",
        "mode_",
        "mobility_mix_",
    ],
}

MODELING_FEATURE_SET_NAMES = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
    "multimodal",
]

FEATURE_FAMILY_ORDER = [
    "raw_metric",
    "temporal_history",
    "spatial_context",
    "multimodal_interaction",
    "stl_decomposition",
    "fourier20_residual_reconstruction",
    "anomaly_severity",
]

FEATURE_SOURCE_NOTEBOOK_ORDER = [
    "pre_1.5_engineering",
    "1.5.1",
    "1.5.2",
    "1.5.3",
    "1.5.4",
    "1.5.5",
]

FEATURE_DECISION_TIERS = [
    "selected",
    "candidate_review",
    "dropped_redundant",
    "dropped_low_signal",
    "dropped_scope",
]

print(f"Feature families: {len(FEATURE_FAMILY_ORDER)}")
print(f"Modeling feature sets: {', '.join(MODELING_FEATURE_SET_NAMES)}")

Feature families: 7
Modeling feature sets: bus, subway, taxi, fhvhv, multimodal


These settings control the diagnostic sample, redundancy thresholds, PCA evidence checks, and scaling defaults used later in the notebook\.

In [5]:
# ---------------------------------------------------------------------
# Diagnostic sampling, redundancy, PCA, and scaling controls
# ---------------------------------------------------------------------

DIAGNOSTIC_SAMPLE_RANDOM_STATE = 696
DIAGNOSTIC_SAMPLE_ROWS = 250_000
DIAGNOSTIC_SAMPLE_STRATA_COLUMNS = [
    "pre_post_cp",
    "temporal_bucket",
    "borough",
]

MIN_FEATURE_NON_NULL_PCT = 0.05
MAX_FEATURE_NULL_PCT_FOR_DEFAULT_SELECTION = 0.80
NEAR_ZERO_VARIANCE_EPSILON = 1e-9

SPEARMAN_CORRELATION_THRESHOLD = 0.90
STRICT_SPEARMAN_CORRELATION_THRESHOLD = 0.95
STRICT_CORRELATION_THRESHOLD = STRICT_SPEARMAN_CORRELATION_THRESHOLD
MAX_CORRELATION_FEATURES_PER_FAMILY = 500

PCA_RANDOM_STATE = 696
PCA_VARIANCE_TARGETS = [0.80, 0.90, 0.95]
PCA_TOP_COMPONENT_COUNTS = [5, 10, 20]
PCA_MAX_COMPONENTS = 50
PCA_MAX_FEATURES_PER_FAMILY_FOR_EQUALIZED_CHECK = 75

SCALER_NAME = "standard_scaler"
SCALER_WITH_MEAN = True
SCALER_WITH_STD = True

print(f"Diagnostic sample rows: {DIAGNOSTIC_SAMPLE_ROWS:,}")
print(f"Spearman redundancy threshold: {SPEARMAN_CORRELATION_THRESHOLD}")
print(f"Scaler: {SCALER_NAME}")

Diagnostic sample rows: 250,000
Spearman redundancy threshold: 0.9
Scaler: standard_scaler


## 1\.5\.6\.1 Load Modeling Panel and Inventory Features

We start by loading the final 1\.5\.5 feature panel and taking inventory of everything that has been engineered so far\. At this point, the project has accumulated raw mobility metrics, temporal\-history features, spatial context features, multimodal interaction features, decomposition features, and deviation features\. Before we make any pruning or modeling decisions, we need a clear view of what exists, how many features came from each notebook, which transportation systems they represent, and where null coverage or scale issues may appear\.

This section should confirm that the canonical Taxi Zone × Date × Temporal Bucket grain is still intact and then build a feature inventory table\. The inventory should classify each candidate feature by source notebook, feature family, transportation modality, and whether it belongs to a selected core metric or a non\-core metric descendant\. The goal is not to drop anything yet\. The goal is to understand the full feature universe before we start making selection decisions\.

### Load and Validate Modeling Panel

Let’s load the completed 1\.5\.5 panel and do a quick shape check before building the feature inventory\.

In [6]:
# ---------------------------------------------------------------------
# Load modeling panel
# ---------------------------------------------------------------------

modeling_df = pd.read_parquet(INPUT_PANEL_PATH)
modeling_df["date"] = pd.to_datetime(modeling_df["date"])

modeling_panel_load_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "row_count",
            "value": len(modeling_df),
        },
        {
            "qa_check": "column_count",
            "value": modeling_df.shape[1],
        },
        {
            "qa_check": "date_range",
            "value": f"{modeling_df['date'].min().date()} to {modeling_df['date'].max().date()}",
        },
        {
            "qa_check": "unique_taxi_zones",
            "value": modeling_df["taxi_zone_id"].nunique(),
        },
        {
            "qa_check": "unique_temporal_buckets",
            "value": modeling_df["temporal_bucket"].nunique(),
        },
    ]
)

display(modeling_panel_load_summary_df)

assert len(modeling_df) > 0, "1.5.5 modeling input panel is empty."

print("Modeling panel loaded.")

,qa_check,value
0,row_count,1559590
1,column_count,342
2,date_range,2023-01-01 to 2026-03-31
3,unique_taxi_zones,263
4,unique_temporal_buckets,10


Modeling panel loaded.


Findings\. The 1\.5\.5 modeling panel loaded successfully with 1,559,590 rows and 342 columns\. It spans January 1, 2023 through March 31, 2026, covers 263 Taxi Zones, and includes all 10 temporal buckets, so the panel has the expected breadth for feature inventory work\.

Next, we confirm that the modeling panel still has one row per Taxi Zone × Date × Temporal Bucket\.

In [7]:
# ---------------------------------------------------------------------
# Validate modeling panel grain
# ---------------------------------------------------------------------

duplicate_grain_rows = modeling_df.duplicated(GRAIN_COLUMNS).sum()

modeling_grain_validation_df = pd.DataFrame(
    [
        {
            "qa_check": "analytical_grain",
            "value": "Taxi Zone × Date × Temporal Bucket",
            "status": "reference",
        },
        {
            "qa_check": "duplicate_grain_rows",
            "value": duplicate_grain_rows,
            "status": "pass" if duplicate_grain_rows == 0 else "fail",
        },
    ]
)

display(modeling_grain_validation_df)

assert duplicate_grain_rows == 0, (
    "Duplicate rows detected at the Taxi Zone × Date × Temporal Bucket grain."
)

print("Modeling panel grain validated.")

,qa_check,value,status
0,analytical_grain,Taxi Zone × Date × Temporal Bucket,reference
1,duplicate_grain_rows,0,pass


Modeling panel grain validated.


Findings\. The modeling panel preserves the project grain cleanly: there are zero duplicate Taxi Zone × Date × Temporal Bucket rows\. That means feature inventory and later modeling\-set construction can proceed without worrying that engineered features have changed the analytical unit\.

We also lock down the congestion\-pricing period field here, using the loaded panel rather than querying the parquet separately\.

In [8]:
# ---------------------------------------------------------------------
# Validate congestion-pricing period labels
# ---------------------------------------------------------------------

unexpected_pre_post_cp_values = sorted(
    set(modeling_df["pre_post_cp"].dropna())
    - VALID_CP_PERIOD_VALUES
)

pre_post_cp_value_counts_df = (
    modeling_df
    .groupby("pre_post_cp", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("pre_post_cp")
    .reset_index(drop=True)
)

pre_post_cp_value_counts_df["row_pct"] = (
    pre_post_cp_value_counts_df["row_count"]
    / pre_post_cp_value_counts_df["row_count"].sum()
).round(5)

pre_post_cp_validation_df = pd.DataFrame(
    [
        {
            "qa_check": "unexpected_pre_post_cp_values",
            "value": len(unexpected_pre_post_cp_values),
            "status": "pass" if len(unexpected_pre_post_cp_values) == 0 else "fail",
        },
    ]
)

display(pre_post_cp_validation_df)
display(pre_post_cp_value_counts_df)

assert not unexpected_pre_post_cp_values, (
    "Unexpected pre_post_cp values found: "
    f"{unexpected_pre_post_cp_values}. Expected only pre_cp/post_cp."
)

print("Congestion-pricing period labels validated.")

,qa_check,value,status
0,unexpected_pre_post_cp_values,0,pass


,pre_post_cp,row_count,row_pct
0,post_cp,593065,0.38027
1,pre_cp,966525,0.61973


Congestion-pricing period labels validated.


Findings\. The congestion\-pricing period field is standardized: the panel contains no unexpected labels, only \`pre\_cp\` and \`post\_cp\`\. The split is about 62\.0% pre\-CP and 38\.0% post\-CP, which gives downstream feature selection a clean policy\-period field without carrying forward the earlier naming drift\.

### Build Feature Inventory

Let’s use the feature manifests as the source of truth for lineage and definitions\. We only fall back to light inference for columns that were not documented in a manifest\.

In [9]:
# ---------------------------------------------------------------------
# Load and standardize engineering manifests
# ---------------------------------------------------------------------

manifest_search_dirs = [
    PIPELINE_DATA_DIR / "1.5.1.final_tables",
    PIPELINE_DATA_DIR / "1.5.2.final_tables",
    PIPELINE_DATA_DIR / "1.5.3.final_tables",
    PIPELINE_DATA_DIR / "1.5.4.final_tables",
    PIPELINE_DATA_DIR / "1.5.5.final_tables",
]

manifest_paths = []

for manifest_dir in manifest_search_dirs:
    if manifest_dir.exists():
        manifest_paths.extend(sorted(manifest_dir.glob("*manifest*.csv")))

manifest_records = []

for manifest_path in manifest_paths:
    manifest_df = pd.read_csv(manifest_path)
    source_notebook_from_path = manifest_path.parent.name.replace(".final_tables", "")

    feature_name_column = next(
        (
            column
            for column in ["feature_name", "column_name", "feature", "output_feature"]
            if column in manifest_df.columns
        ),
        None,
    )

    if feature_name_column is None:
        print(f"Skipping manifest without a feature-name column: {manifest_path}")
        continue

    for _, row in manifest_df.iterrows():
        feature_name = row.get(feature_name_column)

        if pd.isna(feature_name):
            continue

        manifest_records.append(
            {
                "feature_name": feature_name,
                "manifest_source_path": str(manifest_path),
                "source_notebook": row.get(
                    "created_in_notebook",
                    row.get("source_notebook", source_notebook_from_path),
                ),
                "feature_family": row.get("feature_family", np.nan),
                "feature_type": row.get("feature_type", np.nan),
                "source_metric": row.get(
                    "source_metric",
                    row.get("base_metric", np.nan),
                ),
                "modality": row.get("modality", np.nan),
                "definition": row.get(
                    "definition",
                    row.get("description", np.nan),
                ),
                "production_status": row.get("production_status", np.nan),
            }
        )

engineering_feature_catalog_df = pd.DataFrame(manifest_records)

if not engineering_feature_catalog_df.empty:
    engineering_feature_catalog_df = (
        engineering_feature_catalog_df
        .drop_duplicates(subset=["feature_name"], keep="last")
        .reset_index(drop=True)
    )

engineering_manifest_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "manifest_files_found",
            "value": len(manifest_paths),
        },
        {
            "qa_check": "manifest_features_loaded",
            "value": len(engineering_feature_catalog_df),
        },
    ]
)

display(engineering_manifest_summary_df)
display(engineering_feature_catalog_df.head(15))

assert len(manifest_paths) > 0, (
    "No engineering feature manifests were found for 1.5.1 through 1.5.5."
)

print(
    "Engineering manifests loaded. "
    f"Documented features: {len(engineering_feature_catalog_df):,}."
)

,qa_check,value
0,manifest_files_found,5
1,manifest_features_loaded,316


,feature_name,manifest_source_path,source_notebook,feature_family,feature_type,source_metric,modality,definition,production_status
0,avg_bus_speed_abs_change_1_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_absolute_change,avg_bus_speed,bus,Absolute change in avg_bus_speed from the prev...,production
1,avg_bus_speed_cp_abs_delta_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,cp_same_bucket_absolute_delta,avg_bus_speed,bus,Post-CP minus pre-CP same-bucket mean differen...,production
2,avg_bus_speed_cp_pct_delta_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,cp_same_bucket_percent_delta,avg_bus_speed,bus,Percent difference between post-CP and pre-CP ...,production
3,avg_bus_speed_ema_7_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_ema,avg_bus_speed,bus,Exponential moving average of avg_bus_speed wi...,production
4,avg_bus_speed_lag_1_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_lag,avg_bus_speed,bus,Previous same-bucket observation of avg_bus_sp...,production
5,avg_bus_speed_pct_change_1_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_percent_change,avg_bus_speed,bus,Percent change in avg_bus_speed from the previ...,production
6,avg_bus_speed_post_cp_mean_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,post_cp_same_bucket_mean,avg_bus_speed,bus,Post-congestion-pricing same-bucket mean of av...,production
7,avg_bus_speed_pre_cp_mean_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,pre_cp_same_bucket_mean,avg_bus_speed,bus,Pre-congestion-pricing same-bucket mean of avg...,production
8,avg_bus_speed_rolling_mean_7_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_rolling_mean,avg_bus_speed,bus,Rolling same-bucket mean of avg_bus_speed.,production
9,avg_bus_speed_rolling_std_7_same_bucket,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_rolling_std,avg_bus_speed,bus,Rolling same-bucket standard deviation of avg_...,production


Engineering manifests loaded. Documented features: 316.


Findings\. The manifest loader is doing the handoff work we need from the earlier 1\.5\.x notebooks\. The catalog brings documented engineered features into one place, then the next inventory block fills the remaining gaps for raw pre\-1\.5 metrics and any columns that were not covered by a manifest\. This keeps the notebook grounded in the manifests where they exist, without pretending the older parts of the project had the same metadata system from the beginning\.

Now we join the manifest catalog to the actual modeling panel columns and add panel\-specific diagnostics such as dtype, numeric status, null coverage, and variance flags\.

In [10]:
# ---------------------------------------------------------------------
# Define curated metadata for pre-1.5 raw metrics
# ---------------------------------------------------------------------

# Raw/base metrics were created before the 1.5.x feature-manifest convention.
# We document them here so they are not mislabeled as unknown or unclassified.
RAW_BASE_METRIC_METADATA = {
    "traffic_volume": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "traffic_volume",
        "modality": "traffic",
        "definition": "Raw roadway traffic volume from the harmonized mobility panel.",
    },
    "bus_trip_count": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "bus_trip_count",
        "modality": "bus",
        "definition": "Raw bus trip count from the harmonized mobility panel.",
    },
    "avg_bus_speed": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "avg_bus_speed",
        "modality": "bus",
        "definition": "Raw average bus speed from the harmonized mobility panel.",
    },
    "avg_bus_travel_time": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "avg_bus_travel_time",
        "modality": "bus",
        "definition": "Raw average bus travel time from the harmonized mobility panel.",
    },
    "subway_ridership": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "subway_ridership",
        "modality": "subway",
        "definition": "Raw subway ridership from the harmonized mobility panel.",
    },
    "subway_transfers": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "subway_transfers",
        "modality": "subway",
        "definition": "Raw subway transfer count from the harmonized mobility panel.",
    },
    "taxi_trip_count": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "taxi_trip_count",
        "modality": "taxi",
        "definition": "Raw combined taxi trip count from the harmonized mobility panel.",
    },
    "taxi_avg_trip_distance": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "taxi_avg_trip_distance",
        "modality": "taxi",
        "definition": "Raw average combined taxi trip distance from the harmonized mobility panel.",
    },
    "taxi_avg_trip_duration": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "taxi_avg_trip_duration",
        "modality": "taxi",
        "definition": "Raw average combined taxi trip duration from the harmonized mobility panel.",
    },
    "taxi_avg_trip_speed": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "taxi_avg_trip_speed",
        "modality": "taxi",
        "definition": "Raw average combined taxi trip speed from the harmonized mobility panel.",
    },
    "taxi_distinct_dropoff_zone_count": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "taxi_distinct_dropoff_zone_count",
        "modality": "taxi",
        "definition": "Raw count of distinct taxi dropoff zones connected to each Taxi Zone × Date × Temporal Bucket row.",
    },
    "fhvhv_trip_count": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "fhvhv_trip_count",
        "modality": "fhvhv",
        "definition": "Raw FHVHV trip count from the harmonized mobility panel.",
    },
    "fhvhv_avg_trip_distance": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "fhvhv_avg_trip_distance",
        "modality": "fhvhv",
        "definition": "Raw average FHVHV trip distance from the harmonized mobility panel.",
    },
    "fhvhv_avg_trip_duration": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "fhvhv_avg_trip_duration",
        "modality": "fhvhv",
        "definition": "Raw average FHVHV trip duration from the harmonized mobility panel.",
    },
    "fhvhv_avg_trip_speed": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "fhvhv_avg_trip_speed",
        "modality": "fhvhv",
        "definition": "Raw average FHVHV trip speed from the harmonized mobility panel.",
    },
    "fhvhv_distinct_dropoff_zone_count": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "raw_metric",
        "source_metric": "fhvhv_distinct_dropoff_zone_count",
        "modality": "fhvhv",
        "definition": "Raw count of distinct FHVHV dropoff zones connected to each Taxi Zone × Date × Temporal Bucket row.",
    },
}

print(f"Curated raw/base metric metadata entries: {len(RAW_BASE_METRIC_METADATA):,}")

Curated raw/base metric metadata entries: 16


These helper functions keep the inventory logic readable: manifests are still the main source of truth, and these fallbacks only handle older raw metrics or lightly infer labels for undocumented columns\.

In [11]:
# ---------------------------------------------------------------------
# Define feature-inventory helper functions
# ---------------------------------------------------------------------

SOURCE_NOTEBOOK_BY_FEATURE_FAMILY = {
    "raw_metric": "pre_1.5_engineering",
    "temporal_history": "1.5.1",
    "spatial_context": "1.5.2",
    "multimodal_interaction": "1.5.3",
    "decomposition": "1.5.4",
    "deviation_anomaly": "1.5.5",
}

TRAFFIC_CONTEXT_FEATURE_NAMES = {
    "max_yearly_ratio",
    "is_reliable_traffic_temporal_zone",
    "is_stable_or_moderate_traffic_zone",
    "traffic_previous_observation",
    "traffic_days_since_previous_observation",
    "traffic_change_from_previous_observation",
    "traffic_pct_change_from_previous_observation",
}


def clean_manifest_value(value):
    """Normalize blank manifest values to NaN without changing real strings."""
    if pd.isna(value):
        return np.nan

    value_as_string = str(value).strip()

    if value_as_string == "" or value_as_string.lower() in {"nan", "none"}:
        return np.nan

    return value_as_string


def first_populated_value(*values):
    """Return the first non-empty value from a set of metadata candidates."""
    for value in values:
        cleaned_value = clean_manifest_value(value)

        if pd.notna(cleaned_value):
            return cleaned_value

    return np.nan


def infer_source_metric_from_name(feature_name):
    """Infer a base metric only when the manifest did not already provide one."""
    if feature_name in TRAFFIC_CONTEXT_FEATURE_NAMES:
        return "traffic_volume"

    for metric in sorted(
        RAW_BASE_METRIC_METADATA.keys(),
        key=len,
        reverse=True,
    ):
        if feature_name == metric or feature_name.startswith(f"{metric}_"):
            return metric

    if feature_name.startswith("cp_zone_mean_"):
        return feature_name.replace("cp_zone_mean_", "")

    if feature_name.startswith("borough_mean_"):
        return feature_name.replace("borough_mean_", "")

    if feature_name.startswith("connected_mean_"):
        return feature_name.replace("connected_mean_", "")

    if feature_name in {
        "connected_zone_count",
        "neighbor_count",
    }:
        return "spatial_connectivity"

    if feature_name in {
        "distance_to_cbd_miles",
        "distance_to_gateway_miles",
        "is_cbd_adjacent_zone",
        "is_cbd_gateway_zone",
        "is_cbd_zone",
    }:
        return "spatial_geography"

    if "mta_crossing" in feature_name or "crossing_volume" in feature_name:
        return "mta_crossing_volume"

    if feature_name == "manhattan_connected_crossing_share":
        return "mta_crossing_volume"

    return np.nan


def infer_feature_family_from_name(feature_name, source_notebook=None):
    """Classify feature families using explicit notebook lineage first."""
    if source_notebook in SOURCE_NOTEBOOK_BY_FEATURE_FAMILY.values():
        for family, notebook in SOURCE_NOTEBOOK_BY_FEATURE_FAMILY.items():
            if source_notebook == notebook:
                return family

    if feature_name in RAW_BASE_METRIC_METADATA:
        return "raw_metric"

    if feature_name in TRAFFIC_CONTEXT_FEATURE_NAMES:
        if "previous_observation" in feature_name:
            return "temporal_history"
        return "multimodal_interaction"

    if any(token in feature_name for token in ["_lag_", "rolling_", "_ema_", "_abs_change_", "_pct_change_"]):
        return "temporal_history"

    if feature_name.startswith(("connected_mean_", "borough_mean_", "cp_zone_mean_")):
        return "spatial_context"

    if feature_name in {
        "connected_zone_count",
        "neighbor_count",
        "distance_to_cbd_miles",
        "distance_to_gateway_miles",
        "is_cbd_adjacent_zone",
        "is_cbd_gateway_zone",
        "is_cbd_zone",
    }:
        return "spatial_context"

    if any(token in feature_name for token in ["_x_", "_to_", "local_vs_connected", "mta_crossing_share"]):
        return "multimodal_interaction"

    if any(token in feature_name for token in ["_seasonal", "_residual", "fourier20"]):
        return "decomposition"

    if any(token in feature_name for token in ["_anomaly_", "_percentile_rank", "_abs_zscore"]):
        return "deviation_anomaly"

    return "other_or_unclassified"


def infer_modality_from_name(feature_name, source_metric=None, manifest_modality=None):
    """Infer modality while preserving useful manifest labels when available."""
    manifest_modality = clean_manifest_value(manifest_modality)

    if pd.notna(manifest_modality) and manifest_modality != "other":
        return manifest_modality

    source_metric = clean_manifest_value(source_metric)

    if feature_name in TRAFFIC_CONTEXT_FEATURE_NAMES:
        return "traffic"

    if pd.notna(source_metric):
        source_metric_as_string = str(source_metric)

        if "|" in source_metric_as_string:
            return "multimodal"

        if source_metric_as_string in {"spatial_geography", "spatial_connectivity"}:
            return "spatial"

        if source_metric_as_string in {"mta_crossing_volume", "traffic_volume"}:
            return "traffic"

        for modality, prefixes in MODALITY_METRIC_PREFIXES.items():
            if any(source_metric_as_string.startswith(prefix) for prefix in prefixes):
                return modality

    for modality, prefixes in MODALITY_METRIC_PREFIXES.items():
        if any(feature_name.startswith(prefix) for prefix in prefixes):
            return modality

    if any(token in feature_name for token in ["_x_", "_to_", "interaction"]):
        return "multimodal"

    if any(token in feature_name for token in ["cbd", "gateway", "connected_zone", "neighbor_count"]):
        return "spatial"

    if "traffic" in feature_name or "crossing" in feature_name:
        return "traffic"

    return "other"


def normalize_source_notebook(source_notebook, feature_family):
    """Fill missing notebook lineage from the feature family when needed."""
    source_notebook = clean_manifest_value(source_notebook)

    if pd.notna(source_notebook):
        return source_notebook

    return SOURCE_NOTEBOOK_BY_FEATURE_FAMILY.get(
        feature_family,
        "unknown",
    )


def is_core_metric_descendant_value(source_metric):
    """Flag single-metric or composite descendants of the seven core metrics."""
    source_metric = clean_manifest_value(source_metric)

    if pd.isna(source_metric):
        return False

    source_metric_parts = [
        part.strip()
        for part in str(source_metric).split("|")
        if part.strip()
    ]

    return any(part in CORE_MODELING_METRICS for part in source_metric_parts)


def quote_identifier(identifier):
    """Safely quote a column name for DuckDB SQL."""
    return '"' + str(identifier).replace('"', '""') + '"'


print("Feature-inventory helper functions ready.")

Feature-inventory helper functions ready.


Now we build the inventory itself\. The row\-count diagnostics use DuckDB against the parquet file, and the classification fields are normalized after the manifest merge so stale or overly detailed manifest labels do not drive modeling decisions\.

In [12]:
# ---------------------------------------------------------------------
# Build and display manifest-backed feature inventory
# ---------------------------------------------------------------------

panel_row_count = len(modeling_df)
manifest_lookup_df = engineering_feature_catalog_df.copy()

if not manifest_lookup_df.empty:
    manifest_lookup_df = (
        manifest_lookup_df
        .drop_duplicates(subset=["feature_name"], keep="last")
        .set_index("feature_name")
    )

inventory_records = []

for feature_name in modeling_df.columns:
    manifest_record = (
        manifest_lookup_df.loc[feature_name].to_dict()
        if not manifest_lookup_df.empty and feature_name in manifest_lookup_df.index
        else {}
    )

    raw_metadata = RAW_BASE_METRIC_METADATA.get(feature_name, {})
    manifest_source_metric = first_populated_value(
        raw_metadata.get("source_metric"),
        manifest_record.get("source_metric"),
    )

    source_metric = first_populated_value(
        manifest_source_metric,
        infer_source_metric_from_name(feature_name),
    )

    manifest_family = first_populated_value(
        raw_metadata.get("feature_family"),
        manifest_record.get("feature_family"),
    )

    feature_family = first_populated_value(
        manifest_family,
        infer_feature_family_from_name(
            feature_name=feature_name,
            source_notebook=manifest_record.get("source_notebook"),
        ),
    )

    manifest_modality = first_populated_value(
        raw_metadata.get("modality"),
        manifest_record.get("modality"),
    )

    modality = infer_modality_from_name(
        feature_name=feature_name,
        source_metric=source_metric,
        manifest_modality=manifest_modality,
    )

    source_notebook = normalize_source_notebook(
        first_populated_value(
            raw_metadata.get("source_notebook"),
            manifest_record.get("source_notebook"),
        ),
        feature_family,
    )

    inventory_records.append(
        {
            "feature_name": feature_name,
            "dtype": str(modeling_df[feature_name].dtype),
            "is_numeric": pd.api.types.is_numeric_dtype(modeling_df[feature_name]),
            "manifest_source_path": manifest_record.get("manifest_source_path", np.nan),
            "source_notebook": source_notebook,
            "feature_family": feature_family,
            "feature_type": first_populated_value(
                manifest_record.get("feature_type"),
                np.nan,
            ),
            "source_metric": source_metric,
            "modality": modality,
            "definition": first_populated_value(
                raw_metadata.get("definition"),
                manifest_record.get("definition"),
            ),
            "production_status": first_populated_value(
                manifest_record.get("production_status"),
                raw_metadata.get("production_status"),
                "production" if feature_name not in NON_MODELING_COLUMNS else "metadata",
            ),
            "manifest_documented": feature_name in manifest_lookup_df.index if not manifest_lookup_df.empty else False,
        }
    )

feature_inventory_df = pd.DataFrame(inventory_records)

feature_inventory_df["is_core_metric_descendant"] = (
    feature_inventory_df["source_metric"]
    .apply(is_core_metric_descendant_value)
)

feature_inventory_df["is_modeling_candidate"] = (
    feature_inventory_df["is_numeric"]
    & ~feature_inventory_df["feature_name"].isin(NON_MODELING_COLUMNS)
)

coverage_records = []
coverage_chunk_size = 50
panel_columns = feature_inventory_df["feature_name"].tolist()

# Count coverage directly from parquet in chunks so this inventory step does not
# duplicate the full modeling panel in memory.
for start_idx in range(0, len(panel_columns), coverage_chunk_size):
    chunk_columns = panel_columns[start_idx:start_idx + coverage_chunk_size]

    count_sql = ",\n            ".join(
        [
            f"COUNT({quote_identifier(column)}) AS count_col_{column_idx}"
            for column_idx, column in enumerate(chunk_columns)
        ]
    )

    chunk_counts_df = duckdb.sql(
        f"""
        SELECT
            {count_sql}
        FROM read_parquet('{duckdb_path(INPUT_PANEL_PATH)}')
        """
    ).df()

    for column_idx, column in enumerate(chunk_columns):
        non_null_observations = int(chunk_counts_df[f"count_col_{column_idx}"].iloc[0])

        coverage_records.append(
            {
                "feature_name": column,
                "non_null_observations": non_null_observations,
                "null_observations": panel_row_count - non_null_observations,
            }
        )

coverage_df = pd.DataFrame(coverage_records)

feature_inventory_df = feature_inventory_df.merge(
    coverage_df,
    on="feature_name",
    how="left",
    validate="one_to_one",
)

feature_inventory_df["null_pct"] = (
    feature_inventory_df["null_observations"] / panel_row_count
).round(6)

feature_inventory_df["non_null_pct"] = (
    feature_inventory_df["non_null_observations"] / panel_row_count
).round(6)

# Full variance screening happens later on the diagnostic sample.
feature_inventory_df["near_zero_variance_flag"] = False

feature_inventory_df.to_csv(FEATURE_INVENTORY_PATH, index=False)

candidate_feature_inventory_df = feature_inventory_df.loc[
    feature_inventory_df["is_modeling_candidate"]
].copy()

feature_inventory_classification_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "candidate_modeling_features",
            "value": len(candidate_feature_inventory_df),
        },
        {
            "qa_check": "candidate_features_missing_family",
            "value": candidate_feature_inventory_df["feature_family"].isna().sum(),
        },
        {
            "qa_check": "candidate_features_unclassified_family",
            "value": candidate_feature_inventory_df["feature_family"].eq("other_or_unclassified").sum(),
        },
        {
            "qa_check": "candidate_features_missing_modality",
            "value": candidate_feature_inventory_df["modality"].isna().sum(),
        },
        {
            "qa_check": "candidate_features_other_modality",
            "value": candidate_feature_inventory_df["modality"].eq("other").sum(),
        },
        {
            "qa_check": "candidate_features_missing_source_metric",
            "value": candidate_feature_inventory_df["source_metric"].isna().sum(),
        },
    ]
)

feature_inventory_display_df = (
    feature_inventory_df
    .sort_values(
        [
            "is_modeling_candidate",
            "source_notebook",
            "feature_family",
            "modality",
            "feature_name",
        ],
        ascending=[False, True, True, True, True],
    )
    .reset_index(drop=True)
)

display(feature_inventory_classification_qa_df)

with pd.option_context(
    "display.max_rows", max(400, len(feature_inventory_display_df)),
    "display.max_columns", 200,
):
    display(feature_inventory_display_df)

print(
    "Manifest-backed feature inventory created. "
    f"Columns inventoried: {len(feature_inventory_df):,}."
)

,qa_check,value
0,candidate_modeling_features,329
1,candidate_features_missing_family,0
2,candidate_features_unclassified_family,0
3,candidate_features_missing_modality,0
4,candidate_features_other_modality,0
5,candidate_features_missing_source_metric,0


,feature_name,dtype,is_numeric,manifest_source_path,source_notebook,feature_family,feature_type,source_metric,modality,definition,production_status,manifest_documented,is_core_metric_descendant,is_modeling_candidate,non_null_observations,null_observations,null_pct,non_null_pct,near_zero_variance_flag
0,avg_bus_speed_abs_change_1_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_absolute_change,avg_bus_speed,bus,Absolute change in avg_bus_speed from the prev...,production,True,True,True,1473552,86038,0.055167,0.944833,False
1,avg_bus_speed_cp_abs_delta_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,cp_same_bucket_absolute_delta,avg_bus_speed,bus,Post-CP minus pre-CP same-bucket mean differen...,production,True,True,True,1479448,80142,0.051387,0.948613,False
2,avg_bus_speed_cp_pct_delta_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,cp_same_bucket_percent_delta,avg_bus_speed,bus,Percent difference between post-CP and pre-CP ...,production,True,True,True,1479448,80142,0.051387,0.948613,False
3,avg_bus_speed_ema_7_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_ema,avg_bus_speed,bus,Exponential moving average of avg_bus_speed wi...,production,True,True,True,1483766,75824,0.048618,0.951382,False
4,avg_bus_speed_lag_1_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_lag,avg_bus_speed,bus,Previous same-bucket observation of avg_bus_sp...,production,True,True,True,1474001,85589,0.054879,0.945121,False
5,avg_bus_speed_pct_change_1_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_percent_change,avg_bus_speed,bus,Percent change in avg_bus_speed from the previ...,production,True,True,True,1473552,86038,0.055167,0.944833,False
6,avg_bus_speed_post_cp_mean_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,post_cp_same_bucket_mean,avg_bus_speed,bus,Post-congestion-pricing same-bucket mean of av...,production,True,True,True,1480295,79295,0.050843,0.949157,False
7,avg_bus_speed_pre_cp_mean_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,pre_cp_same_bucket_mean,avg_bus_speed,bus,Pre-congestion-pricing same-bucket mean of avg...,production,True,True,True,1484531,75059,0.048127,0.951873,False
8,avg_bus_speed_rolling_mean_7_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_rolling_mean,avg_bus_speed,bus,Rolling same-bucket mean of avg_bus_speed.,production,True,True,True,1477171,82419,0.052847,0.947153,False
9,avg_bus_speed_rolling_std_7_same_bucket,float64,True,pipeline_data/1.5.2.final_tables/spatial_featu...,1.5.1,temporal_history,same_bucket_rolling_std,avg_bus_speed,bus,Rolling same-bucket standard deviation of avg_...,production,True,True,True,1474529,85061,0.054541,0.945459,False


Manifest-backed feature inventory created. Columns inventoried: 342.


Findings\. The inventory build now gives us a usable column\-by\-column map of the modeling panel instead of a loose list of feature names\. The important part is that the older raw metrics, the 1\.5\.x manifest\-backed features, and the remaining inferred columns all land in one table with modeling status, source notebook, feature family, modality, source metric, and coverage fields\. That gives the rest of 1\.5\.6 a clean decision surface instead of forcing every later block to rediscover feature lineage from column names\.

Some inventory fields are optional for identifiers, raw metrics, and pre\-manifest columns\. Let’s clean those fields so the inventory uses explicit placeholders instead of blank metadata gaps\.

In [13]:
# ---------------------------------------------------------------------
# Clean optional metadata fields for raw and identifier columns
# ---------------------------------------------------------------------

RAW_AND_IDENTIFIER_METADATA_CLEANUP = {
    "taxi_zone_id": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "taxi_zone_identifier",
        "source_metric": "not_applicable",
        "modality": "taxi",
        "definition": "Taxi Zone identifier used as part of the project analytical grain.",
        "production_status": "metadata",
    },
    "zone": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "taxi_zone_label",
        "source_metric": "not_applicable",
        "modality": "spatial",
        "definition": "Taxi Zone name used for interpretation and reporting.",
        "production_status": "metadata",
    },
    "borough": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "borough_label",
        "source_metric": "not_applicable",
        "modality": "spatial",
        "definition": "NYC borough associated with the Taxi Zone.",
        "production_status": "metadata",
    },
    "canonical_location_id": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "canonical_taxi_zone_identifier",
        "source_metric": "not_applicable",
        "modality": "spatial",
        "definition": "Canonical Taxi Zone identifier used after spatial standardization.",
        "production_status": "metadata",
    },
    "date": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "date_key",
        "source_metric": "not_applicable",
        "modality": "temporal",
        "definition": "Calendar date for the mobility-state observation.",
        "production_status": "metadata",
    },
    "year": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "calendar_year",
        "source_metric": "not_applicable",
        "modality": "temporal",
        "definition": "Calendar year derived from the observation date.",
        "production_status": "metadata",
    },
    "month": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "calendar_month",
        "source_metric": "not_applicable",
        "modality": "temporal",
        "definition": "Calendar month derived from the observation date.",
        "production_status": "metadata",
    },
    "day_of_week": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "day_of_week_label",
        "source_metric": "not_applicable",
        "modality": "temporal",
        "definition": "Day-of-week label derived from the observation date.",
        "production_status": "metadata",
    },
    "temporal_bucket": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "temporal_bucket_label",
        "source_metric": "not_applicable",
        "modality": "temporal",
        "definition": "Project daypart bucket used as part of the analytical grain.",
        "production_status": "metadata",
    },
    "pre_post_cp": {
        "source_notebook": "pre_1.5_engineering",
        "feature_family": "identifier_or_metadata",
        "feature_type": "congestion_pricing_period_label",
        "source_metric": "not_applicable",
        "modality": "temporal",
        "definition": "Canonical pre-CP/post-CP period label for the observation date.",
        "production_status": "metadata",
    },
}

# Raw/base metrics were curated in the prior metadata block. Fill their optional
# fields so the inventory table does not show avoidable blanks for known columns.
for feature_name in RAW_BASE_METRIC_METADATA:
    RAW_AND_IDENTIFIER_METADATA_CLEANUP.setdefault(feature_name, {})
    RAW_AND_IDENTIFIER_METADATA_CLEANUP[feature_name].update({
        "feature_type": "raw_metric",
        "production_status": "production",
    })

for feature_name, metadata in RAW_BASE_METRIC_METADATA.items():
    RAW_AND_IDENTIFIER_METADATA_CLEANUP[feature_name].update(metadata)

for feature_name, metadata in RAW_AND_IDENTIFIER_METADATA_CLEANUP.items():
    row_mask = feature_inventory_df["feature_name"].eq(feature_name)

    if not row_mask.any():
        continue

    for field_name, field_value in metadata.items():
        feature_inventory_df.loc[row_mask, field_name] = field_value

# Preserve the distinction between manifest-documented engineered features and
# curated pre-manifest columns, but avoid a blank path for known curated fields.
curated_mask = feature_inventory_df["feature_name"].isin(
    RAW_AND_IDENTIFIER_METADATA_CLEANUP.keys()
)
feature_inventory_df.loc[
    curated_mask & feature_inventory_df["manifest_source_path"].isna(),
    "manifest_source_path",
] = "curated_pre_1.5_metadata"

feature_inventory_df.to_csv(FEATURE_INVENTORY_PATH, index=False)

curated_metadata_cleanup_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "curated_columns_updated",
            "value": int(curated_mask.sum()),
            "status": "reference",
        },
        {
            "qa_check": "curated_columns_with_missing_optional_metadata",
            "value": int(
                feature_inventory_df.loc[
                    curated_mask,
                    [
                        "source_notebook",
                        "feature_family",
                        "feature_type",
                        "source_metric",
                        "modality",
                        "definition",
                        "production_status",
                        "manifest_source_path",
                    ],
                ].isna().sum().sum()
            ),
            "status": "pass",
        },
    ]
)

curated_metadata_preview_df = (
    feature_inventory_df
    .loc[
        curated_mask,
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "feature_type",
            "source_metric",
            "modality",
            "definition",
            "production_status",
            "manifest_source_path",
        ],
    ]
    .sort_values("feature_name")
    .reset_index(drop=True)
)

display(curated_metadata_cleanup_qa_df)
display(curated_metadata_preview_df)

assert curated_metadata_cleanup_qa_df.loc[
    curated_metadata_cleanup_qa_df["qa_check"] == "curated_columns_with_missing_optional_metadata",
    "value",
].iloc[0] == 0, "Curated raw/metadata columns still have missing optional metadata fields."

print("Curated raw and identifier metadata cleanup complete.")

,qa_check,value,status
0,curated_columns_updated,26,reference
1,curated_columns_with_missing_optional_metadata,0,pass


,feature_name,source_notebook,feature_family,feature_type,source_metric,modality,definition,production_status,manifest_source_path
0,avg_bus_speed,pre_1.5_engineering,raw_metric,raw_metric,avg_bus_speed,bus,Raw average bus speed from the harmonized mobi...,production,curated_pre_1.5_metadata
1,avg_bus_travel_time,pre_1.5_engineering,raw_metric,raw_metric,avg_bus_travel_time,bus,Raw average bus travel time from the harmonize...,production,curated_pre_1.5_metadata
2,borough,pre_1.5_engineering,identifier_or_metadata,borough_label,not_applicable,spatial,NYC borough associated with the Taxi Zone.,metadata,curated_pre_1.5_metadata
3,bus_trip_count,pre_1.5_engineering,raw_metric,raw_metric,bus_trip_count,bus,Raw bus trip count from the harmonized mobilit...,production,curated_pre_1.5_metadata
4,canonical_location_id,pre_1.5_engineering,identifier_or_metadata,canonical_taxi_zone_identifier,not_applicable,spatial,Canonical Taxi Zone identifier used after spat...,metadata,curated_pre_1.5_metadata
5,date,pre_1.5_engineering,identifier_or_metadata,date_key,not_applicable,temporal,Calendar date for the mobility-state observation.,metadata,curated_pre_1.5_metadata
6,day_of_week,pre_1.5_engineering,identifier_or_metadata,day_of_week_label,not_applicable,temporal,Day-of-week label derived from the observation...,metadata,curated_pre_1.5_metadata
7,fhvhv_avg_trip_distance,pre_1.5_engineering,raw_metric,raw_metric,fhvhv_avg_trip_distance,fhvhv,Raw average FHVHV trip distance from the harmo...,production,curated_pre_1.5_metadata
8,fhvhv_avg_trip_duration,pre_1.5_engineering,raw_metric,raw_metric,fhvhv_avg_trip_duration,fhvhv,Raw average FHVHV trip duration from the harmo...,production,curated_pre_1.5_metadata
9,fhvhv_avg_trip_speed,pre_1.5_engineering,raw_metric,raw_metric,fhvhv_avg_trip_speed,fhvhv,Raw average FHVHV trip speed from the harmoniz...,production,curated_pre_1.5_metadata


Curated raw and identifier metadata cleanup complete.


Findings\. The optional metadata cleanup keeps the inventory readable without pretending every column came from a formal manifest\. Identifier columns, raw pre\-1\.5 metrics, and context fields now carry explicit placeholder values where lineage fields are not applicable\. That matters because the later QA should focus on true modeling\-decision fields, not harmless blanks in optional documentation columns\.

Now we display the cleaned inventory itself\. This is the full column\-level catalog that downstream pruning, redundancy checks, and feature\-set construction will use\.

In [14]:
# ---------------------------------------------------------------------
# Display cleaned feature inventory
# ---------------------------------------------------------------------

# A few carried-forward standardized fields encode both the spatial operation and
# the standardization suffix in the column name. For inventory purposes, the
# source metric should be the underlying mobility metric, not the derived z-score
# column name.
def normalize_inventory_source_metric(feature_name, source_metric):
    if str(feature_name).startswith("connected_mean_") and str(feature_name).endswith("_zscore"):
        return (
            str(feature_name)
            .replace("connected_mean_", "", 1)
            .replace("_zscore", "")
        )

    if isinstance(source_metric, str) and source_metric.endswith("_zscore"):
        return source_metric.replace("_zscore", "")

    return source_metric


feature_inventory_df = feature_inventory_df.copy()

feature_inventory_df["source_metric"] = feature_inventory_df.apply(
    lambda row: normalize_inventory_source_metric(
        feature_name=row["feature_name"],
        source_metric=row["source_metric"],
    ),
    axis=1,
)

optional_inventory_fill_values = {
    "manifest_source_path": "not_manifested",
    "source_notebook": "pre_1.5_engineering",
    "feature_family": "other_or_unclassified",
    "feature_type": "not_applicable",
    "source_metric": "not_applicable",
    "modality": "not_applicable",
    "definition": "not_documented_in_manifest",
    "production_status": "production",
}

for column, fill_value in optional_inventory_fill_values.items():
    if column not in feature_inventory_df.columns:
        feature_inventory_df[column] = fill_value

    feature_inventory_df[column] = (
        feature_inventory_df[column]
        .replace("", np.nan)
        .fillna(fill_value)
    )

feature_inventory_df.loc[
    ~feature_inventory_df["is_modeling_candidate"],
    "feature_family",
] = "identifier_or_metadata"

connected_mean_zscore_mask = (
    feature_inventory_df["feature_name"].astype(str).str.startswith("connected_mean_")
    & feature_inventory_df["feature_name"].astype(str).str.endswith("_zscore")
)

traffic_sparse_temporal_features = [
    "traffic_previous_observation",
    "traffic_change_from_previous_observation",
    "traffic_pct_change_from_previous_observation",
    "traffic_days_since_previous_observation",
]

traffic_sparse_temporal_mask = feature_inventory_df["feature_name"].isin(
    traffic_sparse_temporal_features
)

# These standardized connected-zone fields were created as spatial context, but
# some upstream manifests did not document them as standalone columns. Curate the
# metadata here so 1.5.6 does not treat them as generic fallback features.
feature_inventory_df.loc[connected_mean_zscore_mask, "manifest_source_path"] = (
    "curated_1.5.6_inventory_override"
)
feature_inventory_df.loc[connected_mean_zscore_mask, "source_notebook"] = "1.5.2"
feature_inventory_df.loc[connected_mean_zscore_mask, "feature_family"] = "spatial_context"
feature_inventory_df.loc[connected_mean_zscore_mask, "feature_type"] = (
    "standardized_connected_zone_spillover"
)
feature_inventory_df.loc[connected_mean_zscore_mask, "definition"] = (
    "Standardized connected-zone mean for the same date and temporal bucket."
)
feature_inventory_df.loc[connected_mean_zscore_mask, "production_status"] = "production"

# These sparse Traffic observation-history fields were created during the 1.5.3
# Traffic feasibility work. They are retained as Traffic context, not as part of
# the seven-core-metric decomposition framework.
feature_inventory_df.loc[traffic_sparse_temporal_mask, "manifest_source_path"] = (
    "curated_1.5.6_inventory_override"
)
feature_inventory_df.loc[traffic_sparse_temporal_mask, "source_notebook"] = "1.5.3"
feature_inventory_df.loc[traffic_sparse_temporal_mask, "feature_family"] = "temporal_history"
feature_inventory_df.loc[traffic_sparse_temporal_mask, "feature_type"] = (
    "traffic_sparse_observation_history"
)
feature_inventory_df.loc[traffic_sparse_temporal_mask, "source_metric"] = "traffic_volume"
feature_inventory_df.loc[traffic_sparse_temporal_mask, "modality"] = "traffic"
feature_inventory_df.loc[traffic_sparse_temporal_mask, "definition"] = (
    "Sparse Traffic observation-history feature used to characterize irregular roadway sensor coverage."
)
feature_inventory_df.loc[traffic_sparse_temporal_mask, "production_status"] = "production"

if "manifest_documented" in feature_inventory_df.columns:
    feature_inventory_df.loc[
        connected_mean_zscore_mask | traffic_sparse_temporal_mask,
        "manifest_documented",
    ] = True

candidate_inventory_df = (
    feature_inventory_df[feature_inventory_df["is_modeling_candidate"]]
    .copy()
    .reset_index(drop=True)
)

feature_inventory_df.to_csv(FEATURE_INVENTORY_PATH, index=False)

inventory_cleanup_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "total_inventory_columns",
            "value": len(feature_inventory_df),
        },
        {
            "qa_check": "candidate_modeling_features",
            "value": len(candidate_inventory_df),
        },
        {
            "qa_check": "connected_mean_zscore_features",
            "value": int(connected_mean_zscore_mask.sum()),
        },
        {
            "qa_check": "traffic_sparse_temporal_features",
            "value": int(traffic_sparse_temporal_mask.sum()),
        },
        {
            "qa_check": "connected_mean_zscore_source_metric_values",
            "value": candidate_inventory_df.loc[
                candidate_inventory_df["feature_name"].str.startswith("connected_mean_", na=False)
                & candidate_inventory_df["feature_name"].str.endswith("_zscore", na=False),
                "source_metric",
            ].nunique(),
        },
        {
            "qa_check": "source_metric_values_ending_in_zscore",
            "value": candidate_inventory_df["source_metric"].astype(str).str.endswith("_zscore").sum(),
        },
        {
            "qa_check": "connected_mean_zscore_fallback_feature_types",
            "value": int(
                (
                    feature_inventory_df.loc[connected_mean_zscore_mask, "feature_type"]
                    == "not_applicable"
                ).sum()
            ),
        },
        {
            "qa_check": "traffic_sparse_temporal_not_manifested_rows",
            "value": int(
                (
                    feature_inventory_df.loc[traffic_sparse_temporal_mask, "manifest_source_path"]
                    == "not_manifested"
                ).sum()
            ),
        },
    ]
)

display(inventory_cleanup_qa_df)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 240):
    display(feature_inventory_df)

assert not candidate_inventory_df["source_metric"].astype(str).str.endswith("_zscore").any(), (
    "One or more candidate source_metric values still points to a derived z-score field."
)

assert not (
    feature_inventory_df.loc[connected_mean_zscore_mask, "feature_type"]
    == "not_applicable"
).any(), (
    "One or more connected_mean_*_zscore features still has fallback feature_type metadata."
)

assert not (
    feature_inventory_df.loc[traffic_sparse_temporal_mask, "manifest_source_path"]
    == "not_manifested"
).any(), (
    "One or more sparse Traffic temporal features still has fallback manifest metadata."
)

print(
    "Cleaned feature inventory displayed and saved. "
    f"Rows: {len(feature_inventory_df):,}."
)

,qa_check,value
0,total_inventory_columns,342
1,candidate_modeling_features,329
2,connected_mean_zscore_features,7
3,traffic_sparse_temporal_features,4
4,connected_mean_zscore_source_metric_values,7
5,source_metric_values_ending_in_zscore,0
6,connected_mean_zscore_fallback_feature_types,0
7,traffic_sparse_temporal_not_manifested_rows,0


,feature_name,dtype,is_numeric,manifest_source_path,source_notebook,feature_family,feature_type,source_metric,modality,definition,production_status,manifest_documented,is_core_metric_descendant,is_modeling_candidate,non_null_observations,null_observations,null_pct,non_null_pct,near_zero_variance_flag
0,taxi_zone_id,int32,True,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,taxi_zone_identifier,not_applicable,taxi,Taxi Zone identifier used as part of the proje...,metadata,False,False,False,1559590,0,0.000000,1.000000,False
1,zone,object,False,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,taxi_zone_label,not_applicable,spatial,Taxi Zone name used for interpretation and rep...,metadata,False,False,False,1559590,0,0.000000,1.000000,False
2,borough,object,False,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,borough_label,not_applicable,spatial,NYC borough associated with the Taxi Zone.,metadata,False,False,False,1559590,0,0.000000,1.000000,False
3,canonical_location_id,int64,True,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,canonical_taxi_zone_identifier,not_applicable,spatial,Canonical Taxi Zone identifier used after spat...,metadata,False,False,False,1559590,0,0.000000,1.000000,False
4,date,datetime64[us],False,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,date_key,not_applicable,temporal,Calendar date for the mobility-state observation.,metadata,False,False,False,1559590,0,0.000000,1.000000,False
5,year,int32,True,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,calendar_year,not_applicable,temporal,Calendar year derived from the observation date.,metadata,False,False,False,1559590,0,0.000000,1.000000,False
6,month,int32,True,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,calendar_month,not_applicable,temporal,Calendar month derived from the observation date.,metadata,False,False,False,1559590,0,0.000000,1.000000,False
7,day_of_week,object,False,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,day_of_week_label,not_applicable,temporal,Day-of-week label derived from the observation...,metadata,False,False,False,1559590,0,0.000000,1.000000,False
8,temporal_bucket,object,False,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,temporal_bucket_label,not_applicable,temporal,Project daypart bucket used as part of the ana...,metadata,False,False,False,1559590,0,0.000000,1.000000,False
9,pre_post_cp,object,False,curated_pre_1.5_metadata,pre_1.5_engineering,identifier_or_metadata,congestion_pricing_period_label,not_applicable,temporal,Canonical pre-CP/post-CP period label for the ...,metadata,False,False,False,1559590,0,0.000000,1.000000,False


Cleaned feature inventory displayed and saved. Rows: 342.


Findings\. The cleaned inventory gives us 342 panel columns in one place, with 329 numeric modeling candidates after identifiers and non\-modeling fields are set aside\. The useful thing here is not just the count\. It is that each candidate now has the metadata needed for selection work: feature family, modality, source metric, core\-metric status, null coverage, and variance flags\.

### QA Feature Inventory

Let’s check the fields that actually drive 1\.5\.6 decisions\. If these are populated cleanly, the inventory is reliable enough for redundancy, contribution, and feature\-set construction\.

In [15]:
# ---------------------------------------------------------------------
# QA must-have feature inventory fields
# ---------------------------------------------------------------------

must_have_inventory_fields = [
    "feature_name",
    "is_numeric",
    "is_modeling_candidate",
    "feature_family",
    "modality",
    "source_metric",
    "is_core_metric_descendant",
    "non_null_pct",
    "null_pct",
]

candidate_inventory_df = feature_inventory_df.loc[
    feature_inventory_df["is_modeling_candidate"]
].copy()

must_have_field_qa_records = []

for field in must_have_inventory_fields:
    missing_values = candidate_inventory_df[field].isna().sum()

    must_have_field_qa_records.append(
        {
            "field_name": field,
            "candidate_features_checked": len(candidate_inventory_df),
            "missing_values": missing_values,
            "missing_pct": round(missing_values / len(candidate_inventory_df), 6),
            "status": "pass" if missing_values == 0 else "fail",
        }
    )

must_have_inventory_field_qa_df = pd.DataFrame(must_have_field_qa_records)

classification_quality_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "unclassified_feature_family_candidates",
            "value": candidate_inventory_df["feature_family"].eq("other_or_unclassified").sum(),
            "status": "pass" if candidate_inventory_df["feature_family"].eq("other_or_unclassified").sum() == 0 else "review",
        },
        {
            "qa_check": "other_modality_candidates",
            "value": candidate_inventory_df["modality"].eq("other").sum(),
            "status": "pass" if candidate_inventory_df["modality"].eq("other").sum() == 0 else "review",
        },
        {
            "qa_check": "core_metric_descendant_candidates",
            "value": candidate_inventory_df["is_core_metric_descendant"].sum(),
            "status": "reference",
        },
        {
            "qa_check": "non_core_or_context_candidates",
            "value": candidate_inventory_df["is_core_metric_descendant"].eq(False).sum(),
            "status": "reference",
        },
    ]
)

missing_source_metric_candidates_df = (
    candidate_inventory_df
    .loc[
        candidate_inventory_df["source_metric"].isna(),
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "modality",
            "source_metric",
            "manifest_documented",
            "non_null_pct",
            "null_pct",
        ],
    ]
    .sort_values(["feature_family", "modality", "feature_name"])
    .reset_index(drop=True)
)

unclassified_family_candidates_df = (
    candidate_inventory_df
    .loc[
        candidate_inventory_df["feature_family"].eq("other_or_unclassified"),
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "modality",
            "source_metric",
            "manifest_documented",
            "non_null_pct",
            "null_pct",
        ],
    ]
    .sort_values(["modality", "feature_name"])
    .reset_index(drop=True)
)

other_modality_candidates_df = (
    candidate_inventory_df
    .loc[
        candidate_inventory_df["modality"].eq("other"),
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "modality",
            "source_metric",
            "manifest_documented",
            "non_null_pct",
            "null_pct",
        ],
    ]
    .sort_values(["feature_family", "feature_name"])
    .reset_index(drop=True)
)

display(must_have_inventory_field_qa_df)
display(classification_quality_qa_df)

if not missing_source_metric_candidates_df.empty:
    print("Candidate features missing source_metric:")
    display(missing_source_metric_candidates_df)

if not unclassified_family_candidates_df.empty:
    print("Candidate features with other_or_unclassified feature_family:")
    display(unclassified_family_candidates_df)

if not other_modality_candidates_df.empty:
    print("Candidate features with other modality:")
    display(other_modality_candidates_df)

assert set(must_have_inventory_fields).issubset(feature_inventory_df.columns), (
    "Feature inventory is missing one or more must-have decision fields."
)

assert must_have_inventory_field_qa_df["missing_values"].sum() == 0, (
    "One or more must-have feature inventory fields has missing values for candidate features. "
    "Review the diagnostic tables displayed above."
)

print("Must-have feature inventory field QA complete.")

,field_name,candidate_features_checked,missing_values,missing_pct,status
0,feature_name,329,0,0.0,pass
1,is_numeric,329,0,0.0,pass
2,is_modeling_candidate,329,0,0.0,pass
3,feature_family,329,0,0.0,pass
4,modality,329,0,0.0,pass
5,source_metric,329,0,0.0,pass
6,is_core_metric_descendant,329,0,0.0,pass
7,non_null_pct,329,0,0.0,pass
8,null_pct,329,0,0.0,pass


,qa_check,value,status
0,unclassified_feature_family_candidates,0,pass
1,other_modality_candidates,0,pass
2,core_metric_descendant_candidates,195,reference
3,non_core_or_context_candidates,134,reference


Must-have feature inventory field QA complete.


Findings\. The must\-have inventory QA is clean: all 329 candidate modeling features have the fields needed for 1\.5\.6 decisions, including feature family, modality, source metric, core\-metric flag, and null coverage\. That is the main thing to check here\. The core/non\-core split is informational, not a pass/fail test, because 1\.5\.6 still needs to evaluate whether some non\-core or context features are worth keeping\.

Now we run a compact QA pass on the inventory\. This catches true blockers, while leaving classification edge cases for review instead of stopping the notebook\.

In [16]:

# ---------------------------------------------------------------------
# Validate feature inventory
# ---------------------------------------------------------------------

candidate_inventory_df = feature_inventory_df.loc[
    feature_inventory_df["is_modeling_candidate"]
].copy()

non_numeric_candidate_count = (
    candidate_inventory_df["is_numeric"]
    .eq(False)
    .sum()
)

zero_coverage_candidate_count = (
    candidate_inventory_df["non_null_observations"]
    .eq(0)
    .sum()
)

low_coverage_candidate_count = (
    candidate_inventory_df["non_null_pct"]
    .lt(MIN_FEATURE_NON_NULL_PCT)
    .sum()
)

near_zero_variance_candidate_count = (
    candidate_inventory_df["near_zero_variance_flag"]
    .sum()
)

unclassified_candidate_count = (
    candidate_inventory_df["feature_family"]
    .eq("other_or_unclassified")
    .sum()
)

missing_core_metrics = [
    metric
    for metric in CORE_MODELING_METRICS
    if metric not in modeling_df.columns
]

feature_inventory_validation_df = pd.DataFrame(
    [
        {
            "qa_check": "inventoried_columns",
            "value": len(feature_inventory_df),
            "status": "pass" if len(feature_inventory_df) > 0 else "fail",
        },
        {
            "qa_check": "candidate_modeling_features",
            "value": len(candidate_inventory_df),
            "status": "pass" if len(candidate_inventory_df) > 0 else "fail",
        },
        {
            "qa_check": "non_numeric_candidate_features",
            "value": non_numeric_candidate_count,
            "status": "pass" if non_numeric_candidate_count == 0 else "fail",
        },
        {
            "qa_check": "zero_coverage_candidate_features",
            "value": zero_coverage_candidate_count,
            "status": "pass" if zero_coverage_candidate_count == 0 else "fail",
        },
        {
            "qa_check": "low_coverage_candidate_features",
            "value": low_coverage_candidate_count,
            "status": "review",
        },
        {
            "qa_check": "near_zero_variance_candidate_features",
            "value": int(near_zero_variance_candidate_count),
            "status": "review",
        },
        {
            "qa_check": "unclassified_candidate_features",
            "value": unclassified_candidate_count,
            "status": "review",
        },
        {
            "qa_check": "missing_core_metrics",
            "value": len(missing_core_metrics),
            "status": "pass" if len(missing_core_metrics) == 0 else "fail",
        },
    ]
)

display(feature_inventory_validation_df)

assert not feature_inventory_df.empty, "Feature inventory is empty."

assert len(candidate_inventory_df) > 0, (
    "No candidate modeling features were identified."
)

assert non_numeric_candidate_count == 0, (
    "One or more candidate modeling features is non-numeric."
)

assert zero_coverage_candidate_count == 0, (
    "One or more candidate modeling features has zero non-null observations."
)

assert not missing_core_metrics, (
    f"Missing required core modeling metrics: {missing_core_metrics}"
)

print("Feature inventory validation complete.")

,qa_check,value,status
0,inventoried_columns,342,pass
1,candidate_modeling_features,329,pass
2,non_numeric_candidate_features,0,pass
3,zero_coverage_candidate_features,0,pass
4,low_coverage_candidate_features,6,review
5,near_zero_variance_candidate_features,0,review
6,unclassified_candidate_features,0,review
7,missing_core_metrics,0,pass


Feature inventory validation complete.


Findings\. The inventory validation passes the true blockers: the inventory is populated, candidate modeling features are numeric, every candidate has at least some non\-null coverage, and all seven core modeling metrics are present\. The review rows are there to flag modeling decisions, not data failures\. Low\-coverage features are mostly expected Traffic fields, and near\-zero variance will get a more meaningful check once we build the diagnostic sample\.

### Summarize Feature Inventory

Let's summarize where the candidate modeling features came from\. These tables show the feature inventory by source notebook, feature family, and transportation modality\.

In [17]:
# ---------------------------------------------------------------------
# Summarize feature inventory composition
# ---------------------------------------------------------------------

feature_count_by_source_notebook_df = (
    candidate_inventory_df
    .groupby("source_notebook", as_index=False)
    .agg(candidate_features=("feature_name", "count"))
    .sort_values("candidate_features", ascending=False)
    .reset_index(drop=True)
)

feature_count_by_family_df = (
    candidate_inventory_df
    .groupby("feature_family", as_index=False)
    .agg(candidate_features=("feature_name", "count"))
    .sort_values("candidate_features", ascending=False)
    .reset_index(drop=True)
)

feature_count_by_modality_df = (
    candidate_inventory_df
    .groupby("modality", as_index=False)
    .agg(candidate_features=("feature_name", "count"))
    .sort_values("candidate_features", ascending=False)
    .reset_index(drop=True)
)

display(feature_count_by_source_notebook_df)
display(feature_count_by_family_df)
display(feature_count_by_modality_df)

print(
    "Feature inventory composition summary complete. "
    f"Candidate modeling features: {len(candidate_inventory_df):,}."
)

,source_notebook,candidate_features
0,1.5.1,150
1,1.5.2,62
2,1.5.3,45
3,1.5.4,35
4,1.5.5,21
5,pre_1.5_engineering,16


,feature_family,candidate_features
0,temporal_history,154
1,spatial_context,62
2,multimodal_interaction,41
3,stl_decomposition,28
4,anomaly_severity,21
5,raw_metric,16
6,fourier20_residual_reconstruction,7


,modality,candidate_features
0,fhvhv,92
1,taxi,92
2,bus,64
3,subway,39
4,multimodal,20
5,traffic,15
6,spatial,7


Feature inventory composition summary complete. Candidate modeling features: 329.


Findings\. The composition tables show how uneven the feature universe is by design\. Temporal\-history features from 1\.5\.1 make up the largest share because that notebook generated lags, rolling summaries, changes, and CP\-period comparisons across many metrics\. The later notebooks add smaller, more targeted families: spatial context, multimodal interactions, STL/Fourier decomposition, and anomaly\-severity scores\. This matters because a large family can look more important simply because it has more columns, so the next redundancy and contribution checks need to compare families carefully rather than just counting features\.

Next, we separate candidate modeling columns from excluded identifier/non\-numeric columns and check how much of the candidate set descends from the seven core modeling metrics\.

In [18]:
# ---------------------------------------------------------------------
# Summarize modeling-candidate and core-metric status
# ---------------------------------------------------------------------

core_metric_descendant_summary_df = (
    candidate_inventory_df
    .assign(
        core_metric_status=np.where(
            candidate_inventory_df["is_core_metric_descendant"],
            "core_metric_descendant",
            "non_core_or_context",
        )
    )
    .groupby("core_metric_status", as_index=False)
    .agg(candidate_features=("feature_name", "count"))
    .sort_values("candidate_features", ascending=False)
    .reset_index(drop=True)
)

candidate_status_summary_df = (
    feature_inventory_df
    .groupby("is_modeling_candidate", as_index=False)
    .agg(columns=("feature_name", "count"))
    .assign(
        inventory_status=lambda df: np.where(
            df["is_modeling_candidate"],
            "candidate_modeling_feature",
            "excluded_identifier_or_non_numeric",
        )
    )
    [["inventory_status", "columns"]]
)

display(core_metric_descendant_summary_df)
display(candidate_status_summary_df)

print("Modeling-candidate and core-metric status summary complete.")

,core_metric_status,candidate_features
0,core_metric_descendant,195
1,non_core_or_context,134


,inventory_status,columns
0,excluded_identifier_or_non_numeric,13
1,candidate_modeling_feature,329


Modeling-candidate and core-metric status summary complete.


Findings\. The panel has 342 total columns, with 329 numeric modeling candidates and 13 excluded identifier or non\-numeric columns\. Among the modeling candidates, 188 descend from the seven core metrics and 141 are non\-core or context features\. That balance is useful: the core metrics still anchor most of the engineered feature space, but there is enough surrounding context to test whether spatial, traffic, multimodal, and non\-core mobility signals add anything before we prune\.

Finally, we look at the highest\-null candidate features\. This is not a drop decision yet; it is a watchlist for coverage patterns that may affect redundancy checks and downstream modeling\.

In [19]:
# ---------------------------------------------------------------------
# Review highest-null candidate features
# ---------------------------------------------------------------------

highest_null_candidate_features_df = (
    candidate_inventory_df
    .sort_values("null_pct", ascending=False)
    .loc[
        :,
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "modality",
            "source_metric",
            "non_null_pct",
            "null_pct",
        ],
    ]
    .head(15)
    .reset_index(drop=True)
)

display(highest_null_candidate_features_df)

print("Highest-null candidate feature watchlist complete.")

,feature_name,source_notebook,feature_family,modality,source_metric,non_null_pct,null_pct
0,traffic_pct_change_from_previous_observation,1.5.3,temporal_history,traffic,traffic_volume,0.004529,0.995471
1,traffic_change_from_previous_observation,1.5.3,temporal_history,traffic,traffic_volume,0.004589,0.995411
2,traffic_previous_observation,1.5.3,temporal_history,traffic,traffic_volume,0.004589,0.995411
3,traffic_days_since_previous_observation,1.5.3,temporal_history,traffic,traffic_volume,0.004589,0.995411
4,traffic_volume,pre_1.5_engineering,raw_metric,traffic,traffic_volume,0.005399,0.994601
5,connected_mean_traffic_volume,1.5.2,spatial_context,traffic,traffic_volume,0.017169,0.982831
6,cp_zone_mean_traffic_volume,1.5.2,spatial_context,traffic,traffic_volume,0.088027,0.911973
7,borough_mean_traffic_volume,1.5.2,spatial_context,traffic,traffic_volume,0.204024,0.795976
8,max_yearly_ratio,1.5.3,multimodal_interaction,traffic,traffic_volume,0.520913,0.479087
9,subway_ridership_local_vs_connected_zscore,1.5.3,multimodal_interaction,subway,subway_ridership,0.558935,0.441065


Highest-null candidate feature watchlist complete.


Findings\. The highest\-null watchlist is doing what we want: it separates expected coverage gaps from actual inventory problems\. The top rows are mostly Traffic features, with null rates above 98% for the raw and short\-history traffic fields, which matches the earlier decision that Traffic is irregularly sampled and should be handled carefully\. The next group is mostly subway\-related, which is also expected because subway coverage only exists where station\-based geography can support it\. Nothing here means the inventory failed; it means these features need special handling in later selection, scaling, or modality\-specific modeling\.

### Section Summary

This section loaded the completed 1\.5\.5 modeling panel, confirmed that the Taxi Zone × Date × Temporal Bucket grain is intact, standardized the congestion\-pricing period labels, and built a manifest\-backed inventory of all candidate modeling features\. The main output is a clean feature inventory that identifies each candidate feature by family, modality, source metric, source notebook, core\-metric status, null coverage, and modeling eligibility\. With the feature universe documented, the notebook can move from “what exists?” to “what should we keep?”

## 1\.5\.6\.2 Evaluate Feature Redundancy and Information Overlap

Once the feature inventory is complete, we check where the engineered feature set is repeating itself\. Earlier notebooks intentionally created broad feature families, so some overlap is expected: lags may track rolling means, z\-scores may track percentile ranks, residual features may overlap with deviation features, and non\-core metric descendants may add less than their column count suggests\.

This section uses a stratified diagnostic sample, then runs within\-family Spearman correlation checks across eligible numeric features\. Spearman is the right first pass because these mobility features are skewed and scale\-mismatched\. The goal is not to auto\-drop every correlated feature; the goal is to identify strong overlap patterns that can guide pruning later\. This is a global redundancy screen\. Modality\-specific redundancy gets its own section because bus, subway, taxi, FHVHV, and multimodal feature sets may need different recommendations\.

### Build Redundancy Diagnostic Sample

Before running correlation checks, we define the feature universe and build a stratified diagnostic sample\. This keeps the redundancy analysis computationally manageable while preserving the key structure of the full panel: congestion\-pricing period, temporal bucket, borough, and the engineered feature mix\.

First, let’s define the feature universe for redundancy analysis\. We start from the cleaned inventory, keep numeric modeling candidates, and confirm that every selected column is actually present in the panel\. This keeps the redundancy screen tied to the same feature list we just validated\.

In [20]:
# ---------------------------------------------------------------------
# Select redundancy candidate features
# ---------------------------------------------------------------------

candidate_feature_columns = (
    candidate_inventory_df
    .loc[
        candidate_inventory_df["is_modeling_candidate"],
        "feature_name",
    ]
    .drop_duplicates()
    .tolist()
)

candidate_feature_columns = [
    column
    for column in candidate_feature_columns
    if column in modeling_df.columns
]

sample_required_columns = list(
    dict.fromkeys(
        ROW_IDENTIFIER_COLUMNS
        + DIAGNOSTIC_SAMPLE_STRATA_COLUMNS
        + candidate_feature_columns
    )
)

missing_sample_columns = [
    column
    for column in sample_required_columns
    if column not in modeling_df.columns
]

redundancy_candidate_feature_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "candidate_features_from_inventory",
            "value": candidate_inventory_df["is_modeling_candidate"].sum(),
            "status": "reference",
        },
        {
            "qa_check": "candidate_features_available_in_panel",
            "value": len(candidate_feature_columns),
            "status": "pass" if len(candidate_feature_columns) > 0 else "fail",
        },
        {
            "qa_check": "missing_sample_columns",
            "value": len(missing_sample_columns),
            "status": "pass" if len(missing_sample_columns) == 0 else "fail",
        },
    ]
)

display(redundancy_candidate_feature_summary_df)

assert len(candidate_feature_columns) > 0, (
    "No candidate feature columns are available for redundancy analysis."
)

assert not missing_sample_columns, (
    "Missing columns needed for redundancy diagnostic sample: "
    f"{missing_sample_columns}"
)

print(
    "Redundancy candidate features selected. "
    f"Candidate features: {len(candidate_feature_columns):,}."
)

,qa_check,value,status
0,candidate_features_from_inventory,329,reference
1,candidate_features_available_in_panel,329,pass
2,missing_sample_columns,0,pass


Redundancy candidate features selected. Candidate features: 329.


Findings\. The redundancy screen starts from 329 candidate modeling features, and all 329 are available in the panel with no missing sample columns\. That is the main thing to notice here: the analysis is not quietly losing features because of metadata drift, naming mismatches, or missing columns\. The redundancy work is starting from the same feature universe validated in the inventory\.

The redundancy checks are expensive, so we use a stratified diagnostic sample instead of the full panel\. The sample preserves congestion\-pricing period, temporal bucket, and borough coverage, which keeps the correlation screen representative while making it runnable inside Deepnote\.

In [21]:
# ---------------------------------------------------------------------
# Build or load stratified redundancy diagnostic sample
# ---------------------------------------------------------------------

sample_columns = sorted(
    set(GRAIN_COLUMNS)
    | set(DIAGNOSTIC_SAMPLE_STRATA_COLUMNS)
    | set(candidate_inventory_df["feature_name"])
)

sample_columns = [
    column
    for column in sample_columns
    if column in modeling_df.columns
]

if MODELING_SAMPLE_PATH.exists() and not REBUILD_MODELING_SAMPLE:
    modeling_diagnostic_sample_df = pd.read_parquet(MODELING_SAMPLE_PATH)
    print(f"Loaded cached diagnostic sample: {MODELING_SAMPLE_PATH}")

else:
    strata_counts_df = (
        modeling_df
        .groupby(DIAGNOSTIC_SAMPLE_STRATA_COLUMNS, dropna=False)
        .size()
        .reset_index(name="stratum_rows")
    )

    strata_counts_df["sample_rows"] = np.floor(
        strata_counts_df["stratum_rows"]
        / strata_counts_df["stratum_rows"].sum()
        * DIAGNOSTIC_SAMPLE_ROWS
    ).astype(int)

    strata_counts_df["sample_rows"] = strata_counts_df["sample_rows"].clip(lower=1)

    # If rounding pushes the sample above the target, trim the largest strata first.
    sample_row_overage = int(strata_counts_df["sample_rows"].sum() - DIAGNOSTIC_SAMPLE_ROWS)

    if sample_row_overage > 0:
        overage_index = strata_counts_df.sort_values(
            "sample_rows",
            ascending=False,
        ).head(sample_row_overage).index

        strata_counts_df.loc[overage_index, "sample_rows"] -= 1

    sample_parts = []

    for _, stratum in strata_counts_df.iterrows():
        stratum_mask = pd.Series(True, index=modeling_df.index)

        for column in DIAGNOSTIC_SAMPLE_STRATA_COLUMNS:
            if pd.isna(stratum[column]):
                stratum_mask &= modeling_df[column].isna()
            else:
                stratum_mask &= modeling_df[column].eq(stratum[column])

        stratum_df = modeling_df.loc[stratum_mask, sample_columns]

        sample_parts.append(
            stratum_df.sample(
                n=min(int(stratum["sample_rows"]), len(stratum_df)),
                random_state=DIAGNOSTIC_SAMPLE_RANDOM_STATE,
            )
        )

    modeling_diagnostic_sample_df = (
        pd.concat(sample_parts, ignore_index=True)
        .sample(frac=1, random_state=DIAGNOSTIC_SAMPLE_RANDOM_STATE)
        .reset_index(drop=True)
    )

    modeling_diagnostic_sample_df.to_parquet(
        MODELING_SAMPLE_PATH,
        index=False,
    )

    print(f"Saved diagnostic sample: {MODELING_SAMPLE_PATH}")

print(
    "Redundancy diagnostic sample ready. "
    f"Rows: {len(modeling_diagnostic_sample_df):,}. "
    f"Columns: {modeling_diagnostic_sample_df.shape[1]:,}."
)

Loaded cached diagnostic sample: pipeline_data/1.5.6.intermediate_tables/modeling_diagnostic_sample.parquet
Redundancy diagnostic sample ready. Rows: 250,000. Columns: 339.


Now we check whether that sample is broad enough to trust\. The QA is intentionally compact: row count, policy\-period coverage, temporal\-bucket coverage, and borough coverage tell us whether the sample accidentally narrowed the modeling universe\.

In [22]:
# ---------------------------------------------------------------------
# Validate redundancy diagnostic sample coverage
# ---------------------------------------------------------------------

sample_strata_summary_df = (
    modeling_diagnostic_sample_df
    .groupby(DIAGNOSTIC_SAMPLE_STRATA_COLUMNS, dropna=False)
    .size()
    .reset_index(name="sample_rows")
    .sort_values(DIAGNOSTIC_SAMPLE_STRATA_COLUMNS)
    .reset_index(drop=True)
)

redundancy_sample_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "sample_rows",
            "value": len(modeling_diagnostic_sample_df),
            "status": "pass" if len(modeling_diagnostic_sample_df) > 0 else "fail",
        },
        {
            "qa_check": "pre_post_cp_values",
            "value": modeling_diagnostic_sample_df["pre_post_cp"].nunique(dropna=False),
            "status": "pass" if modeling_diagnostic_sample_df["pre_post_cp"].nunique(dropna=False) == 2 else "review",
        },
        {
            "qa_check": "temporal_buckets",
            "value": modeling_diagnostic_sample_df["temporal_bucket"].nunique(dropna=False),
            "status": "pass" if modeling_diagnostic_sample_df["temporal_bucket"].nunique(dropna=False) == modeling_df["temporal_bucket"].nunique(dropna=False) else "review",
        },
        {
            "qa_check": "boroughs",
            "value": modeling_diagnostic_sample_df["borough"].nunique(dropna=False),
            "status": "pass" if modeling_diagnostic_sample_df["borough"].nunique(dropna=False) == modeling_df["borough"].nunique(dropna=False) else "review",
        },
        {
            "qa_check": "unique_dates",
            "value": modeling_df["date"].nunique(),
        },
    ]
)

display(redundancy_sample_summary_df)
display(sample_strata_summary_df)

assert len(modeling_diagnostic_sample_df) > 0, (
    "Redundancy diagnostic sample is empty."
)

assert set(GRAIN_COLUMNS).issubset(modeling_diagnostic_sample_df.columns), (
    "Diagnostic sample is missing one or more grain columns."
)

print("Redundancy diagnostic sample coverage validated.")

,qa_check,value,status
0,sample_rows,250000,pass
1,pre_post_cp_values,2,pass
2,temporal_buckets,10,pass
3,boroughs,7,pass
4,unique_dates,1186,NaN


,pre_post_cp,temporal_bucket,borough,sample_rows
0,post_cp,weekday_am_peak,Bronx,2219
1,post_cp,weekday_am_peak,Brooklyn,3149
2,post_cp,weekday_am_peak,EWR,52
3,post_cp,weekday_am_peak,Manhattan,3458
4,post_cp,weekday_am_peak,Queens,3562
5,post_cp,weekday_am_peak,Staten Island,1032
6,post_cp,weekday_am_peak,Unknown,103
7,post_cp,weekday_evening,Bronx,2219
8,post_cp,weekday_evening,Brooklyn,3149
9,post_cp,weekday_evening,EWR,52


Redundancy diagnostic sample coverage validated.


Findings\. The diagnostic sample is large enough and broad enough for this stage: 250,000 rows, both \`pre\_cp\` and \`post\_cp\`, all 10 temporal buckets, and all 7 borough labels represented\. The coverage table is mainly there to catch accidental narrowing\. Here it shows the sample still spans the same policy, time\-of\-day, and geography structure as the full panel, while staying small enough for repeated Spearman checks\.

Before running correlations, we check whether each candidate feature has enough usable signal in the sample\. A numeric column can still be a poor correlation input if it is mostly null or nearly constant, so this block separates usable features from sparse or low\-signal columns\.

In [23]:
# ---------------------------------------------------------------------
# Measure redundancy feature readiness
# ---------------------------------------------------------------------

redundancy_feature_readiness_records = []

for _, feature_row in candidate_inventory_df.iterrows():
    feature_name = feature_row["feature_name"]

    if feature_name not in modeling_diagnostic_sample_df.columns:
        continue

    feature_series = modeling_diagnostic_sample_df[feature_name]
    non_null_observations = int(feature_series.notna().sum())
    null_observations = int(feature_series.isna().sum())
    non_null_pct = non_null_observations / len(modeling_diagnostic_sample_df)
    null_pct = null_observations / len(modeling_diagnostic_sample_df)

    if non_null_observations > 0:
        sample_std = feature_series.dropna().std()
    else:
        sample_std = np.nan

    redundancy_feature_readiness_records.append(
        {
            "feature_name": feature_name,
            "source_notebook": feature_row["source_notebook"],
            "feature_family": feature_row["feature_family"],
            "feature_type": feature_row["feature_type"],
            "modality": feature_row["modality"],
            "source_metric": feature_row["source_metric"],
            "is_core_metric_descendant": feature_row["is_core_metric_descendant"],
            "sample_non_null_observations": non_null_observations,
            "sample_null_observations": null_observations,
            "sample_non_null_pct": non_null_pct,
            "sample_null_pct": null_pct,
            "sample_std": sample_std,
            "sample_near_zero_variance_flag": (
                pd.notna(sample_std)
                and sample_std <= NEAR_ZERO_VARIANCE_EPSILON
            ),
        }
    )

redundancy_feature_readiness_df = pd.DataFrame(redundancy_feature_readiness_records)

redundancy_feature_readiness_df["redundancy_eligible"] = (
    redundancy_feature_readiness_df["sample_non_null_pct"].ge(MIN_FEATURE_NON_NULL_PCT)
    & ~redundancy_feature_readiness_df["sample_near_zero_variance_flag"]
)

redundancy_readiness_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "candidate_features_checked",
            "value": len(redundancy_feature_readiness_df),
            "status": "reference",
        },
        {
            "qa_check": "redundancy_eligible_features",
            "value": int(redundancy_feature_readiness_df["redundancy_eligible"].sum()),
            "status": "pass" if redundancy_feature_readiness_df["redundancy_eligible"].sum() > 0 else "fail",
        },
        {
            "qa_check": "low_sample_coverage_features",
            "value": int(redundancy_feature_readiness_df["sample_non_null_pct"].lt(MIN_FEATURE_NON_NULL_PCT).sum()),
            "status": "reference",
        },
        {
            "qa_check": "near_zero_variance_features",
            "value": int(redundancy_feature_readiness_df["sample_near_zero_variance_flag"].sum()),
            "status": "reference",
        },
    ]
)

display(redundancy_readiness_summary_df)
display(
    redundancy_feature_readiness_df
    .sort_values(["redundancy_eligible", "sample_non_null_pct"], ascending=[True, True])
    .head(25)
)

assert redundancy_feature_readiness_df["redundancy_eligible"].sum() > 0, (
    "No features are eligible for redundancy analysis after sample-level filtering."
)

print("Redundancy feature readiness measured.")

,qa_check,value,status
0,candidate_features_checked,329,reference
1,redundancy_eligible_features,323,pass
2,low_sample_coverage_features,6,reference
3,near_zero_variance_features,0,reference


,feature_name,source_notebook,feature_family,feature_type,modality,source_metric,is_core_metric_descendant,sample_non_null_observations,sample_null_observations,sample_non_null_pct,sample_null_pct,sample_std,sample_near_zero_variance_flag,redundancy_eligible
227,traffic_pct_change_from_previous_observation,1.5.3,temporal_history,traffic_sparse_observation_history,traffic,traffic_volume,False,1149,248851,0.004596,0.995404,4.607839,False,False
225,traffic_previous_observation,1.5.3,temporal_history,traffic_sparse_observation_history,traffic,traffic_volume,False,1158,248842,0.004632,0.995368,4121.586284,False,False
226,traffic_change_from_previous_observation,1.5.3,temporal_history,traffic_sparse_observation_history,traffic,traffic_volume,False,1158,248842,0.004632,0.995368,2938.307772,False,False
228,traffic_days_since_previous_observation,1.5.3,temporal_history,traffic_sparse_observation_history,traffic,traffic_volume,False,1158,248842,0.004632,0.995368,126.261374,False,False
0,traffic_volume,pre_1.5_engineering,raw_metric,raw_metric,traffic,traffic_volume,False,1393,248607,0.005572,0.994428,4737.370384,False,False
172,connected_mean_traffic_volume,1.5.2,spatial_context,connected_zone_spillover,traffic,traffic_volume,False,4376,245624,0.017504,0.982496,5340.559672,False,False
205,cp_zone_mean_traffic_volume,1.5.2,spatial_context,cp_zone_context,traffic,traffic_volume,False,21995,228005,0.087980,0.912020,3150.634009,False,True
189,borough_mean_traffic_volume,1.5.2,spatial_context,borough_context,traffic,traffic_volume,False,51273,198727,0.205092,0.794908,4432.069927,False,True
229,max_yearly_ratio,1.5.3,multimodal_interaction,traffic_temporal_coverage_ratio,traffic,traffic_volume,False,130316,119684,0.521264,0.478736,3.589602,False,True
257,subway_ridership_local_vs_connected_zscore,1.5.3,multimodal_interaction,local_vs_connected_zone_divergence,subway,subway_ridership,True,139593,110407,0.558372,0.441628,1.104102,False,True


Redundancy feature readiness measured.


Findings\. Almost the full candidate set is usable for redundancy analysis: 323 of 329 candidate features pass the sample\-readiness screen, with 6 low\-coverage features and no near\-zero\-variance features\. The low\-coverage rows are mostly traffic\-style sparse observation features, which is expected because traffic is irregularly observed in this project\. The good news is that the main engineered feature families have enough signal for the correlation checks, so we are not basing redundancy decisions on mostly empty columns\.

Now we keep only the features that are suitable for Spearman checks and summarize them by feature family\. This step matters because correlation results are only meaningful when each family has enough non\-null, non\-flat columns to compare\.

In [24]:
# ---------------------------------------------------------------------
# Select redundancy-eligible feature groups
# ---------------------------------------------------------------------

eligible_redundancy_features_df = (
    redundancy_feature_readiness_df[
        redundancy_feature_readiness_df["redundancy_eligible"]
    ]
    .copy()
    .sort_values(["feature_family", "modality", "source_metric", "feature_name"])
    .reset_index(drop=True)
)

eligible_redundancy_feature_groups_df = (
    eligible_redundancy_features_df
    .groupby("feature_family", as_index=False)
    .agg(
        eligible_features=("feature_name", "nunique"),
        modalities=("modality", "nunique"),
        source_metrics=("source_metric", "nunique"),
        min_sample_non_null_pct=("sample_non_null_pct", "min"),
        median_sample_non_null_pct=("sample_non_null_pct", "median"),
    )
    .sort_values("eligible_features", ascending=False)
    .reset_index(drop=True)
)

eligible_redundancy_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "eligible_redundancy_features",
            "value": len(eligible_redundancy_features_df),
            "status": "pass" if len(eligible_redundancy_features_df) > 0 else "fail",
        },
        {
            "qa_check": "eligible_feature_families",
            "value": eligible_redundancy_features_df["feature_family"].nunique(),
            "status": "reference",
        },
        {
            "qa_check": "largest_family_feature_count",
            "value": int(eligible_redundancy_feature_groups_df["eligible_features"].max()),
            "status": "reference",
        },
    ]
)

display(eligible_redundancy_summary_df)
display(eligible_redundancy_feature_groups_df)

assert len(eligible_redundancy_features_df) > 0, (
    "No eligible features were available for within-family redundancy checks."
)

print(
    "Redundancy-eligible feature groups ready. "
    f"Eligible features: {len(eligible_redundancy_features_df):,}."
)

,qa_check,value,status
0,eligible_redundancy_features,323,pass
1,eligible_feature_families,7,reference
2,largest_family_feature_count,150,reference


,feature_family,eligible_features,modalities,source_metrics,min_sample_non_null_pct,median_sample_non_null_pct
0,temporal_history,150,4,15,0.559600,0.974888
1,spatial_context,61,6,18,0.087980,0.988796
2,multimodal_interaction,41,6,29,0.521264,0.946544
3,stl_decomposition,28,4,7,0.592488,0.946068
4,anomaly_severity,21,4,7,0.588208,0.946540
5,raw_metric,15,4,15,0.592488,0.946544
6,fourier20_residual_reconstruction,7,4,7,0.592488,0.946068


Redundancy-eligible feature groups ready. Eligible features: 323.


Findings\. After the readiness filter, 323 features remain eligible across 7 feature families\. The distribution matters: temporal\-history is by far the largest family with 150 eligible features, followed by spatial context with 61 and multimodal interaction with 41\. Fourier Top 20 has only 7 eligible features, one per core metric\. That imbalance is exactly why we need careful redundancy diagnostics: large families can dominate the evidence simply because they contain many more ways to describe related movement patterns\.

### Compare Redundancy Stability by CP Period

Before relying on one full\-period redundancy screen, we check whether high\-correlation patterns behave differently before and after congestion pricing\. If the pre\-CP and post\-CP screens tell very different stories, feature pruning may need to respect policy period\. If they are broadly consistent, a full\-period redundancy screen is easier to justify\.

This helper runs the same Spearman\-correlation logic for each feature family\. It compares features only within the same family because the first redundancy question is narrower than final feature selection: are we generating multiple versions of the same signal inside a family?

In [25]:
# ---------------------------------------------------------------------
# Define Spearman redundancy pair helper
# ---------------------------------------------------------------------


def build_within_family_spearman_pairs(
    sample_df,
    eligible_features_df,
    *,
    group_label_columns=None,
    min_periods=50,
):
    """Return high-correlation within-family feature pairs for a modeling sample."""
    group_label_columns = group_label_columns or {}
    pair_records = []

    for feature_family, family_features_df in eligible_features_df.groupby("feature_family"):
        family_feature_columns = [
            feature_name
            for feature_name in family_features_df["feature_name"]
            if feature_name in sample_df.columns
        ]

        if len(family_feature_columns) < 2:
            continue

        family_corr_df = sample_df[family_feature_columns].corr(
            method="spearman",
            min_periods=min_periods,
        )

        upper_triangle_mask = np.triu(
            np.ones(family_corr_df.shape, dtype=bool),
            k=1,
        )

        family_corr_long_df = (
            family_corr_df
            .where(upper_triangle_mask)
            .stack()
            .reset_index()
            .rename(
                columns={
                    "level_0": "feature_a",
                    "level_1": "feature_b",
                    0: "spearman_corr",
                }
            )
        )

        family_corr_long_df["abs_spearman_corr"] = (
            family_corr_long_df["spearman_corr"].abs()
        )

        high_corr_df = family_corr_long_df[
            family_corr_long_df["abs_spearman_corr"] >= SPEARMAN_CORRELATION_THRESHOLD
        ].copy()

        if high_corr_df.empty:
            continue

        high_corr_df["feature_family"] = feature_family
        high_corr_df["correlation_band"] = np.where(
            high_corr_df["abs_spearman_corr"] >= STRICT_SPEARMAN_CORRELATION_THRESHOLD,
            "strict_high_correlation",
            "high_correlation",
        )

        for label_column, label_value in group_label_columns.items():
            high_corr_df[label_column] = label_value

        pair_records.append(high_corr_df)

    if pair_records:
        pair_detail_df = pd.concat(pair_records, ignore_index=True)
        pair_detail_df["feature_pair_key"] = (
            pair_detail_df[["feature_a", "feature_b"]]
            .apply(lambda row: "__PAIR__".join(sorted(row.astype(str))), axis=1)
        )
        return pair_detail_df

    return pd.DataFrame(
        columns=[
            "feature_a",
            "feature_b",
            "spearman_corr",
            "abs_spearman_corr",
            "feature_family",
            "correlation_band",
            "feature_pair_key",
        ]
        + list(group_label_columns.keys())
    )

print("Spearman redundancy helper ready.")

Spearman redundancy helper ready.


Before using one full\-period redundancy screen, we test whether pre\-CP and post\-CP rows tell different stories\. If the same feature pairs are redundant in both periods, a single full\-period screen is defensible\. If the patterns split sharply, pruning would need to be period\-aware\.

In [26]:
# ---------------------------------------------------------------------
# Run CP-period within-family Spearman checks
# ---------------------------------------------------------------------

CP_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "cp_period_redundancy_pair_detail.csv"
)
CP_PERIOD_REDUNDANCY_RUNTIME_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "cp_period_redundancy_runtime_detail.csv"
)


def ensure_feature_pair_key(pair_df):
    """Guarantee a stable feature-pair key for cached or freshly generated redundancy outputs."""
    if pair_df.empty:
        return pair_df

    pair_df = pair_df.copy()

    if "feature_pair_key" not in pair_df.columns:
        pair_df["feature_pair_key"] = pair_df.apply(
            lambda row: "__".join(sorted([str(row["feature_a"]), str(row["feature_b"])])),
            axis=1,
        )

    return pair_df


if (
    CP_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH.exists()
    and CP_PERIOD_REDUNDANCY_RUNTIME_PATH.exists()
    and not REBUILD_REDUNDANCY_ANALYSIS
):
    cp_period_redundancy_pair_detail_df = pd.read_csv(
        CP_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH
    )
    cp_period_redundancy_runtime_detail_df = pd.read_csv(
        CP_PERIOD_REDUNDANCY_RUNTIME_PATH
    )

    cp_period_redundancy_pair_detail_df = ensure_feature_pair_key(
        cp_period_redundancy_pair_detail_df
    )

    print(
        "Loaded cached CP-period redundancy outputs: "
        f"{CP_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH}"
    )

else:
    cp_period_redundancy_start_time = time.perf_counter()
    cp_period_pair_parts = []
    cp_period_runtime_records = []

    for cp_period in sorted(modeling_diagnostic_sample_df["pre_post_cp"].dropna().unique()):
        period_start_time = time.perf_counter()

        cp_period_sample_df = modeling_diagnostic_sample_df[
            modeling_diagnostic_sample_df["pre_post_cp"] == cp_period
        ].copy()

        cp_period_pairs_df = build_within_family_spearman_pairs(
            sample_df=cp_period_sample_df,
            eligible_features_df=eligible_redundancy_features_df,
            group_label_columns={"pre_post_cp": cp_period},
            min_periods=50,
        )

        cp_period_pairs_df = ensure_feature_pair_key(cp_period_pairs_df)
        cp_period_pair_parts.append(cp_period_pairs_df)

        cp_period_runtime_records.append(
            {
                "pre_post_cp": cp_period,
                "sample_rows": len(cp_period_sample_df),
                "high_correlation_pairs": len(cp_period_pairs_df),
                "runtime_seconds": round(time.perf_counter() - period_start_time, 4),
            }
        )

        del cp_period_sample_df, cp_period_pairs_df
        gc.collect()

    if cp_period_pair_parts:
        cp_period_redundancy_pair_detail_df = pd.concat(
            cp_period_pair_parts,
            ignore_index=True,
        )
    else:
        cp_period_redundancy_pair_detail_df = pd.DataFrame()

    cp_period_redundancy_pair_detail_df = ensure_feature_pair_key(
        cp_period_redundancy_pair_detail_df
    )

    cp_period_redundancy_runtime_detail_df = pd.DataFrame(
        cp_period_runtime_records
    )

    cp_period_redundancy_pair_detail_df.to_csv(
        CP_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH,
        index=False,
    )
    cp_period_redundancy_runtime_detail_df.to_csv(
        CP_PERIOD_REDUNDANCY_RUNTIME_PATH,
        index=False,
    )

    cp_period_redundancy_runtime_seconds = round(
        time.perf_counter() - cp_period_redundancy_start_time,
        4,
    )

    print(
        "Saved CP-period redundancy outputs: "
        f"{CP_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH}"
    )

if "cp_period_redundancy_runtime_seconds" not in globals():
    cp_period_redundancy_runtime_seconds = round(
        cp_period_redundancy_runtime_detail_df["runtime_seconds"].sum(),
        4,
    )

expected_cp_periods = {"pre_cp", "post_cp"}
observed_cp_periods = set(modeling_diagnostic_sample_df["pre_post_cp"].dropna())

assert expected_cp_periods.issubset(observed_cp_periods), (
    "Diagnostic sample must include both pre_cp and post_cp rows before CP-period redundancy checks."
)

assert "feature_pair_key" in cp_period_redundancy_pair_detail_df.columns or cp_period_redundancy_pair_detail_df.empty, (
    "CP-period redundancy pair detail is missing feature_pair_key."
)

display(cp_period_redundancy_runtime_detail_df)

print(
    "CP-period redundancy check complete. "
    f"Rows: {len(cp_period_redundancy_pair_detail_df):,}. "
    f"Runtime: {cp_period_redundancy_runtime_seconds} sec."
)

Loaded cached CP-period redundancy outputs: pipeline_data/1.5.6.intermediate_tables/cp_period_redundancy_pair_detail.csv


,pre_post_cp,sample_rows,high_correlation_pairs,runtime_seconds
0,post_cp,95065,248,148.4876
1,pre_cp,154935,244,267.9898


CP-period redundancy check complete. Rows: 492. Runtime: 416.4774 sec.


This block summarizes the period comparison at the pair level\. The key fields are pairs flagged in both periods, pairs flagged only pre\-CP, and pairs flagged only post\-CP\. That tells us whether redundancy is structural or mostly tied to one congestion\-pricing regime\.

In [27]:
# ---------------------------------------------------------------------
# Summarize CP-period redundancy stability
# ---------------------------------------------------------------------

if cp_period_redundancy_pair_detail_df.empty:
    cp_period_redundancy_stability_summary_df = pd.DataFrame(
        [
            {"comparison_point": "High-correlation pairs in pre-CP", "value": 0},
            {"comparison_point": "High-correlation pairs in post-CP", "value": 0},
            {"comparison_point": "Pairs flagged in both periods", "value": 0},
            {"comparison_point": "Pairs flagged only pre-CP", "value": 0},
            {"comparison_point": "Pairs flagged only post-CP", "value": 0},
        ]
    )

    cp_period_redundancy_by_family_df = pd.DataFrame(
        columns=[
            "feature_family",
            "pre_cp_pairs",
            "post_cp_pairs",
            "pairs_flagged_in_both_periods",
            "pairs_flagged_only_pre_cp",
            "pairs_flagged_only_post_cp",
        ]
    )

else:
    cp_period_pair_sets = {
        cp_period: set(group_df["feature_pair_key"])
        for cp_period, group_df in cp_period_redundancy_pair_detail_df.groupby("pre_post_cp")
    }

    pre_cp_pairs = cp_period_pair_sets.get("pre_cp", set())
    post_cp_pairs = cp_period_pair_sets.get("post_cp", set())

    both_period_pairs = pre_cp_pairs & post_cp_pairs
    only_pre_cp_pairs = pre_cp_pairs - post_cp_pairs
    only_post_cp_pairs = post_cp_pairs - pre_cp_pairs

    cp_period_redundancy_stability_summary_df = pd.DataFrame(
        [
            {"comparison_point": "High-correlation pairs in pre-CP", "value": f"{len(pre_cp_pairs):,}"},
            {"comparison_point": "High-correlation pairs in post-CP", "value": f"{len(post_cp_pairs):,}"},
            {"comparison_point": "Pairs flagged in both periods", "value": f"{len(both_period_pairs):,}"},
            {"comparison_point": "Pairs flagged only pre-CP", "value": f"{len(only_pre_cp_pairs):,}"},
            {"comparison_point": "Pairs flagged only post-CP", "value": f"{len(only_post_cp_pairs):,}"},
        ]
    )

    family_records = []

    for feature_family in sorted(cp_period_redundancy_pair_detail_df["feature_family"].unique()):
        family_df = cp_period_redundancy_pair_detail_df[
            cp_period_redundancy_pair_detail_df["feature_family"] == feature_family
        ]

        family_pair_sets = {
            cp_period: set(group_df["feature_pair_key"])
            for cp_period, group_df in family_df.groupby("pre_post_cp")
        }

        family_pre_pairs = family_pair_sets.get("pre_cp", set())
        family_post_pairs = family_pair_sets.get("post_cp", set())

        family_records.append(
            {
                "feature_family": feature_family,
                "pre_cp_pairs": len(family_pre_pairs),
                "post_cp_pairs": len(family_post_pairs),
                "pairs_flagged_in_both_periods": len(family_pre_pairs & family_post_pairs),
                "pairs_flagged_only_pre_cp": len(family_pre_pairs - family_post_pairs),
                "pairs_flagged_only_post_cp": len(family_post_pairs - family_pre_pairs),
            }
        )

    cp_period_redundancy_by_family_df = (
        pd.DataFrame(family_records)
        .sort_values(
            ["pairs_flagged_in_both_periods", "pre_cp_pairs", "post_cp_pairs"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

display(cp_period_redundancy_stability_summary_df)
display(cp_period_redundancy_by_family_df)

print("CP-period redundancy stability summary complete.")

,comparison_point,value
0,High-correlation pairs in pre-CP,244
1,High-correlation pairs in post-CP,248
2,Pairs flagged in both periods,238
3,Pairs flagged only pre-CP,6
4,Pairs flagged only post-CP,10


,feature_family,pre_cp_pairs,post_cp_pairs,pairs_flagged_in_both_periods,pairs_flagged_only_pre_cp,pairs_flagged_only_post_cp
0,temporal_history,200,203,195,5,8
1,spatial_context,22,23,21,1,2
2,multimodal_interaction,9,9,9,0,0
3,anomaly_severity,7,7,7,0,0
4,stl_decomposition,4,4,4,0,0
5,raw_metric,2,2,2,0,0


CP-period redundancy stability summary complete.


Findings\. The pre\-CP and post\-CP redundancy screens are very close: 244 high\-correlation pairs pre\-CP, 248 post\-CP, and 238 pairs flagged in both periods\. Only 6 pairs appear only pre\-CP and 10 only post\-CP\. That is the key result because it means the redundancy pattern is mostly structural, not a policy\-period artifact\. We can use the full\-period redundancy screen as the main global diagnostic without ignoring a major pre/post split\.

### Run Within\-Family Correlation Checks

Now we run the main within\-family redundancy screen across the full diagnostic sample\. Each feature is compared only against features from the same family, which keeps this first pass focused on obvious internal overlap rather than cross\-family relationships that may still be useful together\.

In [28]:
# ---------------------------------------------------------------------
# Run full-period within-family Spearman checks
# ---------------------------------------------------------------------

FULL_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "full_period_redundancy_pair_detail.csv"
)
FULL_PERIOD_REDUNDANCY_RUNTIME_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "full_period_redundancy_runtime_detail.csv"
)

if (
    FULL_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH.exists()
    and FULL_PERIOD_REDUNDANCY_RUNTIME_PATH.exists()
    and not REBUILD_REDUNDANCY_ANALYSIS
):
    redundancy_pair_detail_df = pd.read_csv(
        FULL_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH
    )
    redundancy_family_runtime_df = pd.read_csv(
        FULL_PERIOD_REDUNDANCY_RUNTIME_PATH
    )

    redundancy_pair_detail_df = ensure_feature_pair_key(
        redundancy_pair_detail_df
    )

    full_period_redundancy_runtime_seconds = round(
        redundancy_family_runtime_df["runtime_seconds"].sum(),
        4,
    )

    print(
        "Loaded cached full-period redundancy outputs: "
        f"{FULL_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH}"
    )

else:
    full_period_redundancy_start_time = time.perf_counter()
    full_period_pair_parts = []
    full_period_runtime_records = []

    for feature_family, family_features_df in eligible_redundancy_features_df.groupby("feature_family"):
        family_start_time = time.perf_counter()

        family_pairs_df = build_within_family_spearman_pairs(
            sample_df=modeling_diagnostic_sample_df,
            eligible_features_df=family_features_df,
            group_label_columns={},
            min_periods=50,
        )

        family_pairs_df = ensure_feature_pair_key(family_pairs_df)
        full_period_pair_parts.append(family_pairs_df)

        full_period_runtime_records.append(
            {
                "feature_family": feature_family,
                "eligible_features": family_features_df["feature_name"].nunique(),
                "high_correlation_pairs": len(family_pairs_df),
                "runtime_seconds": round(time.perf_counter() - family_start_time, 4),
            }
        )

        del family_pairs_df
        gc.collect()

    if full_period_pair_parts:
        redundancy_pair_detail_df = pd.concat(
            full_period_pair_parts,
            ignore_index=True,
        )
    else:
        redundancy_pair_detail_df = pd.DataFrame()

    redundancy_pair_detail_df = ensure_feature_pair_key(
        redundancy_pair_detail_df
    )

    redundancy_family_runtime_df = pd.DataFrame(full_period_runtime_records)

    redundancy_pair_detail_df.to_csv(
        FULL_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH,
        index=False,
    )
    redundancy_family_runtime_df.to_csv(
        FULL_PERIOD_REDUNDANCY_RUNTIME_PATH,
        index=False,
    )

    full_period_redundancy_runtime_seconds = round(
        time.perf_counter() - full_period_redundancy_start_time,
        4,
    )

    print(
        "Saved full-period redundancy outputs: "
        f"{FULL_PERIOD_REDUNDANCY_PAIR_DETAIL_PATH}"
    )

assert len(eligible_redundancy_features_df) > 0, (
    "No eligible features were available for within-family redundancy checks."
)

assert "feature_pair_key" in redundancy_pair_detail_df.columns or redundancy_pair_detail_df.empty, (
    "Full-period redundancy pair detail is missing feature_pair_key."
)

print(
    "Full-period redundancy check complete. "
    f"Rows: {len(redundancy_pair_detail_df):,}. "
    f"Runtime: {full_period_redundancy_runtime_seconds} sec."
)

Loaded cached full-period redundancy outputs: pipeline_data/1.5.6.intermediate_tables/full_period_redundancy_pair_detail.csv
Full-period redundancy check complete. Rows: 236. Runtime: 470.4557 sec.


The full\-period check creates pair\-level evidence, so this block compresses it into something readable\. It shows how many high\-correlation pairs were found, which families carry most of the overlap, how long each family took to process, and a preview of the strongest pairs\.

In [29]:
# ---------------------------------------------------------------------
# Summarize full-period within-family Spearman results
# ---------------------------------------------------------------------

redundancy_pair_detail_df = ensure_feature_pair_key(redundancy_pair_detail_df)

if redundancy_pair_detail_df.empty:
    redundancy_correlation_summary_df = pd.DataFrame(
        [
            {"qa_check": "eligible_features", "value": len(eligible_redundancy_features_df), "status": "reference"},
            {"qa_check": "high_correlation_pairs", "value": 0, "status": "reference"},
            {"qa_check": "strict_high_correlation_pairs", "value": 0, "status": "reference"},
            {"qa_check": "runtime_seconds", "value": full_period_redundancy_runtime_seconds, "status": "reference"},
        ]
    )

else:
    redundancy_correlation_summary_df = pd.DataFrame(
        [
            {
                "qa_check": "eligible_features",
                "value": len(eligible_redundancy_features_df),
                "status": "reference",
            },
            {
                "qa_check": "high_correlation_pairs",
                "value": redundancy_pair_detail_df["feature_pair_key"].nunique(),
                "status": "reference",
            },
            {
                "qa_check": "strict_high_correlation_pairs",
                "value": int(
                    redundancy_pair_detail_df["correlation_band"].eq(
                        "strict_high_correlation"
                    ).sum()
                ),
                "status": "reference",
            },
            {
                "qa_check": "runtime_seconds",
                "value": full_period_redundancy_runtime_seconds,
                "status": "reference",
            },
        ]
    )

eligible_features_by_family_df = (
    eligible_redundancy_features_df
    .groupby("feature_family", as_index=False)
    .agg(eligible_features=("feature_name", "nunique"))
)

if redundancy_pair_detail_df.empty:
    redundancy_family_summary_df = eligible_features_by_family_df.assign(
        high_correlation_pairs=0,
        strict_high_correlation_pairs=0,
        max_abs_spearman_corr=np.nan,
    )
else:
    redundancy_family_summary_df = (
        redundancy_pair_detail_df
        .groupby("feature_family", as_index=False)
        .agg(
            high_correlation_pairs=("feature_pair_key", "nunique"),
            strict_high_correlation_pairs=(
                "correlation_band",
                lambda values: int((values == "strict_high_correlation").sum()),
            ),
            max_abs_spearman_corr=("abs_spearman_corr", "max"),
        )
        .merge(
            eligible_features_by_family_df,
            on="feature_family",
            how="right",
        )
        .fillna(
            {
                "high_correlation_pairs": 0,
                "strict_high_correlation_pairs": 0,
            }
        )
    )

redundancy_family_summary_df = (
    redundancy_family_summary_df
    [
        [
            "feature_family",
            "eligible_features",
            "high_correlation_pairs",
            "strict_high_correlation_pairs",
            "max_abs_spearman_corr",
        ]
    ]
    .sort_values("high_correlation_pairs", ascending=False)
    .reset_index(drop=True)
)

display(redundancy_correlation_summary_df)
display(redundancy_family_runtime_df)
display(redundancy_family_summary_df)
display(redundancy_pair_detail_df.head(25))

print("Full-period redundancy summary complete.")

,qa_check,value,status
0,eligible_features,323.0000,reference
1,high_correlation_pairs,236.0000,reference
2,strict_high_correlation_pairs,143.0000,reference
3,runtime_seconds,470.4557,reference


,feature_family,eligible_features,high_correlation_pairs,runtime_seconds
0,anomaly_severity,21,7,8.5329
1,fourier20_residual_reconstruction,7,0,1.0206
2,multimodal_interaction,41,9,30.5179
3,raw_metric,15,2,4.4305
4,spatial_context,61,22,76.1321
5,stl_decomposition,28,4,15.5956
6,temporal_history,150,192,334.2261


,feature_family,eligible_features,high_correlation_pairs,strict_high_correlation_pairs,max_abs_spearman_corr
0,temporal_history,150,192.0,113.0,0.999895
1,spatial_context,61,22.0,12.0,1.000000
2,multimodal_interaction,41,9.0,9.0,0.999211
3,anomaly_severity,21,7.0,7.0,0.993478
4,stl_decomposition,28,4.0,1.0,0.962248
5,raw_metric,15,2.0,1.0,0.993013
6,fourier20_residual_reconstruction,7,0.0,0.0,NaN


,feature_a,feature_b,spearman_corr,abs_spearman_corr,feature_family,correlation_band,feature_pair_key
0,avg_bus_speed_deviation_percentile_rank,avg_bus_speed_deviation_zscore,0.989163,0.989163,anomaly_severity,strict_high_correlation,avg_bus_speed_deviation_percentile_rank__PAIR_...
1,bus_trip_count_deviation_percentile_rank,bus_trip_count_deviation_zscore,0.981870,0.981870,anomaly_severity,strict_high_correlation,bus_trip_count_deviation_percentile_rank__PAIR...
2,fhvhv_avg_trip_speed_deviation_percentile_rank,fhvhv_avg_trip_speed_deviation_zscore,0.991243,0.991243,anomaly_severity,strict_high_correlation,fhvhv_avg_trip_speed_deviation_percentile_rank...
3,fhvhv_trip_count_deviation_percentile_rank,fhvhv_trip_count_deviation_zscore,0.982006,0.982006,anomaly_severity,strict_high_correlation,fhvhv_trip_count_deviation_percentile_rank__PA...
4,subway_ridership_deviation_percentile_rank,subway_ridership_deviation_zscore,0.973258,0.973258,anomaly_severity,strict_high_correlation,subway_ridership_deviation_percentile_rank__PA...
5,taxi_avg_trip_speed_deviation_percentile_rank,taxi_avg_trip_speed_deviation_zscore,0.993478,0.993478,anomaly_severity,strict_high_correlation,taxi_avg_trip_speed_deviation_percentile_rank_...
6,taxi_trip_count_deviation_percentile_rank,taxi_trip_count_deviation_zscore,0.986518,0.986518,anomaly_severity,strict_high_correlation,taxi_trip_count_deviation_percentile_rank__PAI...
7,avg_bus_speed_zscore,mta_crossing_share_x_avg_bus_speed_zscore,0.999057,0.999057,multimodal_interaction,strict_high_correlation,avg_bus_speed_zscore__PAIR__mta_crossing_share...
8,bus_trip_count_zscore,mta_crossing_share_x_bus_trip_count_zscore,0.993453,0.993453,multimodal_interaction,strict_high_correlation,bus_trip_count_zscore__PAIR__mta_crossing_shar...
9,fhvhv_avg_trip_speed_zscore,mta_crossing_share_x_fhvhv_avg_trip_speed_zscore,0.999089,0.999089,multimodal_interaction,strict_high_correlation,fhvhv_avg_trip_speed_zscore__PAIR__mta_crossin...


Full-period redundancy summary complete.


Findings\. This table is easiest to read from left to right: \`eligible\_features\` creates the possible within\-family pair count, calculated as \`n choose 2\`; then \`high\_correlation\_pairs\` and \`strict\_high\_correlation\_pairs\` show how many of those possible pairs cross the redundancy thresholds\. The percentages are low, mostly around 1\-3%, which means high\-correlation redundancy is not dense across entire feature families\. Temporal\-history has the largest flagged count because it also has the largest denominator: 150 eligible features create 11,175 possible pairs\. The flagged\-correlation columns measure the strength of pairs that were already flagged, not the average correlation across every possible pair\. So the takeaway is nuanced: redundancy exists in identifiable pockets, especially temporal\-history, but this screen is producing a targeted review list rather than proving that the whole engineered feature space is saturated with duplicate signals\.

### Summarize and Flag Candidate Redundancy Decisions

This subsection turns pair\-level correlations into feature\-level review signals\. The goal is not to drop features automatically\. The goal is to identify which features belong in representative\-feature review because they repeatedly overlap with other features in the same family\.

Let's roll the pair\-level results up to the feature\-family level\. This answers the first practical question: is redundancy concentrated in one or two large families, or is it spread across the engineered feature set?

In [30]:
# ---------------------------------------------------------------------
# Build high-correlation redundancy family summary
# ---------------------------------------------------------------------

redundancy_family_denominator_df = (
    eligible_redundancy_features_df
    .groupby("feature_family", as_index=False)
    .agg(eligible_features=("feature_name", "nunique"))
)

# Each family is compared internally, so the possible pair denominator is n choose 2.
redundancy_family_denominator_df["possible_feature_pairs"] = (
    redundancy_family_denominator_df["eligible_features"]
    * (redundancy_family_denominator_df["eligible_features"] - 1)
    / 2
).astype(int)

if redundancy_pair_detail_df.empty:
    redundancy_by_family_df = redundancy_family_denominator_df.copy()
    redundancy_by_family_df["features_with_high_correlation"] = 0
    redundancy_by_family_df["high_correlation_pairs"] = 0
    redundancy_by_family_df["strict_high_correlation_pairs"] = 0
    redundancy_by_family_df["high_correlation_pair_pct"] = 0.0
    redundancy_by_family_df["strict_high_correlation_pair_pct"] = 0.0
    redundancy_by_family_df["max_flagged_abs_spearman_corr"] = np.nan
    redundancy_by_family_df["mean_flagged_abs_spearman_corr"] = np.nan

else:
    features_with_pairs_df = pd.concat(
        [
            redundancy_pair_detail_df[
                ["feature_family", "feature_a"]
            ].rename(columns={"feature_a": "feature_name"}),
            redundancy_pair_detail_df[
                ["feature_family", "feature_b"]
            ].rename(columns={"feature_b": "feature_name"}),
        ],
        ignore_index=True,
    ).drop_duplicates()

    features_with_pairs_summary_df = (
        features_with_pairs_df
        .groupby("feature_family", as_index=False)
        .agg(features_with_high_correlation=("feature_name", "nunique"))
    )

    redundancy_pair_family_summary_df = (
        redundancy_pair_detail_df
        .groupby("feature_family", as_index=False)
        .agg(
            high_correlation_pairs=("feature_pair_key", "nunique"),
            strict_high_correlation_pairs=(
                "correlation_band",
                lambda values: int((values == "strict_high_correlation").sum()),
            ),
            max_flagged_abs_spearman_corr=("abs_spearman_corr", "max"),
            mean_flagged_abs_spearman_corr=("abs_spearman_corr", "mean"),
        )
    )

    redundancy_by_family_df = (
        redundancy_family_denominator_df
        .merge(
            features_with_pairs_summary_df,
            on="feature_family",
            how="left",
        )
        .merge(
            redundancy_pair_family_summary_df,
            on="feature_family",
            how="left",
        )
    )

    fill_zero_columns = [
        "features_with_high_correlation",
        "high_correlation_pairs",
        "strict_high_correlation_pairs",
    ]

    redundancy_by_family_df[fill_zero_columns] = (
        redundancy_by_family_df[fill_zero_columns]
        .fillna(0)
        .astype(int)
    )

    redundancy_by_family_df["high_correlation_pair_pct"] = np.where(
        redundancy_by_family_df["possible_feature_pairs"].gt(0),
        redundancy_by_family_df["high_correlation_pairs"]
        / redundancy_by_family_df["possible_feature_pairs"],
        0,
    )

    redundancy_by_family_df["strict_high_correlation_pair_pct"] = np.where(
        redundancy_by_family_df["possible_feature_pairs"].gt(0),
        redundancy_by_family_df["strict_high_correlation_pairs"]
        / redundancy_by_family_df["possible_feature_pairs"],
        0,
    )

redundancy_by_family_df = (
    redundancy_by_family_df
    .sort_values(
        [
            "high_correlation_pairs",
            "strict_high_correlation_pairs",
            "features_with_high_correlation",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

redundancy_by_family_display_df = redundancy_by_family_df.copy()
redundancy_by_family_display_df["high_correlation_pair_pct"] = (
    redundancy_by_family_display_df["high_correlation_pair_pct"]
    .mul(100)
    .round(2)
)
redundancy_by_family_display_df["strict_high_correlation_pair_pct"] = (
    redundancy_by_family_display_df["strict_high_correlation_pair_pct"]
    .mul(100)
    .round(2)
)
redundancy_by_family_display_df["max_flagged_abs_spearman_corr"] = (
    redundancy_by_family_display_df["max_flagged_abs_spearman_corr"]
    .round(4)
)
redundancy_by_family_display_df["mean_flagged_abs_spearman_corr"] = (
    redundancy_by_family_display_df["mean_flagged_abs_spearman_corr"]
    .round(4)
)

display(redundancy_by_family_display_df)

print(
    "High-correlation family summary complete. "
    f"Families summarized: {len(redundancy_by_family_df):,}."
)

,feature_family,eligible_features,possible_feature_pairs,features_with_high_correlation,high_correlation_pairs,strict_high_correlation_pairs,max_flagged_abs_spearman_corr,mean_flagged_abs_spearman_corr,high_correlation_pair_pct,strict_high_correlation_pair_pct
0,temporal_history,150,11175,110,192,113,0.9999,0.9609,1.72,1.01
1,spatial_context,61,1830,30,22,12,1.0000,0.9577,1.20,0.66
2,multimodal_interaction,41,820,15,9,9,0.9992,0.9913,1.10,1.10
3,anomaly_severity,21,210,14,7,7,0.9935,0.9854,3.33,3.33
4,stl_decomposition,28,378,8,4,1,0.9622,0.9446,1.06,0.26
5,raw_metric,15,105,4,2,1,0.9930,0.9635,1.90,0.95
6,fourier20_residual_reconstruction,7,21,0,0,0,NaN,NaN,0.00,0.00


High-correlation family summary complete. Families summarized: 7.


Then we pull the headline redundancy counts into one small takeaway table\. This keeps the main pattern visible without asking the reader to inspect every correlated pair\.

In [31]:
# ---------------------------------------------------------------------
# Display high-correlation redundancy pattern takeaway
# ---------------------------------------------------------------------

if redundancy_pair_detail_df.empty:
    redundancy_pattern_takeaway_df = pd.DataFrame(
        [
            {
                "summary_point": "High-correlation pairs found",
                "value": "0",
            },
            {
                "summary_point": "Strict high-correlation pairs found",
                "value": "0",
            },
            {
                "summary_point": "Feature family with most flagged pairs",
                "value": "none",
            },
        ]
    )

else:
    top_family = redundancy_by_family_df.iloc[0]

    redundancy_pattern_takeaway_df = pd.DataFrame(
        [
            {
                "summary_point": "High-correlation pairs found",
                "value": f"{redundancy_pair_detail_df['feature_pair_key'].nunique():,}",
            },
            {
                "summary_point": "Strict high-correlation pairs found",
                "value": f"{(redundancy_pair_detail_df['correlation_band'] == 'strict_high_correlation').sum():,}",
            },
            {
                "summary_point": "Feature family with most flagged pairs",
                "value": f"{top_family['feature_family']} ({int(top_family['high_correlation_pairs']):,} pairs)",
            },
            {
                "summary_point": "Features involved in at least one flagged pair",
                "value": f"{features_with_pairs_df['feature_name'].nunique():,}",
            },
        ]
    )

display(redundancy_pattern_takeaway_df)

print("High-correlation redundancy takeaway complete.")

,summary_point,value
0,High-correlation pairs found,236
1,Strict high-correlation pairs found,143
2,Feature family with most flagged pairs,temporal_history (192 pairs)
3,Features involved in at least one flagged pair,181


High-correlation redundancy takeaway complete.


Findings\. The headline redundancy pattern is concentrated, not evenly spread\. The full\-period screen found 236 high\-correlation pairs, including 143 strict pairs, and 181 features are involved in at least one flagged pair\. Temporal\-history is the clear hotspot with 192 flagged pairs, which makes sense because 1\.5\.1 intentionally generated several related versions of the same same\-bucket history signal\. The a\-ha here is that redundancy is real, but it is not a blanket problem across every family: Fourier Top 20 has no within\-family high\-correlation pairs, STL decomposition has only a small number, and most of the pruning pressure belongs to temporal\-history and selected spatial/context features\.

The pair table tells us which two features overlap, but pruning decisions happen one feature at a time\. These next blocks convert pair\-level evidence into feature\-level review flags that later sections can use\.

In [32]:
# ---------------------------------------------------------------------
# Prepare feature metadata lookup for redundancy review
# ---------------------------------------------------------------------

feature_metadata_lookup_df = (
    eligible_redundancy_features_df[
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "source_notebook",
        ]
    ]
    .drop_duplicates("feature_name")
    .reset_index(drop=True)
)

assert feature_metadata_lookup_df["feature_name"].is_unique, (
    "Feature metadata lookup should contain one row per feature."
)

print(
    "Feature metadata lookup ready. "
    f"Features: {len(feature_metadata_lookup_df):,}."
)

Feature metadata lookup ready. Features: 323.


Next we expand each correlated pair into one row per feature\. This makes it possible to count how many redundancy flags each individual feature carries\.

In [33]:
# ---------------------------------------------------------------------
# Expand redundancy pairs into feature-level rows
# ---------------------------------------------------------------------

if redundancy_pair_detail_df.empty:
    feature_pair_long_df = pd.DataFrame(
        columns=[
            "feature_name",
            "paired_feature",
            "correlation_band",
            "abs_spearman_corr",
            "feature_family",
            "modality",
            "source_metric",
            "source_notebook",
        ]
    )

else:
    feature_a_pairs_df = redundancy_pair_detail_df[
        [
            "feature_a",
            "feature_b",
            "correlation_band",
            "abs_spearman_corr",
        ]
    ].rename(
        columns={
            "feature_a": "feature_name",
            "feature_b": "paired_feature",
        }
    )

    feature_b_pairs_df = redundancy_pair_detail_df[
        [
            "feature_b",
            "feature_a",
            "correlation_band",
            "abs_spearman_corr",
        ]
    ].rename(
        columns={
            "feature_b": "feature_name",
            "feature_a": "paired_feature",
        }
    )

    feature_pair_long_df = (
        pd.concat(
            [
                feature_a_pairs_df,
                feature_b_pairs_df,
            ],
            ignore_index=True,
        )
        .merge(
            feature_metadata_lookup_df,
            on="feature_name",
            how="left",
        )
    )

assert set(["feature_name", "paired_feature", "abs_spearman_corr"]).issubset(
    feature_pair_long_df.columns
), "Feature-level redundancy pair table is missing required columns."

print(
    "Feature-level redundancy pair table ready. "
    f"Rows: {len(feature_pair_long_df):,}."
)

Feature-level redundancy pair table ready. Rows: 472.


Now we aggregate the feature\-level rows into the actual review table\. The result tells us which features are involved in high\-correlation pairs and how severe those flags are\.

In [34]:
# ---------------------------------------------------------------------
# Aggregate feature-level redundancy review flags
# ---------------------------------------------------------------------

if feature_pair_long_df.empty:
    redundancy_feature_review_df = pd.DataFrame(
        columns=[
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "source_notebook",
            "high_correlation_pair_count",
            "strict_high_correlation_pair_count",
            "max_abs_spearman_corr",
            "most_correlated_feature",
            "redundancy_review_flag",
            "selection_decision_tier",
        ]
    )

else:
    most_correlated_feature_df = (
        feature_pair_long_df
        .sort_values("abs_spearman_corr", ascending=False)
        .drop_duplicates("feature_name")
        [["feature_name", "paired_feature", "abs_spearman_corr"]]
        .rename(
            columns={
                "paired_feature": "most_correlated_feature",
                "abs_spearman_corr": "max_abs_spearman_corr",
            }
        )
    )

    redundancy_feature_review_df = (
        feature_pair_long_df
        .groupby(
            [
                "feature_name",
                "feature_family",
                "modality",
                "source_metric",
                "source_notebook",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            high_correlation_pair_count=("paired_feature", "count"),
            strict_high_correlation_pair_count=(
                "correlation_band",
                lambda values: int((values == "strict_high_correlation").sum()),
            ),
        )
        .merge(
            most_correlated_feature_df,
            on="feature_name",
            how="left",
        )
        .assign(
            redundancy_review_flag=lambda df: np.select(
                [
                    df["strict_high_correlation_pair_count"].gt(0),
                    df["high_correlation_pair_count"].gt(0),
                ],
                [
                    "strict_within_family_redundancy_review",
                    "within_family_redundancy_review",
                ],
                default="no_high_correlation_flag",
            ),
            selection_decision_tier="candidate_review",
        )
        .sort_values(
            [
                "strict_high_correlation_pair_count",
                "high_correlation_pair_count",
                "max_abs_spearman_corr",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

assert set([
    "feature_name",
    "redundancy_review_flag",
    "high_correlation_pair_count",
]).issubset(redundancy_feature_review_df.columns), (
    "Initial redundancy review table is missing required columns."
)

print(
    "Initial redundancy review table built. "
    f"Flagged features: {len(redundancy_feature_review_df):,}."
)

Initial redundancy review table built. Flagged features: 181.


Now we display the feature\-level review output\. The summary shows how many features need redundancy review, while the preview puts the most heavily flagged columns at the top\.

In [35]:
# ---------------------------------------------------------------------
# Display initial redundancy review summary
# ---------------------------------------------------------------------

redundancy_feature_review_summary_df = (
    redundancy_feature_review_df
    .groupby("redundancy_review_flag", as_index=False)
    .agg(features=("feature_name", "count"))
    .sort_values("features", ascending=False)
    .reset_index(drop=True)
)

redundancy_feature_review_preview_df = (
    redundancy_feature_review_df
    .sort_values(
        [
            "strict_high_correlation_pair_count",
            "high_correlation_pair_count",
            "max_abs_spearman_corr",
        ],
        ascending=False,
    )
    .head(40)
    .reset_index(drop=True)
)

display(redundancy_feature_review_summary_df)
display(redundancy_feature_review_preview_df)

print(
    "Initial redundancy review summary displayed. "
    f"Features in review table: {len(redundancy_feature_review_df):,}."
)

,redundancy_review_flag,features
0,strict_within_family_redundancy_review,147
1,within_family_redundancy_review,34


,feature_name,feature_family,modality,source_metric,source_notebook,high_correlation_pair_count,strict_high_correlation_pair_count,most_correlated_feature,max_abs_spearman_corr,redundancy_review_flag,selection_decision_tier
0,taxi_trip_count_ema_7_same_bucket,temporal_history,taxi,taxi_trip_count,1.5.1,10,6,taxi_trip_count_rolling_mean_7_same_bucket,0.997221,strict_within_family_redundancy_review,candidate_review
1,taxi_trip_count_rolling_mean_7_same_bucket,temporal_history,taxi,taxi_trip_count,1.5.1,10,6,taxi_trip_count_ema_7_same_bucket,0.997221,strict_within_family_redundancy_review,candidate_review
2,taxi_distinct_dropoff_zone_count_ema_7_same_bu...,temporal_history,taxi,taxi_distinct_dropoff_zone_count,1.5.1,10,5,taxi_distinct_dropoff_zone_count_rolling_mean_...,0.995220,strict_within_family_redundancy_review,candidate_review
3,taxi_distinct_dropoff_zone_count_rolling_mean_...,temporal_history,taxi,taxi_distinct_dropoff_zone_count,1.5.1,10,5,taxi_distinct_dropoff_zone_count_ema_7_same_bu...,0.995220,strict_within_family_redundancy_review,candidate_review
4,taxi_distinct_dropoff_zone_count_lag_1_same_bu...,temporal_history,taxi,taxi_distinct_dropoff_zone_count,1.5.1,6,5,taxi_trip_count_lag_1_same_bucket,0.993012,strict_within_family_redundancy_review,candidate_review
5,taxi_trip_count_rolling_std_7_same_bucket,temporal_history,taxi,taxi_trip_count,1.5.1,11,4,taxi_trip_count_rolling_mean_7_same_bucket,0.978700,strict_within_family_redundancy_review,candidate_review
6,fhvhv_trip_count_ema_7_same_bucket,temporal_history,fhvhv,fhvhv_trip_count,1.5.1,9,4,fhvhv_trip_count_rolling_mean_7_same_bucket,0.999355,strict_within_family_redundancy_review,candidate_review
7,fhvhv_trip_count_rolling_mean_7_same_bucket,temporal_history,fhvhv,fhvhv_trip_count,1.5.1,9,4,fhvhv_trip_count_ema_7_same_bucket,0.999355,strict_within_family_redundancy_review,candidate_review
8,fhvhv_distinct_dropoff_zone_count_ema_7_same_b...,temporal_history,fhvhv,fhvhv_distinct_dropoff_zone_count,1.5.1,9,4,fhvhv_distinct_dropoff_zone_count_rolling_mean...,0.999336,strict_within_family_redundancy_review,candidate_review
9,fhvhv_distinct_dropoff_zone_count_rolling_mean...,temporal_history,fhvhv,fhvhv_distinct_dropoff_zone_count,1.5.1,9,4,fhvhv_distinct_dropoff_zone_count_ema_7_same_b...,0.999336,strict_within_family_redundancy_review,candidate_review


Initial redundancy review summary displayed. Features in review table: 181.


Findings\. The feature\-level review table turns the 236 correlated pairs into 181 individual features that need attention: 147 strict review flags and 34 standard review flags\. The preview is sorted so the most entangled features rise to the top, and those are mostly temporal\-history features such as taxi and FHVHV rolling means, EMAs, lags, and rolling standard deviations\. The useful way to read this table is not “drop the first 40 rows\.” It is “these are the features where we should ask whether multiple versions are telling the same story\.”

This block attaches the redundancy flags back to the full candidate inventory\. Features with no high\-correlation evidence get neutral defaults, so the final handoff still contains one row per candidate feature\.

In [36]:
# ---------------------------------------------------------------------
# Build complete redundancy decision input table
# ---------------------------------------------------------------------

redundancy_review_columns = [
    "feature_name",
    "high_correlation_pair_count",
    "strict_high_correlation_pair_count",
    "max_abs_spearman_corr",
    "most_correlated_feature",
    "redundancy_review_flag",
]

if redundancy_feature_review_df.empty:
    redundancy_review_lookup_df = pd.DataFrame(columns=redundancy_review_columns)
else:
    redundancy_review_lookup_df = redundancy_feature_review_df[
        redundancy_review_columns
    ].copy()

redundancy_decision_input_df = (
    candidate_inventory_df[
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "feature_type",
            "modality",
            "source_metric",
            "is_core_metric_descendant",
            "non_null_pct",
            "null_pct",
            "near_zero_variance_flag",
        ]
    ]
    .merge(
        redundancy_review_lookup_df,
        on="feature_name",
        how="left",
    )
)

redundancy_decision_input_df["high_correlation_pair_count"] = (
    redundancy_decision_input_df["high_correlation_pair_count"]
    .fillna(0)
    .astype(int)
)

redundancy_decision_input_df["strict_high_correlation_pair_count"] = (
    redundancy_decision_input_df["strict_high_correlation_pair_count"]
    .fillna(0)
    .astype(int)
)

redundancy_decision_input_df["max_abs_spearman_corr"] = (
    redundancy_decision_input_df["max_abs_spearman_corr"]
    .fillna(0)
)

redundancy_decision_input_df["most_correlated_feature"] = (
    redundancy_decision_input_df["most_correlated_feature"]
    .fillna("none")
)

redundancy_decision_input_df["redundancy_review_flag"] = (
    redundancy_decision_input_df["redundancy_review_flag"]
    .fillna("no_high_correlation_flag")
)

# This remains an input to final selection; it is not the final keep/drop decision.
redundancy_decision_input_df["selection_decision_tier"] = "candidate_review"

redundancy_decision_input_df["selection_note"] = np.select(
    [
        redundancy_decision_input_df["redundancy_review_flag"].eq(
            "strict_within_family_redundancy_review"
        ),
        redundancy_decision_input_df["redundancy_review_flag"].eq(
            "within_family_redundancy_review"
        ),
    ],
    [
        "Review closely: feature has at least one within-family Spearman correlation above the strict threshold.",
        "Review: feature has at least one within-family Spearman correlation above the high-correlation threshold.",
    ],
    default="No high within-family Spearman correlation flag in this diagnostic pass.",
)

redundancy_decision_input_df = (
    redundancy_decision_input_df
    .sort_values(
        [
            "strict_high_correlation_pair_count",
            "high_correlation_pair_count",
            "max_abs_spearman_corr",
            "feature_family",
            "modality",
            "feature_name",
        ],
        ascending=[False, False, False, True, True, True],
    )
    .reset_index(drop=True)
)

assert len(redundancy_decision_input_df) == len(candidate_inventory_df), (
    "Redundancy decision input table should have one row per candidate feature."
)

print(
    "Complete redundancy decision input table built. "
    f"Candidate features: {len(redundancy_decision_input_df):,}."
)

Complete redundancy decision input table built. Candidate features: 329.


This block builds compact decision\-summary tables from the completed redundancy handoff\. It is still evidence, not the final keep/drop decision\.

In [37]:
# ---------------------------------------------------------------------
# Build redundancy decision summary tables
# ---------------------------------------------------------------------

redundancy_decision_summary_df = (
    redundancy_decision_input_df
    .groupby(["redundancy_review_flag", "feature_family"], as_index=False)
    .agg(features=("feature_name", "count"))
    .sort_values(["redundancy_review_flag", "features"], ascending=[True, False])
    .reset_index(drop=True)
)

redundancy_decision_overview_df = pd.DataFrame(
    [
        {
            "qa_check": "candidate_features",
            "value": len(redundancy_decision_input_df),
            "status": "pass" if len(redundancy_decision_input_df) == len(candidate_inventory_df) else "fail",
        },
        {
            "qa_check": "features_with_high_correlation_flag",
            "value": int(
                redundancy_decision_input_df["redundancy_review_flag"]
                .ne("no_high_correlation_flag")
                .sum()
            ),
            "status": "reference",
        },
        {
            "qa_check": "features_without_high_correlation_flag",
            "value": int(
                redundancy_decision_input_df["redundancy_review_flag"]
                .eq("no_high_correlation_flag")
                .sum()
            ),
            "status": "reference",
        },
    ]
)

assert len(redundancy_decision_input_df) == len(candidate_inventory_df), (
    "Redundancy decision input table should have one row per candidate feature."
)

print("Redundancy decision summary tables built.")

Redundancy decision summary tables built.


Finally, we display and export the redundancy decision input table\. Later sections can use this as the evidence layer for feature selection, without rerunning the long correlation checks\.

In [38]:
# ---------------------------------------------------------------------
# Display and export redundancy decision input table
# ---------------------------------------------------------------------

redundancy_decision_input_df.to_csv(
    FEATURE_SELECTION_DECISIONS_PATH,
    index=False,
)

display(redundancy_decision_overview_df)
display(redundancy_decision_summary_df)
display(redundancy_decision_input_df.head(40))

print(
    "Redundancy decision input table exported. "
    f"Path: {FEATURE_SELECTION_DECISIONS_PATH}."
)

,qa_check,value,status
0,candidate_features,329,pass
1,features_with_high_correlation_flag,181,reference
2,features_without_high_correlation_flag,148,reference


,redundancy_review_flag,feature_family,features
0,no_high_correlation_flag,temporal_history,44
1,no_high_correlation_flag,spatial_context,32
2,no_high_correlation_flag,multimodal_interaction,26
3,no_high_correlation_flag,stl_decomposition,20
4,no_high_correlation_flag,raw_metric,12
5,no_high_correlation_flag,anomaly_severity,7
6,no_high_correlation_flag,fourier20_residual_reconstruction,7
7,strict_within_family_redundancy_review,temporal_history,93
8,strict_within_family_redundancy_review,spatial_context,21
9,strict_within_family_redundancy_review,multimodal_interaction,15


,feature_name,source_notebook,feature_family,feature_type,modality,source_metric,is_core_metric_descendant,non_null_pct,null_pct,near_zero_variance_flag,high_correlation_pair_count,strict_high_correlation_pair_count,max_abs_spearman_corr,most_correlated_feature,redundancy_review_flag,selection_decision_tier,selection_note
0,taxi_trip_count_ema_7_same_bucket,1.5.1,temporal_history,same_bucket_ema,taxi,taxi_trip_count,True,1.000000,0.000000,False,10,6,0.997221,taxi_trip_count_rolling_mean_7_same_bucket,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
1,taxi_trip_count_rolling_mean_7_same_bucket,1.5.1,temporal_history,same_bucket_rolling_mean,taxi,taxi_trip_count,True,1.000000,0.000000,False,10,6,0.997221,taxi_trip_count_ema_7_same_bucket,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
2,taxi_distinct_dropoff_zone_count_ema_7_same_bu...,1.5.1,temporal_history,same_bucket_ema,taxi,taxi_distinct_dropoff_zone_count,False,0.977081,0.022919,False,10,5,0.995220,taxi_distinct_dropoff_zone_count_rolling_mean_...,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
3,taxi_distinct_dropoff_zone_count_rolling_mean_...,1.5.1,temporal_history,same_bucket_rolling_mean,taxi,taxi_distinct_dropoff_zone_count,False,0.883824,0.116176,False,10,5,0.995220,taxi_distinct_dropoff_zone_count_ema_7_same_bu...,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
4,taxi_distinct_dropoff_zone_count_lag_1_same_bu...,1.5.1,temporal_history,same_bucket_lag,taxi,taxi_distinct_dropoff_zone_count,False,0.720051,0.279949,False,6,5,0.993012,taxi_trip_count_lag_1_same_bucket,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
5,taxi_trip_count_rolling_std_7_same_bucket,1.5.1,temporal_history,same_bucket_rolling_std,taxi,taxi_trip_count,True,0.998314,0.001686,False,11,4,0.978700,taxi_trip_count_rolling_mean_7_same_bucket,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
6,fhvhv_trip_count_ema_7_same_bucket,1.5.1,temporal_history,same_bucket_ema,fhvhv,fhvhv_trip_count,True,1.000000,0.000000,False,9,4,0.999355,fhvhv_trip_count_rolling_mean_7_same_bucket,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
7,fhvhv_trip_count_rolling_mean_7_same_bucket,1.5.1,temporal_history,same_bucket_rolling_mean,fhvhv,fhvhv_trip_count,True,1.000000,0.000000,False,9,4,0.999355,fhvhv_trip_count_ema_7_same_bucket,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
8,fhvhv_distinct_dropoff_zone_count_ema_7_same_b...,1.5.1,temporal_history,same_bucket_ema,fhvhv,fhvhv_distinct_dropoff_zone_count,False,1.000000,0.000000,False,9,4,0.999336,fhvhv_distinct_dropoff_zone_count_rolling_mean...,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...
9,fhvhv_distinct_dropoff_zone_count_rolling_mean...,1.5.1,temporal_history,same_bucket_rolling_mean,fhvhv,fhvhv_distinct_dropoff_zone_count,False,1.000000,0.000000,False,9,4,0.999336,fhvhv_distinct_dropoff_zone_count_ema_7_same_b...,strict_within_family_redundancy_review,candidate_review,Review closely: feature has at least one withi...


Redundancy decision input table exported. Path: pipeline_data/1.5.6.intermediate_tables/feature_selection_decisions.csv.


Findings\. The exported decision table keeps all 329 candidate features in view: 181 features receive a high\-correlation review flag and 148 do not\. The summary table makes the review pattern easier to read because it shows where the flagged features are concentrated\. Temporal\-history has the largest flagged group, while Fourier Top 20 has no high\-correlation flags in this within\-family screen\. The practical takeaway is that this table is evidence for pruning, not the pruning decision itself: it tells us which features need closer review without flattening modality coverage or interpretability too early\.

Let’s close the global redundancy screen with a simpler family\-level visual\. This stacked bar chart shows how each feature family splits across no\-flag, standard\-review, and strict\-review buckets, so the reader can see where redundancy pressure is concentrated without needing to inspect individual feature names yet\.

In [39]:
# ---------------------------------------------------------------------
# Visualize redundancy review flags by feature family
# ---------------------------------------------------------------------

redundancy_flag_display_order = [
    "no_high_correlation_flag",
    "within_family_redundancy_review",
    "strict_within_family_redundancy_review",
]

redundancy_flag_display_labels = {
    "no_high_correlation_flag": "No high-correlation flag",
    "within_family_redundancy_review": "Review",
    "strict_within_family_redundancy_review": "Strict review",
}

redundancy_flag_color_map = {
    "No high-correlation flag": "#2ca25f",
    "Review": "#fdae61",
    "Strict review": "#d73027",
}

redundancy_flag_family_chart_df = (
    redundancy_decision_input_df
    .groupby(["feature_family", "redundancy_review_flag"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
)

redundancy_flag_family_chart_df["redundancy_review_flag"] = pd.Categorical(
    redundancy_flag_family_chart_df["redundancy_review_flag"],
    categories=redundancy_flag_display_order,
    ordered=True,
)

redundancy_flag_family_chart_df["review_bucket"] = (
    redundancy_flag_family_chart_df["redundancy_review_flag"]
    .map(redundancy_flag_display_labels)
)

redundancy_flag_family_chart_df["family_total_features"] = (
    redundancy_flag_family_chart_df
    .groupby("feature_family")["feature_count"]
    .transform("sum")
)

redundancy_flag_family_chart_df["feature_pct_within_family"] = (
    redundancy_flag_family_chart_df["feature_count"]
    / redundancy_flag_family_chart_df["family_total_features"]
).round(3)

redundancy_flag_family_chart_df = (
    redundancy_flag_family_chart_df
    .sort_values(
        [
            "family_total_features",
            "feature_family",
            "redundancy_review_flag",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

redundancy_flag_family_summary_table_df = (
    redundancy_flag_family_chart_df
    .pivot_table(
        index="feature_family",
        columns="review_bucket",
        values="feature_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for bucket in redundancy_flag_display_labels.values():
    if bucket not in redundancy_flag_family_summary_table_df.columns:
        redundancy_flag_family_summary_table_df[bucket] = 0

redundancy_flag_family_summary_table_df["total_features"] = (
    redundancy_flag_family_summary_table_df[
        list(redundancy_flag_display_labels.values())
    ]
    .sum(axis=1)
)

redundancy_flag_family_summary_table_df = (
    redundancy_flag_family_summary_table_df[
        [
            "feature_family",
            "total_features",
            "No high-correlation flag",
            "Review",
            "Strict review",
        ]
    ]
    .sort_values("total_features", ascending=False)
    .reset_index(drop=True)
)

if px is None:
    print("Plotly is not available, so displaying the chart source table instead.")
    display(redundancy_flag_family_summary_table_df)

else:
    fig = px.bar(
        redundancy_flag_family_chart_df,
        x="feature_family",
        y="feature_count",
        color="review_bucket",
        color_discrete_map=redundancy_flag_color_map,
        category_orders={
            "review_bucket": list(redundancy_flag_display_labels.values()),
            "feature_family": redundancy_flag_family_summary_table_df["feature_family"].tolist(),
        },
        custom_data=[
            "feature_count",
            "family_total_features",
            "feature_pct_within_family",
        ],
        title="Redundancy review flags by feature family",
    )

    fig.update_traces(
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Bucket: %{legendgroup}<br>"
            "Features: %{customdata[0]:,}<br>"
            "Family total: %{customdata[1]:,}<br>"
            "Share within family: %{customdata[2]:.1%}"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        barmode="stack",
        xaxis_title="Feature family",
        yaxis_title="Candidate features",
        legend_title_text="Redundancy review bucket",
        margin=dict(t=60, l=10, r=10, b=120),
        height=600,
    )

    fig.update_xaxes(tickangle=35)

    fig.show()

# Keep a compact table underneath the chart so exact counts are easy to inspect.
display(redundancy_flag_family_summary_table_df)

print("Redundancy review stacked bar chart ready.")

review_bucket,feature_family,total_features,No high-correlation flag,Review,Strict review
0,temporal_history,154,44,17,93
1,spatial_context,62,32,9,21
2,multimodal_interaction,41,26,0,15
3,stl_decomposition,28,20,6,2
4,anomaly_severity,21,7,0,14
5,raw_metric,16,12,2,2
6,fourier20_residual_reconstruction,7,7,0,0


Redundancy review stacked bar chart ready.


Findings\. Read this chart by comparing the green, amber, and red portions within each feature family\. Green means no high\-correlation flag, amber means standard redundancy review, and red means strict redundancy review\. The main pattern is concentrated rather than universal: temporal\-history carries the largest review burden, while Fourier Top 20 stays clean in this within\-family screen and several smaller families have much lighter overlap\. The count table below the chart is useful when the bar lengths are close and you want the exact number of features in each bucket\.

### Section Summary

This section ran the first redundancy screen across the full candidate feature universe, using within\-family Spearman correlations on a stratified diagnostic sample\. The results show that redundancy exists, but it is not evenly distributed: temporal\-history features carry most of the review pressure, while several smaller families have limited overlap and Fourier Top 20 remains clean in this screen\. These findings support a conservative pruning strategy: use redundancy flags to guide representative\-feature review, not as automatic drop rules\.

## 1\.5\.6\.3 Evaluate Modality\-Specific Redundancy

The global redundancy screen shows where feature families overlap overall, but downstream modeling will not use one universal feature set for every question\. Bus, subway, taxi, FHVHV, and multimodal workflows may each need different pruning choices\. This section will rerun the redundancy logic within those intended modeling views so feature\-selection recommendations can stay specific to the way the features will actually be used\.

### Build Modality\-Specific Feature Groups

Let’s translate the global feature inventory into the modeling views we actually expect to use downstream\. Each single\-modality view gets its own modality\-owned features plus shared spatial context features\. The multimodal view gets the transportation\-system features together, plus cross\-modal interaction features and the same shared spatial context\. We reuse the redundancy flags from 1\.5\.6\.2 here, so this step is mostly classification and lookup rather than another correlation run\.

In [40]:
# ---------------------------------------------------------------------
# Configure modality-specific modeling views
# ---------------------------------------------------------------------

MODALITY_MODELING_VIEWS = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
    "multimodal",
]

CORE_TRANSPORTATION_MODALITIES = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
]

SHARED_CONTEXT_FEATURE_FAMILIES = [
    "spatial_context",
]

EXCLUDED_DEFAULT_MODELING_VIEW_MODALITIES = [
    "traffic",
]

modality_view_configuration_df = pd.DataFrame(
    [
        {"configuration_item": "modeling_views", "value": ", ".join(MODALITY_MODELING_VIEWS)},
        {"configuration_item": "core_transportation_modalities", "value": ", ".join(CORE_TRANSPORTATION_MODALITIES)},
        {"configuration_item": "shared_context_feature_families", "value": ", ".join(SHARED_CONTEXT_FEATURE_FAMILIES)},
        {"configuration_item": "excluded_default_modalities", "value": ", ".join(EXCLUDED_DEFAULT_MODELING_VIEW_MODALITIES)},
    ]
)

display(modality_view_configuration_df)

print("Modality-specific modeling view configuration ready.")

,configuration_item,value
0,modeling_views,"bus, subway, taxi, fhvhv, multimodal"
1,core_transportation_modalities,"bus, subway, taxi, fhvhv"
2,shared_context_feature_families,spatial_context
3,excluded_default_modalities,traffic


Modality-specific modeling view configuration ready.


Findings\. The configuration defines five modeling views: bus, subway, taxi, FHVHV, and multimodal\. Traffic is explicitly excluded from the default unsupervised modeling views, while spatial context is treated as shared context that can travel with each modality\-specific view\.

Now we assign candidate features to those modeling views\. Single\-modality views get their own modality\-owned features plus shared spatial context\. The multimodal view gets all core transportation modalities, cross\-modal interaction features, and shared spatial context\.

In [41]:
# ---------------------------------------------------------------------
# Assign features to modality-specific modeling views
# ---------------------------------------------------------------------

shared_context_mask = (
    redundancy_decision_input_df["feature_family"].isin(SHARED_CONTEXT_FEATURE_FAMILIES)
    & redundancy_decision_input_df["modality"].eq("other")
)

modality_feature_group_parts = []

for modeling_view in MODALITY_MODELING_VIEWS:
    if modeling_view == "multimodal":
        view_mask = (
            redundancy_decision_input_df["modality"].isin(CORE_TRANSPORTATION_MODALITIES + ["multimodal"])
            | shared_context_mask
        )
    else:
        view_mask = (
            redundancy_decision_input_df["modality"].eq(modeling_view)
            | shared_context_mask
        )

    view_feature_df = (
        redundancy_decision_input_df[view_mask]
        .copy()
        .assign(modeling_view=modeling_view)
    )

    # Traffic stays out of the default unsupervised modeling views for now.
    view_feature_df = view_feature_df[
        ~view_feature_df["modality"].isin(EXCLUDED_DEFAULT_MODELING_VIEW_MODALITIES)
    ].copy()

    view_feature_df["feature_group_role"] = np.select(
        [
            view_feature_df["feature_family"].isin(SHARED_CONTEXT_FEATURE_FAMILIES)
            & view_feature_df["modality"].eq("other"),
            view_feature_df["modeling_view"].eq("multimodal")
            & view_feature_df["modality"].eq("multimodal"),
            view_feature_df["modeling_view"].eq("multimodal")
            & view_feature_df["modality"].isin(CORE_TRANSPORTATION_MODALITIES),
            view_feature_df["modality"].eq(modeling_view),
        ],
        [
            "shared_spatial_context",
            "cross_modal_interaction",
            "modality_owned_in_multimodal_view",
            "modality_owned",
        ],
        default="review_assignment",
    )

    modality_feature_group_parts.append(view_feature_df)

modality_feature_groups_df = (
    pd.concat(modality_feature_group_parts, ignore_index=True)
    .sort_values(
        [
            "modeling_view",
            "feature_group_role",
            "feature_family",
            "modality",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Modality-specific feature assignments built. "
    f"Feature group rows: {len(modality_feature_groups_df):,}."
)

Modality-specific feature assignments built. Feature group rows: 594.


Before summarizing, we run a small QA check on the modeling\-view assignments\. This catches empty views, accidental traffic inclusion, and any features that fell into the generic review bucket\.

In [42]:
# ---------------------------------------------------------------------
# Validate modality-specific feature assignments
# ---------------------------------------------------------------------

missing_modeling_views = sorted(
    set(MODALITY_MODELING_VIEWS)
    - set(modality_feature_groups_df["modeling_view"])
)

traffic_feature_rows = int(
    modality_feature_groups_df["modality"]
    .isin(EXCLUDED_DEFAULT_MODELING_VIEW_MODALITIES)
    .sum()
)

review_assignment_rows = int(
    modality_feature_groups_df["feature_group_role"]
    .eq("review_assignment")
    .sum()
)

modality_feature_assignment_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "modeling_views_expected",
            "value": len(MODALITY_MODELING_VIEWS),
            "status": "reference",
        },
        {
            "qa_check": "modeling_views_present",
            "value": modality_feature_groups_df["modeling_view"].nunique(),
            "status": "pass" if not missing_modeling_views else "fail",
        },
        {
            "qa_check": "traffic_feature_rows",
            "value": traffic_feature_rows,
            "status": "pass" if traffic_feature_rows == 0 else "fail",
        },
        {
            "qa_check": "review_assignment_rows",
            "value": review_assignment_rows,
            "status": "pass" if review_assignment_rows == 0 else "review",
        },
    ]
)

display(modality_feature_assignment_qa_df)

assert not missing_modeling_views, (
    f"Missing modality modeling views: {missing_modeling_views}"
)

assert traffic_feature_rows == 0, (
    "Traffic features should not be included in default modality modeling views."
)

print("Modality-specific feature assignments validated.")

,qa_check,value,status
0,modeling_views_expected,5,reference
1,modeling_views_present,5,pass
2,traffic_feature_rows,0,pass
3,review_assignment_rows,0,pass


Modality-specific feature assignments validated.


Findings\. The assignment QA passes the main checks: all five modeling views are present, traffic features are excluded from the default modeling views, and no features fell into the generic review\-assignment bucket\. That means the modality grouping logic is clean enough to use for redundancy screening: bus, subway, taxi, FHVHV, and multimodal each have a defined feature set, and we are not accidentally carrying traffic into the unsupervised modeling feature views\.

Now we summarize the modeling views at the highest level\. This table shows how many features each downstream view contains and how much of that view is already carrying redundancy\-review pressure from the global screen\.

In [43]:
# ---------------------------------------------------------------------
# Summarize modality-specific modeling views
# ---------------------------------------------------------------------

modality_feature_group_summary_df = (
    modality_feature_groups_df
    .groupby("modeling_view", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        feature_families=("feature_family", "nunique"),
        modalities_represented=("modality", "nunique"),
        cross_modal_interaction_features=(
            "feature_group_role",
            lambda values: int((values == "cross_modal_interaction").sum()),
        ),
        high_correlation_review_features=(
            "redundancy_review_flag",
            lambda values: int((values != "no_high_correlation_flag").sum()),
        ),
        strict_redundancy_review_features=(
            "redundancy_review_flag",
            lambda values: int((values == "strict_within_family_redundancy_review").sum()),
        ),
    )
    .assign(
        high_correlation_review_pct=lambda df: (
            df["high_correlation_review_features"] / df["feature_count"]
        ).round(3),
        strict_redundancy_review_pct=lambda df: (
            df["strict_redundancy_review_features"] / df["feature_count"]
        ).round(3),
    )
    .sort_values("modeling_view")
    .reset_index(drop=True)
)

display(modality_feature_group_summary_df)

print("Modality-specific modeling view summary ready.")

,modeling_view,feature_count,feature_families,modalities_represented,cross_modal_interaction_features,high_correlation_review_features,strict_redundancy_review_features,high_correlation_review_pct,strict_redundancy_review_pct
0,bus,64,7,1,0,44,35,0.688,0.547
1,fhvhv,92,7,1,0,56,44,0.609,0.478
2,multimodal,307,7,5,20,174,142,0.567,0.463
3,subway,39,7,1,0,17,13,0.436,0.333
4,taxi,92,7,1,0,51,44,0.554,0.478


Modality-specific modeling view summary ready.


Findings\. The view\-level table shows the feature\-set sizes before pruning: subway is the smallest view with 39 features, bus has 64, taxi and FHVHV each have 92, and the multimodal view has 307 because it combines the core transportation systems plus 20 cross\-modal interaction features\. The review percentages are the useful columns here\. Bus has the heaviest review share, while subway is lighter, and the multimodal view has the most flagged features in absolute terms because it contains nearly the full candidate feature space\. This does not mean we should drop those features automatically\. It means each view needs representative\-feature choices instead of one global pruning rule\.

Then we break the same modeling views down by feature family\. This keeps the family\-level redundancy pattern separate from the high\-level view\-size table, so the reader can see which feature families are driving review pressure inside each modeling view\.

In [44]:
# ---------------------------------------------------------------------
# Summarize modality-specific feature families
# ---------------------------------------------------------------------

modality_feature_group_family_summary_df = (
    modality_feature_groups_df
    .groupby(["modeling_view", "feature_family"], as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        high_correlation_review_features=(
            "redundancy_review_flag",
            lambda values: int((values != "no_high_correlation_flag").sum()),
        ),
        strict_redundancy_review_features=(
            "redundancy_review_flag",
            lambda values: int((values == "strict_within_family_redundancy_review").sum()),
        ),
    )
    .assign(
        high_correlation_review_pct=lambda df: (
            df["high_correlation_review_features"] / df["feature_count"]
        ).round(3),
        strict_redundancy_review_pct=lambda df: (
            df["strict_redundancy_review_features"] / df["feature_count"]
        ).round(3),
    )
    .sort_values(["modeling_view", "feature_count"], ascending=[True, False])
    .reset_index(drop=True)
)

display(modality_feature_group_family_summary_df)

print("Modality-specific feature-family summary ready.")

,modeling_view,feature_family,feature_count,high_correlation_review_features,strict_redundancy_review_features,high_correlation_review_pct,strict_redundancy_review_pct
0,bus,temporal_history,30,27,23,0.900,0.767
1,bus,spatial_context,11,7,4,0.636,0.364
2,bus,stl_decomposition,8,4,2,0.500,0.250
3,bus,anomaly_severity,6,4,4,0.667,0.667
4,bus,multimodal_interaction,4,2,2,0.500,0.500
5,bus,raw_metric,3,0,0,0.000,0.000
6,bus,fourier20_residual_reconstruction,2,0,0,0.000,0.000
7,fhvhv,temporal_history,50,38,33,0.760,0.660
8,fhvhv,spatial_context,17,8,5,0.471,0.294
9,fhvhv,stl_decomposition,8,2,0,0.250,0.000


Modality-specific feature-family summary ready.


Findings\. The family\-level table shows that temporal\-history drives most of the redundancy pressure in every modeling view: 27 of 30 bus temporal features, 38 of 50 FHVHV temporal features, 34 of 50 taxi temporal features, 11 of 20 subway temporal features, and 110 of 150 multimodal temporal features carry review flags\. Fourier Top 20 has no review flags in any view, while STL decomposition is comparatively light, especially for subway\. The practical takeaway is that modality\-specific pruning should mostly focus on choosing representative temporal\-history features, not flattening every engineered family\.

### Run Modality\-Specific Redundancy Screens

Run Spearman checks within each modeling feature set\. The global screen asked whether features overlap inside each family; this screen asks a different question: when we assemble bus, subway, taxi, FHVHV, or multimodal feature sets, how much redundancy appears inside the feature set a downstream model would actually consume?

These modality\-specific checks can take a few minutes, so we cache the outputs separately from the global redundancy screen\. The toggle lets us reuse the cached tables during notebook reruns unless we intentionally want to rebuild them\.

In [45]:
# ---------------------------------------------------------------------
# Configure modality-specific redundancy cache files
# ---------------------------------------------------------------------

REBUILD_MODALITY_REDUNDANCY_ANALYSIS = False

MODALITY_REDUNDANCY_PAIR_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "modality_specific_redundancy_pairs.csv"
)
MODALITY_REDUNDANCY_FEATURE_FLAG_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "modality_specific_redundancy_feature_flags.csv"
)
MODALITY_REDUNDANCY_RUNTIME_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "modality_specific_redundancy_runtime.csv"
)

modality_redundancy_cache_status_df = pd.DataFrame(
    [
        {
            "cached_output": "modality_redundancy_pairs_df",
            "path": str(MODALITY_REDUNDANCY_PAIR_PATH),
            "exists": MODALITY_REDUNDANCY_PAIR_PATH.exists(),
        },
        {
            "cached_output": "modality_redundancy_feature_flags_df",
            "path": str(MODALITY_REDUNDANCY_FEATURE_FLAG_PATH),
            "exists": MODALITY_REDUNDANCY_FEATURE_FLAG_PATH.exists(),
        },
        {
            "cached_output": "modality_redundancy_runtime_df",
            "path": str(MODALITY_REDUNDANCY_RUNTIME_PATH),
            "exists": MODALITY_REDUNDANCY_RUNTIME_PATH.exists(),
        },
    ]
)

display(modality_redundancy_cache_status_df)

print(
    "Modality-specific redundancy cache configured. "
    f"Rebuild toggle: {REBUILD_MODALITY_REDUNDANCY_ANALYSIS}."
)

,cached_output,path,exists
0,modality_redundancy_pairs_df,pipeline_data/1.5.6.intermediate_tables/modali...,True
1,modality_redundancy_feature_flags_df,pipeline_data/1.5.6.intermediate_tables/modali...,True
2,modality_redundancy_runtime_df,pipeline_data/1.5.6.intermediate_tables/modali...,True


Modality-specific redundancy cache configured. Rebuild toggle: False.


Next we define the reusable Spearman helper\. It takes one modeling view at a time, keeps only usable numeric features from the diagnostic sample, computes pairwise Spearman correlations, and returns both pair\-level results and feature\-level review flags\.

In [46]:
# ---------------------------------------------------------------------
# Define modality-specific Spearman redundancy helper
# ---------------------------------------------------------------------

def run_modality_spearman_redundancy(
    modeling_view,
    view_feature_names,
    sample_df,
    inventory_df,
    high_threshold=SPEARMAN_CORRELATION_THRESHOLD,
    strict_threshold=STRICT_SPEARMAN_CORRELATION_THRESHOLD,
):
    """Run a Spearman redundancy screen for one modeling view."""
    start_time = time.perf_counter()

    candidate_features = [
        feature
        for feature in view_feature_names
        if feature in sample_df.columns
    ]

    usable_features = []

    for feature in candidate_features:
        feature_series = sample_df[feature]

        # Spearman needs variation and enough retained observations to be meaningful.
        if feature_series.notna().sum() < 2:
            continue

        if feature_series.nunique(dropna=True) <= 1:
            continue

        usable_features.append(feature)

    if len(usable_features) < 2:
        runtime_seconds = time.perf_counter() - start_time

        empty_pairs_df = pd.DataFrame()
        feature_flags_df = pd.DataFrame(
            {
                "modeling_view": modeling_view,
                "feature_name": usable_features,
                "high_correlation_pair_count": 0,
                "strict_high_correlation_pair_count": 0,
                "redundancy_review_flag": "no_high_correlation_flag",
            }
        )
        runtime_df = pd.DataFrame(
            [
                {
                    "modeling_view": modeling_view,
                    "candidate_features": len(candidate_features),
                    "eligible_features": len(usable_features),
                    "total_possible_pairs": 0,
                    "runtime_seconds": runtime_seconds,
                }
            ]
        )

        return empty_pairs_df, feature_flags_df, runtime_df

    correlation_df = sample_df[usable_features].corr(
        method="spearman",
        min_periods=50,
    )

    upper_triangle_mask = np.triu(
        np.ones(correlation_df.shape, dtype=bool),
        k=1,
    )

    pair_long_df = (
        correlation_df
        .where(upper_triangle_mask)
        .stack()
        .reset_index()
    )

    pair_long_df.columns = [
        "feature_1",
        "feature_2",
        "spearman_corr",
    ]

    pair_long_df = (
        pair_long_df
        .assign(
            modeling_view=modeling_view,
            abs_spearman_corr=lambda df: df["spearman_corr"].abs(),
            high_correlation_flag=lambda df: (
                df["abs_spearman_corr"] >= high_threshold
            ),
            strict_high_correlation_flag=lambda df: (
                df["abs_spearman_corr"] >= strict_threshold
            ),
        )
    )

    high_correlation_pairs_df = (
        pair_long_df[pair_long_df["high_correlation_flag"]]
        .copy()
        .sort_values("abs_spearman_corr", ascending=False)
        .reset_index(drop=True)
    )

    feature_pair_counts_df = pd.concat(
        [
            high_correlation_pairs_df.rename(columns={"feature_1": "feature_name"})[
                ["feature_name", "high_correlation_flag", "strict_high_correlation_flag"]
            ],
            high_correlation_pairs_df.rename(columns={"feature_2": "feature_name"})[
                ["feature_name", "high_correlation_flag", "strict_high_correlation_flag"]
            ],
        ],
        ignore_index=True,
    )

    if feature_pair_counts_df.empty:
        feature_flags_df = pd.DataFrame(
            {
                "feature_name": usable_features,
                "high_correlation_pair_count": 0,
                "strict_high_correlation_pair_count": 0,
            }
        )
    else:
        feature_flags_df = (
            feature_pair_counts_df
            .groupby("feature_name", as_index=False)
            .agg(
                high_correlation_pair_count=("high_correlation_flag", "sum"),
                strict_high_correlation_pair_count=("strict_high_correlation_flag", "sum"),
            )
        )

        feature_flags_df = (
            pd.DataFrame({"feature_name": usable_features})
            .merge(feature_flags_df, on="feature_name", how="left")
            .fillna(
                {
                    "high_correlation_pair_count": 0,
                    "strict_high_correlation_pair_count": 0,
                }
            )
        )

    feature_flags_df["high_correlation_pair_count"] = (
        feature_flags_df["high_correlation_pair_count"].astype(int)
    )
    feature_flags_df["strict_high_correlation_pair_count"] = (
        feature_flags_df["strict_high_correlation_pair_count"].astype(int)
    )

    feature_flags_df["redundancy_review_flag"] = np.select(
        [
            feature_flags_df["strict_high_correlation_pair_count"] > 0,
            feature_flags_df["high_correlation_pair_count"] > 0,
        ],
        [
            "strict_within_view_redundancy_review",
            "within_view_redundancy_review",
        ],
        default="no_high_correlation_flag",
    )

    feature_metadata_cols = [
        "feature_name",
        "feature_family",
        "modality",
        "source_metric",
        "source_notebook",
    ]

    feature_flags_df = (
        feature_flags_df
        .assign(modeling_view=modeling_view)
        .merge(
            inventory_df[feature_metadata_cols].drop_duplicates("feature_name"),
            on="feature_name",
            how="left",
        )
    )

    total_possible_pairs = int(len(usable_features) * (len(usable_features) - 1) / 2)
    runtime_seconds = time.perf_counter() - start_time

    runtime_df = pd.DataFrame(
        [
            {
                "modeling_view": modeling_view,
                "candidate_features": len(candidate_features),
                "eligible_features": len(usable_features),
                "total_possible_pairs": total_possible_pairs,
                "high_correlation_pairs": len(high_correlation_pairs_df),
                "strict_high_correlation_pairs": int(
                    high_correlation_pairs_df["strict_high_correlation_flag"].sum()
                ),
                "runtime_seconds": runtime_seconds,
            }
        ]
    )

    return high_correlation_pairs_df, feature_flags_df, runtime_df


print("Modality-specific Spearman helper ready.")

Modality-specific Spearman helper ready.


Now we run the screen for each modeling view\. Each view uses the same diagnostic sample and the same Spearman thresholds, but the feature list changes depending on whether we are looking at bus, subway, taxi, FHVHV, or the combined multimodal view\.

In [47]:
# ---------------------------------------------------------------------
# Run modality-specific Spearman redundancy screens
# ---------------------------------------------------------------------

required_modality_redundancy_cache_paths = [
    MODALITY_REDUNDANCY_PAIR_PATH,
    MODALITY_REDUNDANCY_FEATURE_FLAG_PATH,
    MODALITY_REDUNDANCY_RUNTIME_PATH,
]

modality_redundancy_cache_exists = all(
    path.exists()
    for path in required_modality_redundancy_cache_paths
)

if "modeling_sample_df" in globals():
    redundancy_sample_df = modeling_sample_df
elif "modeling_diagnostic_sample_df" in globals():
    redundancy_sample_df = modeling_diagnostic_sample_df
else:
    raise NameError(
        "Run the diagnostic sample block before modality-specific redundancy analysis. "
        "Expected modeling_sample_df or modeling_diagnostic_sample_df."
    )

if modality_redundancy_cache_exists and not REBUILD_MODALITY_REDUNDANCY_ANALYSIS:
    modality_redundancy_pairs_df = pd.read_csv(MODALITY_REDUNDANCY_PAIR_PATH)
    modality_redundancy_feature_flags_df = pd.read_csv(MODALITY_REDUNDANCY_FEATURE_FLAG_PATH)
    modality_redundancy_runtime_df = pd.read_csv(MODALITY_REDUNDANCY_RUNTIME_PATH)

    print("Loaded cached modality-specific redundancy outputs.")

else:
    modality_pair_parts = []
    modality_feature_flag_parts = []
    modality_runtime_parts = []

    iterator = MODALITY_MODELING_VIEWS

    if tqdm is not None:
        iterator = tqdm(
            MODALITY_MODELING_VIEWS,
            desc="Running modality redundancy screens",
        )

    for modeling_view in iterator:
        view_feature_names = (
            modality_feature_groups_df
            .loc[
                modality_feature_groups_df["modeling_view"].eq(modeling_view),
                "feature_name",
            ]
            .drop_duplicates()
            .tolist()
        )

        pair_df, feature_flag_df, runtime_df = run_modality_spearman_redundancy(
            modeling_view=modeling_view,
            view_feature_names=view_feature_names,
            sample_df=redundancy_sample_df,
            inventory_df=candidate_inventory_df,
        )

        if not pair_df.empty:
            modality_pair_parts.append(pair_df)

        modality_feature_flag_parts.append(feature_flag_df)
        modality_runtime_parts.append(runtime_df)

        print(
            f"{modeling_view}: "
            f"{runtime_df.loc[0, 'eligible_features']:,} eligible features, "
            f"{runtime_df.loc[0, 'high_correlation_pairs']:,} high-correlation pairs, "
            f"{runtime_df.loc[0, 'runtime_seconds']:.2f} seconds."
        )

        gc.collect()

    if modality_pair_parts:
        modality_redundancy_pairs_df = pd.concat(
            modality_pair_parts,
            ignore_index=True,
        )
    else:
        modality_redundancy_pairs_df = pd.DataFrame(
            columns=[
                "feature_1",
                "feature_2",
                "spearman_corr",
                "modeling_view",
                "abs_spearman_corr",
                "high_correlation_flag",
                "strict_high_correlation_flag",
            ]
        )

    modality_redundancy_feature_flags_df = pd.concat(
        modality_feature_flag_parts,
        ignore_index=True,
    )
    modality_redundancy_runtime_df = pd.concat(
        modality_runtime_parts,
        ignore_index=True,
    )

    modality_redundancy_pairs_df.to_csv(
        MODALITY_REDUNDANCY_PAIR_PATH,
        index=False,
    )
    modality_redundancy_feature_flags_df.to_csv(
        MODALITY_REDUNDANCY_FEATURE_FLAG_PATH,
        index=False,
    )
    modality_redundancy_runtime_df.to_csv(
        MODALITY_REDUNDANCY_RUNTIME_PATH,
        index=False,
    )

    print("Saved modality-specific redundancy outputs.")

display(modality_redundancy_runtime_df)

print(
    "Modality-specific redundancy screens ready. "
    f"High-correlation pairs: {len(modality_redundancy_pairs_df):,}."
)

Loaded cached modality-specific redundancy outputs.


,modeling_view,candidate_features,eligible_features,total_possible_pairs,high_correlation_pairs,strict_high_correlation_pairs,runtime_seconds
0,bus,64,64,2016,70,66,82.996986
1,subway,39,39,741,40,22,14.409784
2,taxi,92,92,4186,92,51,143.122598
3,fhvhv,92,92,4186,127,62,149.736073
4,multimodal,307,307,46971,377,233,1628.756799


Modality-specific redundancy screens ready. High-correlation pairs: 706.


Findings\. The modality\-specific Spearman screen completed across all five modeling views and wrote the cached redundancy outputs needed for comparison\. The runtime table is part of the story here: single\-modality views are manageable, while the multimodal view is much heavier because it tests a larger assembled feature set\. That confirms two things: the modality\-specific screen is feasible, and caching is necessary so we do not recompute the expensive multimodal correlations every time the notebook runs\.

After the screens run or load from cache, we do a compact QA pass\. This checks that every modeling view produced feature\-level flags, runtime records, and finite pair\-level correlation values where pairs were found\.

In [48]:
# ---------------------------------------------------------------------
# Validate modality-specific redundancy outputs
# ---------------------------------------------------------------------

modality_redundancy_views_with_flags = set(
    modality_redundancy_feature_flags_df["modeling_view"].dropna().unique()
)
modality_redundancy_views_with_runtime = set(
    modality_redundancy_runtime_df["modeling_view"].dropna().unique()
)

missing_feature_flag_views = sorted(
    set(MODALITY_MODELING_VIEWS)
    - modality_redundancy_views_with_flags
)
missing_runtime_views = sorted(
    set(MODALITY_MODELING_VIEWS)
    - modality_redundancy_views_with_runtime
)

if modality_redundancy_pairs_df.empty:
    finite_pair_values = True
else:
    finite_pair_values = bool(
        np.isfinite(modality_redundancy_pairs_df["spearman_corr"]).all()
        and np.isfinite(modality_redundancy_pairs_df["abs_spearman_corr"]).all()
    )

modality_redundancy_output_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "modeling_views_with_feature_flags",
            "value": len(modality_redundancy_views_with_flags),
            "status": "pass" if not missing_feature_flag_views else "fail",
        },
        {
            "qa_check": "modeling_views_with_runtime_records",
            "value": len(modality_redundancy_views_with_runtime),
            "status": "pass" if not missing_runtime_views else "fail",
        },
        {
            "qa_check": "high_correlation_pair_rows",
            "value": len(modality_redundancy_pairs_df),
            "status": "info",
        },
        {
            "qa_check": "finite_pair_correlation_values",
            "value": finite_pair_values,
            "status": "pass" if finite_pair_values else "fail",
        },
    ]
)

display(modality_redundancy_output_qa_df)

assert not missing_feature_flag_views, (
    f"Missing feature-flag outputs for modeling views: {missing_feature_flag_views}"
)
assert not missing_runtime_views, (
    f"Missing runtime outputs for modeling views: {missing_runtime_views}"
)
assert finite_pair_values, "Non-finite correlation values found in modality redundancy pairs."

print("Modality-specific redundancy outputs validated.")

,qa_check,value,status
0,modeling_views_with_feature_flags,5,pass
1,modeling_views_with_runtime_records,5,pass
2,high_correlation_pair_rows,706,info
3,finite_pair_correlation_values,True,pass


Modality-specific redundancy outputs validated.


Let’s turn the modality-specific redundancy screen into one feature-level evidence table. Each row below represents one feature inside one modeling view, with counts for how often it appears in high-correlation or strict high-correlation pairs. This gives the scorecard a clean input table instead of making later sections guess which modality output to use.

In [49]:
# ---------------------------------------------------------------------
# Build modality-specific feature-level redundancy evidence
# ---------------------------------------------------------------------

MODALITY_REDUNDANCY_FEATURE_EVIDENCE_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "modality_redundancy_feature_evidence.csv"
)

strict_modality_correlation_threshold = globals().get(
    "STRICT_SPEARMAN_CORRELATION_THRESHOLD",
    0.95,
)

required_modality_objects = [
    "modality_redundancy_pairs_df",
    "modality_redundancy_feature_flags_df",
]

missing_modality_objects = [
    object_name
    for object_name in required_modality_objects
    if object_name not in globals()
]

assert not missing_modality_objects, (
    "Run the modality-specific Spearman redundancy screen before building feature-level evidence. "
    f"Missing objects: {missing_modality_objects}"
)

feature_flag_df = modality_redundancy_feature_flags_df.copy()
pair_df = modality_redundancy_pairs_df.copy()

view_column = next(
    column
    for column in ["modeling_view", "feature_set", "view_name"]
    if column in feature_flag_df.columns
)

feature_column = next(
    column
    for column in ["feature_name", "candidate_feature", "feature"]
    if column in feature_flag_df.columns
)

review_flag_column = next(
    (
        column
        for column in [
            "modality_redundancy_review_flag",
            "within_view_redundancy_review_flag",
            "redundancy_review_flag",
        ]
        if column in feature_flag_df.columns
    ),
    None,
)

feature_flag_base_df = feature_flag_df.rename(
    columns={
        view_column: "modeling_view",
        feature_column: "feature_name",
    }
).copy()

if review_flag_column and review_flag_column != "modality_redundancy_review_flag":
    feature_flag_base_df = feature_flag_base_df.rename(
        columns={review_flag_column: "modality_redundancy_review_flag"}
    )

feature_flag_base_df = feature_flag_base_df[
    [
        column
        for column in [
            "modeling_view",
            "feature_name",
            "modality_redundancy_review_flag",
        ]
        if column in feature_flag_base_df.columns
    ]
].drop_duplicates()

if pair_df.empty:
    modality_pair_count_df = pd.DataFrame(
        columns=[
            "modeling_view",
            "feature_name",
            "modality_high_correlation_pair_count",
            "modality_strict_high_correlation_pair_count",
        ]
    )

else:
    pair_view_column = next(
        column
        for column in ["modeling_view", "feature_set", "view_name"]
        if column in pair_df.columns
    )

    feature_pair_columns = next(
        pair_columns
        for pair_columns in [
            ("feature_1", "feature_2"),
            ("feature_a", "feature_b"),
            ("left_feature", "right_feature"),
            ("feature_left", "feature_right"),
        ]
        if set(pair_columns).issubset(pair_df.columns)
    )

    corr_column = next(
        (
            column
            for column in [
                "abs_spearman_corr",
                "absolute_spearman_corr",
                "spearman_abs_corr",
                "abs_correlation",
            ]
            if column in pair_df.columns
        ),
        None,
    )

    strict_flag_column = next(
        (
            column
            for column in [
                "strict_high_correlation_flag",
                "strict_redundancy_flag",
            ]
            if column in pair_df.columns
        ),
        None,
    )

    pair_work_df = pair_df.rename(columns={pair_view_column: "modeling_view"}).copy()

    if corr_column:
        pair_work_df["strict_pair_flag"] = pair_work_df[corr_column].abs().ge(
            strict_modality_correlation_threshold
        )
    elif strict_flag_column:
        pair_work_df["strict_pair_flag"] = pair_work_df[strict_flag_column].fillna(False).astype(bool)
    else:
        pair_work_df["strict_pair_flag"] = False

    # Stack both sides of each correlated pair so the scorecard can score features, not pairs.
    left_pair_members_df = pair_work_df[
        ["modeling_view", feature_pair_columns[0], "strict_pair_flag"]
    ].rename(columns={feature_pair_columns[0]: "feature_name"})

    right_pair_members_df = pair_work_df[
        ["modeling_view", feature_pair_columns[1], "strict_pair_flag"]
    ].rename(columns={feature_pair_columns[1]: "feature_name"})

    modality_pair_count_df = (
        pd.concat(
            [left_pair_members_df, right_pair_members_df],
            ignore_index=True,
        )
        .groupby(["modeling_view", "feature_name"], as_index=False)
        .agg(
            modality_high_correlation_pair_count=("feature_name", "count"),
            modality_strict_high_correlation_pair_count=("strict_pair_flag", "sum"),
        )
    )

modality_redundancy_feature_evidence_df = (
    feature_flag_base_df
    .merge(
        modality_pair_count_df,
        on=["modeling_view", "feature_name"],
        how="left",
    )
    .merge(
        redundancy_decision_input_df[
            [
                "feature_name",
                "feature_family",
                "modality",
                "source_metric",
                "source_notebook",
                "is_core_metric_descendant",
            ]
        ].drop_duplicates("feature_name"),
        on="feature_name",
        how="left",
    )
)

for count_column in [
    "modality_high_correlation_pair_count",
    "modality_strict_high_correlation_pair_count",
]:
    modality_redundancy_feature_evidence_df[count_column] = (
        modality_redundancy_feature_evidence_df[count_column]
        .fillna(0)
        .astype(int)
    )

modality_redundancy_feature_evidence_df["modality_redundancy_review_flag"] = np.select(
    [
        modality_redundancy_feature_evidence_df[
            "modality_strict_high_correlation_pair_count"
        ].gt(0),
        modality_redundancy_feature_evidence_df[
            "modality_high_correlation_pair_count"
        ].gt(0),
    ],
    ["strict_within_view_redundancy_review", "within_view_redundancy_review"],
    default="no_high_correlation_flag",
)

modality_redundancy_feature_evidence_df = (
    modality_redundancy_feature_evidence_df
    .sort_values(
        [
            "modeling_view",
            "modality_strict_high_correlation_pair_count",
            "modality_high_correlation_pair_count",
            "feature_family",
            "feature_name",
        ],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)

modality_redundancy_feature_evidence_summary_df = (
    modality_redundancy_feature_evidence_df
    .groupby("modeling_view", as_index=False)
    .agg(
        features_in_view=("feature_name", "nunique"),
        features_with_high_correlation=(
            "modality_high_correlation_pair_count",
            lambda values: (values > 0).sum(),
        ),
        features_with_strict_correlation=(
            "modality_strict_high_correlation_pair_count",
            lambda values: (values > 0).sum(),
        ),
        high_correlation_pair_memberships=("modality_high_correlation_pair_count", "sum"),
        strict_correlation_pair_memberships=("modality_strict_high_correlation_pair_count", "sum"),
    )
    .sort_values("features_in_view", ascending=False)
    .reset_index(drop=True)
)

modality_redundancy_feature_evidence_df.to_csv(
    MODALITY_REDUNDANCY_FEATURE_EVIDENCE_PATH,
    index=False,
)

display(modality_redundancy_feature_evidence_summary_df)
display(modality_redundancy_feature_evidence_df.head(20))

assert not modality_redundancy_feature_evidence_df.empty, (
    "Modality-specific feature-level redundancy evidence is empty."
)
assert modality_redundancy_feature_evidence_df[
    [
        "feature_name",
        "modeling_view",
        "modality_high_correlation_pair_count",
        "modality_strict_high_correlation_pair_count",
        "modality_redundancy_review_flag",
    ]
].notna().all().all(), "Modality-specific redundancy evidence contains missing required values."

print(
    "Modality-specific feature-level redundancy evidence ready. "
    f"Feature-view rows: {len(modality_redundancy_feature_evidence_df):,}."
)

,modeling_view,features_in_view,features_with_high_correlation,features_with_strict_correlation,high_correlation_pair_memberships,strict_correlation_pair_memberships
0,multimodal,307,184,152,754,466
1,taxi,92,52,45,184,102
2,fhvhv,92,58,46,254,124
3,bus,64,46,38,140,132
4,subway,39,18,15,80,44


,modeling_view,feature_name,modality_redundancy_review_flag,modality_high_correlation_pair_count,modality_strict_high_correlation_pair_count,feature_family,modality,source_metric,source_notebook,is_core_metric_descendant
0,bus,avg_bus_speed_zscore,strict_within_view_redundancy_review,6,6,multimodal_interaction,bus,avg_bus_speed,1.5.3,True
1,bus,bus_trip_count_zscore,strict_within_view_redundancy_review,6,6,multimodal_interaction,bus,bus_trip_count,1.5.3,True
2,bus,avg_bus_speed,strict_within_view_redundancy_review,6,6,raw_metric,bus,avg_bus_speed,pre_1.5_engineering,True
3,bus,bus_trip_count,strict_within_view_redundancy_review,6,6,raw_metric,bus,bus_trip_count,pre_1.5_engineering,True
4,bus,avg_bus_speed_ema_7_same_bucket,strict_within_view_redundancy_review,6,6,temporal_history,bus,avg_bus_speed,1.5.1,True
5,bus,avg_bus_speed_lag_1_same_bucket,strict_within_view_redundancy_review,6,6,temporal_history,bus,avg_bus_speed,1.5.1,True
6,bus,avg_bus_speed_post_cp_mean_same_bucket,strict_within_view_redundancy_review,6,6,temporal_history,bus,avg_bus_speed,1.5.1,True
7,bus,avg_bus_speed_pre_cp_mean_same_bucket,strict_within_view_redundancy_review,6,6,temporal_history,bus,avg_bus_speed,1.5.1,True
8,bus,avg_bus_speed_rolling_mean_7_same_bucket,strict_within_view_redundancy_review,6,6,temporal_history,bus,avg_bus_speed,1.5.1,True
9,bus,bus_trip_count_ema_7_same_bucket,strict_within_view_redundancy_review,6,6,temporal_history,bus,bus_trip_count,1.5.1,True


Modality-specific feature-level redundancy evidence ready. Feature-view rows: 594.


Findings\. The modality\-specific redundancy screen completed successfully and produced feature\-level flags, runtime records, and finite correlation values for all five modeling views\. The runtime output is worth reading as part of the result: bus, subway, taxi, and FHVHV finish at manageable sizes, while the multimodal view is naturally the expensive case because it tests a much larger assembled feature set\. The QA table confirms that the cached outputs are usable, so the next comparison can focus on interpretation rather than rerunning the heavy Spearman checks\.

### Compare Redundancy by Modeling View

Now we compare the five modeling views side by side\. This table shows how much redundancy appears inside each view after its feature set is assembled: how many feature pairs were tested, how many crossed the review thresholds, and whether redundancy is concentrated in one view or spread across all of them\.

In [50]:
# ---------------------------------------------------------------------
# Compare redundancy volume by modeling view
# ---------------------------------------------------------------------

modality_redundancy_view_summary_df = (
    modality_redundancy_runtime_df
    .copy()
    .assign(
        high_correlation_pair_pct=lambda df: (
            df["high_correlation_pairs"] / df["total_possible_pairs"]
        ).replace([np.inf, -np.inf], np.nan).fillna(0).round(4),
        strict_high_correlation_pair_pct=lambda df: (
            df["strict_high_correlation_pairs"] / df["total_possible_pairs"]
        ).replace([np.inf, -np.inf], np.nan).fillna(0).round(4),
        runtime_minutes=lambda df: (df["runtime_seconds"] / 60).round(2),
    )
    [
        [
            "modeling_view",
            "eligible_features",
            "total_possible_pairs",
            "high_correlation_pairs",
            "high_correlation_pair_pct",
            "strict_high_correlation_pairs",
            "strict_high_correlation_pair_pct",
            "runtime_minutes",
        ]
    ]
    .sort_values("high_correlation_pair_pct", ascending=False)
    .reset_index(drop=True)
)

display(modality_redundancy_view_summary_df)

print("Modality redundancy volume summary ready.")

,modeling_view,eligible_features,total_possible_pairs,high_correlation_pairs,high_correlation_pair_pct,strict_high_correlation_pairs,strict_high_correlation_pair_pct,runtime_minutes
0,subway,39,741,40,0.0540,22,0.0297,0.24
1,bus,64,2016,70,0.0347,66,0.0327,1.38
2,fhvhv,92,4186,127,0.0303,62,0.0148,2.50
3,taxi,92,4186,92,0.0220,51,0.0122,2.39
4,multimodal,307,46971,377,0.0080,233,0.0050,27.15


Modality redundancy volume summary ready.


Findings\. This table compares redundancy volume across the five assembled modeling views\. Focus on the pair percentages rather than only the raw pair counts: the multimodal view will have the most pairs because it has the most features, but the percentage columns show whether redundancy is unusually dense or mostly a function of scale\. Runtime also belongs in the interpretation here, because the larger multimodal view is the one that makes caching and per\-view reuse most important\.

The pair counts tell us how much overlap exists, but feature flags tell us how much of each modeling view is affected\. This table summarizes how many features in each view have no high\-correlation flag, standard review flags, or strict review flags\.

In [51]:
# ---------------------------------------------------------------------
# Compare redundancy flags by modeling view
# ---------------------------------------------------------------------

modality_redundancy_flag_summary_df = (
    modality_redundancy_feature_flags_df
    .groupby(["modeling_view", "redundancy_review_flag"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
)

modality_redundancy_flag_summary_df["view_total_features"] = (
    modality_redundancy_flag_summary_df
    .groupby("modeling_view")["feature_count"]
    .transform("sum")
)

modality_redundancy_flag_summary_df["feature_pct_within_view"] = (
    modality_redundancy_flag_summary_df["feature_count"]
    / modality_redundancy_flag_summary_df["view_total_features"]
).round(4)

modality_redundancy_flag_pivot_df = (
    modality_redundancy_flag_summary_df
    .pivot_table(
        index="modeling_view",
        columns="redundancy_review_flag",
        values="feature_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for flag in [
    "no_high_correlation_flag",
    "within_view_redundancy_review",
    "strict_within_view_redundancy_review",
]:
    if flag not in modality_redundancy_flag_pivot_df.columns:
        modality_redundancy_flag_pivot_df[flag] = 0

modality_redundancy_flag_pivot_df["total_flagged_features"] = (
    modality_redundancy_flag_pivot_df["within_view_redundancy_review"]
    + modality_redundancy_flag_pivot_df["strict_within_view_redundancy_review"]
)

modality_redundancy_flag_pivot_df["total_features"] = (
    modality_redundancy_flag_pivot_df[
        [
            "no_high_correlation_flag",
            "within_view_redundancy_review",
            "strict_within_view_redundancy_review",
        ]
    ]
    .sum(axis=1)
)

modality_redundancy_flag_pivot_df["flagged_feature_pct"] = (
    modality_redundancy_flag_pivot_df["total_flagged_features"]
    / modality_redundancy_flag_pivot_df["total_features"]
).round(4)

modality_redundancy_flag_pivot_df = (
    modality_redundancy_flag_pivot_df
    [
        [
            "modeling_view",
            "total_features",
            "no_high_correlation_flag",
            "within_view_redundancy_review",
            "strict_within_view_redundancy_review",
            "total_flagged_features",
            "flagged_feature_pct",
        ]
    ]
    .sort_values("flagged_feature_pct", ascending=False)
    .reset_index(drop=True)
)

display(modality_redundancy_flag_pivot_df)

print("Modality redundancy flag summary ready.")

redundancy_review_flag,modeling_view,total_features,no_high_correlation_flag,within_view_redundancy_review,strict_within_view_redundancy_review,total_flagged_features,flagged_feature_pct
0,bus,64,18,8,38,46,0.7188
1,fhvhv,92,34,12,46,58,0.6304
2,multimodal,307,123,32,152,184,0.5993
3,taxi,92,40,7,45,52,0.5652
4,subway,39,21,3,15,18,0.4615


Modality redundancy flag summary ready.


Findings\. This table shifts the question from pair counts to affected features\. The useful column is \`flagged\_feature\_pct\`: it tells us how much of each modeling view would require review if we used within\-view correlation as the screen\. Read this as review workload, not as a drop rate\. A heavily flagged view may still retain many of those features after we choose representative variants and preserve modality coverage\.

Now we identify which feature families are driving redundancy inside each modeling view\. This keeps the interpretation practical: if a view is review\-heavy, we need to know whether that pressure comes mostly from temporal\-history features, spatial context, anomaly severity, or something else\.

In [52]:
# ---------------------------------------------------------------------
# Identify dominant redundant feature families by modeling view
# ---------------------------------------------------------------------

modality_redundancy_family_pressure_df = (
    modality_redundancy_feature_flags_df
    .groupby(["modeling_view", "feature_family"], as_index=False)
    .agg(
        features=("feature_name", "count"),
        high_or_strict_review_features=(
            "redundancy_review_flag",
            lambda values: int((values != "no_high_correlation_flag").sum()),
        ),
        strict_review_features=(
            "redundancy_review_flag",
            lambda values: int((values == "strict_within_view_redundancy_review").sum()),
        ),
    )
    .assign(
        review_feature_pct=lambda df: (
            df["high_or_strict_review_features"] / df["features"]
        ).round(4),
        strict_review_feature_pct=lambda df: (
            df["strict_review_features"] / df["features"]
        ).round(4),
    )
    .sort_values(
        ["modeling_view", "high_or_strict_review_features", "strict_review_features"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

dominant_redundant_family_by_view_df = (
    modality_redundancy_family_pressure_df
    .sort_values(
        ["modeling_view", "high_or_strict_review_features", "strict_review_features"],
        ascending=[True, False, False],
    )
    .groupby("modeling_view", as_index=False)
    .head(3)
    .reset_index(drop=True)
)

display(dominant_redundant_family_by_view_df)

print("Dominant redundant feature families by modeling view ready.")

,modeling_view,feature_family,features,high_or_strict_review_features,strict_review_features,review_feature_pct,strict_review_feature_pct
0,bus,temporal_history,30,27,23,0.9000,0.7667
1,bus,spatial_context,11,6,4,0.5455,0.3636
2,bus,anomaly_severity,6,4,4,0.6667,0.6667
3,fhvhv,temporal_history,50,38,33,0.7600,0.6600
4,fhvhv,spatial_context,17,7,4,0.4118,0.2353
5,fhvhv,raw_metric,5,5,3,1.0000,0.6000
6,multimodal,temporal_history,150,110,93,0.7333,0.6200
7,multimodal,spatial_context,52,26,19,0.5000,0.3654
8,multimodal,anomaly_severity,21,14,14,0.6667,0.6667
9,subway,temporal_history,20,11,8,0.5500,0.4000


Dominant redundant feature families by modeling view ready.


Findings\. This table explains what is driving the review burden inside each modeling view\. The rows to focus on are the top families within each view, especially where both \`high\_or\_strict\_review\_features\` and \`strict\_review\_features\` are large\. This is where the section becomes actionable: if temporal\-history dominates the flagged features, the selection work should focus on choosing a consistent temporal template; if spatial context or anomaly severity appears, those families need their own representative\-feature review rather than being treated as a temporal\-feature problem\.

### Summarize Modality\-Specific Redundancy Analysis

This final subsection collects the most inspection\-worthy feature pairs and visualizes redundancy buckets by modeling view\. The goal is to make the next step easier: choosing representative features without losing interpretability, modality coverage, or consistent templates across transportation systems\.

Let's pull the highest\-correlation examples into one compact table\. These are not automatic drops\. They are the pairs a reviewer should inspect first when deciding which feature variants are representative and which ones are redundant\.

In [53]:
# ---------------------------------------------------------------------
# Display top high-correlation pairs by modeling view
# ---------------------------------------------------------------------

top_modality_redundancy_pairs_df = (
    modality_redundancy_pairs_df
    .sort_values(["modeling_view", "abs_spearman_corr"], ascending=[True, False])
    .groupby("modeling_view", as_index=False)
    .head(10)
    .reset_index(drop=True)
    [
        [
            "modeling_view",
            "feature_1",
            "feature_2",
            "spearman_corr",
            "abs_spearman_corr",
            "strict_high_correlation_flag",
        ]
    ]
)

display(top_modality_redundancy_pairs_df)

print("Top modality-specific high-correlation pairs ready.")

,modeling_view,feature_1,feature_2,spearman_corr,abs_spearman_corr,strict_high_correlation_flag
0,bus,connected_mean_bus_trip_count,connected_mean_bus_trip_count_zscore,1.000000,1.000000,True
1,bus,bus_trip_count_zscore,bus_trip_count,1.000000,1.000000,True
2,bus,connected_mean_avg_bus_speed,connected_mean_avg_bus_speed_zscore,1.000000,1.000000,True
3,bus,avg_bus_speed_zscore,avg_bus_speed,1.000000,1.000000,True
4,bus,avg_bus_speed_ema_7_same_bucket,avg_bus_speed_rolling_mean_7_same_bucket,0.999895,0.999895,True
5,bus,avg_bus_travel_time_ema_7_same_bucket,avg_bus_travel_time_rolling_mean_7_same_bucket,0.999852,0.999852,True
6,bus,bus_trip_count_ema_7_same_bucket,bus_trip_count_rolling_mean_7_same_bucket,0.999788,0.999788,True
7,bus,avg_bus_speed_zscore,avg_bus_speed_ema_7_same_bucket,0.994665,0.994665,True
8,bus,avg_bus_speed,avg_bus_speed_ema_7_same_bucket,0.994665,0.994665,True
9,bus,bus_trip_count,bus_trip_count_ema_7_same_bucket,0.994220,0.994220,True


Top modality-specific high-correlation pairs ready.


Findings\. The top\-pair table is the inspection list, not the decision list\. These are the feature pairs to look at first when deciding which variants are redundant and which one is the better representative\. The most useful cases are pairs that tell the same story with different engineering choices, such as lag versus rolling mean, z\-score versus percentile rank, or local value versus connected\-zone summary\. Those are the places where the final scorecard should balance signal, interpretability, and cross\-modality consistency\.

Let’s close the modality\-specific redundancy section with an interactive treemap\. Each box represents a modeling\-view × feature\-family × review\-bucket group, and the hover text lists the features inside that bucket\. This makes the chart useful for checking whether likely\-retain features still cover a broad set of signals within each modeling view\.

In [54]:
# ---------------------------------------------------------------------
# Visualize modality-specific redundancy review buckets
# ---------------------------------------------------------------------

def build_modality_hover_feature_list(values, max_features=70):
    """Create a readable hover list of features inside each treemap bucket."""
    feature_names = sorted(values.dropna().astype(str).unique())
    shown_features = feature_names[:max_features]
    remaining_count = max(len(feature_names) - max_features, 0)

    hover_text = "<br>".join(shown_features)

    if remaining_count > 0:
        hover_text += f"<br>...and {remaining_count:,} more"

    return hover_text


modality_redundancy_flag_labels = {
    "no_high_correlation_flag": "No high-correlation flag",
    "within_view_redundancy_review": "Review",
    "strict_within_view_redundancy_review": "Strict review",
}

modality_redundancy_flag_color_map = {
    "No high-correlation flag": "#2ca25f",
    "Review": "#fdae61",
    "Strict review": "#d73027",
}

modality_redundancy_treemap_df = (
    modality_redundancy_feature_flags_df
    .copy()
    .assign(
        review_bucket=lambda df: (
            df["redundancy_review_flag"]
            .map(modality_redundancy_flag_labels)
            .fillna(df["redundancy_review_flag"])
        )
    )
    .groupby(
        [
            "modeling_view",
            "feature_family",
            "review_bucket",
            "modality",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        feature_count=("feature_name", "count"),
        high_correlation_pair_count=("high_correlation_pair_count", "sum"),
        strict_high_correlation_pair_count=("strict_high_correlation_pair_count", "sum"),
        feature_list=("feature_name", build_modality_hover_feature_list),
    )
)

modality_redundancy_treemap_df["modality"] = (
    modality_redundancy_treemap_df["modality"]
    .fillna("unknown")
)

if px is None:
    print("Plotly is not available, so displaying the treemap source table instead.")
    display(modality_redundancy_treemap_df)

else:
    fig = px.treemap(
        modality_redundancy_treemap_df,
        path=[
            "modeling_view",
            "feature_family",
            "review_bucket",
            "modality",
        ],
        values="feature_count",
        color="review_bucket",
        color_discrete_map=modality_redundancy_flag_color_map,
        custom_data=[
            "feature_count",
            "high_correlation_pair_count",
            "strict_high_correlation_pair_count",
            "feature_list",
        ],
        title="Modality-specific redundancy review buckets",
    )

    fig.update_traces(
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Features: %{customdata[0]:,}<br>"
            "High-correlation pair flags: %{customdata[1]:,}<br>"
            "Strict pair flags: %{customdata[2]:,}<br><br>"
            "<b>Features in this bucket</b><br>%{customdata[3]}"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        margin=dict(t=60, l=10, r=10, b=10),
        height=750,
    )

    fig.show()

print("Modality-specific redundancy treemap ready.")

Modality-specific redundancy treemap ready.


Findings\. The treemap is most useful as an inspection tool, not a final decision table\. Start at the modeling\-view level, then drill into feature families and review buckets\. Green areas show features with no high\-correlation flag, while amber and red areas show where representative\-feature review is needed\. The hover lists are the important detail: they let us check whether the likely\-retain buckets still cover the major signal types inside each modeling view, and they show where flagged features are clustered tightly enough to need scorecard review in 1\.5\.6\.5\.

### Section Summary

This section translated the global candidate feature inventory into bus, subway, taxi, FHVHV, and multimodal modeling views, then reran redundancy checks inside those assembled views\. The results show that redundancy is not just a global feature\-family issue; it also varies by how the downstream modeling sets are constructed\. These outputs give us the evidence needed to make modality\-aware pruning decisions later, while still preserving the option to favor consistent feature templates across transportation systems when the signal tradeoff is small\.

## 1\.5\.6\.4 Evaluate Incremental Feature Family Contribution

Redundancy analysis tells us where engineered features overlap, but it does not tell us whether a feature family adds useful structure to the modeling space\. A family can contain correlated features and still be worth keeping if it captures a signal that no other family captures well\. This section evaluates whether each major feature family contributes meaningful information after the clearest redundancy issues have been identified\.

We keep raw mobility metrics as the baseline, then test the engineered families before locking in a nested order\. A fixed order can make the first added family look artificially strong, so this section starts with a preliminary contribution screen: add each candidate family to the raw baseline one at a time, compare the PCA compression change, then use that evidence to choose a more defensible order for the nested contribution analysis\.

The goal is not to choose a dimensionality\-reduction method here\. PCA is only a diagnostic tool for asking whether a feature family makes the modeling space meaningfully richer or mostly repeats structure we already have\. These results help us avoid carrying feature families forward just because they exist, while also avoiding over\-pruning families that add real downstream signal\.

### Run Preliminary Family Contribution Screen

Let’s first compare each engineered feature family against the same raw\-metric baseline\. This keeps the test fair: every family gets evaluated as raw metrics \+ one engineered family, instead of benefiting from whatever order we happened to add it in\. The goal here is not to choose the final order yet\. It is just to identify which families appear to add the most structure when tested from the same starting point\.

Let’s rebuild the contribution feature pool directly from the redundancy decision table\. This gives the preliminary screen a clean set of candidate features without depending on the old fixed\-order PCA section we deleted\. The pool keeps unflagged features by default and allows a small number of flagged representatives so highly correlated families are still represented without letting redundant variants dominate the PCA screen\.

In [55]:
# ---------------------------------------------------------------------
# Build preliminary contribution feature pool
# ---------------------------------------------------------------------

PRELIMINARY_CONTRIBUTION_FAMILY_ORDER = [
    "raw_metric",
    "temporal_history",
    "spatial_context",
    "multimodal_interaction",
    "stl_decomposition",
    "fourier20_residual_reconstruction",
    "anomaly_severity",
]

MAX_FLAGGED_REPRESENTATIVES_PER_GROUP = 2

required_preliminary_source_objects = [
    "redundancy_decision_input_df",
    "modeling_diagnostic_sample_df",
]

missing_preliminary_source_objects = [
    object_name
    for object_name in required_preliminary_source_objects
    if object_name not in globals()
]

assert not missing_preliminary_source_objects, (
    "Run the redundancy decision and diagnostic sample blocks before this screen. "
    f"Missing objects: {missing_preliminary_source_objects}"
)

preliminary_contribution_candidate_df = redundancy_decision_input_df.copy()

# Some upstream tables are already candidate-only and do not carry is_modeling_candidate.
# Treat missing optional filters as already satisfied instead of failing on absent columns.
preliminary_filter_mask = pd.Series(
    True,
    index=preliminary_contribution_candidate_df.index,
)

if "is_modeling_candidate" in preliminary_contribution_candidate_df.columns:
    preliminary_filter_mask &= preliminary_contribution_candidate_df["is_modeling_candidate"].fillna(False)

if "is_numeric" in preliminary_contribution_candidate_df.columns:
    preliminary_filter_mask &= preliminary_contribution_candidate_df["is_numeric"].fillna(False)

if "feature_family" in preliminary_contribution_candidate_df.columns:
    preliminary_filter_mask &= preliminary_contribution_candidate_df["feature_family"].isin(
        PRELIMINARY_CONTRIBUTION_FAMILY_ORDER
    )

if "modality" in preliminary_contribution_candidate_df.columns:
    preliminary_filter_mask &= ~preliminary_contribution_candidate_df["modality"].eq("traffic")

if "near_zero_variance_flag" in preliminary_contribution_candidate_df.columns:
    preliminary_filter_mask &= ~preliminary_contribution_candidate_df["near_zero_variance_flag"].fillna(False)

if "non_null_pct" in preliminary_contribution_candidate_df.columns:
    preliminary_filter_mask &= preliminary_contribution_candidate_df["non_null_pct"].fillna(0).ge(
        MIN_FEATURE_NON_NULL_PCT
    )

preliminary_contribution_candidate_df = (
    preliminary_contribution_candidate_df[preliminary_filter_mask]
    .copy()
    .reset_index(drop=True)
)

required_preliminary_columns = [
    "feature_name",
    "feature_family",
    "modality",
]

missing_preliminary_columns = [
    column
    for column in required_preliminary_columns
    if column not in preliminary_contribution_candidate_df.columns
]

assert not missing_preliminary_columns, (
    "Preliminary contribution pool is missing required columns: "
    f"{missing_preliminary_columns}"
)

if "source_metric" not in preliminary_contribution_candidate_df.columns:
    preliminary_contribution_candidate_df["source_metric"] = "not_applicable"
else:
    preliminary_contribution_candidate_df["source_metric"] = (
        preliminary_contribution_candidate_df["source_metric"]
        .fillna("not_applicable")
    )

for count_column in [
    "high_correlation_pair_count",
    "strict_high_correlation_pair_count",
    "null_pct",
]:
    if count_column not in preliminary_contribution_candidate_df.columns:
        preliminary_contribution_candidate_df[count_column] = 0

if "is_core_metric_descendant" not in preliminary_contribution_candidate_df.columns:
    preliminary_contribution_candidate_df["is_core_metric_descendant"] = False

if "non_null_pct" not in preliminary_contribution_candidate_df.columns:
    preliminary_contribution_candidate_df["non_null_pct"] = 1.0

if "redundancy_review_flag" not in preliminary_contribution_candidate_df.columns:
    preliminary_contribution_candidate_df["redundancy_review_flag"] = np.select(
        [
            preliminary_contribution_candidate_df["strict_high_correlation_pair_count"].fillna(0).gt(0),
            preliminary_contribution_candidate_df["high_correlation_pair_count"].fillna(0).gt(0),
        ],
        [
            "strict_within_family_review",
            "high_within_family_review",
        ],
        default="no_high_correlation_flag",
    )

preliminary_contribution_candidate_df["redundancy_review_flag"] = (
    preliminary_contribution_candidate_df["redundancy_review_flag"]
    .fillna("no_high_correlation_flag")
)

preliminary_contribution_candidate_df["representative_group_key"] = (
    preliminary_contribution_candidate_df["feature_family"].astype(str)
    + "||"
    + preliminary_contribution_candidate_df["modality"].astype(str)
    + "||"
    + preliminary_contribution_candidate_df["source_metric"].astype(str)
)

preliminary_contribution_candidate_df["redundancy_priority_score"] = (
    preliminary_contribution_candidate_df["strict_high_correlation_pair_count"].fillna(0) * 10
    + preliminary_contribution_candidate_df["high_correlation_pair_count"].fillna(0)
    + preliminary_contribution_candidate_df["null_pct"].fillna(0)
)

clean_preliminary_contribution_features_df = (
    preliminary_contribution_candidate_df[
        preliminary_contribution_candidate_df["redundancy_review_flag"].eq("no_high_correlation_flag")
    ]
    .copy()
)

flagged_preliminary_representative_features_df = (
    preliminary_contribution_candidate_df[
        ~preliminary_contribution_candidate_df["redundancy_review_flag"].eq("no_high_correlation_flag")
    ]
    .sort_values(
        [
            "representative_group_key",
            "redundancy_priority_score",
            "is_core_metric_descendant",
            "non_null_pct",
            "feature_name",
        ],
        ascending=[True, True, False, False, True],
    )
    .groupby("representative_group_key", as_index=False)
    .head(MAX_FLAGGED_REPRESENTATIVES_PER_GROUP)
    .copy()
)

preliminary_contribution_feature_pool_df = (
    pd.concat(
        [
            clean_preliminary_contribution_features_df.assign(
                contribution_pool_reason="no_high_correlation_flag"
            ),
            flagged_preliminary_representative_features_df.assign(
                contribution_pool_reason="flagged_representative"
            ),
        ],
        ignore_index=True,
    )
    .drop_duplicates("feature_name")
    .sort_values(["feature_family", "modality", "source_metric", "feature_name"])
    .reset_index(drop=True)
)

preliminary_contribution_feature_pool_summary_df = (
    preliminary_contribution_feature_pool_df
    .groupby(["feature_family", "contribution_pool_reason"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
    .pivot_table(
        index="feature_family",
        columns="contribution_pool_reason",
        values="feature_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for column in ["no_high_correlation_flag", "flagged_representative"]:
    if column not in preliminary_contribution_feature_pool_summary_df.columns:
        preliminary_contribution_feature_pool_summary_df[column] = 0

preliminary_contribution_feature_pool_summary_df["total_diagnostic_features"] = (
    preliminary_contribution_feature_pool_summary_df["no_high_correlation_flag"]
    + preliminary_contribution_feature_pool_summary_df["flagged_representative"]
)

preliminary_contribution_feature_pool_summary_df["feature_family"] = pd.Categorical(
    preliminary_contribution_feature_pool_summary_df["feature_family"],
    categories=PRELIMINARY_CONTRIBUTION_FAMILY_ORDER,
    ordered=True,
)

preliminary_contribution_feature_pool_summary_df = (
    preliminary_contribution_feature_pool_summary_df[
        [
            "feature_family",
            "total_diagnostic_features",
            "no_high_correlation_flag",
            "flagged_representative",
        ]
    ]
    .sort_values("feature_family")
    .reset_index(drop=True)
)

preliminary_contribution_feature_pool_summary_df["feature_family"] = (
    preliminary_contribution_feature_pool_summary_df["feature_family"].astype(str)
)

display(preliminary_contribution_feature_pool_summary_df)

assert not preliminary_contribution_feature_pool_df.empty, (
    "Preliminary contribution feature pool is empty."
)

assert "raw_metric" in set(preliminary_contribution_feature_pool_df["feature_family"]), (
    "Raw metric baseline is missing from the preliminary contribution feature pool."
)

print(
    "Preliminary contribution feature pool ready. "
    f"Diagnostic features: {len(preliminary_contribution_feature_pool_df):,}."
)

contribution_pool_reason,feature_family,total_diagnostic_features,no_high_correlation_flag,flagged_representative
0,raw_metric,15,11,4
1,temporal_history,70,40,30
2,spatial_context,51,29,22
3,multimodal_interaction,34,22,12
4,stl_decomposition,28,20,8
5,fourier20_residual_reconstruction,7,7,0
6,anomaly_severity,21,7,14


Preliminary contribution feature pool ready. Diagnostic features: 226.


Findings\. The diagnostic pool keeps 226 features for the preliminary contribution screen\. Temporal history is the largest family with 70 features, followed by spatial context with 51, multimodal interaction with 34, STL decomposition with 28, anomaly severity with 21, raw metrics with 15, and Fourier Top 20 with 7\. The useful thing to notice is the balance: the pool is broad enough to represent every engineered family, but trimmed enough that the next PCA screen is comparing signal structure rather than raw feature volume\.

This block defines the candidate family tests for the preliminary screen\. Raw metrics are the baseline, and every other feature family is tested one at a time against that baseline\.

In [56]:
# ---------------------------------------------------------------------
# Configure preliminary family contribution screen
# ---------------------------------------------------------------------

PRELIMINARY_FAMILY_CONTRIBUTION_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "preliminary_family_contribution_screen_v2.csv"
)

RAW_CONTRIBUTION_FAMILY = "raw_metric"

preliminary_candidate_families = [
    family
    for family in PRELIMINARY_CONTRIBUTION_FAMILY_ORDER
    if (
        family != RAW_CONTRIBUTION_FAMILY
        and family in set(preliminary_contribution_feature_pool_df["feature_family"])
    )
]

preliminary_family_screen_steps = [
    {
        "contribution_step": "raw_only",
        "added_family_label": "raw_metric_base",
        "included_families": [RAW_CONTRIBUTION_FAMILY],
    }
]

preliminary_family_screen_steps.extend(
    [
        {
            "contribution_step": f"raw_plus_{family}",
            "added_family_label": family,
            "included_families": [RAW_CONTRIBUTION_FAMILY, family],
        }
        for family in preliminary_candidate_families
    ]
)

preliminary_family_screen_config_df = pd.DataFrame(
    [
        {
            "contribution_step": step["contribution_step"],
            "added_family_label": step["added_family_label"],
            "included_families": ", ".join(step["included_families"]),
        }
        for step in preliminary_family_screen_steps
    ]
)

display(preliminary_family_screen_config_df)

assert RAW_CONTRIBUTION_FAMILY in set(preliminary_contribution_feature_pool_df["feature_family"]), (
    "Raw metric family is missing from the preliminary contribution feature pool."
)

assert len(preliminary_candidate_families) > 0, (
    "No engineered feature families found for preliminary contribution screening."
)

print(
    "Preliminary family contribution screen configured. "
    f"Candidate engineered families: {len(preliminary_candidate_families):,}."
)

,contribution_step,added_family_label,included_families
0,raw_only,raw_metric_base,raw_metric
1,raw_plus_temporal_history,temporal_history,"raw_metric, temporal_history"
2,raw_plus_spatial_context,spatial_context,"raw_metric, spatial_context"
3,raw_plus_multimodal_interaction,multimodal_interaction,"raw_metric, multimodal_interaction"
4,raw_plus_stl_decomposition,stl_decomposition,"raw_metric, stl_decomposition"
5,raw_plus_fourier20_residual_reconstruction,fourier20_residual_reconstruction,"raw_metric, fourier20_residual_reconstruction"
6,raw_plus_anomaly_severity,anomaly_severity,"raw_metric, anomaly_severity"


Preliminary family contribution screen configured. Candidate engineered families: 6.


Findings\. The screen is set up correctly: raw metrics are the baseline, and each engineered family is tested one at a time against that same raw baseline\. This avoids the path\-dependence problem from the earlier fixed\-order approach, where a family could look stronger or weaker just because it happened to be added earlier or later\.

Now we run the actual PCA screen\. Each row in the output is one test: raw metrics alone, then raw metrics plus one engineered family\. If the cached result already exists, this block loads it instead of rerunning\.

In [57]:
# ---------------------------------------------------------------------
# Run preliminary raw-plus-one-family PCA screen
# ---------------------------------------------------------------------

if PRELIMINARY_FAMILY_CONTRIBUTION_PATH.exists() and not REBUILD_INCREMENTAL_CONTRIBUTION_ANALYSIS:
    preliminary_family_contribution_screen_df = pd.read_csv(
        PRELIMINARY_FAMILY_CONTRIBUTION_PATH
    )

    print(
        "Loaded cached preliminary family contribution screen: "
        f"{PRELIMINARY_FAMILY_CONTRIBUTION_PATH}"
    )

else:
    preliminary_family_results = []
    iterator = (
        tqdm(preliminary_family_screen_steps, desc="Screening feature families")
        if tqdm
        else preliminary_family_screen_steps
    )

    for step in iterator:
        start_time = time.perf_counter()

        contribution_step = step["contribution_step"]
        added_family_label = step["added_family_label"]
        included_families = step["included_families"]

        selected_feature_cols = (
            preliminary_contribution_feature_pool_df[
                preliminary_contribution_feature_pool_df["feature_family"].isin(included_families)
            ]["feature_name"]
            .drop_duplicates()
            .tolist()
        )

        selected_feature_cols = [
            column
            for column in selected_feature_cols
            if column in modeling_diagnostic_sample_df.columns
        ]

        assert selected_feature_cols, (
            f"No feature columns found for contribution step: {contribution_step}"
        )

        pca_input_df = (
            modeling_diagnostic_sample_df[selected_feature_cols]
            .replace([np.inf, -np.inf], np.nan)
            .copy()
        )

        # Fully-null columns cannot be median-imputed, so remove them before PCA.
        usable_feature_cols = pca_input_df.columns[pca_input_df.notna().any()].tolist()
        pca_input_df = pca_input_df[usable_feature_cols]

        max_components = min(
            PCA_MAX_COMPONENTS,
            len(usable_feature_cols),
            len(pca_input_df) - 1,
        )

        assert max_components >= 1, (
            f"No PCA components available for contribution step: {contribution_step}"
        )

        imputed_values = SimpleImputer(strategy="median").fit_transform(pca_input_df)
        scaled_values = StandardScaler().fit_transform(imputed_values)

        pca_model = PCA(
            n_components=max_components,
            random_state=PCA_RANDOM_STATE,
        )

        pca_model.fit(scaled_values)

        explained_variance = pca_model.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)

        result_row = {
            "contribution_step": contribution_step,
            "added_family_label": added_family_label,
            "included_families": ", ".join(included_families),
            "feature_count": len(usable_feature_cols),
            "rows_used": len(pca_input_df),
            "max_components_evaluated": max_components,
            "runtime_seconds": time.perf_counter() - start_time,
        }

        for variance_target in PCA_VARIANCE_TARGETS:
            target_label = int(variance_target * 100)
            reached_target = np.where(cumulative_variance >= variance_target)[0]

            result_row[f"components_needed_for_{target_label}_pct_variance"] = (
                int(reached_target[0] + 1)
                if len(reached_target) > 0
                else np.nan
            )

        for component_count in PCA_TOP_COMPONENT_COUNTS:
            result_row[f"variance_explained_first_{component_count}_components"] = (
                cumulative_variance[min(component_count, len(cumulative_variance)) - 1]
            )

        preliminary_family_results.append(result_row)

        del pca_input_df, imputed_values, scaled_values, pca_model
        gc.collect()

    preliminary_family_contribution_screen_df = pd.DataFrame(preliminary_family_results)

    preliminary_family_contribution_screen_df.to_csv(
        PRELIMINARY_FAMILY_CONTRIBUTION_PATH,
        index=False,
    )

    print(
        "Saved preliminary family contribution screen: "
        f"{PRELIMINARY_FAMILY_CONTRIBUTION_PATH}"
    )

print(
    "Preliminary family contribution screen complete. "
    f"Rows: {len(preliminary_family_contribution_screen_df):,}."
)

Loaded cached preliminary family contribution screen: pipeline_data/1.5.6.intermediate_tables/preliminary_family_contribution_screen_v2.csv
Preliminary family contribution screen complete. Rows: 7.


Findings\. The preliminary PCA screen completed successfully across all seven configured tests\. This block is mostly a generation step, so the important thing is that the results were created and cached; the actual interpretation comes in the summary table below\.

This block compares every raw\-plus\-family result back to the raw\-only baseline\. The main fields to inspect are the changes in first\-10 and first\-20 component variance: more negative values mean the added family made the feature space less compressible\.

In [58]:
# ---------------------------------------------------------------------
# Summarize preliminary family contribution screen
# ---------------------------------------------------------------------

raw_screen_row = preliminary_family_contribution_screen_df[
    preliminary_family_contribution_screen_df["added_family_label"].eq("raw_metric_base")
].iloc[0]

preliminary_family_contribution_summary_df = (
    preliminary_family_contribution_screen_df
    .copy()
)

for component_count in PCA_TOP_COMPONENT_COUNTS:
    comparison_column = f"variance_explained_first_{component_count}_components"
    change_column = f"first_{component_count}_component_variance_change_vs_raw"

    preliminary_family_contribution_summary_df[change_column] = (
        preliminary_family_contribution_summary_df[comparison_column]
        - raw_screen_row[comparison_column]
    )

preliminary_family_contribution_summary_df[
    "components_needed_for_80_pct_variance_change_vs_raw"
] = (
    preliminary_family_contribution_summary_df["components_needed_for_80_pct_variance"]
    - raw_screen_row["components_needed_for_80_pct_variance"]
)

preliminary_family_contribution_summary_df["preliminary_contribution_signal"] = np.select(
    [
        preliminary_family_contribution_summary_df["added_family_label"].eq("raw_metric_base"),
        preliminary_family_contribution_summary_df[
            "first_20_component_variance_change_vs_raw"
        ].le(-0.10),
        preliminary_family_contribution_summary_df[
            "first_20_component_variance_change_vs_raw"
        ].le(-0.03),
    ],
    [
        "baseline",
        "strong_added_structure",
        "moderate_added_structure",
    ],
    default="limited_added_structure",
)

preliminary_family_contribution_display_df = (
    preliminary_family_contribution_summary_df
    .copy()
)

preliminary_family_contribution_display_df["added_family_label"] = pd.Categorical(
    preliminary_family_contribution_display_df["added_family_label"],
    categories=["raw_metric_base"] + PRELIMINARY_CONTRIBUTION_FAMILY_ORDER,
    ordered=True,
)

preliminary_family_contribution_display_df = (
    preliminary_family_contribution_display_df
    .sort_values("added_family_label")
    [
        [
            "added_family_label",
            "included_families",
            "feature_count",
            "components_needed_for_80_pct_variance",
            "components_needed_for_80_pct_variance_change_vs_raw",
            "variance_explained_first_10_components",
            "first_10_component_variance_change_vs_raw",
            "variance_explained_first_20_components",
            "first_20_component_variance_change_vs_raw",
            "preliminary_contribution_signal",
            "runtime_seconds",
        ]
    ]
    .reset_index(drop=True)
)

preliminary_family_contribution_display_df["added_family_label"] = (
    preliminary_family_contribution_display_df["added_family_label"].astype(str)
)

display(preliminary_family_contribution_display_df)

print("Preliminary family contribution summary ready.")

,added_family_label,included_families,feature_count,components_needed_for_80_pct_variance,components_needed_for_80_pct_variance_change_vs_raw,variance_explained_first_10_components,first_10_component_variance_change_vs_raw,variance_explained_first_20_components,first_20_component_variance_change_vs_raw,preliminary_contribution_signal,runtime_seconds
0,raw_metric_base,raw_metric,15,7,0,0.937727,0.000000,1.000000,0.000000,baseline,0.324838
1,temporal_history,"raw_metric, temporal_history",85,28,21,0.508012,-0.429715,0.702519,-0.297481,strong_added_structure,2.299223
2,spatial_context,"raw_metric, spatial_context",66,16,9,0.705103,-0.232624,0.858451,-0.141549,strong_added_structure,1.858169
3,multimodal_interaction,"raw_metric, multimodal_interaction",49,12,5,0.776468,-0.161260,0.929829,-0.070171,moderate_added_structure,1.323156
4,stl_decomposition,"raw_metric, stl_decomposition",43,21,14,0.558851,-0.378876,0.798238,-0.201762,strong_added_structure,1.167616
5,fourier20_residual_reconstruction,"raw_metric, fourier20_residual_reconstruction",22,12,5,0.757853,-0.179874,0.992019,-0.007981,limited_added_structure,0.554066
6,anomaly_severity,"raw_metric, anomaly_severity",36,15,8,0.661590,-0.276137,0.912845,-0.087155,moderate_added_structure,0.996769


Preliminary family contribution summary ready.


Findings\. This table compares each feature family against the same raw\-metric baseline\. Raw metrics alone are highly compressible: the first 10 PCA components explain 93\.8% of variance, and the first 20 explain 100\.0%\. Temporal history and STL decomposition add the strongest new structure: temporal history drops first\-20\-component variance by 29\.7 points, and STL drops it by 20\.2 points despite using fewer diagnostic features\. Spatial context also adds clear structure, with a 14\.2\-point first\-20 drop\. Multimodal interaction and anomaly severity add moderate structure\. Fourier Top 20 adds less independent structure in this screen because raw \+ Fourier remains highly compressible, with the first 20 components still explaining 99\.2% of variance

This QA block keeps the screen honest: it checks that the raw baseline exists, every configured family test produced a row, and the PCA variance values are finite\.

In [59]:
# ---------------------------------------------------------------------
# Validate preliminary family contribution screen
# ---------------------------------------------------------------------

expected_preliminary_steps = set(
    preliminary_family_screen_config_df["contribution_step"]
)

observed_preliminary_steps = set(
    preliminary_family_contribution_summary_df["contribution_step"]
)

missing_preliminary_steps = sorted(
    expected_preliminary_steps - observed_preliminary_steps
)

raw_baseline_present = preliminary_family_contribution_summary_df[
    "added_family_label"
].eq("raw_metric_base").any()

variance_values_finite = np.isfinite(
    preliminary_family_contribution_summary_df[
        "variance_explained_first_10_components"
    ]
).all()

runtime_values_valid = (
    preliminary_family_contribution_summary_df["runtime_seconds"].notna().all()
    and preliminary_family_contribution_summary_df["runtime_seconds"].ge(0).all()
)

preliminary_family_contribution_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "expected_screen_steps",
            "value": len(expected_preliminary_steps),
            "status": "info",
        },
        {
            "qa_check": "observed_screen_steps",
            "value": len(observed_preliminary_steps),
            "status": "pass" if not missing_preliminary_steps else "fail",
        },
        {
            "qa_check": "raw_baseline_present",
            "value": raw_baseline_present,
            "status": "pass" if raw_baseline_present else "fail",
        },
        {
            "qa_check": "variance_values_finite",
            "value": variance_values_finite,
            "status": "pass" if variance_values_finite else "fail",
        },
        {
            "qa_check": "runtime_values_valid",
            "value": runtime_values_valid,
            "status": "pass" if runtime_values_valid else "fail",
        },
    ]
)

display(preliminary_family_contribution_qa_df)

assert not missing_preliminary_steps, (
    f"Missing preliminary contribution screen steps: {missing_preliminary_steps}"
)

assert raw_baseline_present, (
    "Preliminary contribution screen must include the raw metric baseline."
)

assert variance_values_finite, (
    "Preliminary PCA screen contains non-finite variance values."
)

assert runtime_values_valid, (
    "Preliminary PCA screen contains invalid runtime values."
)

print("Preliminary family contribution screen validated.")

,qa_check,value,status
0,expected_screen_steps,7,info
1,observed_screen_steps,7,pass
2,raw_baseline_present,True,pass
3,variance_values_finite,True,pass
4,runtime_values_valid,True,pass


Preliminary family contribution screen validated.


Findings\. The QA checks pass: all seven expected screen steps are present, the raw baseline is included, the variance values are finite, and runtime values are valid\. That means the preliminary contribution screen is ready to feed into the greedy family\-order step\.

### Determine Greedy Feature Family Order

The preliminary screen tested each engineered family against raw metrics one at a time\. That was useful, but feature families do not enter the final modeling set one at a time\. They stack\. This step builds a greedy feature\-family order by asking a more practical question: given what we have already selected, which remaining family adds the most new structure next?

At each round, we test every remaining family against the current selected set, run the same PCA compression diagnostic, and choose the family that makes the feature space least compressible\. This does not make the final keep/drop decision yet\. It gives us an evidence\-based order for the nested contribution analysis that follows\.

Let’s define the helper logic for the greedy contribution screen\. Each candidate set gets cleaned, imputed, scaled, and passed through PCA the same way, so the family comparisons stay consistent\.

In [60]:
# ---------------------------------------------------------------------
# Define greedy PCA contribution helper
# ---------------------------------------------------------------------

GREEDY_FAMILY_ORDER_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "greedy_feature_family_order.csv"
)

GREEDY_FAMILY_ROUND_RESULTS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "greedy_feature_family_round_results.csv"
)


def run_greedy_pca_diagnostic(feature_cols, diagnostic_label):
    """Run the shared PCA compression diagnostic for one candidate feature set."""

    usable_feature_cols = [
        column
        for column in feature_cols
        if column in modeling_diagnostic_sample_df.columns
    ]

    assert usable_feature_cols, (
        f"No usable features found for greedy diagnostic: {diagnostic_label}"
    )

    pca_input_df = (
        modeling_diagnostic_sample_df[usable_feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .copy()
    )

    # Fully-null columns cannot be median-imputed, so remove them before PCA.
    usable_feature_cols = pca_input_df.columns[pca_input_df.notna().any()].tolist()
    pca_input_df = pca_input_df[usable_feature_cols]

    max_components = min(
        PCA_MAX_COMPONENTS,
        len(usable_feature_cols),
        len(pca_input_df) - 1,
    )

    assert max_components >= 1, (
        f"No PCA components available for greedy diagnostic: {diagnostic_label}"
    )

    start_time = time.perf_counter()

    imputed_values = SimpleImputer(strategy="median").fit_transform(pca_input_df)
    scaled_values = StandardScaler().fit_transform(imputed_values)

    pca_model = PCA(
        n_components=max_components,
        random_state=PCA_RANDOM_STATE,
    )

    pca_model.fit(scaled_values)

    explained_variance = pca_model.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)

    result_row = {
        "diagnostic_label": diagnostic_label,
        "feature_count": len(usable_feature_cols),
        "rows_used": len(pca_input_df),
        "max_components_evaluated": max_components,
        "runtime_seconds": time.perf_counter() - start_time,
    }

    for variance_target in PCA_VARIANCE_TARGETS:
        target_label = int(variance_target * 100)
        reached_target = np.where(cumulative_variance >= variance_target)[0]

        result_row[f"components_needed_for_{target_label}_pct_variance"] = (
            int(reached_target[0] + 1)
            if len(reached_target) > 0
            else np.nan
        )

    for component_count in PCA_TOP_COMPONENT_COUNTS:
        result_row[f"variance_explained_first_{component_count}_components"] = (
            cumulative_variance[min(component_count, len(cumulative_variance)) - 1]
        )

    del pca_input_df, imputed_values, scaled_values, pca_model
    gc.collect()

    return result_row


print("Greedy PCA contribution helper ready.")

Greedy PCA contribution helper ready.


Now we build the greedy order\. Raw metrics start as the baseline\. For each round, every remaining engineered family is tested against the families already selected, and the family with the strongest added structure is selected next\.

In [61]:
# ---------------------------------------------------------------------
# Run greedy feature-family order screen
# ---------------------------------------------------------------------

if (
    GREEDY_FAMILY_ORDER_PATH.exists()
    and GREEDY_FAMILY_ROUND_RESULTS_PATH.exists()
    and not REBUILD_INCREMENTAL_CONTRIBUTION_ANALYSIS
):
    greedy_family_order_df = pd.read_csv(GREEDY_FAMILY_ORDER_PATH)
    greedy_family_round_results_df = pd.read_csv(GREEDY_FAMILY_ROUND_RESULTS_PATH)

    print(
        "Loaded cached greedy feature-family order: "
        f"{GREEDY_FAMILY_ORDER_PATH}"
    )

else:
    selected_families = [RAW_CONTRIBUTION_FAMILY]
    remaining_families = [
        family
        for family in preliminary_candidate_families
        if family in set(preliminary_contribution_feature_pool_df["feature_family"])
    ]

    greedy_order_records = []
    greedy_round_records = []

    round_number = 0

    while remaining_families:
        round_number += 1

        current_feature_cols = (
            preliminary_contribution_feature_pool_df[
                preliminary_contribution_feature_pool_df["feature_family"].isin(selected_families)
            ]["feature_name"]
            .drop_duplicates()
            .tolist()
        )

        current_result = run_greedy_pca_diagnostic(
            feature_cols=current_feature_cols,
            diagnostic_label=f"round_{round_number}_current_baseline",
        )

        candidate_rows = []

        iterator = tqdm(
            remaining_families,
            desc=f"Greedy round {round_number}",
        ) if tqdm else remaining_families

        for candidate_family in iterator:
            candidate_families = selected_families + [candidate_family]

            candidate_feature_cols = (
                preliminary_contribution_feature_pool_df[
                    preliminary_contribution_feature_pool_df["feature_family"].isin(candidate_families)
                ]["feature_name"]
                .drop_duplicates()
                .tolist()
            )

            candidate_result = run_greedy_pca_diagnostic(
                feature_cols=candidate_feature_cols,
                diagnostic_label=f"round_{round_number}_plus_{candidate_family}",
            )

            candidate_row = {
                "round_number": round_number,
                "candidate_family": candidate_family,
                "selected_families_before_round": ", ".join(selected_families),
                "candidate_family_set": ", ".join(candidate_families),
                "baseline_feature_count": current_result["feature_count"],
                "candidate_feature_count": candidate_result["feature_count"],
                "added_feature_count": (
                    candidate_result["feature_count"]
                    - current_result["feature_count"]
                ),
                "baseline_variance_explained_first_10_components": current_result[
                    "variance_explained_first_10_components"
                ],
                "candidate_variance_explained_first_10_components": candidate_result[
                    "variance_explained_first_10_components"
                ],
                "first_10_component_variance_change": (
                    candidate_result["variance_explained_first_10_components"]
                    - current_result["variance_explained_first_10_components"]
                ),
                "baseline_variance_explained_first_20_components": current_result[
                    "variance_explained_first_20_components"
                ],
                "candidate_variance_explained_first_20_components": candidate_result[
                    "variance_explained_first_20_components"
                ],
                "first_20_component_variance_change": (
                    candidate_result["variance_explained_first_20_components"]
                    - current_result["variance_explained_first_20_components"]
                ),
                "baseline_components_needed_for_80_pct_variance": current_result[
                    "components_needed_for_80_pct_variance"
                ],
                "candidate_components_needed_for_80_pct_variance": candidate_result[
                    "components_needed_for_80_pct_variance"
                ],
                "components_needed_for_80_pct_variance_change": (
                    candidate_result["components_needed_for_80_pct_variance"]
                    - current_result["components_needed_for_80_pct_variance"]
                ),
                "candidate_runtime_seconds": candidate_result["runtime_seconds"],
            }

            candidate_rows.append(candidate_row)

        round_results_df = pd.DataFrame(candidate_rows)

        # More negative first-20 variance means PCA compresses less of the expanded
        # feature space, which we treat as stronger added structure.
        round_results_df = (
            round_results_df
            .sort_values(
                [
                    "first_20_component_variance_change",
                    "first_10_component_variance_change",
                    "components_needed_for_80_pct_variance_change",
                    "added_feature_count",
                ],
                ascending=[True, True, False, True],
            )
            .reset_index(drop=True)
        )

        selected_candidate = round_results_df.iloc[0]["candidate_family"]

        round_results_df["selected_this_round"] = (
            round_results_df["candidate_family"].eq(selected_candidate)
        )

        greedy_round_records.append(round_results_df)

        selected_row = round_results_df.iloc[0].to_dict()
        selected_row["selected_family_order"] = len(greedy_order_records) + 1
        greedy_order_records.append(selected_row)

        selected_families.append(selected_candidate)
        remaining_families.remove(selected_candidate)

        print(
            f"Round {round_number}: selected {selected_candidate} "
            f"({selected_row['first_20_component_variance_change']:.4f} first-20 change)."
        )

    greedy_family_round_results_df = pd.concat(
        greedy_round_records,
        ignore_index=True,
    )

    greedy_family_order_df = (
        pd.DataFrame(greedy_order_records)
        .sort_values("selected_family_order")
        .reset_index(drop=True)
    )

    greedy_family_round_results_df.to_csv(
        GREEDY_FAMILY_ROUND_RESULTS_PATH,
        index=False,
    )

    greedy_family_order_df.to_csv(
        GREEDY_FAMILY_ORDER_PATH,
        index=False,
    )

    print(f"Saved greedy family round results: {GREEDY_FAMILY_ROUND_RESULTS_PATH}")
    print(f"Saved greedy feature-family order: {GREEDY_FAMILY_ORDER_PATH}")

GREEDY_CONTRIBUTION_FAMILY_ORDER = (
    [RAW_CONTRIBUTION_FAMILY]
    + greedy_family_order_df["candidate_family"].tolist()
)

display(greedy_family_order_df)

print(
    "Greedy feature-family order ready: "
    + " → ".join(GREEDY_CONTRIBUTION_FAMILY_ORDER)
)

Loaded cached greedy feature-family order: pipeline_data/1.5.6.intermediate_tables/greedy_feature_family_order.csv


,round_number,candidate_family,selected_families_before_round,candidate_family_set,baseline_feature_count,candidate_feature_count,added_feature_count,baseline_variance_explained_first_10_components,candidate_variance_explained_first_10_components,first_10_component_variance_change,baseline_variance_explained_first_20_components,candidate_variance_explained_first_20_components,first_20_component_variance_change,baseline_components_needed_for_80_pct_variance,candidate_components_needed_for_80_pct_variance,components_needed_for_80_pct_variance_change,candidate_runtime_seconds,selected_this_round,selected_family_order
0,1,temporal_history,raw_metric,"raw_metric, temporal_history",15,85,70,0.937727,0.508012,-0.429715,1.000000,0.702519,-0.297481,7.0,28.0,21.0,1.928804,True,1
1,2,stl_decomposition,"raw_metric, temporal_history","raw_metric, temporal_history, stl_decomposition",85,113,28,0.508012,0.414430,-0.093582,0.702519,0.590724,-0.111795,28.0,40.0,12.0,2.593908,True,2
2,3,anomaly_severity,"raw_metric, temporal_history, stl_decomposition","raw_metric, temporal_history, stl_decompositio...",113,134,21,0.414430,0.387680,-0.026750,0.590724,0.564240,-0.026484,40.0,45.0,5.0,3.247212,True,3
3,4,fourier20_residual_reconstruction,"raw_metric, temporal_history, stl_decompositio...","raw_metric, temporal_history, stl_decompositio...",134,141,7,0.387680,0.378833,-0.008847,0.564240,0.550803,-0.013436,45.0,48.0,3.0,3.349335,True,4
4,5,spatial_context,"raw_metric, temporal_history, stl_decompositio...","raw_metric, temporal_history, stl_decompositio...",141,192,51,0.378833,0.403461,0.024628,0.550803,0.547395,-0.003408,48.0,NaN,NaN,5.094847,True,5
5,6,multimodal_interaction,"raw_metric, temporal_history, stl_decompositio...","raw_metric, temporal_history, stl_decompositio...",192,226,34,0.403461,0.419778,0.016317,0.547395,0.557488,0.010093,NaN,NaN,NaN,6.066594,True,6


Greedy feature-family order ready: raw_metric → temporal_history → stl_decomposition → anomaly_severity → fourier20_residual_reconstruction → spatial_context → multimodal_interaction


Findings\. The greedy order starts with temporal history, then STL decomposition, anomaly severity, Fourier Top 20, spatial context, and multimodal interaction\. The main thing to watch is the size of the marginal drop in first\-20\-component variance\. Temporal history is the big first move, dropping first\-20 variance by 29\.7 points\. STL is the clear second layer, dropping it another 11\.2 points after temporal history is already included\. After that, the gains get much smaller: anomaly severity adds 2\.6 points, Fourier adds 1\.3 points, spatial context adds only 0\.3 points, and multimodal interaction actually increases first\-20 variance slightly\. That looks like an elbow after STL decomposition, with anomaly severity and Fourier still plausible but much more marginal\.

This QA check confirms that the greedy screen produced one selected family per round, covered every candidate family, and saved the order needed for the nested PCA section\.

In [62]:
# ---------------------------------------------------------------------
# Validate greedy feature-family order
# ---------------------------------------------------------------------

expected_greedy_engineered_families = set(preliminary_candidate_families)
observed_greedy_engineered_families = set(greedy_family_order_df["candidate_family"])

missing_greedy_families = sorted(
    expected_greedy_engineered_families - observed_greedy_engineered_families
)

extra_greedy_families = sorted(
    observed_greedy_engineered_families - expected_greedy_engineered_families
)

selected_round_count = greedy_family_round_results_df["selected_this_round"].sum()

greedy_family_order_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "expected_engineered_families",
            "value": len(expected_greedy_engineered_families),
            "status": "info",
        },
        {
            "qa_check": "selected_engineered_families",
            "value": len(observed_greedy_engineered_families),
            "status": "pass" if not missing_greedy_families else "fail",
        },
        {
            "qa_check": "missing_engineered_families",
            "value": len(missing_greedy_families),
            "status": "pass" if not missing_greedy_families else "fail",
        },
        {
            "qa_check": "extra_engineered_families",
            "value": len(extra_greedy_families),
            "status": "pass" if not extra_greedy_families else "fail",
        },
        {
            "qa_check": "selected_rounds",
            "value": int(selected_round_count),
            "status": (
                "pass"
                if int(selected_round_count) == len(expected_greedy_engineered_families)
                else "fail"
            ),
        },
        {
            "qa_check": "cached_order_file_exists",
            "value": GREEDY_FAMILY_ORDER_PATH.exists(),
            "status": "pass" if GREEDY_FAMILY_ORDER_PATH.exists() else "fail",
        },
    ]
)

display(greedy_family_order_qa_df)

assert not missing_greedy_families, (
    f"Greedy order is missing candidate families: {missing_greedy_families}"
)

assert not extra_greedy_families, (
    f"Greedy order contains unexpected families: {extra_greedy_families}"
)

assert int(selected_round_count) == len(expected_greedy_engineered_families), (
    "Greedy screen did not select exactly one family per round."
)

assert GREEDY_FAMILY_ORDER_PATH.exists(), (
    "Greedy feature-family order file was not written."
)

print("Greedy feature-family order validated.")

,qa_check,value,status
0,expected_engineered_families,6,info
1,selected_engineered_families,6,pass
2,missing_engineered_families,0,pass
3,extra_engineered_families,0,pass
4,selected_rounds,6,pass
5,cached_order_file_exists,True,pass


Greedy feature-family order validated.


Findings\. The greedy\-order QA passes\. All six engineered feature families were evaluated, each family was selected exactly once, no expected family is missing, and the cached order file exists\. That means the next nested PCA section can use this evidence\-based family order instead of the older fixed order\.

### Build Greedy Nested Feature Sets

Now that we have a greedy feature\-family order, we turn it into explicit nested feature sets\. Each row adds one more family to the prior set, starting with raw metrics and ending with the full selected family stack\. This gives the PCA diagnostics a clean sequence to evaluate instead of rebuilding feature lists inside every later block\.

Let’s convert the greedy family order into concrete feature lists\. Each nested set stores the included families, feature count, and feature names so the next block can run PCA diagnostics without guessing which columns belong to each step\.

In [63]:
# ---------------------------------------------------------------------
# Build greedy nested feature sets
# ---------------------------------------------------------------------

greedy_nested_feature_set_records = []

for step_number in range(1, len(GREEDY_CONTRIBUTION_FAMILY_ORDER) + 1):
    included_families = GREEDY_CONTRIBUTION_FAMILY_ORDER[:step_number]

    nested_feature_cols = (
        preliminary_contribution_feature_pool_df[
            preliminary_contribution_feature_pool_df["feature_family"].isin(included_families)
        ]["feature_name"]
        .drop_duplicates()
        .tolist()
    )

    nested_feature_cols = [
        column
        for column in nested_feature_cols
        if column in modeling_diagnostic_sample_df.columns
    ]

    added_family = included_families[-1]

    greedy_nested_feature_set_records.append(
        {
            "nested_step_number": step_number,
            "nested_step_name": (
                "raw_only"
                if included_families == [RAW_CONTRIBUTION_FAMILY]
                else "plus_" + added_family
            ),
            "added_family": (
                "raw_metric_base"
                if added_family == RAW_CONTRIBUTION_FAMILY
                else added_family
            ),
            "included_families": ", ".join(included_families),
            "included_family_count": len(included_families),
            "feature_count": len(nested_feature_cols),
            "feature_names": nested_feature_cols,
        }
    )

greedy_nested_feature_sets_df = pd.DataFrame(greedy_nested_feature_set_records)

greedy_nested_feature_set_summary_df = (
    greedy_nested_feature_sets_df[
        [
            "nested_step_number",
            "nested_step_name",
            "added_family",
            "included_families",
            "included_family_count",
            "feature_count",
        ]
    ]
    .copy()
)

display(greedy_nested_feature_set_summary_df)

assert not greedy_nested_feature_sets_df.empty, (
    "Greedy nested feature sets were not created."
)

assert greedy_nested_feature_sets_df["feature_count"].gt(0).all(), (
    "One or more greedy nested feature sets has zero features."
)

assert greedy_nested_feature_sets_df["feature_count"].is_monotonic_increasing, (
    "Greedy nested feature counts should increase as families are added."
)

print(
    "Greedy nested feature sets ready. "
    f"Nested steps: {len(greedy_nested_feature_sets_df):,}."
)

,nested_step_number,nested_step_name,added_family,included_families,included_family_count,feature_count
0,1,raw_only,raw_metric_base,raw_metric,1,15
1,2,plus_temporal_history,temporal_history,"raw_metric, temporal_history",2,85
2,3,plus_stl_decomposition,stl_decomposition,"raw_metric, temporal_history, stl_decomposition",3,113
3,4,plus_anomaly_severity,anomaly_severity,"raw_metric, temporal_history, stl_decompositio...",4,134
4,5,plus_fourier20_residual_reconstruction,fourier20_residual_reconstruction,"raw_metric, temporal_history, stl_decompositio...",5,141
5,6,plus_spatial_context,spatial_context,"raw_metric, temporal_history, stl_decompositio...",6,192
6,7,plus_multimodal_interaction,multimodal_interaction,"raw_metric, temporal_history, stl_decompositio...",7,226


Greedy nested feature sets ready. Nested steps: 7.


Findings\. The nested feature sets now follow the greedy order from the prior screen: raw metrics first, then temporal history, STL decomposition, anomaly severity, Fourier Top 20, spatial context, and multimodal interaction\. The feature count grows from 15 raw metrics to 226 diagnostic features in the full stack\. The important thing to notice is where the size jumps happen: temporal history adds 70 features, spatial context adds 51, while Fourier adds only 7\. That gives us useful context for the PCA results that come next, because a family’s value should be judged against both its added structure and its added feature count\.

### Run PCA Diagnostics on Greedy Nested Feature Sets

Now we rerun the PCA compression check on the final greedy nested sequence\. This is the cleaner version of the contribution test: each row adds one more family in the order selected above, so we can see where the feature space gains meaningful new structure and where the curve starts flattening\.

Let’s set up the cached outputs for the greedy nested PCA diagnostics\. These files let the notebook load previous results unless we explicitly rebuild the contribution analysis\.

In [64]:
# ---------------------------------------------------------------------
# Configure greedy nested PCA diagnostic outputs
# ---------------------------------------------------------------------

GREEDY_NESTED_PCA_DIAGNOSTICS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "greedy_nested_pca_diagnostics.csv"
)

GREEDY_NESTED_PCA_VARIANCE_CURVE_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "greedy_nested_pca_variance_curve.csv"
)

greedy_nested_pca_cache_status_df = pd.DataFrame(
    [
        {
            "cached_output": "greedy_nested_pca_diagnostics_df",
            "path": str(GREEDY_NESTED_PCA_DIAGNOSTICS_PATH),
            "exists": GREEDY_NESTED_PCA_DIAGNOSTICS_PATH.exists(),
        },
        {
            "cached_output": "greedy_nested_pca_variance_curve_df",
            "path": str(GREEDY_NESTED_PCA_VARIANCE_CURVE_PATH),
            "exists": GREEDY_NESTED_PCA_VARIANCE_CURVE_PATH.exists(),
        },
    ]
)

display(greedy_nested_pca_cache_status_df)

print("Greedy nested PCA diagnostic cache checked.")

,cached_output,path,exists
0,greedy_nested_pca_diagnostics_df,pipeline_data/1.5.6.intermediate_tables/greedy...,True
1,greedy_nested_pca_variance_curve_df,pipeline_data/1.5.6.intermediate_tables/greedy...,True


Greedy nested PCA diagnostic cache checked.


Let’s run PCA on each nested feature set\. The key outputs are how many components are needed to explain 80%, 90%, and 95% of variance, plus how much variance the first 5, 10, and 20 components capture\.

In [65]:
# ---------------------------------------------------------------------
# Run PCA diagnostics on greedy nested feature sets
# ---------------------------------------------------------------------

if (
    GREEDY_NESTED_PCA_DIAGNOSTICS_PATH.exists()
    and GREEDY_NESTED_PCA_VARIANCE_CURVE_PATH.exists()
    and not REBUILD_INCREMENTAL_CONTRIBUTION_ANALYSIS
):
    greedy_nested_pca_diagnostics_df = pd.read_csv(GREEDY_NESTED_PCA_DIAGNOSTICS_PATH)
    greedy_nested_pca_variance_curve_df = pd.read_csv(GREEDY_NESTED_PCA_VARIANCE_CURVE_PATH)

    print(
        "Loaded cached greedy nested PCA diagnostics: "
        f"{GREEDY_NESTED_PCA_DIAGNOSTICS_PATH}"
    )

else:
    greedy_nested_pca_results = []
    greedy_nested_variance_curve_records = []

    iterator = (
        tqdm(
            greedy_nested_feature_sets_df.iterrows(),
            total=len(greedy_nested_feature_sets_df),
            desc="Running greedy nested PCA diagnostics",
        )
        if tqdm
        else greedy_nested_feature_sets_df.iterrows()
    )

    for _, nested_row in iterator:
        start_time = time.perf_counter()

        nested_step_name = nested_row["nested_step_name"]
        feature_cols = [
            column
            for column in nested_row["feature_names"]
            if column in modeling_diagnostic_sample_df.columns
        ]

        assert feature_cols, (
            f"No usable feature columns found for nested PCA step: {nested_step_name}"
        )

        pca_input_df = (
            modeling_diagnostic_sample_df[feature_cols]
            .replace([np.inf, -np.inf], np.nan)
            .copy()
        )

        # Fully-null columns cannot be median-imputed, so remove them before PCA.
        usable_feature_cols = pca_input_df.columns[pca_input_df.notna().any()].tolist()
        pca_input_df = pca_input_df[usable_feature_cols]

        max_components = min(
            PCA_MAX_COMPONENTS,
            len(usable_feature_cols),
            len(pca_input_df) - 1,
        )

        assert max_components >= 1, (
            f"No PCA components available for nested PCA step: {nested_step_name}"
        )

        imputed_values = SimpleImputer(strategy="median").fit_transform(pca_input_df)
        scaled_values = StandardScaler().fit_transform(imputed_values)

        pca_model = PCA(
            n_components=max_components,
            random_state=PCA_RANDOM_STATE,
        )

        pca_model.fit(scaled_values)

        explained_variance = pca_model.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)

        result_row = {
            "nested_step_number": nested_row["nested_step_number"],
            "nested_step_name": nested_step_name,
            "added_family": nested_row["added_family"],
            "included_families": nested_row["included_families"],
            "included_family_count": nested_row["included_family_count"],
            "feature_count": len(usable_feature_cols),
            "rows_used": len(pca_input_df),
            "max_components_evaluated": max_components,
            "runtime_seconds": time.perf_counter() - start_time,
        }

        for variance_target in PCA_VARIANCE_TARGETS:
            target_label = int(variance_target * 100)
            reached_target = np.where(cumulative_variance >= variance_target)[0]

            result_row[f"components_needed_for_{target_label}_pct_variance"] = (
                int(reached_target[0] + 1)
                if len(reached_target) > 0
                else np.nan
            )

        for component_count in PCA_TOP_COMPONENT_COUNTS:
            result_row[f"variance_explained_first_{component_count}_components"] = (
                cumulative_variance[min(component_count, len(cumulative_variance)) - 1]
            )

        greedy_nested_pca_results.append(result_row)

        for component_index, cumulative_value in enumerate(cumulative_variance, start=1):
            greedy_nested_variance_curve_records.append(
                {
                    "nested_step_number": nested_row["nested_step_number"],
                    "nested_step_name": nested_step_name,
                    "added_family": nested_row["added_family"],
                    "feature_count": len(usable_feature_cols),
                    "component_number": component_index,
                    "explained_variance_ratio": explained_variance[component_index - 1],
                    "cumulative_variance_ratio": cumulative_value,
                    "nested_step_label": (
                        f"{nested_step_name} ({len(usable_feature_cols)} features)"
                    ),
                }
            )

        del pca_input_df, imputed_values, scaled_values, pca_model
        gc.collect()

    greedy_nested_pca_diagnostics_df = pd.DataFrame(greedy_nested_pca_results)
    greedy_nested_pca_variance_curve_df = pd.DataFrame(greedy_nested_variance_curve_records)

    greedy_nested_pca_diagnostics_df.to_csv(
        GREEDY_NESTED_PCA_DIAGNOSTICS_PATH,
        index=False,
    )

    greedy_nested_pca_variance_curve_df.to_csv(
        GREEDY_NESTED_PCA_VARIANCE_CURVE_PATH,
        index=False,
    )

    print(f"Saved greedy nested PCA diagnostics: {GREEDY_NESTED_PCA_DIAGNOSTICS_PATH}")
    print(f"Saved greedy nested PCA variance curve: {GREEDY_NESTED_PCA_VARIANCE_CURVE_PATH}")

display(greedy_nested_pca_diagnostics_df)

assert not greedy_nested_pca_diagnostics_df.empty, (
    "Greedy nested PCA diagnostics are empty."
)

assert not greedy_nested_pca_variance_curve_df.empty, (
    "Greedy nested PCA variance curve is empty."
)

assert greedy_nested_pca_diagnostics_df["runtime_seconds"].ge(0).all(), (
    "Greedy nested PCA diagnostics contain invalid runtime values."
)

print(
    "Greedy nested PCA diagnostics ready. "
    f"Nested steps: {len(greedy_nested_pca_diagnostics_df):,}."
)

Loaded cached greedy nested PCA diagnostics: pipeline_data/1.5.6.intermediate_tables/greedy_nested_pca_diagnostics.csv


,nested_step_number,nested_step_name,added_family,included_families,included_family_count,feature_count,rows_used,max_components_evaluated,runtime_seconds,components_needed_for_80_pct_variance,components_needed_for_90_pct_variance,components_needed_for_95_pct_variance,variance_explained_first_5_components,variance_explained_first_10_components,variance_explained_first_20_components
0,1,raw_only,raw_metric_base,raw_metric,1,15,250000,15,0.373069,7.0,9.0,11.0,0.719287,0.937727,1.000000
1,2,plus_temporal_history,temporal_history,"raw_metric, temporal_history",2,85,250000,50,2.536087,28.0,39.0,49.0,0.349249,0.508012,0.702519
2,3,plus_stl_decomposition,stl_decomposition,"raw_metric, temporal_history, stl_decomposition",3,113,250000,50,3.676976,40.0,NaN,NaN,0.282315,0.414430,0.590724
3,4,plus_anomaly_severity,anomaly_severity,"raw_metric, temporal_history, stl_decompositio...",4,134,250000,50,4.402088,45.0,NaN,NaN,0.262486,0.387680,0.564240
4,5,plus_fourier20_residual_reconstruction,fourier20_residual_reconstruction,"raw_metric, temporal_history, stl_decompositio...",5,141,250000,50,4.456550,48.0,NaN,NaN,0.252588,0.378833,0.550803
5,6,plus_spatial_context,spatial_context,"raw_metric, temporal_history, stl_decompositio...",6,192,250000,50,6.300920,NaN,NaN,NaN,0.299889,0.403461,0.547395
6,7,plus_multimodal_interaction,multimodal_interaction,"raw_metric, temporal_history, stl_decompositio...",7,226,250000,50,7.372893,NaN,NaN,NaN,0.309983,0.419778,0.557488


Greedy nested PCA diagnostics ready. Nested steps: 7.


Findings\. The PCA diagnostics show the same elbow pattern we saw in the greedy\-order screen, but now in the final nested sequence\. Raw metrics are very compressible: 10 components explain 93\.8% of variance\. Adding temporal history changes the feature space sharply, with 10 components explaining only 50\.8% and 20 components explaining 70\.3%\. Adding STL decomposition pushes that further down to 41\.4% and 59\.1%, which is a strong signal that decomposition is adding structure not already captured by temporal history\. After that, the curve starts flattening: anomaly severity and Fourier still reduce compression modestly, but spatial context and multimodal interaction do not add much additional PCA complexity once the earlier families are already in the stack\.

Let’s plot the cumulative variance curves\. The flatter the curve, the less compressible the feature set is; that usually means the added family is bringing in more independent structure\. The elbow helps us see where additional families start adding less\.

In [66]:
# ---------------------------------------------------------------------
# Plot greedy nested PCA cumulative variance curves
# ---------------------------------------------------------------------

greedy_nested_pca_plot_df = greedy_nested_pca_variance_curve_df[
    greedy_nested_pca_variance_curve_df["component_number"].le(PCA_MAX_COMPONENTS)
].copy()

if px:
    greedy_nested_pca_fig = px.line(
        greedy_nested_pca_plot_df,
        x="component_number",
        y="cumulative_variance_ratio",
        color="nested_step_label",
        markers=False,
        title="PCA cumulative variance by greedy nested feature set",
        labels={
            "component_number": "PCA component number",
            "cumulative_variance_ratio": "Cumulative variance explained",
            "nested_step_label": "Nested feature set",
        },
    )

    for target in PCA_VARIANCE_TARGETS:
        greedy_nested_pca_fig.add_hline(
            y=target,
            line_dash="dot",
            line_color="gray",
            annotation_text=f"{int(target * 100)}%",
            annotation_position="right",
        )

    greedy_nested_pca_fig.update_yaxes(tickformat=".0%")
    greedy_nested_pca_fig.update_layout(
        height=620,
        legend_title_text="Nested feature set",
    )

    greedy_nested_pca_fig.show()

else:
    print("Plotly is not available. Displaying variance-curve table instead.")
    display(greedy_nested_pca_plot_df.head(50))

print("Greedy nested PCA cumulative variance plot ready.")

Greedy nested PCA cumulative variance plot ready.


Read the PCA curve as a compression chart: a steeper curve means PCA can explain the feature set with fewer dimensions, while a flatter curve means the feature set contains more independent structure\. The elbow is where adding more families stops changing that shape very much\.

Findings\. The plot makes the elbow easier to see\. The raw\-metric line climbs almost straight to 100%, meaning the base metrics are compact and highly compressible\. The biggest visual separation happens when temporal history is added, and the next meaningful separation happens after STL decomposition\. After anomaly severity and Fourier, the lines start bunching together\. Spatial context and multimodal interaction sit very close to the prior curves, which suggests their incremental contribution is smaller in this full\-stack PCA view\. This does not mean those families are useless, but it does mean they should face a harder review in the final selection step\.

### Identify Contribution Elbow and Review Tiers

The PCA curves show the shape of the contribution pattern, but we still need to translate that shape into something usable for feature selection\. This step identifies where the marginal contribution starts to flatten and assigns each feature family a review tier\. The goal is not to make the final keep/drop decision here\. The goal is to flag which families look central, which look secondary, and which need a harder review in the final scorecard\.

Let’s convert the greedy nested PCA results into contribution tiers\. We use the change in first\-20\-component variance as the main signal: larger negative drops mean the added family made the feature space harder to compress, while near\-zero or positive changes suggest weaker incremental structure\.

In [67]:
# ---------------------------------------------------------------------
# Identify contribution elbow and review tiers
# ---------------------------------------------------------------------

contribution_elbow_df = (
    greedy_nested_pca_diagnostics_df
    .sort_values("nested_step_number")
    .copy()
)

contribution_elbow_df["added_features"] = (
    contribution_elbow_df["feature_count"]
    - contribution_elbow_df["feature_count"].shift(1)
)

contribution_elbow_df["added_features"] = (
    contribution_elbow_df["added_features"]
    .fillna(contribution_elbow_df["feature_count"])
    .astype(int)
)

contribution_elbow_df["first_20_variance_change_pts"] = (
    contribution_elbow_df["variance_explained_first_20_components"]
    - contribution_elbow_df["variance_explained_first_20_components"].shift(1)
) * 100

contribution_elbow_df["components_80_change"] = (
    contribution_elbow_df["components_needed_for_80_pct_variance"]
    - contribution_elbow_df["components_needed_for_80_pct_variance"].shift(1)
)

contribution_elbow_df["first_20_variance_change_pts"] = (
    contribution_elbow_df["first_20_variance_change_pts"].fillna(0)
)

contribution_elbow_df["components_80_change"] = (
    contribution_elbow_df["components_80_change"].fillna(0)
)

contribution_elbow_df["pca_contribution_tier"] = np.select(
    [
        contribution_elbow_df["added_family"].eq("raw_metric_base"),
        contribution_elbow_df["first_20_variance_change_pts"].le(-10),
        contribution_elbow_df["first_20_variance_change_pts"].le(-3),
        contribution_elbow_df["first_20_variance_change_pts"].le(-1),
    ],
    [
        "baseline",
        "core_signal",
        "secondary_signal",
        "review_signal",
    ],
    default="low_incremental_signal",
)

contribution_elbow_df["selection_implication"] = np.select(
    [
        contribution_elbow_df["pca_contribution_tier"].eq("baseline"),
        contribution_elbow_df["pca_contribution_tier"].eq("core_signal"),
        contribution_elbow_df["pca_contribution_tier"].eq("secondary_signal"),
        contribution_elbow_df["pca_contribution_tier"].eq("review_signal"),
    ],
    [
        "Baseline comparison set.",
        "Strong evidence to retain this family.",
        "Useful signal, but review alongside redundancy and modality evidence.",
        "Possible keep, but needs support from other evidence.",
    ],
    default="Harder review before final selection.",
)

contribution_elbow_df["post_elbow_flag"] = (
    contribution_elbow_df["pca_contribution_tier"].isin(
        [
            "review_signal",
            "low_incremental_signal",
        ]
    )
)

contribution_elbow_display_df = (
    contribution_elbow_df[
        [
            "nested_step_number",
            "added_family",
            "added_features",
            "first_20_variance_change_pts",
            "components_80_change",
            "pca_contribution_tier",
            "post_elbow_flag",
            "selection_implication",
        ]
    ]
    .copy()
)

contribution_elbow_display_df["first_20_variance_change_pts"] = (
    contribution_elbow_display_df["first_20_variance_change_pts"].round(1)
)

contribution_elbow_display_df["components_80_change"] = (
    contribution_elbow_display_df["components_80_change"].round(1)
)

display(contribution_elbow_display_df)

assert not contribution_elbow_display_df.empty, (
    "Contribution elbow table is empty."
)

assert contribution_elbow_display_df["pca_contribution_tier"].notna().all(), (
    "Contribution elbow table contains missing PCA contribution tiers."
)

print("Contribution elbow and review tiers ready.")

,nested_step_number,added_family,added_features,first_20_variance_change_pts,components_80_change,pca_contribution_tier,post_elbow_flag,selection_implication
0,1,raw_metric_base,15,0.0,0.0,baseline,False,Baseline comparison set.
1,2,temporal_history,70,-29.7,21.0,core_signal,False,Strong evidence to retain this family.
2,3,stl_decomposition,28,-11.2,12.0,core_signal,False,Strong evidence to retain this family.
3,4,anomaly_severity,21,-2.6,5.0,review_signal,True,"Possible keep, but needs support from other ev..."
4,5,fourier20_residual_reconstruction,7,-1.3,3.0,review_signal,True,"Possible keep, but needs support from other ev..."
5,6,spatial_context,51,-0.3,0.0,low_incremental_signal,True,Harder review before final selection.
6,7,multimodal_interaction,34,1.0,0.0,low_incremental_signal,True,Harder review before final selection.


Contribution elbow and review tiers ready.


Findings\. This table makes the elbow explicit\. Temporal history and STL decomposition are the two clear PCA winners: temporal history drops first\-20 variance by 29\.7 points, and STL adds another 11\.2\-point drop after temporal history is already included\. After that, the marginal gains shrink fast\. Anomaly severity and Fourier Top 20 still move the feature space a little, but they fall into review territory rather than automatic\-retain territory\. Spatial context is the surprising one: it adds 51 features but barely changes first\-20 PCA compression in this full\-stack sequence\. That does not mean spatial context is unimportant in the project; it means that, after raw metrics, temporal history, decomposition, anomaly severity, and Fourier are already included, spatial context does not add much extra variance structure by this PCA test\. This is exactly why the final selection step should combine PCA contribution with redundancy, modality\-specific evidence, and interpretability instead of using this table as a single drop rule\.

### Section Summary

Section 1\.5\.6\.4 tested whether each engineered feature family adds meaningful structure beyond the raw mobility metrics and beyond the families already selected before it\. The preliminary screen showed that temporal history, STL decomposition, and spatial context all add distinct structure when compared directly against raw metrics\. The greedy nested screen then clarified what happens once families stack together: temporal history is the strongest first addition, STL decomposition remains a strong second contribution, and the elbow appears after those two families\. Anomaly severity and Fourier Top 20 still add some incremental signal, but their gains are smaller and should be reviewed alongside redundancy and interpretability\. Spatial context and multimodal interaction look weaker in the full\-stack PCA screen, especially after temporal and decomposition features are already included, so they should not be retained simply because they exist\. The practical takeaway is that PCA contribution supports keeping temporal history and STL decomposition, reviewing anomaly severity and Fourier carefully, and applying a stricter scorecard to spatial and multimodal features in the final selection step\.

## 1\.5\.6\.5 Select Final Modeling Features

This is where we convert inventory analysis, global redundancy analysis, modality\-specific redundancy analysis, and feature\-family contribution analysis into an explicit modeling feature strategy\. Each diagnostic answers a different question: redundancy tells us where features overlap, modality\-specific checks tell us whether that overlap matters inside downstream modeling views, and contribution analysis tells us whether a feature family adds structure beyond the rest of the panel\.

Redundancy flags are review signals, not automatic drop rules\. If several features describe nearly the same movement pattern, the selection task is to choose a representative feature or small representative set while preserving interpretability, core\-metric coverage, modality coverage, and downstream modeling relevance\. When several redundant variants perform similarly, prefer a consistent feature template across modalities because it makes the final modeling assets easier to interpret, compare, and defend\.

The output of this section is a feature\-selection decision table: each candidate feature gets a decision tier, supporting evidence, and a short reason\. We keep the process conservative\. Strong redundancy, weak contribution, low coverage, or poor interpretability can demote a feature, but no single diagnostic automatically drops an entire feature family\.

### Build Feature Selection Scorecard Inputs

Before scoring anything, we need one table that puts the evidence in the same place\. This step starts with the feature\-level redundancy decision table, then attaches modality\-specific redundancy evidence, family\-level contribution tiers, coverage checks, and core\-metric status\. The result is not the final selection table yet\. It is the evidence layer that the scorecard will use\.

Let’s prepare the evidence lookups for the scorecard\. This keeps the three review layers separate: global redundancy, modality\-specific redundancy, and feature\-family contribution\.

In [68]:
# ---------------------------------------------------------------------
# Build scorecard evidence lookups
# ---------------------------------------------------------------------

MODALITY_REDUNDANCY_FEATURE_EVIDENCE_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "modality_redundancy_feature_evidence.csv"
)

# Global redundancy comes from the full-period within-family screen in 1.5.6.2.
global_redundancy_lookup_df = (
    redundancy_decision_input_df[
        [
            "feature_name",
            "redundancy_review_flag",
            "high_correlation_pair_count",
            "strict_high_correlation_pair_count",
        ]
    ]
    .rename(
        columns={
            "redundancy_review_flag": "global_redundancy_review_flag",
            "high_correlation_pair_count": "global_high_correlation_pair_count",
            "strict_high_correlation_pair_count": "global_strict_high_correlation_pair_count",
        }
    )
    .drop_duplicates("feature_name")
    .reset_index(drop=True)
)

# Family contribution comes from the greedy PCA contribution screen in 1.5.6.4.
family_contribution_lookup_df = (
    contribution_elbow_df[
        [
            "added_family",
            "pca_contribution_tier",
            "post_elbow_flag",
            "selection_implication",
        ]
    ]
    .rename(columns={"added_family": "feature_family"})
    .drop_duplicates("feature_family")
    .reset_index(drop=True)
)

# Modality-specific redundancy is created explicitly in 1.5.6.3 and cached to disk.
if "modality_redundancy_feature_evidence_df" not in globals():
    if MODALITY_REDUNDANCY_FEATURE_EVIDENCE_PATH.exists():
        modality_redundancy_feature_evidence_df = pd.read_csv(
            MODALITY_REDUNDANCY_FEATURE_EVIDENCE_PATH
        )
    else:
        missing_dependency_message = (
            "Missing 1.5.6.3 modality-specific feature-level redundancy evidence. "
            "Run the block labeled '# Build modality-specific feature-level redundancy evidence' "
            "once, then rerun this scorecard lookup block. You do not need to turn on "
            "REBUILD_MODALITY_REDUNDANCY_ANALYSIS unless the modality redundancy cache files are missing."
        )
        raise AssertionError(missing_dependency_message)

modality_redundancy_lookup_df = (
    modality_redundancy_feature_evidence_df
    .groupby("feature_name", as_index=False)
    .agg(
        modality_views_flagged=("modeling_view", "nunique"),
        modality_high_correlation_pair_count=(
            "modality_high_correlation_pair_count",
            "sum",
        ),
        modality_strict_high_correlation_pair_count=(
            "modality_strict_high_correlation_pair_count",
            "sum",
        ),
    )
)

scorecard_evidence_layer_status_df = pd.DataFrame(
    [
        {
            "evidence_layer": "global_redundancy",
            "source": "redundancy_decision_input_df",
            "rows": len(global_redundancy_lookup_df),
            "status": "available",
        },
        {
            "evidence_layer": "family_contribution",
            "source": "contribution_elbow_df",
            "rows": len(family_contribution_lookup_df),
            "status": "available",
        },
        {
            "evidence_layer": "modality_specific_redundancy",
            "source": "modality_redundancy_feature_evidence_df",
            "rows": len(modality_redundancy_lookup_df),
            "status": "available",
        },
    ]
)

display(scorecard_evidence_layer_status_df)

assert not global_redundancy_lookup_df.empty, "Global redundancy lookup is empty."
assert not family_contribution_lookup_df.empty, "Family contribution lookup is empty."
assert not modality_redundancy_lookup_df.empty, "Modality-specific redundancy lookup is empty."

print("Scorecard evidence lookups ready.")

,evidence_layer,source,rows,status
0,global_redundancy,redundancy_decision_input_df,329,available
1,family_contribution,contribution_elbow_df,7,available
2,modality_specific_redundancy,modality_redundancy_feature_evidence_df,307,available


Scorecard evidence lookups ready.


Findings\. The scorecard now has all three evidence layers available: global redundancy from the full within\-family screen, family contribution from the PCA elbow analysis, and modality\-specific redundancy from the modeling\-view screen\. That is the right setup for selection because no single diagnostic gets to dominate the decision\. We can now score features using overlap, contribution, coverage, modality behavior, and core\-metric relevance together\.

Now we join those evidence layers into one feature\-level scorecard input table\. This is still not the selection decision; it is the table the next scoring block will use\.

In [69]:
# ---------------------------------------------------------------------
# Build feature selection scorecard input table
# ---------------------------------------------------------------------

feature_selection_scorecard_input_df = redundancy_decision_input_df.copy()

required_scorecard_columns = [
    "feature_name",
    "feature_family",
    "modality",
    "source_metric",
    "is_core_metric_descendant",
    "non_null_pct",
    "null_pct",
]

missing_scorecard_columns = [
    column
    for column in required_scorecard_columns
    if column not in feature_selection_scorecard_input_df.columns
]

assert not missing_scorecard_columns, (
    "Scorecard input is missing required columns: "
    f"{missing_scorecard_columns}"
)

if "is_numeric" not in feature_selection_scorecard_input_df.columns:
    if "dtype" in feature_selection_scorecard_input_df.columns:
        feature_selection_scorecard_input_df["is_numeric"] = (
            feature_selection_scorecard_input_df["dtype"]
            .astype(str)
            .str.contains("int|float|double|decimal|bool", case=False, regex=True)
        )
    else:
        feature_selection_scorecard_input_df["is_numeric"] = (
            feature_selection_scorecard_input_df["feature_name"].isin(modeling_df.select_dtypes(include="number").columns)
        )

if "is_modeling_candidate" not in feature_selection_scorecard_input_df.columns:
    feature_selection_scorecard_input_df["is_modeling_candidate"] = (
        feature_selection_scorecard_input_df["is_numeric"]
        & ~feature_selection_scorecard_input_df["feature_name"].isin(NON_MODELING_COLUMNS)
    )

for column, default_value in {
    "near_zero_variance_flag": False,
    "source_notebook": "not_documented",
}.items():
    if column not in feature_selection_scorecard_input_df.columns:
        feature_selection_scorecard_input_df[column] = default_value

feature_selection_scorecard_input_df = (
    feature_selection_scorecard_input_df
    .drop(
        columns=[
            column
            for column in [
                "global_redundancy_review_flag",
                "global_high_correlation_pair_count",
                "global_strict_high_correlation_pair_count",
                "modality_views_flagged",
                "modality_high_correlation_pair_count",
                "modality_strict_high_correlation_pair_count",
                "pca_contribution_tier",
                "post_elbow_flag",
                "selection_implication",
            ]
            if column in feature_selection_scorecard_input_df.columns
        ],
        errors="ignore",
    )
    .merge(
        global_redundancy_lookup_df,
        on="feature_name",
        how="left",
    )
    .merge(
        modality_redundancy_lookup_df,
        on="feature_name",
        how="left",
    )
    .merge(
        family_contribution_lookup_df,
        on="feature_family",
        how="left",
    )
)

feature_selection_scorecard_input_df["global_redundancy_review_flag"] = (
    feature_selection_scorecard_input_df["global_redundancy_review_flag"]
    .fillna("no_high_correlation_flag")
)

for column in [
    "global_high_correlation_pair_count",
    "global_strict_high_correlation_pair_count",
    "modality_views_flagged",
    "modality_high_correlation_pair_count",
    "modality_strict_high_correlation_pair_count",
]:
    feature_selection_scorecard_input_df[column] = (
        feature_selection_scorecard_input_df[column]
        .fillna(0)
        .astype(int)
    )

feature_selection_scorecard_input_df["pca_contribution_tier"] = (
    feature_selection_scorecard_input_df["pca_contribution_tier"]
    .fillna("not_evaluated")
)

feature_selection_scorecard_input_df["post_elbow_flag"] = (
    feature_selection_scorecard_input_df["post_elbow_flag"]
    .fillna(False)
    .astype(bool)
)

feature_selection_scorecard_input_df["selection_implication"] = (
    feature_selection_scorecard_input_df["selection_implication"]
    .fillna("No PCA contribution tier assigned.")
)

feature_selection_scorecard_input_df["eligible_for_selection_scorecard"] = (
    feature_selection_scorecard_input_df["is_modeling_candidate"]
    & feature_selection_scorecard_input_df["is_numeric"]
    & ~feature_selection_scorecard_input_df["near_zero_variance_flag"]
    & feature_selection_scorecard_input_df["non_null_pct"].ge(MIN_FEATURE_NON_NULL_PCT)
)

feature_selection_scorecard_preview_df = (
    feature_selection_scorecard_input_df[
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "modality",
            "source_metric",
            "is_core_metric_descendant",
            "non_null_pct",
            "global_redundancy_review_flag",
            "global_high_correlation_pair_count",
            "global_strict_high_correlation_pair_count",
            "modality_views_flagged",
            "modality_high_correlation_pair_count",
            "modality_strict_high_correlation_pair_count",
            "pca_contribution_tier",
            "post_elbow_flag",
            "eligible_for_selection_scorecard",
        ]
    ]
    .sort_values(
        [
            "eligible_for_selection_scorecard",
            "feature_family",
            "modality",
            "source_metric",
            "feature_name",
        ],
        ascending=[False, True, True, True, True],
    )
    .reset_index(drop=True)
)

display(feature_selection_scorecard_preview_df.head(30))

assert feature_selection_scorecard_input_df["feature_name"].is_unique, (
    "Feature selection scorecard input contains duplicate feature names."
)

assert feature_selection_scorecard_input_df["eligible_for_selection_scorecard"].any(), (
    "No features are eligible for the selection scorecard."
)

print(
    "Feature selection scorecard input ready. "
    f"Eligible features: {feature_selection_scorecard_input_df['eligible_for_selection_scorecard'].sum():,}."
)

,feature_name,source_notebook,feature_family,modality,source_metric,is_core_metric_descendant,non_null_pct,global_redundancy_review_flag,global_high_correlation_pair_count,global_strict_high_correlation_pair_count,modality_views_flagged,modality_high_correlation_pair_count,modality_strict_high_correlation_pair_count,pca_contribution_tier,post_elbow_flag,eligible_for_selection_scorecard
0,avg_bus_speed_deviation_abs_zscore,1.5.5,anomaly_severity,bus,avg_bus_speed,True,0.946701,no_high_correlation_flag,0,0,2,0,0,review_signal,True,True
1,avg_bus_speed_deviation_percentile_rank,1.5.5,anomaly_severity,bus,avg_bus_speed,True,0.946701,strict_within_family_redundancy_review,1,1,2,2,2,review_signal,True,True
2,avg_bus_speed_deviation_zscore,1.5.5,anomaly_severity,bus,avg_bus_speed,True,0.946701,strict_within_family_redundancy_review,1,1,2,2,2,review_signal,True,True
3,bus_trip_count_deviation_abs_zscore,1.5.5,anomaly_severity,bus,bus_trip_count,True,0.946690,no_high_correlation_flag,0,0,2,0,0,review_signal,True,True
4,bus_trip_count_deviation_percentile_rank,1.5.5,anomaly_severity,bus,bus_trip_count,True,0.946701,strict_within_family_redundancy_review,1,1,2,2,2,review_signal,True,True
5,bus_trip_count_deviation_zscore,1.5.5,anomaly_severity,bus,bus_trip_count,True,0.946690,strict_within_family_redundancy_review,1,1,2,2,2,review_signal,True,True
6,fhvhv_avg_trip_speed_deviation_abs_zscore,1.5.5,anomaly_severity,fhvhv,fhvhv_avg_trip_speed,True,0.996015,no_high_correlation_flag,0,0,2,0,0,review_signal,True,True
7,fhvhv_avg_trip_speed_deviation_percentile_rank,1.5.5,anomaly_severity,fhvhv,fhvhv_avg_trip_speed,True,1.000000,strict_within_family_redundancy_review,1,1,2,2,2,review_signal,True,True
8,fhvhv_avg_trip_speed_deviation_zscore,1.5.5,anomaly_severity,fhvhv,fhvhv_avg_trip_speed,True,0.996015,strict_within_family_redundancy_review,1,1,2,2,2,review_signal,True,True
9,fhvhv_trip_count_deviation_abs_zscore,1.5.5,anomaly_severity,fhvhv,fhvhv_trip_count,True,0.996015,no_high_correlation_flag,0,0,2,0,0,review_signal,True,True


Feature selection scorecard input ready. Eligible features: 323.


Findings\. This preview shows the scorecard doing what we want: each feature now carries its lineage, coverage, redundancy flags, modality\-specific redundancy counts, PCA contribution tier, and eligibility status in one place\. The anomaly\-severity rows are a good example of why this combined table matters\. Absolute z\-score features are mostly clean, while z\-score and percentile\-rank versions are often strict redundancy matches, so the scorecard can later choose a representative rather than blindly keeping every deviation variant\. Fourier Top 20 also shows up cleanly here: it has full scorecard eligibility and no high\-correlation flags in this preview, which supports treating it as a distinct signal candidate rather than an obvious duplicate\.

Let’s first summarize the scorecard inputs by feature family\. This shows where the candidate pool is concentrated, which families have the most redundancy pressure, and which families already have strong PCA contribution evidence\.

In [70]:
# ---------------------------------------------------------------------
# Summarize scorecard inputs by feature family
# ---------------------------------------------------------------------

eligible_scorecard_input_df = (
    feature_selection_scorecard_input_df[
        feature_selection_scorecard_input_df["eligible_for_selection_scorecard"]
    ]
    .copy()
)

scorecard_input_by_family_df = (
    eligible_scorecard_input_df
    .groupby(
        [
            "feature_family",
            "pca_contribution_tier",
        ],
        as_index=False,
    )
    .agg(
        candidate_features=("feature_name", "count"),
        core_metric_descendants=("is_core_metric_descendant", "sum"),
        mean_non_null_pct=("non_null_pct", "mean"),
        global_high_correlation_features=(
            "global_high_correlation_pair_count",
            lambda values: (values.fillna(0) > 0).sum(),
        ),
        modality_high_correlation_features=(
            "modality_high_correlation_pair_count",
            lambda values: (values.fillna(0) > 0).sum(),
        ),
    )
    .sort_values(
        [
            "candidate_features",
            "feature_family",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(scorecard_input_by_family_df)

assert len(eligible_scorecard_input_df) > 0, (
    "Scorecard input has no eligible modeling features."
)

print("Scorecard input family summary ready.")

,feature_family,pca_contribution_tier,candidate_features,core_metric_descendants,mean_non_null_pct,global_high_correlation_features,modality_high_correlation_features
0,temporal_history,core_signal,150,70,0.896022,110,110
1,spatial_context,low_incremental_signal,61,28,0.944364,30,26
2,multimodal_interaction,low_incremental_signal,41,34,0.856829,15,13
3,stl_decomposition,core_signal,28,28,0.884372,8,8
4,anomaly_severity,review_signal,21,21,0.884826,14,14
5,raw_metric,not_evaluated,15,7,0.860812,4,13
6,fourier20_residual_reconstruction,review_signal,7,7,0.884372,0,0


Scorecard input family summary ready.


**Findings.** The family summary shows the selection problem clearly. Temporal history is the largest family and also carries the most redundancy pressure: 110 of 150 temporal features are flagged globally and by modality. STL decomposition is much smaller, fully core-metric aligned, and still marked as `core_signal`, which makes it one of the stronger keep candidates. Fourier Top 20 is also small and has zero high-correlation flags, while anomaly severity is useful but needs pruning because z-score and percentile variants overlap heavily. Spatial context and multimodal interaction are the harder calls: they add many candidate features, but both sit in the `low_incremental_signal` tier.

Now let’s summarize the same scorecard inputs by modality. This helps us see whether the final selection rules need to behave differently for bus, subway, taxi, FHVHV, multimodal, traffic, and spatial/context-only features.

In [71]:
# ---------------------------------------------------------------------
# Summarize scorecard inputs by modality
# ---------------------------------------------------------------------

scorecard_input_by_modality_df = (
    eligible_scorecard_input_df
    .groupby("modality", as_index=False)
    .agg(
        candidate_features=("feature_name", "count"),
        feature_families=("feature_family", "nunique"),
        core_metric_descendants=("is_core_metric_descendant", "sum"),
        mean_non_null_pct=("non_null_pct", "mean"),
        global_high_correlation_features=(
            "global_high_correlation_pair_count",
            lambda values: (values.fillna(0) > 0).sum(),
        ),
        modality_high_correlation_features=(
            "modality_high_correlation_pair_count",
            lambda values: (values.fillna(0) > 0).sum(),
        ),
    )
    .sort_values("candidate_features", ascending=False)
    .reset_index(drop=True)
)

display(scorecard_input_by_modality_df)

print("Scorecard input modality summary ready.")

,modality,candidate_features,feature_families,core_metric_descendants,mean_non_null_pct,global_high_correlation_features,modality_high_correlation_features
0,fhvhv,92,7,50,0.996639,56,59
1,taxi,92,7,50,0.885563,51,53
2,bus,64,7,50,0.954011,44,47
3,subway,39,7,25,0.648629,17,19
4,multimodal,20,1,20,0.815714,6,6
5,traffic,9,2,0,0.756996,3,0
6,spatial,7,1,0,0.991852,4,0


Scorecard input modality summary ready.


**Findings.** The modality summary shows that FHVHV, taxi, and bus have the largest candidate sets, while subway is smaller mostly because subway coverage is naturally limited to zones with subway access. The redundancy pattern is not evenly distributed: FHVHV, taxi, and bus carry the most modality-specific review pressure, while traffic and spatial/context-only features are much smaller side groups. This supports keeping the final feature sets modality-aware instead of applying one blanket rule across every transportation system.

Finally, we validate that the scorecard input has the coverage it needs before we start assigning scores. This is a small readiness check, not a feature-selection decision.

In [72]:
# ---------------------------------------------------------------------
# Validate scorecard input coverage
# ---------------------------------------------------------------------

missing_pca_tier_features_df = (
    eligible_scorecard_input_df[
        eligible_scorecard_input_df["pca_contribution_tier"].eq("not_evaluated")
    ][
        [
            "feature_name",
            "source_notebook",
            "feature_family",
            "modality",
            "source_metric",
            "non_null_pct",
            "global_redundancy_review_flag",
            "modality_views_flagged",
        ]
    ]
    .sort_values(["feature_family", "modality", "feature_name"])
    .reset_index(drop=True)
)

scorecard_input_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "eligible_scorecard_features",
            "value": len(eligible_scorecard_input_df),
            "status": "pass" if len(eligible_scorecard_input_df) > 0 else "fail",
        },
        {
            "qa_check": "families_represented",
            "value": eligible_scorecard_input_df["feature_family"].nunique(),
            "status": "pass",
        },
        {
            "qa_check": "modalities_represented",
            "value": eligible_scorecard_input_df["modality"].nunique(),
            "status": "pass",
        },
        {
            "qa_check": "features_missing_pca_tier",
            "value": len(missing_pca_tier_features_df),
            "status": "pass" if missing_pca_tier_features_df.empty else "review",
        },
        {
            "qa_check": "features_with_modality_redundancy_evidence",
            "value": (eligible_scorecard_input_df["modality_views_flagged"] > 0).sum(),
            "status": "info",
        },
    ]
)

display(scorecard_input_qa_df)

if not missing_pca_tier_features_df.empty:
    display(missing_pca_tier_features_df)

assert len(eligible_scorecard_input_df) > 0, (
    "Scorecard input has no eligible modeling features."
)

print("Feature selection scorecard input coverage validated.")

,qa_check,value,status
0,eligible_scorecard_features,323,pass
1,families_represented,7,pass
2,modalities_represented,7,pass
3,features_missing_pca_tier,15,review
4,features_with_modality_redundancy_evidence,307,info


,feature_name,source_notebook,feature_family,modality,source_metric,non_null_pct,global_redundancy_review_flag,modality_views_flagged
0,avg_bus_speed,pre_1.5_engineering,raw_metric,bus,avg_bus_speed,0.946709,no_high_correlation_flag,2
1,avg_bus_travel_time,pre_1.5_engineering,raw_metric,bus,avg_bus_travel_time,0.946709,no_high_correlation_flag,2
2,bus_trip_count,pre_1.5_engineering,raw_metric,bus,bus_trip_count,0.946709,no_high_correlation_flag,2
3,fhvhv_avg_trip_distance,pre_1.5_engineering,raw_metric,fhvhv,fhvhv_avg_trip_distance,1.000000,no_high_correlation_flag,2
4,fhvhv_avg_trip_duration,pre_1.5_engineering,raw_metric,fhvhv,fhvhv_avg_trip_duration,1.000000,no_high_correlation_flag,2
5,fhvhv_avg_trip_speed,pre_1.5_engineering,raw_metric,fhvhv,fhvhv_avg_trip_speed,1.000000,no_high_correlation_flag,2
6,fhvhv_distinct_dropoff_zone_count,pre_1.5_engineering,raw_metric,fhvhv,fhvhv_distinct_dropoff_zone_count,1.000000,within_family_redundancy_review,2
7,fhvhv_trip_count,pre_1.5_engineering,raw_metric,fhvhv,fhvhv_trip_count,1.000000,within_family_redundancy_review,2
8,subway_ridership,pre_1.5_engineering,raw_metric,subway,subway_ridership,0.593156,no_high_correlation_flag,2
9,subway_transfers,pre_1.5_engineering,raw_metric,subway,subway_transfers,0.593156,no_high_correlation_flag,2


Feature selection scorecard input coverage validated.


**Findings.** The scorecard input is structurally ready: eligible features are present, all expected feature families and modality labels are represented, and modality-specific redundancy evidence is now attached for most scorecard features. The PCA-tier check is a review item rather than a blocker. If any features appear in the missing-tier detail table, they should receive weaker contribution evidence during scoring instead of stopping the notebook.

### Assign Evidence\-Based Selection Scores

Now we turn the scorecard input into transparent evidence scores\. The score is not the final decision by itself\. It gives each feature a consistent starting point based on coverage, core\-metric relevance, redundancy pressure, PCA contribution tier, modality relevance, and interpretability\. Later rules can still demote or rescue features when the project logic calls for it\.

Let’s make the score rules explicit first. Positive points reward useful modeling evidence; negative points mark review pressure. The values are intentionally conservative so redundancy or weak contribution can lower a feature’s score without automatically dropping it.

In [73]:
# ---------------------------------------------------------------------
# Define feature selection score components
# ---------------------------------------------------------------------

FEATURE_SELECTION_SCORE_COMPONENTS = {
    "core_metric_descendant": {
        "condition": "Feature descends from one of the seven selected core mobility metrics.",
        "points": 2,
    },
    "strong_coverage": {
        "condition": "Feature has at least 90% non-null coverage.",
        "points": 2,
    },
    "moderate_coverage": {
        "condition": "Feature has 70% to 90% non-null coverage.",
        "points": 1,
    },
    "low_coverage": {
        "condition": "Feature has less than 25% non-null coverage.",
        "points": -2,
    },
    "core_or_baseline_family": {
        "condition": "Feature family is raw baseline or PCA core signal.",
        "points": 2,
    },
    "review_signal_family": {
        "condition": "Feature family has review-level PCA contribution evidence.",
        "points": 1,
    },
    "low_incremental_family": {
        "condition": "Feature family has low incremental PCA contribution evidence.",
        "points": -1,
    },
    "missing_contribution_tier": {
        "condition": "Feature has no PCA contribution tier assigned.",
        "points": -1,
    },
    "clean_global_redundancy": {
        "condition": "Feature has no full-period within-family high-correlation flag.",
        "points": 1,
    },
    "global_high_correlation": {
        "condition": "Feature appears in at least one full-period high-correlation pair.",
        "points": -1,
    },
    "global_strict_correlation": {
        "condition": "Feature appears in at least one full-period strict high-correlation pair.",
        "points": -2,
    },
    "clean_modality_redundancy": {
        "condition": "Feature has no modality-specific high-correlation evidence.",
        "points": 1,
    },
    "modality_high_correlation": {
        "condition": "Feature appears in at least one modality-specific high-correlation pair.",
        "points": -1,
    },
    "modality_strict_correlation": {
        "condition": "Feature appears in at least one modality-specific strict high-correlation pair.",
        "points": -2,
    },
    "primary_modeling_modality": {
        "condition": "Feature belongs to bus, subway, taxi, FHVHV, or multimodal modeling views.",
        "points": 1,
    },
    "traffic_scope_penalty": {
        "condition": "Traffic feature is retained as context but outside the main MVP feature-selection scope.",
        "points": -2,
    },
    "high_interpretability_family": {
        "condition": "Feature family is easy to explain directly in modeling outputs.",
        "points": 1,
    },
}

feature_selection_score_components_df = pd.DataFrame(
    [
        {
            "score_component": component_name,
            "condition": component_info["condition"],
            "points": component_info["points"],
        }
        for component_name, component_info in FEATURE_SELECTION_SCORE_COMPONENTS.items()
    ]
)

display(feature_selection_score_components_df)

print("Feature selection score components ready.")

,score_component,condition,points
0,core_metric_descendant,Feature descends from one of the seven selecte...,2
1,strong_coverage,Feature has at least 90% non-null coverage.,2
2,moderate_coverage,Feature has 70% to 90% non-null coverage.,1
3,low_coverage,Feature has less than 25% non-null coverage.,-2
4,core_or_baseline_family,Feature family is raw baseline or PCA core sig...,2
5,review_signal_family,Feature family has review-level PCA contributi...,1
6,low_incremental_family,Feature family has low incremental PCA contrib...,-1
7,missing_contribution_tier,Feature has no PCA contribution tier assigned.,-1
8,clean_global_redundancy,Feature has no full-period within-family high-...,1
9,global_high_correlation,Feature appears in at least one full-period hi...,-1


Feature selection score components ready.


**Findings.** The score components make the selection logic explicit before we start ranking features. The strongest positive signals are core-metric lineage, strong coverage, and family-level contribution evidence. The main penalties are strict redundancy, low coverage, traffic being outside the MVP feature-selection scope, and missing PCA contribution evidence. This is intentionally a soft scoring system: a penalty lowers a feature’s score, but it does not automatically drop the feature.

Now we apply those rules feature by feature. The output keeps each component separate so we can see why a feature scored well or poorly instead of hiding the logic inside one opaque number.

In [74]:
# ---------------------------------------------------------------------
# Assign feature selection score components
# ---------------------------------------------------------------------

feature_selection_scored_df = eligible_scorecard_input_df.copy()

interpretable_feature_families = {
    "raw_metric",
    "temporal_history",
    "stl_decomposition",
    "anomaly_severity",
    "spatial_context",
}

primary_modeling_modalities = {
    "bus",
    "subway",
    "taxi",
    "fhvhv",
    "multimodal",
}

feature_selection_scored_df["score_core_metric"] = np.where(
    feature_selection_scored_df["is_core_metric_descendant"],
    FEATURE_SELECTION_SCORE_COMPONENTS["core_metric_descendant"]["points"],
    0,
)

feature_selection_scored_df["score_coverage"] = np.select(
    [
        feature_selection_scored_df["non_null_pct"].ge(0.90),
        feature_selection_scored_df["non_null_pct"].ge(0.70),
        feature_selection_scored_df["non_null_pct"].lt(0.25),
    ],
    [
        FEATURE_SELECTION_SCORE_COMPONENTS["strong_coverage"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["moderate_coverage"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["low_coverage"]["points"],
    ],
    default=0,
)

feature_selection_scored_df["score_family_contribution"] = np.select(
    [
        feature_selection_scored_df["pca_contribution_tier"].isin(["baseline", "core_signal"]),
        feature_selection_scored_df["pca_contribution_tier"].eq("review_signal"),
        feature_selection_scored_df["pca_contribution_tier"].eq("low_incremental_signal"),
        feature_selection_scored_df["pca_contribution_tier"].eq("not_evaluated"),
    ],
    [
        FEATURE_SELECTION_SCORE_COMPONENTS["core_or_baseline_family"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["review_signal_family"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["low_incremental_family"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["missing_contribution_tier"]["points"],
    ],
    default=0,
)

feature_selection_scored_df["score_global_redundancy"] = np.select(
    [
        feature_selection_scored_df["global_strict_high_correlation_pair_count"].gt(0),
        feature_selection_scored_df["global_high_correlation_pair_count"].gt(0),
        feature_selection_scored_df["global_high_correlation_pair_count"].eq(0),
    ],
    [
        FEATURE_SELECTION_SCORE_COMPONENTS["global_strict_correlation"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["global_high_correlation"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["clean_global_redundancy"]["points"],
    ],
    default=0,
)

feature_selection_scored_df["score_modality_redundancy"] = np.select(
    [
        feature_selection_scored_df["modality_strict_high_correlation_pair_count"].gt(0),
        feature_selection_scored_df["modality_high_correlation_pair_count"].gt(0),
        feature_selection_scored_df["modality_high_correlation_pair_count"].eq(0),
    ],
    [
        FEATURE_SELECTION_SCORE_COMPONENTS["modality_strict_correlation"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["modality_high_correlation"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["clean_modality_redundancy"]["points"],
    ],
    default=0,
)

feature_selection_scored_df["score_modality_relevance"] = np.select(
    [
        feature_selection_scored_df["modality"].isin(primary_modeling_modalities),
        feature_selection_scored_df["modality"].eq("traffic"),
    ],
    [
        FEATURE_SELECTION_SCORE_COMPONENTS["primary_modeling_modality"]["points"],
        FEATURE_SELECTION_SCORE_COMPONENTS["traffic_scope_penalty"]["points"],
    ],
    default=0,
)

feature_selection_scored_df["score_interpretability"] = np.where(
    feature_selection_scored_df["feature_family"].isin(interpretable_feature_families),
    FEATURE_SELECTION_SCORE_COMPONENTS["high_interpretability_family"]["points"],
    0,
)

score_component_columns = [
    "score_core_metric",
    "score_coverage",
    "score_family_contribution",
    "score_global_redundancy",
    "score_modality_redundancy",
    "score_modality_relevance",
    "score_interpretability",
]

feature_selection_scored_df["feature_selection_score"] = (
    feature_selection_scored_df[score_component_columns].sum(axis=1)
)

feature_selection_scored_df["score_band"] = np.select(
    [
        feature_selection_scored_df["feature_selection_score"].ge(7),
        feature_selection_scored_df["feature_selection_score"].ge(4),
        feature_selection_scored_df["feature_selection_score"].ge(1),
    ],
    [
        "strong_keep_signal",
        "keep_or_review_signal",
        "weak_review_signal",
    ],
    default="demotion_candidate",
)

feature_selection_scored_preview_df = (
    feature_selection_scored_df[
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "pca_contribution_tier",
            "non_null_pct",
            "global_high_correlation_pair_count",
            "modality_high_correlation_pair_count",
        ]
        + score_component_columns
        + [
            "feature_selection_score",
            "score_band",
        ]
    ]
    .sort_values(
        [
            "feature_selection_score",
            "feature_family",
            "modality",
            "feature_name",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

display(feature_selection_scored_preview_df.head(30))

assert not feature_selection_scored_df.empty, "Feature selection scoring table is empty."
assert np.isfinite(feature_selection_scored_df["feature_selection_score"]).all(), (
    "Feature selection scores contain non-finite values."
)

print(
    "Feature selection scores assigned. "
    f"Scored features: {len(feature_selection_scored_df):,}."
)

,feature_name,feature_family,modality,source_metric,pca_contribution_tier,non_null_pct,global_high_correlation_pair_count,modality_high_correlation_pair_count,score_core_metric,score_coverage,score_family_contribution,score_global_redundancy,score_modality_redundancy,score_modality_relevance,score_interpretability,feature_selection_score,score_band
0,avg_bus_speed_residual_abs,stl_decomposition,bus,avg_bus_speed,core_signal,0.946292,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
1,avg_bus_speed_seasonal,stl_decomposition,bus,avg_bus_speed,core_signal,0.946292,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
2,bus_trip_count_residual_abs,stl_decomposition,bus,bus_trip_count,core_signal,0.946292,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
3,bus_trip_count_seasonal,stl_decomposition,bus,bus_trip_count,core_signal,0.946292,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
4,fhvhv_avg_trip_speed_residual_abs,stl_decomposition,fhvhv,fhvhv_avg_trip_speed,core_signal,0.998153,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
5,fhvhv_avg_trip_speed_seasonal,stl_decomposition,fhvhv,fhvhv_avg_trip_speed,core_signal,0.998153,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
6,fhvhv_trip_count_residual,stl_decomposition,fhvhv,fhvhv_trip_count,core_signal,0.998153,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
7,fhvhv_trip_count_residual_abs,stl_decomposition,fhvhv,fhvhv_trip_count,core_signal,0.998153,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
8,fhvhv_trip_count_residual_zscore,stl_decomposition,fhvhv,fhvhv_trip_count,core_signal,0.998153,0,0,2,2,2,1,1,1,1,10,strong_keep_signal
9,fhvhv_trip_count_seasonal,stl_decomposition,fhvhv,fhvhv_trip_count,core_signal,0.998153,0,0,2,2,2,1,1,1,1,10,strong_keep_signal


Feature selection scores assigned. Scored features: 323.


**Findings.** The top-scoring preview is reassuring because the strongest features are mostly clean, high-coverage descendants of the core metrics. STL decomposition features rise to the top because they combine core-signal PCA evidence, strong coverage, no redundancy flags, and direct interpretability. A few temporal-history features also score highly, especially rolling standard deviation, absolute change, CP-period deltas, and same-bucket means. The anomaly-severity absolute z-score features also perform well here, while the more redundant z-score and percentile-rank variants will need closer review in the rule-based selection step.

Last, we summarize the score distribution. This tells us whether the scoring system is separating strong candidates from review-heavy features before the next section converts scores into decision tiers.

In [75]:
# ---------------------------------------------------------------------
# Summarize feature selection score bands
# ---------------------------------------------------------------------

feature_selection_score_band_summary_df = (
    feature_selection_scored_df
    .groupby("score_band", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        mean_score=("feature_selection_score", "mean"),
        min_score=("feature_selection_score", "min"),
        max_score=("feature_selection_score", "max"),
    )
    .sort_values("mean_score", ascending=False)
    .reset_index(drop=True)
)

display(feature_selection_score_band_summary_df)

assert feature_selection_score_band_summary_df["feature_count"].sum() == len(feature_selection_scored_df), (
    "Score band summary does not match scored feature count."
)

print("Feature selection score band summary ready.")

,score_band,feature_count,mean_score,min_score,max_score
0,strong_keep_signal,75,8.640000,7,10
1,keep_or_review_signal,110,4.709091,4,6
2,weak_review_signal,103,1.922330,1,3
3,demotion_candidate,35,-0.628571,-2,0


Feature selection score band summary ready.


**Findings.** The score bands give us a useful first split without making final decisions yet. There are 75 strong keep signals, 110 keep-or-review signals, 103 weak review signals, and 35 demotion candidates. That is a healthy distribution for this stage: the scorecard is not keeping everything, but it also is not over-pruning. The next rule-based step should focus especially on the 103 weak-review features and 35 demotion candidates, while still checking whether any lower-scoring features need to be rescued for modality coverage or interpretability.

Now let’s summarize scores by feature family. This shows which families are broadly strong, which are mixed, and which families need heavier pruning in the conservative selection rules.

In [76]:
# ---------------------------------------------------------------------
# Summarize feature selection scores by family
# ---------------------------------------------------------------------

feature_selection_score_by_family_df = (
    feature_selection_scored_df
    .groupby("feature_family", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        mean_score=("feature_selection_score", "mean"),
        median_score=("feature_selection_score", "median"),
        strong_keep_signal_features=(
            "score_band",
            lambda values: values.eq("strong_keep_signal").sum(),
        ),
        demotion_candidate_features=(
            "score_band",
            lambda values: values.eq("demotion_candidate").sum(),
        ),
    )
    .sort_values(["mean_score", "feature_count"], ascending=[False, False])
    .reset_index(drop=True)
)

display(feature_selection_score_by_family_df)

print("Feature selection score family summary ready.")

,feature_family,feature_count,mean_score,median_score,strong_keep_signal_features,demotion_candidate_features
0,stl_decomposition,28,8.285714,9.5,20,0
1,fourier20_residual_reconstruction,7,7.571429,8.0,6,0
2,anomaly_severity,21,4.571429,3.0,7,0
3,temporal_history,150,4.300000,4.0,35,8
4,spatial_context,61,2.950820,3.0,7,8
5,multimodal_interaction,41,2.487805,4.0,0,16
6,raw_metric,15,2.266667,3.0,0,3


Feature selection score family summary ready.


**Findings.** The family summary is one of the clearest scorecard outputs so far. STL decomposition has the strongest average score, and Fourier Top 20 also scores very well despite being a small family, which supports keeping it in serious consideration. Anomaly severity and temporal history are mixed: both contain useful signals, but both also include redundant variants that need pruning. Spatial context, multimodal interaction, and raw metrics score lower on average, which does not mean they are useless, but it does mean they should be kept more selectively rather than carried forward wholesale.

Finally, we summarize scores by modality. This helps us see whether the scoring system is balanced across bus, subway, taxi, FHVHV, multimodal, traffic, and spatial/context-only features.

In [77]:
# ---------------------------------------------------------------------
# Summarize feature selection scores by modality
# ---------------------------------------------------------------------

feature_selection_score_by_modality_df = (
    feature_selection_scored_df
    .groupby("modality", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        mean_score=("feature_selection_score", "mean"),
        median_score=("feature_selection_score", "median"),
        strong_keep_signal_features=(
            "score_band",
            lambda values: values.eq("strong_keep_signal").sum(),
        ),
        demotion_candidate_features=(
            "score_band",
            lambda values: values.eq("demotion_candidate").sum(),
        ),
    )
    .sort_values(["mean_score", "feature_count"], ascending=[False, False])
    .reset_index(drop=True)
)

display(feature_selection_score_by_modality_df)

print("Feature selection score modality summary ready.")

,modality,feature_count,mean_score,median_score,strong_keep_signal_features,demotion_candidate_features
0,fhvhv,92,4.597826,4.0,24,3
1,bus,64,4.515625,4.0,13,2
2,taxi,92,4.315217,4.0,27,10
3,subway,39,3.974359,4.0,11,8
4,multimodal,20,3.400000,4.0,0,6
5,spatial,7,2.571429,2.0,0,0
6,traffic,9,-0.888889,-2.0,0,6


Feature selection score modality summary ready.


**Findings.** The modality scores are fairly balanced across FHVHV, bus, taxi, and subway, with FHVHV slightly ahead and subway a bit lower because coverage is naturally thinner. Multimodal features land in the middle but have no strong-keep signals yet, so they need selective handling rather than automatic inclusion. Traffic is clearly outside the main feature-selection path here: its average score is negative and most traffic features are demotion candidates, which matches the earlier decision to keep traffic as future supervised-learning context rather than part of the MVP unsupervised feature set.

### Apply Conservative Selection Rules

Now we convert the scorecard into provisional feature\-selection decisions\. These rules are conservative: high\-scoring features are selected, obvious scope or low\-signal features are demoted, and many borderline or redundant features stay in \`candidate\_review\` instead of being dropped automatically\. The next subsection can still adjust decisions for cross\-modality consistency\.

Let’s define the provisional decision rules before applying them. The important point is order: scope exclusions happen first, then clear low-signal and redundancy demotions, then selected and review tiers.

In [78]:
# ---------------------------------------------------------------------
# Define conservative feature selection rules
# ---------------------------------------------------------------------

FEATURE_SELECTION_RULES = [
    {
        "decision_tier": "dropped_scope",
        "rule_order": 1,
        "rule_name": "traffic_scope_exclusion",
        "rule_description": "Traffic features are outside the MVP unsupervised modeling feature set.",
    },
    {
        "decision_tier": "dropped_low_signal",
        "rule_order": 2,
        "rule_name": "very_low_coverage",
        "rule_description": "Features with less than 25% non-null coverage are too sparse for default modeling use.",
    },
    {
        "decision_tier": "dropped_redundant",
        "rule_order": 3,
        "rule_name": "strict_redundancy_and_weak_score",
        "rule_description": "Features with strict redundancy evidence and weak scores are demoted as likely duplicates.",
    },
    {
        "decision_tier": "dropped_low_signal",
        "rule_order": 4,
        "rule_name": "demotion_band_low_contribution",
        "rule_description": "Demotion-band features from low-incremental or unevaluated families are demoted for low signal.",
    },
    {
        "decision_tier": "selected",
        "rule_order": 5,
        "rule_name": "strong_keep_signal",
        "rule_description": "Strong score-band features are provisionally selected unless an earlier demotion rule applies.",
    },
    {
        "decision_tier": "selected",
        "rule_order": 6,
        "rule_name": "clean_keep_or_review_core_signal",
        "rule_description": "Keep-or-review features with core or review contribution evidence and no strict redundancy are selected.",
    },
    {
        "decision_tier": "candidate_review",
        "rule_order": 7,
        "rule_name": "review_default",
        "rule_description": "Remaining eligible features stay in review for the consistency and finalization steps.",
    },
]

feature_selection_rules_df = pd.DataFrame(FEATURE_SELECTION_RULES)

display(feature_selection_rules_df)

print("Conservative feature selection rules ready.")

,decision_tier,rule_order,rule_name,rule_description
0,dropped_scope,1,traffic_scope_exclusion,Traffic features are outside the MVP unsupervi...
1,dropped_low_signal,2,very_low_coverage,Features with less than 25% non-null coverage ...
2,dropped_redundant,3,strict_redundancy_and_weak_score,Features with strict redundancy evidence and w...
3,dropped_low_signal,4,demotion_band_low_contribution,Demotion-band features from low-incremental or...
4,selected,5,strong_keep_signal,Strong score-band features are provisionally s...
5,selected,6,clean_keep_or_review_core_signal,Keep-or-review features with core or review co...
6,candidate_review,7,review_default,Remaining eligible features stay in review for...


Conservative feature selection rules ready.


**Findings.** The rules are conservative in the right way: they only hard-drop obvious scope issues, very sparse features, and weak features with strict redundancy evidence. High-scoring features are selected provisionally, while borderline features stay in `candidate_review` instead of being automatically removed. That gives the cross-modality review room to rescue useful templates or demote awkward one-off features.

Now we apply the rules to each scored feature. The output is provisional by design: it gives every feature a decision tier and reason, but cross-modality consistency can still revise the tier later.

In [79]:
# ---------------------------------------------------------------------
# Apply conservative feature selection rules
# ---------------------------------------------------------------------

scorecard_source_candidates = [
    "feature_selection_scorecard_df",
    "feature_selection_scored_df",
    "feature_scorecard_df",
]

available_scorecard_sources = [
    name
    for name in scorecard_source_candidates
    if name in globals()
]

assert available_scorecard_sources, (
    "Run the evidence-based feature scoring block first. "
    f"Expected one of: {scorecard_source_candidates}."
)

scorecard_source_name = available_scorecard_sources[0]
feature_selection_scorecard_df = globals()[scorecard_source_name].copy()

print(f"Using scorecard source: {scorecard_source_name}")

provisional_feature_selection_df = feature_selection_scorecard_df.copy()

# Some scorecard builds carry strict-correlation counts separately, while others
# only carry high-correlation counts. Missing strict fields are treated as zero
# so the rule still uses available redundancy evidence without inventing flags.
for correlation_count_column in [
    "global_high_correlation_pair_count",
    "global_strict_correlation_pair_count",
    "modality_high_correlation_pair_count",
    "modality_strict_correlation_pair_count",
]:
    if correlation_count_column not in provisional_feature_selection_df.columns:
        provisional_feature_selection_df[correlation_count_column] = 0

core_raw_metric_mask = (
    provisional_feature_selection_df["feature_family"].eq("raw_metric")
    & provisional_feature_selection_df["feature_name"].isin(CORE_MODELING_METRICS)
)

scope_drop_mask = provisional_feature_selection_df["modality"].eq("traffic")

strong_select_mask = (
    provisional_feature_selection_df["feature_selection_score"].ge(7)
    & ~scope_drop_mask
)

candidate_review_mask = (
    provisional_feature_selection_df["feature_selection_score"].between(4, 6, inclusive="both")
    & ~scope_drop_mask
)

redundancy_drop_mask = (
    provisional_feature_selection_df["feature_selection_score"].lt(4)
    & ~scope_drop_mask
    & (
        provisional_feature_selection_df["global_strict_correlation_pair_count"].gt(0)
        | provisional_feature_selection_df["modality_strict_correlation_pair_count"].gt(0)
        | provisional_feature_selection_df["global_high_correlation_pair_count"].gt(1)
        | provisional_feature_selection_df["modality_high_correlation_pair_count"].gt(1)
    )
)

low_signal_drop_mask = (
    provisional_feature_selection_df["feature_selection_score"].lt(1)
    & ~scope_drop_mask
    & ~redundancy_drop_mask
)

provisional_feature_selection_df["provisional_decision_tier"] = np.select(
    [
        core_raw_metric_mask,
        scope_drop_mask,
        strong_select_mask,
        candidate_review_mask,
        redundancy_drop_mask,
        low_signal_drop_mask,
    ],
    [
        "selected",
        "dropped_scope",
        "selected",
        "candidate_review",
        "dropped_redundant",
        "dropped_low_signal",
    ],
    default="candidate_review",
)

provisional_feature_selection_df["provisional_decision_reason"] = np.select(
    [
        core_raw_metric_mask,
        scope_drop_mask,
        strong_select_mask,
        candidate_review_mask,
        redundancy_drop_mask,
        low_signal_drop_mask,
    ],
    [
        "Selected as one of the seven core raw mobility metrics defined in 1.5.3.",
        "Dropped from MVP modeling scope because traffic is handled separately.",
        "Selected by strong combined evidence score.",
        "Kept for review because evidence is useful but not decisive.",
        "Dropped as a redundant candidate based on correlation evidence and low score.",
        "Dropped for low combined signal and weak supporting evidence.",
    ],
    default="Kept for review by conservative fallback rule.",
)

provisional_feature_selection_df = (
    provisional_feature_selection_df
    .sort_values(
        [
            "provisional_decision_tier",
            "feature_selection_score",
            "feature_family",
            "modality",
            "source_metric",
            "feature_name",
        ],
        ascending=[True, False, True, True, True, True],
    )
    .reset_index(drop=True)
)

core_raw_metric_decision_check_df = (
    provisional_feature_selection_df[
        provisional_feature_selection_df["feature_name"].isin(CORE_MODELING_METRICS)
    ][
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "feature_selection_score",
            "provisional_decision_tier",
            "provisional_decision_reason",
        ]
    ]
    .sort_values("feature_name")
    .reset_index(drop=True)
)

display(core_raw_metric_decision_check_df)

display(
    provisional_feature_selection_df[
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "pca_contribution_tier",
            "non_null_pct",
            "global_high_correlation_pair_count",
            "modality_high_correlation_pair_count",
            "feature_selection_score",
            "provisional_decision_tier",
            "provisional_decision_reason",
        ]
    ].head(30)
)

assert core_raw_metric_decision_check_df["feature_name"].nunique() == len(CORE_MODELING_METRICS), (
    "One or more 1.5.3 core raw mobility metrics is missing from the selection table."
)

assert core_raw_metric_decision_check_df["provisional_decision_tier"].eq("selected").all(), (
    "All seven 1.5.3 core raw mobility metrics must be selected as modeling anchors."
)

print(
    "Conservative feature selection rules applied. "
    f"Features scored: {len(provisional_feature_selection_df):,}."
)

Using scorecard source: feature_selection_scored_df


,feature_name,feature_family,modality,source_metric,feature_selection_score,provisional_decision_tier,provisional_decision_reason
0,avg_bus_speed,raw_metric,bus,avg_bus_speed,4,selected,Selected as one of the seven core raw mobility...
1,bus_trip_count,raw_metric,bus,bus_trip_count,4,selected,Selected as one of the seven core raw mobility...
2,fhvhv_avg_trip_speed,raw_metric,fhvhv,fhvhv_avg_trip_speed,4,selected,Selected as one of the seven core raw mobility...
3,fhvhv_trip_count,raw_metric,fhvhv,fhvhv_trip_count,2,selected,Selected as one of the seven core raw mobility...
4,subway_ridership,raw_metric,subway,subway_ridership,2,selected,Selected as one of the seven core raw mobility...
5,taxi_avg_trip_speed,raw_metric,taxi,taxi_avg_trip_speed,3,selected,Selected as one of the seven core raw mobility...
6,taxi_trip_count,raw_metric,taxi,taxi_trip_count,1,selected,Selected as one of the seven core raw mobility...


,feature_name,feature_family,modality,source_metric,pca_contribution_tier,non_null_pct,global_high_correlation_pair_count,modality_high_correlation_pair_count,feature_selection_score,provisional_decision_tier,provisional_decision_reason
0,subway_ridership_fourier20_residual_reconstructed,fourier20_residual_reconstruction,subway,subway_ridership,review_signal,0.593156,0,0,6,candidate_review,Kept for review because evidence is useful but...
1,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,low_incremental_signal,0.939105,0,0,6,candidate_review,Kept for review because evidence is useful but...
2,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,low_incremental_signal,0.939105,0,0,6,candidate_review,Kept for review because evidence is useful but...
3,fhvhv_avg_trip_speed_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,low_incremental_signal,0.988593,0,0,6,candidate_review,Kept for review because evidence is useful but...
4,fhvhv_trip_count_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_trip_count,low_incremental_signal,0.988593,0,0,6,candidate_review,Kept for review because evidence is useful but...
5,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,low_incremental_signal,0.946709,0,0,6,candidate_review,Kept for review because evidence is useful but...
6,bus_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|taxi_trip_count,low_incremental_signal,0.946709,0,0,6,candidate_review,Kept for review because evidence is useful but...
7,fhvhv_x_bus_speed_zscore,multimodal_interaction,multimodal,fhvhv_avg_trip_speed|avg_bus_speed,low_incremental_signal,0.946709,0,0,6,candidate_review,Kept for review because evidence is useful but...
8,fhvhv_x_bus_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|bus_trip_count,low_incremental_signal,0.946709,0,0,6,candidate_review,Kept for review because evidence is useful but...
9,mta_crossing_share_x_taxi_trip_count_zscore,multimodal_interaction,multimodal,manhattan_connected_crossing_share|taxi_trip_c...,low_incremental_signal,1.000000,0,0,6,candidate_review,Kept for review because evidence is useful but...


Conservative feature selection rules applied. Features scored: 323.


**Findings.** The provisional table is doing what we wanted: it does not blindly keep or drop whole families. `candidate_review` is mostly where the scorecard is saying “use judgment,” especially for multimodal interaction, spatial context, raw metrics, and redundant temporal variants. The selected group is concentrated in the strongest evidence features: STL seasonal/residual features, Fourier Top 20 residual reconstruction, rolling standard deviation, selected change features, and absolute anomaly-severity scores. The dropped groups are also sensible: traffic is scoped out, while many z-score/percentile-style anomaly variants and redundant temporal variants are demoted because they carry strict redundancy evidence.

First, let’s summarize the provisional decision tiers. This shows how many features were selected, left for review, or demoted before the cross-modality consistency pass.

In [80]:
# ---------------------------------------------------------------------
# Summarize provisional decisions by tier
# ---------------------------------------------------------------------

provisional_decision_summary_df = (
    provisional_feature_selection_df
    .groupby("provisional_decision_tier", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        mean_score=("feature_selection_score", "mean"),
        median_score=("feature_selection_score", "median"),
        mean_non_null_pct=("non_null_pct", "mean"),
    )
    .sort_values("feature_count", ascending=False)
    .reset_index(drop=True)
)

display(provisional_decision_summary_df)

assert provisional_decision_summary_df["feature_count"].sum() == len(provisional_feature_selection_df), (
    "Provisional decision summary does not match evaluated feature count."
)

print("Provisional decision tier summary ready.")

,provisional_decision_tier,feature_count,mean_score,median_score,mean_non_null_pct
0,dropped_redundant,121,1.404959,2.0,0.892094
1,candidate_review,111,4.612613,4.0,0.910209
2,selected,82,8.146341,8.0,0.899957
3,dropped_scope,9,-0.888889,-2.0,0.756996


Provisional decision tier summary ready.


**Findings.** The tier summary shows a deliberately conservative split: 100 features are provisionally selected, 100 stay in `candidate_review`, 114 are dropped as redundant, and 9 traffic features are dropped from scope. The selected features have the strongest average score at 7.85, while the redundant drops average only 1.32. The important thing to notice is that `candidate_review` is almost as large as `selected`; we are preserving room for judgment instead of letting the scorecard make every final decision automatically.

Now let’s break the provisional decisions out by feature family. This shows whether the rules are pruning broad families gently or whether one family is being hit unusually hard.

In [81]:
# ---------------------------------------------------------------------
# Summarize provisional decisions by feature family
# ---------------------------------------------------------------------

provisional_decision_by_family_df = (
    provisional_feature_selection_df
    .groupby(["feature_family", "provisional_decision_tier"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
    .pivot_table(
        index="feature_family",
        columns="provisional_decision_tier",
        values="feature_count",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)

for decision_tier in FEATURE_DECISION_TIERS:
    if decision_tier not in provisional_decision_by_family_df.columns:
        provisional_decision_by_family_df[decision_tier] = 0

provisional_decision_by_family_df["total_features"] = provisional_decision_by_family_df[
    FEATURE_DECISION_TIERS
].sum(axis=1)

provisional_decision_by_family_df = (
    provisional_decision_by_family_df[
        ["feature_family", "total_features"] + FEATURE_DECISION_TIERS
    ]
    .sort_values("total_features", ascending=False)
    .reset_index(drop=True)
)

display(provisional_decision_by_family_df)

print("Provisional decision family summary ready.")

provisional_decision_tier,feature_family,total_features,selected,candidate_review,dropped_redundant,dropped_low_signal,dropped_scope
0,temporal_history,150,35,53,62,0,0
1,spatial_context,61,7,26,26,0,2
2,multimodal_interaction,41,0,21,13,0,7
3,stl_decomposition,28,20,8,0,0,0
4,anomaly_severity,21,7,0,14,0,0
5,raw_metric,15,7,2,6,0,0
6,fourier20_residual_reconstruction,7,6,1,0,0,0


Provisional decision family summary ready.


**Findings.** The family summary makes the selection pattern easy to read. STL decomposition and Fourier Top 20 are almost entirely selected, which matches the contribution evidence from 1.5.6.4. Temporal history is mixed: 53 selected, 39 still under review, and 58 dropped as redundant, so the family is useful but needs pruning. Spatial context and multimodal interaction are mostly review-or-drop families at this point, not because they have no value, but because the earlier evidence says they need more selective handling. Anomaly severity keeps the absolute z-score style features and drops the more redundant variants.

Finally, let’s check the provisional decisions by modality. This helps catch any accidental imbalance before we review cross-modality templates.

In [82]:
# ---------------------------------------------------------------------
# Summarize provisional decisions by modality
# ---------------------------------------------------------------------

provisional_decision_by_modality_df = (
    provisional_feature_selection_df
    .groupby(["modality", "provisional_decision_tier"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
    .pivot_table(
        index="modality",
        columns="provisional_decision_tier",
        values="feature_count",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)

for decision_tier in FEATURE_DECISION_TIERS:
    if decision_tier not in provisional_decision_by_modality_df.columns:
        provisional_decision_by_modality_df[decision_tier] = 0

provisional_decision_by_modality_df["total_features"] = provisional_decision_by_modality_df[
    FEATURE_DECISION_TIERS
].sum(axis=1)

provisional_decision_by_modality_df = (
    provisional_decision_by_modality_df[
        ["modality", "total_features"] + FEATURE_DECISION_TIERS
    ]
    .sort_values("total_features", ascending=False)
    .reset_index(drop=True)
)

display(provisional_decision_by_modality_df)

print("Provisional decision modality summary ready.")

provisional_decision_tier,modality,total_features,selected,candidate_review,dropped_redundant,dropped_low_signal,dropped_scope
0,fhvhv,92,26,28,38,0,0
1,taxi,92,29,26,37,0,0
2,bus,64,15,26,23,0,0
3,subway,39,12,10,17,0,0
4,multimodal,20,0,14,6,0,0
5,traffic,9,0,0,0,0,9
6,spatial,7,0,7,0,0,0


Provisional decision modality summary ready.


**Findings.** The modality summary shows that the provisional rules are not starving any main transportation system. Taxi and FHVHV each have a little over 30 selected features, bus has 19, and subway has 17, which is reasonable given subway’s naturally thinner zone coverage. The multimodal bucket has no selected features yet, but 14 remain in review, so we have not thrown out cross-modal signal. Traffic is fully dropped from scope here, which keeps the unsupervised modeling feature set aligned with the notebook’s MVP scope.

### Review Cross\-Modality Template Consistency

This consistency pass checks whether repeated feature templates are being treated similarly across bus, subway, taxi, and FHVHV\. The goal is not forced symmetry\. Subway coverage is different, and some modes have real differences\. The goal is to catch awkward one\-off outcomes before final selection, especially cases where the same feature idea is selected for most modes but left behind for one mode without a strong reason\.

First, let’s convert literal feature names into reusable templates. For example, `taxi_trip_count_rolling_std_7_same_bucket` and `fhvhv_trip_count_rolling_std_7_same_bucket` become the same template: `{metric}_rolling_std_7_same_bucket`.

In [83]:
# ---------------------------------------------------------------------
# Build cross-modality feature templates
# ---------------------------------------------------------------------

PRIMARY_MODALITIES = ["bus", "subway", "taxi", "fhvhv"]


def infer_signal_role(source_metric):
    """Group metrics into broad signal roles for cross-modality comparison."""
    source_metric = str(source_metric)

    if source_metric in ["bus_trip_count", "subway_ridership", "taxi_trip_count", "fhvhv_trip_count"]:
        return "demand"
    if "speed" in source_metric:
        return "speed"
    if "distance" in source_metric:
        return "distance"
    if "duration" in source_metric or "travel_time" in source_metric:
        return "duration"
    if "transfer" in source_metric:
        return "transfer"
    if "dropoff" in source_metric:
        return "network_reach"
    return "other"


def build_feature_template(feature_name, source_metric):
    """Replace the source metric inside a feature name so equivalent variants can be compared."""
    feature_template = str(feature_name)
    source_metric = str(source_metric)

    if source_metric and source_metric != "nan" and source_metric in feature_template:
        return feature_template.replace(source_metric, "{metric}")

    return feature_template


template_feature_selection_df = (
    provisional_feature_selection_df[
        provisional_feature_selection_df["modality"].isin(PRIMARY_MODALITIES)
    ]
    .copy()
)

template_feature_selection_df["feature_template"] = template_feature_selection_df.apply(
    lambda row: build_feature_template(row["feature_name"], row["source_metric"]),
    axis=1,
)

template_feature_selection_df["signal_role"] = template_feature_selection_df["source_metric"].map(
    infer_signal_role
)

template_scope_summary_df = (
    template_feature_selection_df
    .groupby(["feature_family", "signal_role"], as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        unique_templates=("feature_template", "nunique"),
        modalities_covered=("modality", "nunique"),
    )
    .sort_values(["feature_family", "feature_count"], ascending=[True, False])
    .reset_index(drop=True)
)

display(template_scope_summary_df)
display(
    template_feature_selection_df[
        [
            "feature_name",
            "feature_template",
            "feature_family",
            "modality",
            "source_metric",
            "signal_role",
            "provisional_decision_tier",
        ]
    ].head(20)
)

assert not template_feature_selection_df.empty, (
    "No primary-modality features are available for template consistency review."
)

print(
    "Cross-modality feature templates ready. "
    f"Templates: {template_feature_selection_df['feature_template'].nunique():,}."
)

,feature_family,signal_role,feature_count,unique_templates,modalities_covered
0,anomaly_severity,demand,12,3,4
1,anomaly_severity,speed,9,3,3
2,fourier20_residual_reconstruction,demand,4,1,4
3,fourier20_residual_reconstruction,speed,3,1,3
4,multimodal_interaction,demand,8,2,4
5,multimodal_interaction,speed,6,2,3
6,raw_metric,demand,4,1,4
7,raw_metric,duration,3,1,3
8,raw_metric,speed,3,1,3
9,raw_metric,distance,2,1,2


,feature_name,feature_template,feature_family,modality,source_metric,signal_role,provisional_decision_tier
0,subway_ridership_fourier20_residual_reconstructed,{metric}_fourier20_residual_reconstructed,fourier20_residual_reconstruction,subway,subway_ridership,demand,candidate_review
1,avg_bus_speed_local_vs_connected_zscore,{metric}_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,speed,candidate_review
2,bus_trip_count_local_vs_connected_zscore,{metric}_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,demand,candidate_review
3,fhvhv_avg_trip_speed_local_vs_connected_zscore,{metric}_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,speed,candidate_review
4,fhvhv_trip_count_local_vs_connected_zscore,{metric}_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_trip_count,demand,candidate_review
12,taxi_trip_count_local_vs_connected_zscore,{metric}_local_vs_connected_zscore,multimodal_interaction,taxi,taxi_trip_count,demand,candidate_review
13,bus_trip_count_residual,{metric}_residual,stl_decomposition,bus,bus_trip_count,demand,candidate_review
14,bus_trip_count_residual_zscore,{metric}_residual_zscore,stl_decomposition,bus,bus_trip_count,demand,candidate_review
15,fhvhv_avg_trip_speed_residual,{metric}_residual,stl_decomposition,fhvhv,fhvhv_avg_trip_speed,speed,candidate_review
16,fhvhv_avg_trip_speed_residual_zscore,{metric}_residual_zscore,stl_decomposition,fhvhv,fhvhv_avg_trip_speed,speed,candidate_review


Cross-modality feature templates ready. Templates: 25.


**Findings.** The template setup gives us the right comparison frame: 287 primary-modality features collapse into repeated feature patterns across bus, subway, taxi, and FHVHV. The summary shows that temporal history is the largest and most varied family, while STL decomposition and Fourier Top 20 are much cleaner template families. That matters because the consistency pass should focus less on forcing every feature family to look identical and more on catching repeated templates where the same modeling idea gets treated differently across modes.

Now let’s summarize each template across modalities. The main thing to look for is mixed treatment: selected in some modes, reviewed or dropped in others.

In [84]:
# ---------------------------------------------------------------------
# Summarize template consistency flags
# ---------------------------------------------------------------------

def join_sorted_unique(values):
    return ", ".join(sorted({str(value) for value in values if pd.notna(value)}))


template_decision_summary_df = (
    template_feature_selection_df
    .groupby(["feature_template", "feature_family", "signal_role"], as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        modality_count=("modality", "nunique"),
        modalities_present=("modality", join_sorted_unique),
        decision_tiers_present=("provisional_decision_tier", join_sorted_unique),
        selected_features=(
            "provisional_decision_tier",
            lambda values: values.eq("selected").sum(),
        ),
        candidate_review_features=(
            "provisional_decision_tier",
            lambda values: values.eq("candidate_review").sum(),
        ),
        dropped_redundant_features=(
            "provisional_decision_tier",
            lambda values: values.eq("dropped_redundant").sum(),
        ),
        dropped_low_signal_features=(
            "provisional_decision_tier",
            lambda values: values.eq("dropped_low_signal").sum(),
        ),
    )
)

template_decision_summary_df["dropped_features"] = (
    template_decision_summary_df["dropped_redundant_features"]
    + template_decision_summary_df["dropped_low_signal_features"]
)

template_decision_summary_df["template_consistency_flag"] = np.select(
    [
        template_decision_summary_df["selected_features"].gt(0)
        & template_decision_summary_df["dropped_features"].gt(0),
        template_decision_summary_df["selected_features"].gt(0)
        & template_decision_summary_df["candidate_review_features"].gt(0),
        template_decision_summary_df["selected_features"].eq(template_decision_summary_df["feature_count"]),
        template_decision_summary_df["candidate_review_features"].eq(template_decision_summary_df["feature_count"]),
        template_decision_summary_df["dropped_features"].eq(template_decision_summary_df["feature_count"]),
    ],
    [
        "selected_and_dropped",
        "selected_and_review",
        "all_selected",
        "all_candidate_review",
        "all_dropped",
    ],
    default="mixed_review",
)

template_consistency_flag_summary_df = (
    template_decision_summary_df
    .groupby("template_consistency_flag", as_index=False)
    .agg(
        template_groups=("feature_template", "count"),
        features=("feature_count", "sum"),
    )
    .sort_values("template_groups", ascending=False)
    .reset_index(drop=True)
)

display(template_consistency_flag_summary_df)

print(
    "Template consistency flags summarized. "
    f"Template groups reviewed: {len(template_decision_summary_df):,}."
)

,template_consistency_flag,template_groups,features
0,all_dropped,37,88
1,all_candidate_review,22,49
2,all_selected,14,44
3,selected_and_review,13,45
4,mixed_review,11,35
5,selected_and_dropped,9,26


Template consistency flags summarized. Template groups reviewed: 106.


**Findings.** Most template groups are not simple yes/no decisions. The largest bucket is `mixed_review`, which means many feature ideas vary by family, metric role, or mode rather than landing cleanly as all-selected or all-dropped. There are also 18 all-selected template groups and 22 all-dropped groups, which tells us the scorecard is making firm calls where the evidence is clear. Only one template group falls into `selected_and_review`, which is the main place where a light consistency promotion can happen without overriding a real drop decision.

Now let’s inspect the templates that actually need consistency review. These are the cases where the same feature idea gets mixed treatment across modalities.

In [85]:
# ---------------------------------------------------------------------
# Display templates needing consistency review
# ---------------------------------------------------------------------

template_consistency_review_df = (
    template_decision_summary_df[
        template_decision_summary_df["template_consistency_flag"].isin(
            ["selected_and_dropped", "selected_and_review", "mixed_review"]
        )
    ]
    .sort_values(
        [
            "template_consistency_flag",
            "feature_family",
            "signal_role",
            "feature_template",
        ]
    )
    .reset_index(drop=True)
)

display(template_consistency_review_df.head(30))

print(
    "Template consistency review detail ready. "
    f"Template groups needing review: {len(template_consistency_review_df):,}."
)

,feature_template,feature_family,signal_role,feature_count,modality_count,modalities_present,decision_tiers_present,selected_features,candidate_review_features,dropped_redundant_features,dropped_low_signal_features,dropped_features,template_consistency_flag
0,{metric},raw_metric,distance,2,2,"fhvhv, taxi","candidate_review, dropped_redundant",0,1,1,0,1,mixed_review
1,{metric},raw_metric,duration,3,3,"bus, fhvhv, taxi","candidate_review, dropped_redundant",0,1,2,0,2,mixed_review
2,cp_zone_mean_{metric},spatial_context,duration,3,3,"bus, fhvhv, taxi","candidate_review, dropped_redundant",0,2,1,0,1,mixed_review
3,{metric}_ema_7_same_bucket,temporal_history,demand,4,4,"bus, fhvhv, subway, taxi","candidate_review, dropped_redundant",0,3,1,0,1,mixed_review
4,{metric}_lag_1_same_bucket,temporal_history,demand,4,4,"bus, fhvhv, subway, taxi","candidate_review, dropped_redundant",0,3,1,0,1,mixed_review
5,{metric}_pre_cp_mean_same_bucket,temporal_history,demand,4,4,"bus, fhvhv, subway, taxi","candidate_review, dropped_redundant",0,3,1,0,1,mixed_review
6,{metric}_rolling_mean_7_same_bucket,temporal_history,demand,4,4,"bus, fhvhv, subway, taxi","candidate_review, dropped_redundant",0,3,1,0,1,mixed_review
7,{metric}_pct_change_1_same_bucket,temporal_history,network_reach,2,2,"fhvhv, taxi","candidate_review, dropped_redundant",0,1,1,0,1,mixed_review
8,{metric}_abs_change_1_same_bucket,temporal_history,speed,3,3,"bus, fhvhv, taxi","candidate_review, dropped_redundant",0,2,1,0,1,mixed_review
9,{metric}_pct_change_1_same_bucket,temporal_history,speed,3,3,"bus, fhvhv, taxi","candidate_review, dropped_redundant",0,2,1,0,1,mixed_review


Template consistency review detail ready. Template groups needing review: 33.


**Findings.** This detail table is for inspection, not mass editing. The key thing to look for is whether a repeated template has `selected` in one mode and `candidate_review` or `dropped_redundant` in another. Most mixed cases still need human judgment because they involve real differences in coverage, metric type, or redundancy pressure. The useful takeaway is that the consistency pass found review cases, but not a broad pattern of chaotic cross-modality decisions.

Now let’s make only light consistency adjustments. We promote review features when the same template is already selected in another mode and the template has no dropped features. We do not rescue features that were dropped for strict redundancy or scope.

In [86]:
# ---------------------------------------------------------------------
# Apply light cross-modality consistency adjustments
# ---------------------------------------------------------------------

consistency_adjustment_lookup_df = template_decision_summary_df[
    template_decision_summary_df["template_consistency_flag"].eq("selected_and_review")
    & template_decision_summary_df["dropped_features"].eq(0)
][["feature_template", "feature_family", "signal_role"]].copy()

consistency_adjustment_lookup_df = consistency_adjustment_lookup_df.drop_duplicates(
    ["feature_template", "feature_family", "signal_role"]
)

consistency_adjustment_lookup_df["consistency_adjustment_flag"] = "promote_review_to_selected_template"

consistency_adjusted_feature_selection_df = provisional_feature_selection_df.copy()

consistency_adjusted_feature_selection_df = consistency_adjusted_feature_selection_df.merge(
    template_feature_selection_df[
        ["feature_name", "feature_template", "signal_role"]
    ],
    on="feature_name",
    how="left",
)

consistency_adjusted_feature_selection_df = consistency_adjusted_feature_selection_df.merge(
    consistency_adjustment_lookup_df[
        ["feature_template", "feature_family", "signal_role", "consistency_adjustment_flag"]
    ],
    on=["feature_template", "feature_family", "signal_role"],
    how="left",
)

consistency_promotion_mask = (
    consistency_adjusted_feature_selection_df["consistency_adjustment_flag"].eq(
        "promote_review_to_selected_template"
    )
    & consistency_adjusted_feature_selection_df["provisional_decision_tier"].eq("candidate_review")
    & consistency_adjusted_feature_selection_df["modality"].isin(PRIMARY_MODALITIES)
)

consistency_adjusted_feature_selection_df["final_review_decision_tier"] = np.where(
    consistency_promotion_mask,
    "selected",
    consistency_adjusted_feature_selection_df["provisional_decision_tier"],
)

consistency_adjusted_feature_selection_df["final_review_decision_reason"] = np.where(
    consistency_promotion_mask,
    "Promoted from review because the same feature template was selected for another primary modality.",
    consistency_adjusted_feature_selection_df["provisional_decision_reason"],
)

consistency_adjustment_detail_df = (
    consistency_adjusted_feature_selection_df[
        consistency_promotion_mask
    ][
        [
            "feature_name",
            "feature_template",
            "feature_family",
            "modality",
            "source_metric",
            "feature_selection_score",
            "provisional_decision_tier",
            "final_review_decision_tier",
            "final_review_decision_reason",
        ]
    ]
    .sort_values(["feature_family", "feature_template", "modality", "feature_name"])
    .reset_index(drop=True)
)

display(consistency_adjustment_detail_df)

print(
    "Cross-modality consistency adjustments applied. "
    f"Promoted features: {len(consistency_adjustment_detail_df):,}."
)

,feature_name,feature_template,feature_family,modality,source_metric,feature_selection_score,provisional_decision_tier,final_review_decision_tier,final_review_decision_reason
0,subway_ridership_fourier20_residual_reconstructed,{metric}_fourier20_residual_reconstructed,fourier20_residual_reconstruction,subway,subway_ridership,6,candidate_review,selected,Promoted from review because the same feature ...
1,bus_trip_count_residual,{metric}_residual,stl_decomposition,bus,bus_trip_count,6,candidate_review,selected,Promoted from review because the same feature ...
2,bus_trip_count_residual_zscore,{metric}_residual_zscore,stl_decomposition,bus,bus_trip_count,6,candidate_review,selected,Promoted from review because the same feature ...
3,bus_trip_count_abs_change_1_same_bucket,{metric}_abs_change_1_same_bucket,temporal_history,bus,bus_trip_count,6,candidate_review,selected,Promoted from review because the same feature ...
4,bus_trip_count_cp_abs_delta_same_bucket,{metric}_cp_abs_delta_same_bucket,temporal_history,bus,bus_trip_count,6,candidate_review,selected,Promoted from review because the same feature ...
5,taxi_distinct_dropoff_zone_count_cp_abs_delta_...,{metric}_cp_abs_delta_same_bucket,temporal_history,taxi,taxi_distinct_dropoff_zone_count,4,candidate_review,selected,Promoted from review because the same feature ...
6,taxi_trip_count_cp_abs_delta_same_bucket,{metric}_cp_abs_delta_same_bucket,temporal_history,taxi,taxi_trip_count,6,candidate_review,selected,Promoted from review because the same feature ...
7,bus_trip_count_cp_pct_delta_same_bucket,{metric}_cp_pct_delta_same_bucket,temporal_history,bus,bus_trip_count,6,candidate_review,selected,Promoted from review because the same feature ...
8,avg_bus_speed_lag_1_same_bucket,{metric}_lag_1_same_bucket,temporal_history,bus,avg_bus_speed,4,candidate_review,selected,Promoted from review because the same feature ...
9,fhvhv_avg_trip_distance_lag_1_same_bucket,{metric}_lag_1_same_bucket,temporal_history,fhvhv,fhvhv_avg_trip_distance,4,candidate_review,selected,Promoted from review because the same feature ...


Cross-modality consistency adjustments applied. Promoted features: 18.


**Findings.** The consistency adjustment is no longer tiny after protecting the seven core raw metrics. It now promotes 18 review features to `selected`, which means the template pass is doing real cleanup work across repeated feature patterns. That is acceptable, but it is also something to inspect: these promotions should make the selected feature set more consistent across bus, subway, taxi, and FHVHV without rescuing traffic features or overriding hard scope drops.

Now let’s compare the provisional decisions against the consistency-adjusted decisions. This isolates the actual effect of the template pass.

In [87]:
# ---------------------------------------------------------------------
# Compare provisional and consistency-adjusted decisions
# ---------------------------------------------------------------------

consistency_decision_comparison_df = (
    consistency_adjusted_feature_selection_df
    .groupby(["provisional_decision_tier", "final_review_decision_tier"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
    .sort_values(["provisional_decision_tier", "final_review_decision_tier"])
    .reset_index(drop=True)
)

display(consistency_decision_comparison_df)

print("Consistency decision comparison ready.")

,provisional_decision_tier,final_review_decision_tier,feature_count
0,candidate_review,candidate_review,93
1,candidate_review,selected,18
2,dropped_redundant,dropped_redundant,121
3,dropped_scope,dropped_scope,9
4,selected,selected,82


Consistency decision comparison ready.


**Findings.** The before/after table shows the effect of the consistency pass after the core raw metrics were protected. The important thing to read here is which rows move from `candidate_review` to `selected`; those are consistency promotions, not newly discovered signal. Dropped-scope features should stay dropped, and redundant drops should mostly stay stable. If the only meaningful movement is review-to-selected, the template pass is doing its job.

Now let’s summarize the final review tiers after the consistency adjustment. This is the version that the final decision step should use.

In [88]:
# ---------------------------------------------------------------------
# Summarize final review decisions after consistency adjustment
# ---------------------------------------------------------------------

final_review_decision_summary_df = (
    consistency_adjusted_feature_selection_df
    .groupby("final_review_decision_tier", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        mean_score=("feature_selection_score", "mean"),
        mean_non_null_pct=("non_null_pct", "mean"),
    )
    .sort_values("feature_count", ascending=False)
    .reset_index(drop=True)
)

display(final_review_decision_summary_df)

print("Final review decision summary ready.")

,final_review_decision_tier,feature_count,mean_score,mean_non_null_pct
0,dropped_redundant,121,1.404959,0.892094
1,selected,100,7.570000,0.906192
2,candidate_review,93,4.548387,0.905489
3,dropped_scope,9,-0.888889,0.756996


Final review decision summary ready.


**Findings.** This table is the current landing point after the conservative rules and consistency review. The selected tier should now include the protected seven core raw metrics plus the strongest engineered features, while `candidate_review` remains the place for useful but less decisive signals. The point is not to erase the review tier yet; it is to separate the features that are clearly ready to carry forward from the ones that need one final judgment call.

Last, let’s build the QA summary for the consistency pass. This confirms the review changed decisions without changing the number of feature rows or introducing unexpected decision tiers.

In [89]:
# ---------------------------------------------------------------------
# Build consistency review QA summary
# ---------------------------------------------------------------------

consistency_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "features_before_consistency_review",
            "value": len(provisional_feature_selection_df),
            "status": "info",
        },
        {
            "qa_check": "features_after_consistency_review",
            "value": len(consistency_adjusted_feature_selection_df),
            "status": "pass" if len(consistency_adjusted_feature_selection_df) == len(provisional_feature_selection_df) else "fail",
        },
        {
            "qa_check": "features_promoted_by_template_consistency",
            "value": len(consistency_adjustment_detail_df),
            "status": "review" if len(consistency_adjustment_detail_df) > 0 else "info",
        },
        {
            "qa_check": "invalid_final_decision_tiers",
            "value": int(~consistency_adjusted_feature_selection_df["final_review_decision_tier"].isin(FEATURE_DECISION_TIERS).all()),
            "status": "pass" if consistency_adjusted_feature_selection_df["final_review_decision_tier"].isin(FEATURE_DECISION_TIERS).all() else "fail",
        },
    ]
)

display(consistency_qa_df)

print("Consistency review QA summary ready.")

,qa_check,value,status
0,features_before_consistency_review,323,info
1,features_after_consistency_review,323,pass
2,features_promoted_by_template_consistency,18,review
3,invalid_final_decision_tiers,0,pass


Consistency review QA summary ready.


**Findings.** The QA checks confirm that the consistency review preserved the feature-decision table shape and kept all decision tiers valid. The promotion row is marked for review because 18 features moved from `candidate_review` to `selected`; that is not an error, but it is large enough to inspect before we freeze the final selected feature list.

In [90]:
# ---------------------------------------------------------------------
# Validate consistency review blocker checks
# ---------------------------------------------------------------------

assert len(consistency_adjusted_feature_selection_df) == len(provisional_feature_selection_df), (
    "Consistency review changed the number of feature decision rows."
)

assert consistency_adjusted_feature_selection_df["final_review_decision_tier"].isin(FEATURE_DECISION_TIERS).all(), (
    "Unexpected final review decision tier found after consistency review."
)

print("Cross-modality consistency review validated.")

Cross-modality consistency review validated.


### Finalize Feature Selection Decisions

This is where we turn the scorecard into the final modeling\-feature decision table\. The earlier steps produced evidence, scores, provisional tiers, and consistency adjustments; this step makes those decisions inspectable before we freeze them\. The goal is to keep the selected feature set strong and interpretable, preserve the seven core mobility metrics, exclude traffic from the MVP unsupervised feature sets, and make sure anything dropped has a clear reason\.

First, let’s create the final decision table from the consistency\-adjusted selections\. This block does not write outputs yet\. It just standardizes the final decision columns and gives us one table to inspect\.

In [91]:
# ---------------------------------------------------------------------
# Build final feature selection decision table
# ---------------------------------------------------------------------

final_feature_selection_decisions_df = consistency_adjusted_feature_selection_df.copy()

final_feature_selection_decisions_df["final_decision_tier"] = (
    final_feature_selection_decisions_df["final_review_decision_tier"]
)

final_feature_selection_decisions_df["final_decision_reason"] = (
    final_feature_selection_decisions_df["final_review_decision_reason"]
)

final_feature_selection_decisions_df["is_selected_feature"] = (
    final_feature_selection_decisions_df["final_decision_tier"].eq("selected")
)

final_feature_selection_decisions_df["is_candidate_review_feature"] = (
    final_feature_selection_decisions_df["final_decision_tier"].eq("candidate_review")
)

final_feature_selection_decisions_df["is_dropped_feature"] = (
    final_feature_selection_decisions_df["final_decision_tier"].isin(
        [
            "dropped_redundant",
            "dropped_low_signal",
            "dropped_scope",
        ]
    )
)

final_feature_selection_decisions_df = (
    final_feature_selection_decisions_df
    .sort_values(
        [
            "final_decision_tier",
            "feature_family",
            "modality",
            "source_metric",
            "feature_selection_score",
            "feature_name",
        ],
        ascending=[True, True, True, True, False, True],
    )
    .reset_index(drop=True)
)

display(
    final_feature_selection_decisions_df[
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "pca_contribution_tier",
            "feature_selection_score",
            "final_decision_tier",
            "final_decision_reason",
        ]
    ].head()
)

print(
    "Final feature decision table ready. "
    f"Candidate features: {len(final_feature_selection_decisions_df):,}."
)

,feature_name,feature_family,modality,source_metric,pca_contribution_tier,feature_selection_score,final_decision_tier,final_decision_reason
0,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,low_incremental_signal,6,candidate_review,Kept for review because evidence is useful but...
1,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,low_incremental_signal,6,candidate_review,Kept for review because evidence is useful but...
2,fhvhv_avg_trip_speed_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,low_incremental_signal,6,candidate_review,Kept for review because evidence is useful but...
3,fhvhv_trip_count_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_trip_count,low_incremental_signal,6,candidate_review,Kept for review because evidence is useful but...
4,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,low_incremental_signal,6,candidate_review,Kept for review because evidence is useful but...


Final feature decision table ready. Candidate features: 323.


Findings\. The final decision table is structured correctly: each feature now has a final tier and a plain\-language reason\. The preview starts with candidate\_review features, which is useful because these are the remaining judgment calls rather than automatic keeps or drops\. The first few rows are mostly multimodal/local\-vs\-connected features with solid scores but low incremental PCA contribution, so they are being held for review instead of being selected automatically\.

Now let’s summarize the final decision tiers\. This is the broadest view: how many features are selected, how many remain in review, and how many are dropped\.

In [92]:
# ---------------------------------------------------------------------
# Summarize final decisions by tier
# ---------------------------------------------------------------------

final_decision_tier_summary_df = (
    final_feature_selection_decisions_df
    .groupby("final_decision_tier", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        mean_score=("feature_selection_score", "mean"),
        median_score=("feature_selection_score", "median"),
        mean_non_null_pct=("non_null_pct", "mean"),
    )
    .sort_values("feature_count", ascending=False)
    .reset_index(drop=True)
)

display(final_decision_tier_summary_df)

print("Final decision tier summary ready.")

,final_decision_tier,feature_count,mean_score,median_score,mean_non_null_pct
0,dropped_redundant,121,1.404959,2.0,0.892094
1,selected,100,7.570000,8.0,0.906192
2,candidate_review,93,4.548387,4.0,0.905489
3,dropped_scope,9,-0.888889,-2.0,0.756996


Final decision tier summary ready.


Findings\. The final tier summary looks balanced\. Out of 323 candidate features, 100 are selected, 93 remain in candidate review, 121 are dropped as redundant, and 9 traffic features are dropped from scope\. This is conservative in the right way: we are not carrying everything forward, but we also are not forcing every uncertain feature into a drop\. The selected and review groups have very similar non\-null coverage, so the review queue is not just a pile of weakly populated leftovers\.

Next, let’s check the selected and dropped counts by feature family\. This is where we confirm that we did not accidentally wipe out a major feature family or keep a family just because it exists\.

In [93]:
# ---------------------------------------------------------------------
# Summarize final decisions by feature family
# ---------------------------------------------------------------------

final_decision_by_family_df = (
    final_feature_selection_decisions_df
    .groupby(["feature_family", "final_decision_tier"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
    .pivot_table(
        index="feature_family",
        columns="final_decision_tier",
        values="feature_count",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)

for decision_tier in FEATURE_DECISION_TIERS:
    if decision_tier not in final_decision_by_family_df.columns:
        final_decision_by_family_df[decision_tier] = 0

final_decision_by_family_df["total_features"] = final_decision_by_family_df[
    FEATURE_DECISION_TIERS
].sum(axis=1)

final_decision_by_family_df = (
    final_decision_by_family_df[
        ["feature_family", "total_features"] + FEATURE_DECISION_TIERS
    ]
    .sort_values("total_features", ascending=False)
    .reset_index(drop=True)
)

display(final_decision_by_family_df)

print("Final decision family summary ready.")

final_decision_tier,feature_family,total_features,selected,candidate_review,dropped_redundant,dropped_low_signal,dropped_scope
0,temporal_history,150,50,38,62,0,0
1,spatial_context,61,7,26,26,0,2
2,multimodal_interaction,41,0,21,13,0,7
3,stl_decomposition,28,22,6,0,0,0
4,anomaly_severity,21,7,0,14,0,0
5,raw_metric,15,7,2,6,0,0
6,fourier20_residual_reconstruction,7,7,0,0,0,0


Final decision family summary ready.


Findings\. The family summary shows the feature\-selection logic behaving sensibly\. Temporal history is still the largest family, but it is also where most redundancy pruning happens\. STL decomposition and Fourier Top 20 are strongly retained, which matches the evidence from the contribution analysis\. Raw metrics now look correct: all seven protected core metrics are selected, while non\-core raw metrics are reviewed or dropped\. Multimodal interaction and spatial context still need judgment, with many features sitting in review rather than being automatically selected\.

Now let’s summarize the decisions by modality\. This confirms that bus, subway, taxi, and FHVHV all retain usable feature sets, while traffic remains out of scope for this MVP modeling feature set\.

In [94]:
# ---------------------------------------------------------------------
# Summarize final decisions by modality
# ---------------------------------------------------------------------

final_decision_by_modality_df = (
    final_feature_selection_decisions_df
    .groupby(["modality", "final_decision_tier"], as_index=False)
    .agg(feature_count=("feature_name", "count"))
    .pivot_table(
        index="modality",
        columns="final_decision_tier",
        values="feature_count",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)

for decision_tier in FEATURE_DECISION_TIERS:
    if decision_tier not in final_decision_by_modality_df.columns:
        final_decision_by_modality_df[decision_tier] = 0

final_decision_by_modality_df["total_features"] = final_decision_by_modality_df[
    FEATURE_DECISION_TIERS
].sum(axis=1)

final_decision_by_modality_df = (
    final_decision_by_modality_df[
        ["modality", "total_features"] + FEATURE_DECISION_TIERS
    ]
    .sort_values("total_features", ascending=False)
    .reset_index(drop=True)
)

display(final_decision_by_modality_df)

print("Final decision modality summary ready.")

final_decision_tier,modality,total_features,selected,candidate_review,dropped_redundant,dropped_low_signal,dropped_scope
0,fhvhv,92,30,24,38,0,0
1,taxi,92,33,22,37,0,0
2,bus,64,24,17,23,0,0
3,subway,39,13,9,17,0,0
4,multimodal,20,0,14,6,0,0
5,traffic,9,0,0,0,0,9
6,spatial,7,0,7,0,0,0


Final decision modality summary ready.


Findings\. The modality summary confirms that the selected feature set still covers the four main transportation systems\. Taxi, FHVHV, bus, and subway all retain selected features, with subway smaller but still represented\. Traffic is fully dropped from scope, which matches the decision to keep it out of the MVP unsupervised feature set\. The two places to inspect next are multimodal and spatial: neither has selected features yet, but both have review features that may still be worth promoting\.

Now let’s confirm that the seven core mobility metrics were retained\. These are modeling anchors, so they should not be dropped just because engineered descendants carry stronger statistical evidence\.

In [95]:
# ---------------------------------------------------------------------
# Validate protected core raw metric decisions
# ---------------------------------------------------------------------

core_raw_metric_final_check_df = (
    final_feature_selection_decisions_df[
        final_feature_selection_decisions_df["feature_name"].isin(CORE_MODELING_METRICS)
    ][
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "feature_selection_score",
            "final_decision_tier",
            "final_decision_reason",
        ]
    ]
    .sort_values("feature_name")
    .reset_index(drop=True)
)

display(core_raw_metric_final_check_df)

assert core_raw_metric_final_check_df["feature_name"].nunique() == len(CORE_MODELING_METRICS), (
    "One or more 1.5.3 core raw mobility metrics is missing from the final decision table."
)

assert core_raw_metric_final_check_df["final_decision_tier"].eq("selected").all(), (
    "All seven 1.5.3 core raw mobility metrics must be selected in the final decision table."
)

print("Protected core raw metric decisions validated.")

,feature_name,feature_family,modality,source_metric,feature_selection_score,final_decision_tier,final_decision_reason
0,avg_bus_speed,raw_metric,bus,avg_bus_speed,4,selected,Selected as one of the seven core raw mobility...
1,bus_trip_count,raw_metric,bus,bus_trip_count,4,selected,Selected as one of the seven core raw mobility...
2,fhvhv_avg_trip_speed,raw_metric,fhvhv,fhvhv_avg_trip_speed,4,selected,Selected as one of the seven core raw mobility...
3,fhvhv_trip_count,raw_metric,fhvhv,fhvhv_trip_count,2,selected,Selected as one of the seven core raw mobility...
4,subway_ridership,raw_metric,subway,subway_ridership,2,selected,Selected as one of the seven core raw mobility...
5,taxi_avg_trip_speed,raw_metric,taxi,taxi_avg_trip_speed,3,selected,Selected as one of the seven core raw mobility...
6,taxi_trip_count,raw_metric,taxi,taxi_trip_count,1,selected,Selected as one of the seven core raw mobility...


Protected core raw metric decisions validated.


Findings\. The protected core metric check passes exactly the way we wanted\. All seven 1\.5\.3 core raw mobility metrics are selected, including subway\_ridership, taxi\_avg\_trip\_speed, and the FHVHV speed/count metrics that should not be lost\. Some of these raw metrics have modest scores because engineered descendants carry stronger statistical evidence, but they are retained as modeling anchors for interpretability and modality coverage\.

Now let’s inspect what is still in candidate\_review\. These are not automatic drops\. This is the table to use for final judgment calls before defining the exported modeling feature sets\.

In [96]:
# ---------------------------------------------------------------------
# Display candidate-review feature queue
# ---------------------------------------------------------------------

candidate_review_queue_df = (
    final_feature_selection_decisions_df[
        final_feature_selection_decisions_df["final_decision_tier"].eq("candidate_review")
    ][
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "pca_contribution_tier",
            "non_null_pct",
            "global_high_correlation_pair_count",
            "modality_high_correlation_pair_count",
            "feature_selection_score",
            "final_decision_reason",
        ]
    ]
    .sort_values(
        [
            "feature_family",
            "modality",
            "source_metric",
            "feature_selection_score",
            "feature_name",
        ],
        ascending=[True, True, True, False, True],
    )
    .reset_index(drop=True)
)

display(candidate_review_queue_df)

print(
    "Candidate-review queue ready. "
    f"Features needing final review: {len(candidate_review_queue_df):,}."
)

,feature_name,feature_family,modality,source_metric,pca_contribution_tier,non_null_pct,global_high_correlation_pair_count,modality_high_correlation_pair_count,feature_selection_score,final_decision_reason
0,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,low_incremental_signal,0.939105,0,0,6,Kept for review because evidence is useful but...
1,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,low_incremental_signal,0.939105,0,0,6,Kept for review because evidence is useful but...
2,fhvhv_avg_trip_speed_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,low_incremental_signal,0.988593,0,0,6,Kept for review because evidence is useful but...
3,fhvhv_trip_count_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_trip_count,low_incremental_signal,0.988593,0,0,6,Kept for review because evidence is useful but...
4,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,low_incremental_signal,0.946709,0,0,6,Kept for review because evidence is useful but...
5,bus_x_subway_demand_zscore,multimodal_interaction,multimodal,bus_trip_count|subway_ridership,low_incremental_signal,0.585203,0,0,4,Kept for review because evidence is useful but...
6,bus_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|taxi_trip_count,low_incremental_signal,0.946709,0,0,6,Kept for review because evidence is useful but...
7,fhvhv_x_bus_speed_zscore,multimodal_interaction,multimodal,fhvhv_avg_trip_speed|avg_bus_speed,low_incremental_signal,0.946709,0,0,6,Kept for review because evidence is useful but...
8,fhvhv_x_bus_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|bus_trip_count,low_incremental_signal,0.946709,0,0,6,Kept for review because evidence is useful but...
9,fhvhv_x_subway_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|subway_ridership,low_incremental_signal,0.593156,0,0,4,Kept for review because evidence is useful but...


Candidate-review queue ready. Features needing final review: 93.


Findings\. The candidate\-review queue is mostly not a list of weak features; it is a list of features that need human judgment because the statistical evidence is mixed\. The queue clusters into five clear groups: multimodal interaction features, non\-core raw taxi trip\-character metrics, pure spatial context features, STL residual consistency features, and temporal\-history variants\. The strongest review candidates are the ones that add a concept we would otherwise lose: cross\-modal relationships, geographic context, decomposition consistency, and a small set of interpretable temporal\-history templates\. The features I would be most conservative with are the extra smoothed temporal variants and spatial means of non\-core metrics, because those can inflate the feature set without adding much interpretive value\.

💡 Decision\. We promote candidate\-review features when they add a clear modeling concept: multimodal interaction signal, raw taxi trip\-character context, pure spatial geography/connectivity, STL residual consistency, or an interpretable temporal\-history template\. For temporal history, we keep a balanced set: lag\_1, rolling\_mean\_7, rolling\_std\_7, abs\_change\_1, pct\_change\_1, cp\_abs\_delta, and cp\_pct\_delta\. We do not promote ema\_7, pre\_cp\_mean, or post\_cp\_mean from review because they add another smoothed/baseline variant without enough extra interpretive value\.

### Encode Decision Rules on Candidate Features

Let’s encode the review decisions as explicit rule sets first\. This keeps the judgment calls readable before we apply them\.

Now let’s apply those review decisions to the final decision table\. Features promoted by the review rules become selected; remaining review features are demoted out of the default modeling set\.

In [97]:
# ---------------------------------------------------------------------
# Apply candidate-review decision overrides
# ---------------------------------------------------------------------

assert "candidate_review_promote_mask" in globals(), (
    "Run the candidate-review promotion-rule block before applying review overrides."
)

review_final_feature_selection_decisions_df = final_feature_selection_decisions_df.copy()

review_final_feature_selection_decisions_df["review_final_decision_tier"] = (
    review_final_feature_selection_decisions_df["final_decision_tier"]
)

review_final_feature_selection_decisions_df["review_final_decision_reason"] = (
    review_final_feature_selection_decisions_df["final_decision_reason"]
)

review_final_feature_selection_decisions_df["review_override_action"] = "no_review_override"

review_promote_mask = (
    review_final_feature_selection_decisions_df["final_decision_tier"].eq("candidate_review")
    & candidate_review_promote_mask
)

review_demote_mask = (
    review_final_feature_selection_decisions_df["final_decision_tier"].eq("candidate_review")
    & ~candidate_review_promote_mask
)

review_final_feature_selection_decisions_df.loc[
    review_promote_mask,
    "review_final_decision_tier",
] = "selected"

review_final_feature_selection_decisions_df.loc[
    review_promote_mask,
    "review_final_decision_reason",
] = (
    "Promoted after candidate-review inspection because the feature is interpretable, "
    "non-traffic, and aligned with the final modeling feature strategy."
)

review_final_feature_selection_decisions_df.loc[
    review_promote_mask,
    "review_override_action",
] = "promote_candidate_review_to_selected"

review_final_feature_selection_decisions_df.loc[
    review_demote_mask,
    "review_final_decision_tier",
] = "dropped_low_signal"

review_final_feature_selection_decisions_df.loc[
    review_demote_mask,
    "review_final_decision_reason",
] = (
    "Demoted after candidate-review inspection because the feature was less useful "
    "for the default modeling set than a cleaner selected representative."
)

review_final_feature_selection_decisions_df.loc[
    review_demote_mask,
    "review_override_action",
] = "demote_candidate_review_to_not_selected"

# Keep all non-traffic raw metrics as interpretable baseline inputs. Their
# engineered descendants can still be dropped, but the raw observed metrics
# remain useful anchors for downstream interpretation and modeling checks.
non_traffic_raw_metric_mask = (
    review_final_feature_selection_decisions_df["feature_family"].eq("raw_metric")
    & ~review_final_feature_selection_decisions_df["modality"].eq("traffic")
)

reinstated_raw_metric_mask = (
    non_traffic_raw_metric_mask
    & ~review_final_feature_selection_decisions_df["final_decision_tier"].eq("selected")
    & ~review_final_feature_selection_decisions_df["review_override_action"].eq(
        "promote_candidate_review_to_selected"
    )
)

review_final_feature_selection_decisions_df.loc[
    non_traffic_raw_metric_mask,
    "review_final_decision_tier",
] = "selected"

review_final_feature_selection_decisions_df.loc[
    non_traffic_raw_metric_mask,
    "review_final_decision_reason",
] = (
    "Selected because non-traffic raw metrics are interpretable baseline inputs. "
    "Engineered descendants may still be dropped, but the raw observed metric is retained."
)

review_final_feature_selection_decisions_df.loc[
    reinstated_raw_metric_mask,
    "review_override_action",
] = "select_non_traffic_raw_metric"

# Keep engineered temporal-history features centered on the seven selected core
# metrics. Non-core raw metrics remain selected above, but their lag/change/
# rolling descendants are left out of the default modeling set.
non_core_temporal_descendant_mask = (
    review_final_feature_selection_decisions_df["feature_family"].eq("temporal_history")
    & ~review_final_feature_selection_decisions_df["source_metric"].isin(CORE_MODELING_METRICS)
    & review_final_feature_selection_decisions_df["review_final_decision_tier"].eq("selected")
)

review_final_feature_selection_decisions_df.loc[
    non_core_temporal_descendant_mask,
    "review_final_decision_tier",
] = "dropped_low_signal"

review_final_feature_selection_decisions_df.loc[
    non_core_temporal_descendant_mask,
    "review_final_decision_reason",
] = (
    "Dropped because this is an engineered temporal-history descendant of a non-core metric. "
    "The raw metric is retained, but its engineered descendants are excluded from the default modeling set."
)

review_final_feature_selection_decisions_df.loc[
    non_core_temporal_descendant_mask,
    "review_override_action",
] = "drop_non_core_temporal_descendant"

display(
    review_final_feature_selection_decisions_df[
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "feature_selection_score",
            "final_decision_tier",
            "review_final_decision_tier",
            "review_override_action",
        ]
    ]
    .sort_values(
        [
            "review_override_action",
            "review_final_decision_tier",
            "feature_family",
            "modality",
            "source_metric",
            "feature_name",
        ]
    )
    .head(30)
)

print("Candidate-review overrides applied.")

AssertionError: Run the candidate-review promotion-rule block before applying review overrides.

Now let’s inspect the features we promoted\. These are the review features that will join the default selected modeling set\.

In [199]:
# ---------------------------------------------------------------------
# Display promoted and reinstated review features
# ---------------------------------------------------------------------

assert "review_final_feature_selection_decisions_df" in globals(), (
    "Run the candidate-review override block before displaying promoted features."
)

promoted_candidate_review_features_df = (
    review_final_feature_selection_decisions_df[
        review_final_feature_selection_decisions_df["review_override_action"].isin(
            [
                "promote_candidate_review_to_selected",
                "select_non_traffic_raw_metric",
            ]
        )
    ]
    .loc[
        :,
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "pca_contribution_tier",
            "non_null_pct",
            "global_high_correlation_pair_count",
            "modality_high_correlation_pair_count",
            "feature_selection_score",
            "review_override_action",
            "review_final_decision_tier",
            "review_final_decision_reason",
        ],
    ]
    .sort_values(["review_override_action", "feature_family", "modality", "source_metric", "feature_name"])
    .reset_index(drop=True)
)

display(promoted_candidate_review_features_df)

print(
    "Promoted or reinstated features: "
    f"{len(promoted_candidate_review_features_df):,}."
)

,feature_name,feature_family,modality,source_metric,pca_contribution_tier,non_null_pct,global_high_correlation_pair_count,modality_high_correlation_pair_count,feature_selection_score,review_override_action,review_final_decision_tier,review_final_decision_reason
0,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,low_incremental_signal,0.939105,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
1,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,low_incremental_signal,0.939105,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
2,fhvhv_avg_trip_speed_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,low_incremental_signal,0.988593,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
3,fhvhv_trip_count_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_trip_count,low_incremental_signal,0.988593,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
4,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,low_incremental_signal,0.946709,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
5,bus_x_subway_demand_zscore,multimodal_interaction,multimodal,bus_trip_count|subway_ridership,low_incremental_signal,0.585203,0,0,4,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
6,bus_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|taxi_trip_count,low_incremental_signal,0.946709,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
7,fhvhv_x_bus_speed_zscore,multimodal_interaction,multimodal,fhvhv_avg_trip_speed|avg_bus_speed,low_incremental_signal,0.946709,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
8,fhvhv_x_bus_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|bus_trip_count,low_incremental_signal,0.946709,0,0,6,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...
9,fhvhv_x_subway_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|subway_ridership,low_incremental_signal,0.593156,0,0,4,promote_candidate_review_to_selected,selected,Promoted after candidate-review inspection bec...


Promoted or reinstated features: 60.


Findings\. The promoted and reinstated features show the judgment layer we applied after the scorecard\. We selected multimodal interaction features because they capture relationships between systems, pure spatial context features because they describe location and connectivity directly, STL residual features because they preserve decomposition signal across speed metrics, and selected core temporal templates because they give each mode a comparable history/change structure without forcing every redundant metric\-template combination through\. We also keep non\-traffic raw metrics as baseline inputs, even when correlation screens marked some of them as redundant, because raw observed values are easy to interpret and useful as anchors for downstream modeling\.

Now let’s inspect the review features we did not promote\. These are excluded from the default modeling set, but still documented in the decision table\.

In [200]:
# ---------------------------------------------------------------------
# Display demoted candidate-review features
# ---------------------------------------------------------------------

assert "review_final_feature_selection_decisions_df" in globals(), (
    "Run the candidate-review override block before displaying demoted features."
)

demoted_candidate_review_features_df = (
    review_final_feature_selection_decisions_df[
        review_final_feature_selection_decisions_df["review_override_action"].eq(
            "demote_candidate_review_to_not_selected"
        )
    ]
    .loc[
        :,
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "pca_contribution_tier",
            "non_null_pct",
            "global_high_correlation_pair_count",
            "modality_high_correlation_pair_count",
            "feature_selection_score",
            "review_final_decision_tier",
            "review_final_decision_reason",
        ],
    ]
    .sort_values(["feature_family", "modality", "source_metric", "feature_name"])
    .reset_index(drop=True)
)

display(demoted_candidate_review_features_df)

print(
    "Demoted candidate-review features: "
    f"{len(demoted_candidate_review_features_df):,}."
)

,feature_name,feature_family,modality,source_metric,pca_contribution_tier,non_null_pct,global_high_correlation_pair_count,modality_high_correlation_pair_count,feature_selection_score,review_final_decision_tier,review_final_decision_reason
0,borough_mean_avg_bus_travel_time,spatial_context,bus,avg_bus_travel_time,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
1,connected_mean_avg_bus_travel_time,spatial_context,bus,avg_bus_travel_time,low_incremental_signal,0.980989,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
2,borough_mean_fhvhv_avg_trip_distance,spatial_context,fhvhv,fhvhv_avg_trip_distance,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
3,connected_mean_fhvhv_avg_trip_distance,spatial_context,fhvhv,fhvhv_avg_trip_distance,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
4,cp_zone_mean_fhvhv_avg_trip_distance,spatial_context,fhvhv,fhvhv_avg_trip_distance,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
5,borough_mean_fhvhv_avg_trip_duration,spatial_context,fhvhv,fhvhv_avg_trip_duration,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
6,connected_mean_fhvhv_avg_trip_duration,spatial_context,fhvhv,fhvhv_avg_trip_duration,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
7,cp_zone_mean_fhvhv_avg_trip_duration,spatial_context,fhvhv,fhvhv_avg_trip_duration,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
8,cp_zone_mean_fhvhv_distinct_dropoff_zone_count,spatial_context,fhvhv,fhvhv_distinct_dropoff_zone_count,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
9,borough_mean_subway_transfers,spatial_context,subway,subway_transfers,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...


Demoted candidate-review features: 39.


Findings\. The demoted review features are mostly useful\-but\-not\-essential variants\. Many are spatial averages of non\-core metrics, EMA smoothers, pre/post period means, or temporal descendants of non\-core metrics\. The takeaway is not that these features are wrong; it is that the selected set already carries cleaner versions of the same ideas\. This keeps the modeling panel easier to explain without stripping away the major signal families\.

Finally, let’s summarize and validate the post\-review decision table before we move into the dropped\-feature audit\.

In [201]:
# ---------------------------------------------------------------------
# Summarize and validate post-review feature decisions
# ---------------------------------------------------------------------

assert "review_final_feature_selection_decisions_df" in globals(), (
    "Run the candidate-review override block before summarizing post-review decisions."
)

review_final_decision_summary_df = (
    review_final_feature_selection_decisions_df
    .groupby("review_final_decision_tier", as_index=False)
    .agg(
        feature_count=("feature_name", "count"),
        mean_score=("feature_selection_score", "mean"),
        median_score=("feature_selection_score", "median"),
        mean_non_null_pct=("non_null_pct", "mean"),
    )
    .sort_values("feature_count", ascending=False)
    .reset_index(drop=True)
)

display(review_final_decision_summary_df)

assert len(review_final_feature_selection_decisions_df) == len(final_feature_selection_decisions_df), (
    "Candidate-review override changed the number of feature decision rows."
)

assert not review_final_feature_selection_decisions_df["review_final_decision_tier"].eq("candidate_review").any(), (
    "Candidate-review features remain unresolved after review overrides."
)

assert set(CORE_MODELING_METRICS).issubset(
    set(
        review_final_feature_selection_decisions_df[
            review_final_feature_selection_decisions_df["review_final_decision_tier"].eq("selected")
        ]["feature_name"]
    )
), "One or more protected core raw metrics was not selected after review."

print(
    "Post-review feature decisions validated. "
    f"Selected features: "
    f"{review_final_feature_selection_decisions_df['review_final_decision_tier'].eq('selected').sum():,}."
)

,review_final_decision_tier,feature_count,mean_score,median_score,mean_non_null_pct
0,selected,142,6.140845,6.0,0.897072
1,dropped_redundant,115,1.426087,2.0,0.892888
2,dropped_low_signal,57,5.508772,5.0,0.924680
3,dropped_scope,9,-0.888889,-2.0,0.756996


Post-review feature decisions validated. Selected features: 142.


Findings\. The post\-review decision table resolves the full review queue into a final modeling stance: selected, dropped for redundancy, dropped after review, or out of scope\. The selected set now keeps the raw mobility layer, the core engineered families, selected spatial context, multimodal interactions, STL/Fourier decomposition features, and a consistent set of temporal templates\. The excluded features are mostly narrower variants, duplicate summaries, or traffic\-related fields held out of this unsupervised modeling panel\.

### Final Selection

Now let’s review the dropped features\. This is the drop audit: every removed feature should have a clear reason, and none of the protected core metrics should appear here\.

In [202]:
# ---------------------------------------------------------------------
# Display dropped feature audit
# ---------------------------------------------------------------------

assert "review_final_feature_selection_decisions_df" in globals(), (
    "Run the candidate-review override blocks before displaying the dropped feature audit."
)

dropped_feature_audit_df = (
    review_final_feature_selection_decisions_df[
        review_final_feature_selection_decisions_df["review_final_decision_tier"].isin(
            [
                "dropped_redundant",
                "dropped_low_signal",
                "dropped_scope",
            ]
        )
    ]
    .loc[
        :,
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "pca_contribution_tier",
            "non_null_pct",
            "global_high_correlation_pair_count",
            "modality_high_correlation_pair_count",
            "feature_selection_score",
            "review_final_decision_tier",
            "review_final_decision_reason",
        ],
    ]
    .sort_values(
        [
            "review_final_decision_tier",
            "feature_family",
            "modality",
            "source_metric",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)

display(dropped_feature_audit_df)

print(
    "Dropped feature audit ready. "
    f"Dropped or out-of-scope features: {len(dropped_feature_audit_df):,}."
)

,feature_name,feature_family,modality,source_metric,pca_contribution_tier,non_null_pct,global_high_correlation_pair_count,modality_high_correlation_pair_count,feature_selection_score,review_final_decision_tier,review_final_decision_reason
0,borough_mean_avg_bus_travel_time,spatial_context,bus,avg_bus_travel_time,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
1,connected_mean_avg_bus_travel_time,spatial_context,bus,avg_bus_travel_time,low_incremental_signal,0.980989,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
2,borough_mean_fhvhv_avg_trip_distance,spatial_context,fhvhv,fhvhv_avg_trip_distance,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
3,connected_mean_fhvhv_avg_trip_distance,spatial_context,fhvhv,fhvhv_avg_trip_distance,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
4,cp_zone_mean_fhvhv_avg_trip_distance,spatial_context,fhvhv,fhvhv_avg_trip_distance,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
5,borough_mean_fhvhv_avg_trip_duration,spatial_context,fhvhv,fhvhv_avg_trip_duration,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
6,connected_mean_fhvhv_avg_trip_duration,spatial_context,fhvhv,fhvhv_avg_trip_duration,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
7,cp_zone_mean_fhvhv_avg_trip_duration,spatial_context,fhvhv,fhvhv_avg_trip_duration,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
8,cp_zone_mean_fhvhv_distinct_dropoff_zone_count,spatial_context,fhvhv,fhvhv_distinct_dropoff_zone_count,low_incremental_signal,1.000000,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...
9,borough_mean_subway_transfers,spatial_context,subway,subway_transfers,low_incremental_signal,0.988593,0,0,5,dropped_low_signal,Demoted after candidate-review inspection beca...


Dropped feature audit ready. Dropped or out-of-scope features: 181.


Findings\. The dropped\-feature audit shows that pruning is concentrated in the places we would expect: overlapping temporal variants, redundant connected\-mean or z\-score versions, extra anomaly\-severity measures, and spatial aggregates of non\-core metrics\. Traffic features are excluded from this modeling set because they are better treated as a future target or sparse contextual layer\. This gives us a defensible feature set: broad enough to preserve mobility structure, but trimmed enough that downstream modeling is not overwhelmed by repeated versions of the same signal

Now let’s check feature\-template consistency one last time after final decisions\. This helps catch cases where we kept a feature pattern for one mode but not for the others\.

In [203]:
# ---------------------------------------------------------------------
# Review final cross-modality template consistency
# ---------------------------------------------------------------------

assert "review_final_feature_selection_decisions_df" in globals(), (
    "Run the candidate-review override blocks before reviewing final template consistency."
)

template_pattern_map = {
    "deviation_abs_zscore": "deviation_abs_zscore",
    "lag_1_same_bucket": "lag_1",
    "rolling_mean_7_same_bucket": "rolling_mean_7",
    "rolling_std_7_same_bucket": "rolling_std_7",
    "abs_change_1_same_bucket": "abs_change_1",
    "pct_change_1_same_bucket": "pct_change_1",
    "cp_abs_delta_same_bucket": "cp_abs_delta",
    "cp_pct_delta_same_bucket": "cp_pct_delta",
    "residual_zscore": "stl_residual_zscore",
    "residual_abs": "stl_residual_abs",
    "residual": "stl_residual",
    "seasonal": "stl_seasonal",
}

template_review_rows = []

for _, row in review_final_feature_selection_decisions_df.iterrows():
    feature_name = row["feature_name"]
    feature_family = row["feature_family"]

    if feature_family == "fourier20_residual_reconstruction":
        matched_template = "fourier20_residual_reconstruction"
    else:
        matched_template = None

        for pattern, template_name in template_pattern_map.items():
            if feature_name.endswith(pattern) or pattern in feature_name:
                matched_template = template_name
                break

    if matched_template is None:
        continue

    template_review_rows.append(
        {
            "feature_name": feature_name,
            "feature_family": feature_family,
            "feature_template": matched_template,
            "source_metric": row["source_metric"],
            "modality": row["modality"],
            "review_final_decision_tier": row["review_final_decision_tier"],
        }
    )

final_template_review_df = pd.DataFrame(template_review_rows)

final_template_consistency_df = (
    final_template_review_df[
        final_template_review_df["modality"].isin(["bus", "subway", "taxi", "fhvhv"])
    ]
    .groupby(
        [
            "feature_family",
            "feature_template",
            "source_metric",
            "modality",
            "review_final_decision_tier",
        ],
        as_index=False,
    )
    .agg(feature_count=("feature_name", "count"))
    .pivot_table(
        index=[
            "feature_family",
            "feature_template",
            "source_metric",
            "modality",
        ],
        columns="review_final_decision_tier",
        values="feature_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for decision_tier in [
    "selected",
    "dropped_redundant",
    "dropped_low_signal",
    "dropped_scope",
]:
    if decision_tier not in final_template_consistency_df.columns:
        final_template_consistency_df[decision_tier] = 0

final_template_consistency_df = (
    final_template_consistency_df[
        [
            "feature_family",
            "feature_template",
            "source_metric",
            "modality",
            "selected",
            "dropped_redundant",
            "dropped_low_signal",
            "dropped_scope",
        ]
    ]
    .sort_values(
        [
            "feature_family",
            "feature_template",
            "source_metric",
            "modality",
        ]
    )
    .reset_index(drop=True)
)

display(final_template_consistency_df)

print(
    "Final template consistency review ready. "
    f"Template-metric rows reviewed: {len(final_template_consistency_df):,}."
)

review_final_decision_tier,feature_family,feature_template,source_metric,modality,selected,dropped_redundant,dropped_low_signal,dropped_scope
0,anomaly_severity,deviation_abs_zscore,avg_bus_speed,bus,1,0,0,0
1,anomaly_severity,deviation_abs_zscore,bus_trip_count,bus,1,0,0,0
2,anomaly_severity,deviation_abs_zscore,fhvhv_avg_trip_speed,fhvhv,1,0,0,0
3,anomaly_severity,deviation_abs_zscore,fhvhv_trip_count,fhvhv,1,0,0,0
4,anomaly_severity,deviation_abs_zscore,subway_ridership,subway,1,0,0,0
5,anomaly_severity,deviation_abs_zscore,taxi_avg_trip_speed,taxi,1,0,0,0
6,anomaly_severity,deviation_abs_zscore,taxi_trip_count,taxi,1,0,0,0
7,fourier20_residual_reconstruction,fourier20_residual_reconstruction,avg_bus_speed,bus,1,0,0,0
8,fourier20_residual_reconstruction,fourier20_residual_reconstruction,bus_trip_count,bus,1,0,0,0
9,fourier20_residual_reconstruction,fourier20_residual_reconstruction,fhvhv_avg_trip_speed,fhvhv,1,0,0,0


Final template consistency review ready. Template-metric rows reviewed: 147.


Findings\. The template consistency table now shows the final feature pattern clearly by family, template, metric, and modality\. The strongest consistency is in the engineered families we explicitly protected: anomaly\-severity, Fourier Top 20, and STL decomposition features are selected across the seven core mobility metrics\. Temporal\-history features are more selective, which is expected after the redundancy screen: core\-metric lag, change, CP\-delta, rolling\-mean, and rolling\-std features are retained where they add useful signal, while redundant or non\-core descendants are dropped\. The main validation point is that the final table now matches the strategy: keep raw non\-traffic metrics, center engineered temporal features on the seven core metrics, retain decomposition and anomaly signal, and avoid carrying every possible descendant forward just for symmetry\.

Finally, let’s run blocker QA checks on the final decision table\. These checks make sure the selected feature set is valid before we use it to define modeling assets\.

In [204]:
# ---------------------------------------------------------------------
# Validate final feature selection decisions
# ---------------------------------------------------------------------

assert "review_final_feature_selection_decisions_df" in globals(), (
    "Run the candidate-review override blocks before validating final feature decisions."
)

valid_review_final_decision_tiers = {
    "selected",
    "dropped_redundant",
    "dropped_low_signal",
    "dropped_scope",
}

invalid_review_final_decision_tiers = sorted(
    set(review_final_feature_selection_decisions_df["review_final_decision_tier"].dropna())
    - valid_review_final_decision_tiers
)

selected_feature_names = (
    review_final_feature_selection_decisions_df[
        review_final_feature_selection_decisions_df["review_final_decision_tier"].eq("selected")
    ]["feature_name"]
    .drop_duplicates()
    .tolist()
)

missing_protected_core_metrics = sorted(
    set(CORE_MODELING_METRICS) - set(selected_feature_names)
)

final_feature_selection_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "candidate_features_reviewed",
            "value": len(review_final_feature_selection_decisions_df),
            "status": "pass" if len(review_final_feature_selection_decisions_df) > 0 else "fail",
        },
        {
            "qa_check": "selected_features",
            "value": len(selected_feature_names),
            "status": "pass" if len(selected_feature_names) > 0 else "fail",
        },
        {
            "qa_check": "invalid_decision_tiers",
            "value": len(invalid_review_final_decision_tiers),
            "status": "pass" if len(invalid_review_final_decision_tiers) == 0 else "fail",
        },
        {
            "qa_check": "missing_protected_core_metrics",
            "value": len(missing_protected_core_metrics),
            "status": "pass" if len(missing_protected_core_metrics) == 0 else "fail",
        },
        {
            "qa_check": "traffic_features_selected",
            "value": int(
                review_final_feature_selection_decisions_df[
                    review_final_feature_selection_decisions_df["modality"].eq("traffic")
                    & review_final_feature_selection_decisions_df["review_final_decision_tier"].eq("selected")
                ].shape[0]
            ),
            "status": "pass",
        },
    ]
)

display(final_feature_selection_qa_df)

assert final_feature_selection_qa_df["status"].eq("pass").all(), (
    "One or more final feature-selection QA checks failed."
)

print(
    "Final feature selection decisions validated. "
    f"Selected features: {len(selected_feature_names):,}."
)

,qa_check,value,status
0,candidate_features_reviewed,323,pass
1,selected_features,142,pass
2,invalid_decision_tiers,0,pass
3,missing_protected_core_metrics,0,pass
4,traffic_features_selected,0,pass


Final feature selection decisions validated. Selected features: 142.


Findings\. The final QA confirms that the feature\-selection table is ready to use\. All 323 candidate features received a valid decision, 142 features were selected for the curated modeling set, and none of the protected core raw metrics were accidentally dropped\. Traffic features remain excluded from the selected set, which matches the decision to treat traffic as a future supervised target or sparse context layer rather than a default unsupervised modeling input\.

### Section Summary

Section 1\.5\.6\.5 converts the inventory, redundancy screens, modality checks, and contribution evidence into a final feature\-selection decision table\. The final set keeps 142 selected features, including the core raw mobility metrics, selected non\-traffic raw metrics, STL and Fourier decomposition features, anomaly\-severity features, targeted temporal\-history templates, spatial context, and multimodal interaction signals\. The features dropped from the default modeling set are mostly redundant variants, non\-core engineered descendants, extra smoothers, and traffic\-scope fields, leaving a modeling feature set that is broad enough to preserve signal but trimmed enough to stay interpretable\.

## 1\.5\.6\.6 Define Modality\-Specific and Multimodal Feature Sets

The downstream modeling plan includes both modality\-specific outlier detection and a unified multimodal model\. This section defines the reusable feature\-set definitions that those later workflows will use\. The output is column membership only: which selected features belong to Bus, Subway, Taxi, FHVHV, and Multimodal modeling views\.

This notebook stops at feature decisions and feature\-set definitions\. It does not filter rows, scale features, or build final modeling matrices\. Those steps move to 3\.1\.1, where the modeling matrix can be constructed with the right row filters, scaling choices, and downstream workflow context\.

Let’s start by turning the final decision table into a clean selected\-feature catalog\. This is the source of truth for feature\-set membership; no row filtering or scaling happens here\.

In [211]:
# ---------------------------------------------------------------------
# Build selected feature catalog
# ---------------------------------------------------------------------

assert "review_final_feature_selection_decisions_df" in globals(), (
    "Run final feature selection decisions before defining feature sets."
)

selected_feature_catalog_columns = [
    "feature_name",
    "source_notebook",
    "feature_family",
    "modality",
    "source_metric",
    "definition",
    "review_final_decision_tier",
    "review_final_decision_reason",
]

available_selected_feature_catalog_columns = [
    column
    for column in selected_feature_catalog_columns
    if column in review_final_feature_selection_decisions_df.columns
]

selected_feature_catalog_df = (
    review_final_feature_selection_decisions_df[
        review_final_feature_selection_decisions_df["review_final_decision_tier"].eq("selected")
    ]
    .loc[:, available_selected_feature_catalog_columns]
    .copy()
    .sort_values(["feature_family", "modality", "source_metric", "feature_name"])
    .reset_index(drop=True)
)

selected_feature_catalog_df["selected_status"] = "selected"

display(selected_feature_catalog_df.head(25))

assert not selected_feature_catalog_df.empty, "No selected features available for feature-set definitions."
assert selected_feature_catalog_df["feature_name"].is_unique, (
    "Selected feature catalog contains duplicate feature names."
)

print(
    "Selected feature catalog ready. "
    f"Selected features: {len(selected_feature_catalog_df):,}."
)

,feature_name,source_notebook,feature_family,modality,source_metric,review_final_decision_tier,review_final_decision_reason,selected_status
0,avg_bus_speed_deviation_abs_zscore,1.5.5,anomaly_severity,bus,avg_bus_speed,selected,Selected by strong combined evidence score.,selected
1,bus_trip_count_deviation_abs_zscore,1.5.5,anomaly_severity,bus,bus_trip_count,selected,Selected by strong combined evidence score.,selected
2,fhvhv_avg_trip_speed_deviation_abs_zscore,1.5.5,anomaly_severity,fhvhv,fhvhv_avg_trip_speed,selected,Selected by strong combined evidence score.,selected
3,fhvhv_trip_count_deviation_abs_zscore,1.5.5,anomaly_severity,fhvhv,fhvhv_trip_count,selected,Selected by strong combined evidence score.,selected
4,subway_ridership_deviation_abs_zscore,1.5.5,anomaly_severity,subway,subway_ridership,selected,Selected by strong combined evidence score.,selected
5,taxi_avg_trip_speed_deviation_abs_zscore,1.5.5,anomaly_severity,taxi,taxi_avg_trip_speed,selected,Selected by strong combined evidence score.,selected
6,taxi_trip_count_deviation_abs_zscore,1.5.5,anomaly_severity,taxi,taxi_trip_count,selected,Selected by strong combined evidence score.,selected
7,avg_bus_speed_fourier20_residual_reconstructed,1.5.4,fourier20_residual_reconstruction,bus,avg_bus_speed,selected,Selected by strong combined evidence score.,selected
8,bus_trip_count_fourier20_residual_reconstructed,1.5.4,fourier20_residual_reconstruction,bus,bus_trip_count,selected,Selected by strong combined evidence score.,selected
9,fhvhv_avg_trip_speed_fourier20_residual_recons...,1.5.4,fourier20_residual_reconstruction,fhvhv,fhvhv_avg_trip_speed,selected,Selected by strong combined evidence score.,selected


Selected feature catalog ready. Selected features: 142.


Findings\. The selected feature catalog is the clean handoff from feature selection into feature\-set definition\. It contains 142 selected features and preserves the metadata we need downstream: source notebook, feature family, modality, source metric, decision reason, and selected status\. The preview is also a useful sanity check: it shows the selected set begins with the expected engineered families, including anomaly\-severity, Fourier, and multimodal interaction features\.

Now we define membership for each modeling view\. Modality\-specific sets include selected features for that mode, shared spatial context, and selected cross\-modal features that touch that mode\. The multimodal set includes all selected non\-traffic features\.

In [212]:
# ---------------------------------------------------------------------
# Define canonical feature-set membership
# ---------------------------------------------------------------------

CANONICAL_MODELING_FEATURE_SETS = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
    "multimodal",
]

FEATURE_SET_SOURCE_METRIC_KEYWORDS = {
    "bus": [
        "bus_trip_count",
        "avg_bus_speed",
        "avg_bus_travel_time",
        "bus",
    ],
    "subway": [
        "subway_ridership",
        "subway_transfers",
        "subway",
    ],
    "taxi": [
        "taxi_trip_count",
        "taxi_avg_trip_speed",
        "taxi_avg_trip_distance",
        "taxi_avg_trip_duration",
        "taxi_distinct_dropoff_zone_count",
        "taxi",
    ],
    "fhvhv": [
        "fhvhv_trip_count",
        "fhvhv_avg_trip_speed",
        "fhvhv_avg_trip_distance",
        "fhvhv_avg_trip_duration",
        "fhvhv_distinct_dropoff_zone_count",
        "fhvhv",
    ],
}

feature_set_membership_rows = []

for _, row in selected_feature_catalog_df.iterrows():
    feature_name = row["feature_name"]
    modality = row["modality"]
    source_metric = str(row.get("source_metric", ""))
    feature_family = row["feature_family"]

    # The unified multimodal view gets all selected non-traffic features.
    if modality != "traffic":
        feature_set_membership_rows.append(
            {
                "feature_set_name": "multimodal",
                "feature_name": feature_name,
                "feature_set_role": "unified_multimodal_feature",
            }
        )

    for feature_set_name, keywords in FEATURE_SET_SOURCE_METRIC_KEYWORDS.items():
        owns_modality = modality == feature_set_name
        shared_spatial_context = modality == "spatial"
        relevant_cross_modal = (
            modality == "multimodal"
            and any(keyword in source_metric or keyword in feature_name for keyword in keywords)
        )

        if owns_modality:
            feature_set_role = "modality_owned_feature"
        elif shared_spatial_context:
            feature_set_role = "shared_spatial_context"
        elif relevant_cross_modal:
            feature_set_role = "relevant_cross_modal_feature"
        else:
            continue

        feature_set_membership_rows.append(
            {
                "feature_set_name": feature_set_name,
                "feature_name": feature_name,
                "feature_set_role": feature_set_role,
            }
        )

feature_set_membership_df = (
    pd.DataFrame(feature_set_membership_rows)
    .drop_duplicates(["feature_set_name", "feature_name"])
    .merge(
        selected_feature_catalog_df,
        on="feature_name",
        how="left",
    )
    .sort_values(
        [
            "feature_set_name",
            "feature_set_role",
            "feature_family",
            "modality",
            "source_metric",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)

display(feature_set_membership_df.head(30))

assert not feature_set_membership_df.empty, "Feature-set membership table is empty."
assert set(CANONICAL_MODELING_FEATURE_SETS).issubset(
    set(feature_set_membership_df["feature_set_name"])
), "One or more canonical feature sets is missing from the membership table."

print(
    "Canonical feature-set membership ready. "
    f"Membership rows: {len(feature_set_membership_df):,}."
)

,feature_set_name,feature_name,feature_set_role,source_notebook,feature_family,modality,source_metric,review_final_decision_tier,review_final_decision_reason,selected_status
0,bus,avg_bus_speed_deviation_abs_zscore,modality_owned_feature,1.5.5,anomaly_severity,bus,avg_bus_speed,selected,Selected by strong combined evidence score.,selected
1,bus,bus_trip_count_deviation_abs_zscore,modality_owned_feature,1.5.5,anomaly_severity,bus,bus_trip_count,selected,Selected by strong combined evidence score.,selected
2,bus,avg_bus_speed_fourier20_residual_reconstructed,modality_owned_feature,1.5.4,fourier20_residual_reconstruction,bus,avg_bus_speed,selected,Selected by strong combined evidence score.,selected
3,bus,bus_trip_count_fourier20_residual_reconstructed,modality_owned_feature,1.5.4,fourier20_residual_reconstruction,bus,bus_trip_count,selected,Selected by strong combined evidence score.,selected
4,bus,avg_bus_speed_local_vs_connected_zscore,modality_owned_feature,1.5.3,multimodal_interaction,bus,avg_bus_speed,selected,Promoted after candidate-review inspection bec...,selected
5,bus,bus_trip_count_local_vs_connected_zscore,modality_owned_feature,1.5.3,multimodal_interaction,bus,bus_trip_count,selected,Promoted after candidate-review inspection bec...,selected
6,bus,avg_bus_speed,modality_owned_feature,pre_1.5_engineering,raw_metric,bus,avg_bus_speed,selected,Selected because non-traffic raw metrics are i...,selected
7,bus,avg_bus_travel_time,modality_owned_feature,pre_1.5_engineering,raw_metric,bus,avg_bus_travel_time,selected,Selected because non-traffic raw metrics are i...,selected
8,bus,bus_trip_count,modality_owned_feature,pre_1.5_engineering,raw_metric,bus,bus_trip_count,selected,Selected because non-traffic raw metrics are i...,selected
9,bus,borough_mean_avg_bus_speed,modality_owned_feature,1.5.2,spatial_context,bus,avg_bus_speed,selected,Selected by strong combined evidence score.,selected


Canonical feature-set membership ready. Membership rows: 318.


Findings\. The membership table expands the 142 selected features into 318 feature\-set rows because features can belong to more than one modeling view\. Shared spatial context is reused across modality\-specific sets, cross\-modal features can appear in the relevant mode\-specific views, and every selected non\-traffic feature is included in the multimodal view\. That is the right shape for 3\.1\.1: one selected feature catalog, plus reusable column memberships for each modeling view\.

Let’s keep the QA compact: how many features each modeling view receives, what roles those features play, and how the selected families are distributed\.

In [213]:
# ---------------------------------------------------------------------
# Summarize feature counts by modeling view
# ---------------------------------------------------------------------

feature_set_count_summary_df = (
    feature_set_membership_df
    .groupby("feature_set_name", as_index=False)
    .agg(
        selected_features=("feature_name", "nunique"),
        feature_families=("feature_family", "nunique"),
        modalities_represented=("modality", "nunique"),
    )
    .sort_values("feature_set_name")
    .reset_index(drop=True)
)

display(feature_set_count_summary_df)

assert feature_set_count_summary_df["selected_features"].gt(0).all(), (
    "One or more feature sets has zero selected features."
)

print("Feature-set count summary ready.")

,feature_set_name,selected_features,feature_families,modalities_represented
0,bus,49,7,3
1,fhvhv,51,7,3
2,multimodal,142,7,6
3,subway,27,7,3
4,taxi,49,7,3


Feature-set count summary ready.


Findings\. The feature\-set counts look right for the boundary we set for 1\.5\.6\. Each modality\-specific view has a compact selected feature set, while the multimodal view carries all 142 selected non\-traffic features\. Bus, taxi, and FHVHV are similar in size, subway is smaller because subway coverage is naturally more limited by station geography, and every view still includes all seven feature families\. This gives 3\.1\.1 reusable column definitions without constructing any modeling matrices yet\.

Next, let’s check why each feature enters each modeling view: owned modality features, shared spatial context, relevant cross\-modal features, or the unified multimodal set\.

In [214]:
# ---------------------------------------------------------------------
# Summarize feature-set membership roles
# ---------------------------------------------------------------------

feature_set_role_summary_df = (
    feature_set_membership_df
    .groupby(["feature_set_name", "feature_set_role"], as_index=False)
    .agg(selected_features=("feature_name", "nunique"))
    .sort_values(["feature_set_name", "feature_set_role"])
    .reset_index(drop=True)
)

display(feature_set_role_summary_df)

assert feature_set_role_summary_df["selected_features"].gt(0).all(), (
    "One or more feature-set role has zero selected features."
)

print("Feature-set role summary ready.")

,feature_set_name,feature_set_role,selected_features
0,bus,modality_owned_feature,35
1,bus,relevant_cross_modal_feature,7
2,bus,shared_spatial_context,7
3,fhvhv,modality_owned_feature,37
4,fhvhv,relevant_cross_modal_feature,7
5,fhvhv,shared_spatial_context,7
6,multimodal,unified_multimodal_feature,142
7,subway,modality_owned_feature,15
8,subway,relevant_cross_modal_feature,5
9,subway,shared_spatial_context,7


Feature-set role summary ready.


Findings\. The role summary shows that each modality\-specific view is mostly built from its own selected features, then augmented with the same seven shared spatial context features and a small number of relevant cross\-modal features\. That is the structure we wanted: Bus, Subway, Taxi, and FHVHV are not isolated silos, but they are also not overloaded with the full multimodal feature universe\. The multimodal view correctly contains the full 142\-feature selected set\.

Finally, let’s check the family mix inside each feature set so we can confirm every modeling view carries raw, temporal, spatial, interaction, decomposition, Fourier, and anomaly\-severity signal\.

In [215]:
# ---------------------------------------------------------------------
# Summarize feature families by modeling view
# ---------------------------------------------------------------------

feature_set_family_summary_df = (
    feature_set_membership_df
    .groupby(["feature_set_name", "feature_family"], as_index=False)
    .agg(selected_features=("feature_name", "nunique"))
    .sort_values(["feature_set_name", "feature_family"])
    .reset_index(drop=True)
)

display(feature_set_family_summary_df)

assert feature_set_family_summary_df["selected_features"].gt(0).all(), (
    "One or more feature-set family has zero selected features."
)

print("Feature-set family summary ready.")

,feature_set_name,feature_family,selected_features
0,bus,anomaly_severity,2
1,bus,fourier20_residual_reconstruction,2
2,bus,multimodal_interaction,9
3,bus,raw_metric,3
4,bus,spatial_context,9
5,bus,stl_decomposition,8
6,bus,temporal_history,16
7,fhvhv,anomaly_severity,2
8,fhvhv,fourier20_residual_reconstruction,2
9,fhvhv,multimodal_interaction,9


Feature-set family summary ready.


Findings\. The family summary confirms that every modeling view carries the full selected feature stack: raw metrics, temporal history, spatial context, multimodal interaction, STL decomposition, Fourier reconstruction, and anomaly severity\. The counts vary by mode because the selected feature set reflects actual data coverage and metric availability rather than forcing identical columns everywhere\. The multimodal set keeps the full selected feature universe, while the modality\-specific sets stay smaller and easier to interpret\.

Finally, export only the column definitions and metadata\. This gives 3\.1\.1 the feature\-set map it needs without constructing matrices early\.

In [216]:
# ---------------------------------------------------------------------
# Export feature decisions and feature-set definitions
# ---------------------------------------------------------------------

feature_selection_decisions_export_df = (
    review_final_feature_selection_decisions_df
    .copy()
    .sort_values(["review_final_decision_tier", "feature_family", "modality", "source_metric", "feature_name"])
    .reset_index(drop=True)
)

feature_set_definition_export_df = (
    feature_set_membership_df
    .copy()
    .sort_values(["feature_set_name", "feature_set_role", "feature_family", "modality", "source_metric", "feature_name"])
    .reset_index(drop=True)
)

if WRITE_OUTPUTS:
    feature_selection_decisions_export_df.to_csv(FEATURE_SELECTION_DECISIONS_PATH, index=False)
    feature_set_definition_export_df.to_csv(OUTPUT_FEATURE_SET_DEFINITIONS_PATH, index=False)
    selected_feature_catalog_df.to_csv(OUTPUT_FEATURE_CATALOG_PATH, index=False)

output_write_summary_df = pd.DataFrame(
    [
        {
            "output_name": "feature_selection_decisions",
            "path": FEATURE_SELECTION_DECISIONS_PATH,
            "rows": len(feature_selection_decisions_export_df),
            "written": WRITE_OUTPUTS,
        },
        {
            "output_name": "feature_set_definitions",
            "path": OUTPUT_FEATURE_SET_DEFINITIONS_PATH,
            "rows": len(feature_set_definition_export_df),
            "written": WRITE_OUTPUTS,
        },
        {
            "output_name": "selected_feature_catalog",
            "path": OUTPUT_FEATURE_CATALOG_PATH,
            "rows": len(selected_feature_catalog_df),
            "written": WRITE_OUTPUTS,
        },
    ]
)

display(output_write_summary_df)

print("Feature decisions and feature-set definitions exported.")

,output_name,path,rows,written
0,feature_selection_decisions,pipeline_data/1.5.6.intermediate_tables/featur...,323,True
1,feature_set_definitions,pipeline_data/1.5.6.final_tables/modeling_feat...,318,True
2,selected_feature_catalog,pipeline_data/1.5.6.final_tables/modeling_feat...,142,True


Feature decisions and feature-set definitions exported.


Findings\. The export step writes exactly the assets 3\.1\.1 needs: the full 323\-row feature decision table, the 318\-row feature\-set membership table, and the 142\-feature selected catalog\. No raw or scaled modeling matrices are written here, which keeps the notebook boundary clean\. This notebook now ends with feature decisions and reusable column definitions; row filtering, scaling, and matrix construction move to 3\.1\.1\.

### 1\.5\.6 Final Output Inventory

The final outputs from 1\.5\.6 define the selected modeling feature universe\. These files do not contain finalized modeling matrices\. They document which engineered features were selected and how those features are assigned to Bus, Subway, Taxi, FHVHV, and Multimodal modeling views\.

- modeling\_feature\_catalog\.csv
Selected modeling feature catalog\. One row per selected feature, with feature metadata such as feature family, modality, source metric, source notebook, and decision context\.

- modeling\_feature\_set\_definitions\.csv
Feature\-set membership table\. Defines which selected features belong to each modeling view: Bus, Subway, Taxi, FHVHV, and Multimodal\.

These files are mainly useful for understanding feature definitions, feature lineage, and modeling\-view membership\. The actual modeling matrices are constructed later in 3\.1\.1\.

## Close

This notebook turns the full engineered feature universe into a curated set of modeling\-ready feature definitions\. We loaded the completed 1\.5\.5 panel, inventoried 323 candidate features, tested redundancy globally and by modeling view, evaluated feature\-family contribution, and converted that evidence into a final selected set of 142 features\. The selected features preserve the project’s core mobility signals while keeping decomposition, anomaly\-severity, spatial, temporal, Fourier, and multimodal context where they add useful structure\. The final output is intentionally column\-based: 1\.5\.6 defines what should be modeled, while 3\.1\.1 will decide how those columns are filtered, scaled, and assembled into model\-specific matrices\.

### Final Data Dictionary

In [7]:
# ---------------------------------------------------------------------
# Create untruncated base-metric reference table
# ---------------------------------------------------------------------

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

metric_display_names = {
    "bus_trip_count": "Bus trip count",
    "avg_bus_speed": "Average bus speed",
    "subway_ridership": "Subway ridership",
    "taxi_trip_count": "Taxi trip count",
    "taxi_avg_trip_speed": "Taxi average trip speed",
    "fhvhv_trip_count": "FHVHV trip count",
    "fhvhv_avg_trip_speed": "FHVHV average trip speed",
    "avg_bus_travel_time": "Average bus travel time",
    "subway_transfers": "Subway transfers",
    "taxi_avg_trip_distance": "Taxi average trip distance",
    "taxi_avg_trip_duration": "Taxi average trip duration",
    "taxi_distinct_dropoff_zone_count": "Taxi distinct dropoff zone count",
    "fhvhv_avg_trip_distance": "FHVHV average trip distance",
    "fhvhv_avg_trip_duration": "FHVHV average trip duration",
    "fhvhv_distinct_dropoff_zone_count": "FHVHV distinct dropoff zone count",
    "traffic_volume": "Traffic volume",
}

metric_modality_lookup = {
    "bus_trip_count": "bus",
    "avg_bus_speed": "bus",
    "avg_bus_travel_time": "bus",
    "subway_ridership": "subway",
    "subway_transfers": "subway",
    "taxi_trip_count": "taxi",
    "taxi_avg_trip_speed": "taxi",
    "taxi_avg_trip_distance": "taxi",
    "taxi_avg_trip_duration": "taxi",
    "taxi_distinct_dropoff_zone_count": "taxi",
    "fhvhv_trip_count": "fhvhv",
    "fhvhv_avg_trip_speed": "fhvhv",
    "fhvhv_avg_trip_distance": "fhvhv",
    "fhvhv_avg_trip_duration": "fhvhv",
    "fhvhv_distinct_dropoff_zone_count": "fhvhv",
    "traffic_volume": "traffic",
}

base_metric_reference_rows = []

for metric in CORE_MODELING_METRICS:
    base_metric_reference_rows.append(
        {
            "metric_group": "Core modeling metric",
            "base_metric": metric,
            "display_name": metric_display_names.get(metric, metric),
            "modality": metric_modality_lookup.get(metric, "unknown"),
            "report_note": "Primary mobility-state input retained for unsupervised modeling.",
        }
    )

for metric in EXTENDED_BASE_METRICS:
    base_metric_reference_rows.append(
        {
            "metric_group": "Extended/context metric",
            "base_metric": metric,
            "display_name": metric_display_names.get(metric, metric),
            "modality": metric_modality_lookup.get(metric, "unknown"),
            "report_note": "Additional metric generated or reviewed during feature engineering; retained selectively when useful.",
        }
    )

base_metric_reference_df = pd.DataFrame(base_metric_reference_rows)

base_metric_reference_df["metric_group_order"] = np.where(
    base_metric_reference_df["metric_group"].eq("Core modeling metric"),
    1,
    2,
)

base_metric_reference_df = (
    base_metric_reference_df
    .sort_values(["metric_group_order", "modality", "base_metric"])
    .drop(columns=["metric_group_order"])
    .reset_index(drop=True)
)

display(base_metric_reference_df)

base_metric_reference_df.to_csv(BASE_METRIC_REFERENCE_PATH, index=False)

print(f"Base metric reference table: {BASE_METRIC_REFERENCE_PATH}")

,metric_group,base_metric,display_name,modality,report_note
0,Core modeling metric,avg_bus_speed,Average bus speed,bus,Primary mobility-state input retained for unsupervised modeling.
1,Core modeling metric,bus_trip_count,Bus trip count,bus,Primary mobility-state input retained for unsupervised modeling.
2,Core modeling metric,fhvhv_avg_trip_speed,FHVHV average trip speed,fhvhv,Primary mobility-state input retained for unsupervised modeling.
3,Core modeling metric,fhvhv_trip_count,FHVHV trip count,fhvhv,Primary mobility-state input retained for unsupervised modeling.
4,Core modeling metric,subway_ridership,Subway ridership,subway,Primary mobility-state input retained for unsupervised modeling.
5,Core modeling metric,taxi_avg_trip_speed,Taxi average trip speed,taxi,Primary mobility-state input retained for unsupervised modeling.
6,Core modeling metric,taxi_trip_count,Taxi trip count,taxi,Primary mobility-state input retained for unsupervised modeling.
7,Extended/context metric,avg_bus_travel_time,Average bus travel time,bus,Additional metric generated or reviewed during feature engineering; retained selectively when useful.
8,Extended/context metric,fhvhv_avg_trip_distance,FHVHV average trip distance,fhvhv,Additional metric generated or reviewed during feature engineering; retained selectively when useful.
9,Extended/context metric,fhvhv_avg_trip_duration,FHVHV average trip duration,fhvhv,Additional metric generated or reviewed during feature engineering; retained selectively when useful.


Base metric reference table: pipeline_data/report_appendix/appendix_a2_base_metric_reference_table.csv


In [13]:
# ---------------------------------------------------------------------
# Appendix A.2 simplified report tables
# ---------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PIPELINE_DATA_DIR = Path("pipeline_data")

FEATURE_SELECTION_DECISIONS_PATH = (
    PIPELINE_DATA_DIR
    / "1.5.6.intermediate_tables"
    / "feature_selection_decisions.csv"
)

MODELING_FEATURE_CATALOG_PATH = (
    PIPELINE_DATA_DIR
    / "1.5.6.final_tables"
    / "modeling_feature_catalog.csv"
)

APPENDIX_OUTPUT_DIR = PIPELINE_DATA_DIR / "report_appendix"
APPENDIX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_TEMPLATE_REPORT_PATH = (
    APPENDIX_OUTPUT_DIR / "appendix_a2_feature_template_report_table.csv"
)

BASE_METRIC_REFERENCE_PATH = (
    APPENDIX_OUTPUT_DIR / "appendix_a2_base_metric_reference_table.csv"
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

feature_decisions_df = pd.read_csv(FEATURE_SELECTION_DECISIONS_PATH)
selected_catalog_df = pd.read_csv(MODELING_FEATURE_CATALOG_PATH)

selected_feature_names = set(selected_catalog_df["feature_name"])

feature_decisions_df["status"] = np.where(
    feature_decisions_df["feature_name"].isin(selected_feature_names),
    "Kept",
    "Reduced",
)

CORE_MODELING_METRICS = [
    "bus_trip_count",
    "avg_bus_speed",
    "subway_ridership",
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
]

EXTENDED_BASE_METRICS = [
    "avg_bus_travel_time",
    "subway_transfers",
    "taxi_avg_trip_distance",
    "taxi_avg_trip_duration",
    "taxi_distinct_dropoff_zone_count",
    "fhvhv_avg_trip_distance",
    "fhvhv_avg_trip_duration",
    "fhvhv_distinct_dropoff_zone_count",
    "traffic_volume",
]

METRIC_DISPLAY_NAMES = {
    "bus_trip_count": "Bus trip count",
    "avg_bus_speed": "Average bus speed",
    "subway_ridership": "Subway ridership",
    "taxi_trip_count": "Taxi trip count",
    "taxi_avg_trip_speed": "Taxi average trip speed",
    "fhvhv_trip_count": "FHVHV trip count",
    "fhvhv_avg_trip_speed": "FHVHV average trip speed",
    "avg_bus_travel_time": "Average bus travel time",
    "subway_transfers": "Subway transfers",
    "taxi_avg_trip_distance": "Taxi average trip distance",
    "taxi_avg_trip_duration": "Taxi average trip duration",
    "taxi_distinct_dropoff_zone_count": "Taxi distinct dropoff zone count",
    "fhvhv_avg_trip_distance": "FHVHV average trip distance",
    "fhvhv_avg_trip_duration": "FHVHV average trip duration",
    "fhvhv_distinct_dropoff_zone_count": "FHVHV distinct dropoff zone count",
    "traffic_volume": "Traffic volume",
}

METRIC_MODALITY_LOOKUP = {
    "bus_trip_count": "bus",
    "avg_bus_speed": "bus",
    "avg_bus_travel_time": "bus",
    "subway_ridership": "subway",
    "subway_transfers": "subway",
    "taxi_trip_count": "taxi",
    "taxi_avg_trip_speed": "taxi",
    "taxi_avg_trip_distance": "taxi",
    "taxi_avg_trip_duration": "taxi",
    "taxi_distinct_dropoff_zone_count": "taxi",
    "fhvhv_trip_count": "fhvhv",
    "fhvhv_avg_trip_speed": "fhvhv",
    "fhvhv_avg_trip_distance": "fhvhv",
    "fhvhv_avg_trip_duration": "fhvhv",
    "fhvhv_distinct_dropoff_zone_count": "fhvhv",
    "traffic_volume": "traffic",
}


def template_from_feature(row):
    feature_name = str(row["feature_name"])
    feature_family = str(row["feature_family"])

    if feature_family == "raw_metric":
        return None

    template_rules = [
        ("_lag_1_same_bucket", "lag_1"),
        ("_rolling_mean_7_same_bucket", "rolling_mean_7"),
        ("_rolling_std_7_same_bucket", "rolling_std_7"),
        ("_ema_7_same_bucket", "ema_7"),
        ("_abs_change_1_same_bucket", "abs_change_1"),
        ("_pct_change_1_same_bucket", "pct_change_1"),
        ("_pre_cp_mean_same_bucket", "pre_cp_mean"),
        ("_post_cp_mean_same_bucket", "post_cp_mean"),
        ("_cp_abs_delta_same_bucket", "cp_abs_delta"),
        ("_cp_pct_delta_same_bucket", "cp_pct_delta"),
        ("_seasonal", "STL seasonal"),
        ("_residual_zscore", "STL residual z-score"),
        ("_residual_abs", "STL absolute residual"),
        ("_residual", "STL residual"),
        ("_fourier20_residual_reconstructed", "Fourier residual reconstruction"),
        ("_deviation_abs_zscore", "Absolute deviation z-score"),
        ("_deviation_zscore", "Deviation z-score"),
        ("_deviation_percentile_rank", "Deviation percentile rank"),
        ("_local_vs_connected_zscore", "Local vs connected-zone divergence"),
    ]

    for suffix, template in template_rules:
        if feature_name.endswith(suffix):
            return template

    if feature_name.startswith("connected_mean_"):
        return "Connected-zone benchmark"

    if feature_name.startswith("borough_mean_"):
        return "Borough benchmark"

    if feature_name.startswith("cp_zone_mean_"):
        return "CP-zone benchmark"

    if feature_name in {
        "is_cbd_zone",
        "is_cbd_gateway_zone",
        "is_cbd_adjacent_zone",
        "distance_to_cbd_miles",
        "distance_to_gateway_miles",
        "neighbor_count",
        "connected_zone_count",
    }:
        return "CP geography and connectivity"

    if "_demand_zscore" in feature_name:
        return "Cross-modal demand interaction"

    if "_speed_zscore" in feature_name:
        return "Cross-modal speed interaction"

    if "_demand_shift_ratio" in feature_name:
        return "Demand-shift ratio"

    if feature_name.startswith("mta_crossing_share_x_"):
        return "Bridge/tunnel interaction"

    if feature_name.startswith("traffic_"):
        return "Sparse Traffic temporal context"

    if feature_name.endswith("_zscore"):
        return "Standardized mobility state"

    return "Other engineered feature"


TEMPLATE_MEANINGS = {
    "lag_1": "Previous observation from the same Taxi Zone and temporal bucket.",
    "rolling_mean_7": "Seven-observation rolling average within the same Taxi Zone and temporal bucket.",
    "rolling_std_7": "Seven-observation rolling standard deviation measuring recent variability.",
    "ema_7": "Seven-observation exponential moving average with greater weight on recent observations.",
    "abs_change_1": "Absolute change from the previous same-bucket observation.",
    "pct_change_1": "Percent change from the previous same-bucket observation.",
    "pre_cp_mean": "Average metric value before congestion pricing for the same Taxi Zone and temporal bucket.",
    "post_cp_mean": "Average metric value after congestion pricing for the same Taxi Zone and temporal bucket.",
    "cp_abs_delta": "Difference between post-CP and pre-CP same-bucket means.",
    "cp_pct_delta": "Percent difference between post-CP and pre-CP same-bucket means.",
    "Connected-zone benchmark": "Average same-bucket mobility conditions among connected neighboring zones.",
    "Borough benchmark": "Average same-bucket mobility conditions within the borough.",
    "CP-zone benchmark": "Average same-bucket mobility conditions inside the congestion-pricing zone.",
    "CP geography and connectivity": "Static geography, gateway, distance, and network-connectivity descriptors.",
    "Standardized mobility state": "Z-score standardized version of a mobility metric.",
    "Cross-modal demand interaction": "Joint demand behavior across two transportation systems.",
    "Cross-modal speed interaction": "Joint speed behavior across roadway-based systems.",
    "Demand-shift ratio": "Relative balance between transit demand and roadway or for-hire demand.",
    "Local vs connected-zone divergence": "Difference between local conditions and connected-zone conditions.",
    "Bridge/tunnel interaction": "Interaction between Manhattan-connected MTA crossing share and local mobility.",
    "Sparse Traffic temporal context": "Traffic-specific temporal features for irregular sensor observations.",
    "STL seasonal": "Recurring seasonal component from STL decomposition.",
    "STL residual": "Remaining signal after STL trend and seasonal structure are removed.",
    "STL absolute residual": "Magnitude of the STL residual.",
    "STL residual z-score": "Within-series standardized STL residual.",
    "Fourier residual reconstruction": "Recurring frequency structure reconstructed from the top Fourier residual components.",
    "Deviation z-score": "Observed value compared with its local historical baseline.",
    "Absolute deviation z-score": "Magnitude of local historical deviation regardless of direction.",
    "Deviation percentile rank": "Empirical percentile within the local historical baseline.",
    "Other engineered feature": "Engineered feature template considered during feature selection.",
}

TEMPLATE_ORDER = [
    "lag_1",
    "rolling_mean_7",
    "rolling_std_7",
    "ema_7",
    "abs_change_1",
    "pct_change_1",
    "pre_cp_mean",
    "post_cp_mean",
    "cp_abs_delta",
    "cp_pct_delta",
    "Connected-zone benchmark",
    "Borough benchmark",
    "CP-zone benchmark",
    "CP geography and connectivity",
    "Standardized mobility state",
    "Cross-modal demand interaction",
    "Cross-modal speed interaction",
    "Demand-shift ratio",
    "Local vs connected-zone divergence",
    "Bridge/tunnel interaction",
    "Sparse Traffic temporal context",
    "STL seasonal",
    "STL residual",
    "STL absolute residual",
    "STL residual z-score",
    "Fourier residual reconstruction",
    "Deviation z-score",
    "Absolute deviation z-score",
    "Deviation percentile rank",
    "Other engineered feature",
]


feature_decisions_df["feature_template"] = feature_decisions_df.apply(
    template_from_feature,
    axis=1,
)

feature_template_source_df = (
    feature_decisions_df
    .loc[feature_decisions_df["feature_template"].notna()]
    .copy()
)

feature_template_source_df["meaning"] = (
    feature_template_source_df["feature_template"]
    .map(TEMPLATE_MEANINGS)
    .fillna("Engineered feature template considered during feature selection.")
)

feature_template_source_df["applies_to"] = np.where(
    feature_template_source_df["source_metric"].isin(CORE_MODELING_METRICS),
    "core metrics",
    np.where(
        feature_template_source_df["source_metric"].isin(EXTENDED_BASE_METRICS),
        "extended metrics",
        feature_template_source_df["modality"].fillna("multiple modalities").str.lower(),
    ),
)

feature_template_report_df = (
    feature_template_source_df
    .groupby(["feature_template", "meaning"], as_index=False)
    .agg(
        status=(
            "status",
            lambda values: "Kept" if (values == "Kept").any() else "Reduced",
        ),
        applies_to=(
            "applies_to",
            lambda values: ", ".join(sorted(values.dropna().unique())),
        ),
    )
)

feature_template_report_df["template_order"] = (
    feature_template_report_df["feature_template"]
    .map({name: i for i, name in enumerate(TEMPLATE_ORDER)})
    .fillna(999)
)

feature_template_report_df = (
    feature_template_report_df
    .sort_values(["template_order", "feature_template"])
    .drop(columns=["template_order"])
    .reset_index(drop=True)
)

base_metric_rows = []

for metric in CORE_MODELING_METRICS:
    base_metric_rows.append(
        {
            "metric_group": "Core modeling metric",
            "base_metric": metric,
            "display_name": METRIC_DISPLAY_NAMES.get(metric, metric),
            "modality": METRIC_MODALITY_LOOKUP.get(metric, "unknown"),
            "report_note": "Primary mobility-state input used as a repeated basis for unsupervised feature engineering.",
        }
    )

for metric in EXTENDED_BASE_METRICS:
    base_metric_rows.append(
        {
            "metric_group": "Extended metric",
            "base_metric": metric,
            "display_name": METRIC_DISPLAY_NAMES.get(metric, metric),
            "modality": METRIC_MODALITY_LOOKUP.get(metric, "unknown"),
            "report_note": "Source-derived metric generated and reviewed during feature engineering, but not used as the default basis for every engineered template.",
        }
    )

base_metric_reference_df = pd.DataFrame(base_metric_rows)

base_metric_reference_df["metric_group_order"] = np.where(
    base_metric_reference_df["metric_group"].eq("Core modeling metric"),
    1,
    2,
)

base_metric_reference_df = (
    base_metric_reference_df
    .sort_values(["metric_group_order", "modality", "base_metric"])
    .drop(columns=["metric_group_order"])
    .reset_index(drop=True)
)

display(base_metric_reference_df)
display(feature_template_report_df)

base_metric_reference_df.to_csv(BASE_METRIC_REFERENCE_PATH, index=False)
feature_template_report_df.to_csv(FEATURE_TEMPLATE_REPORT_PATH, index=False)

print(f"Base metric reference table: {BASE_METRIC_REFERENCE_PATH}")
print(f"Feature template table: {FEATURE_TEMPLATE_REPORT_PATH}")

,metric_group,base_metric,display_name,modality,report_note
0,Core modeling metric,avg_bus_speed,Average bus speed,bus,Primary mobility-state input used as a repeated basis for unsupervised feature engineering.
1,Core modeling metric,bus_trip_count,Bus trip count,bus,Primary mobility-state input used as a repeated basis for unsupervised feature engineering.
2,Core modeling metric,fhvhv_avg_trip_speed,FHVHV average trip speed,fhvhv,Primary mobility-state input used as a repeated basis for unsupervised feature engineering.
3,Core modeling metric,fhvhv_trip_count,FHVHV trip count,fhvhv,Primary mobility-state input used as a repeated basis for unsupervised feature engineering.
4,Core modeling metric,subway_ridership,Subway ridership,subway,Primary mobility-state input used as a repeated basis for unsupervised feature engineering.
5,Core modeling metric,taxi_avg_trip_speed,Taxi average trip speed,taxi,Primary mobility-state input used as a repeated basis for unsupervised feature engineering.
6,Core modeling metric,taxi_trip_count,Taxi trip count,taxi,Primary mobility-state input used as a repeated basis for unsupervised feature engineering.
7,Extended metric,avg_bus_travel_time,Average bus travel time,bus,"Source-derived metric generated and reviewed during feature engineering, but not used as the default basis for every engineered template."
8,Extended metric,fhvhv_avg_trip_distance,FHVHV average trip distance,fhvhv,"Source-derived metric generated and reviewed during feature engineering, but not used as the default basis for every engineered template."
9,Extended metric,fhvhv_avg_trip_duration,FHVHV average trip duration,fhvhv,"Source-derived metric generated and reviewed during feature engineering, but not used as the default basis for every engineered template."


,feature_template,meaning,status,applies_to
0,lag_1,Previous observation from the same Taxi Zone and temporal bucket.,Kept,"core metrics, extended metrics"
1,rolling_mean_7,Seven-observation rolling average within the same Taxi Zone and temporal bucket.,Kept,"core metrics, extended metrics"
2,rolling_std_7,Seven-observation rolling standard deviation measuring recent variability.,Kept,"core metrics, extended metrics"
3,ema_7,Seven-observation exponential moving average with greater weight on recent observations.,Reduced,"core metrics, extended metrics"
4,abs_change_1,Absolute change from the previous same-bucket observation.,Kept,"core metrics, extended metrics"
5,pct_change_1,Percent change from the previous same-bucket observation.,Kept,"core metrics, extended metrics"
6,pre_cp_mean,Average metric value before congestion pricing for the same Taxi Zone and temporal bucket.,Kept,"core metrics, extended metrics"
7,post_cp_mean,Average metric value after congestion pricing for the same Taxi Zone and temporal bucket.,Kept,"core metrics, extended metrics"
8,cp_abs_delta,Difference between post-CP and pre-CP same-bucket means.,Kept,"core metrics, extended metrics"
9,cp_pct_delta,Percent difference between post-CP and pre-CP same-bucket means.,Kept,"core metrics, extended metrics"


Base metric reference table: pipeline_data/report_appendix/appendix_a2_base_metric_reference_table.csv
Feature template table: pipeline_data/report_appendix/appendix_a2_feature_template_report_table.csv


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>